# Fruit Ninja

## Overview
This notebook demonstrates a hands-free Fruit Ninja-style game powered by a local Ultralytics YOLOv8 pose model on an AMD GPU through ROCm. You play by moving your wrists in front of a webcam: the model finds your hands, and the game turns those positions into blades that slice fruit and dodge bombs.

**🎯 What you will learn**

- How a webcam frame becomes a model input (resize, color, normalization, tensor placement).
- How PyTorch HIP executes pose inference on the `cuda:0` ROCm device.
- How public `Results` fields expose person boxes and COCO wrist keypoints, and why we use wrist indices 9 and 10.
- How detection, physics, collisions, scoring, and UI combine into a real-time game loop.

## Prerequisites
- The project Docker image built from the provided `Dockerfile` with the base-image tag matching the host GPU.
- Internet access on the first run so the notebook can download and checksum the pinned YOLOv8 pose model.
- A working webcam accessible at index 0 for interactive play, or a readable video file supplied through `FRUIT_NINJA_VIDEO_SOURCE` for a finite run.

The finite source and frame-count environment seams described in Step 8 are for smoke runs; ordinary use needs no environment variables.

![Fruit Ninja Flow](images/fruit_ninja_flow.png)

## Step 1: Download Runtime Assets, Import Libraries, and Configure Parameters

The first code cell is self-contained. It downloads the checksum-pinned YOLOv8 pose model into `runtime_assets/` when missing, restores the bundled fruit/bomb sprites there, validates the active ROCm device, and constructs the detector. Valid files are reused on later runs; no repository-side Python helper is required.


In [ ]:
# Core imports
import json, os, platform, random, math, time
import numpy as np
import cv2
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple, Optional
from PIL import Image as PILImage
from IPython.display import display, Image as DisplayImage, clear_output

import hashlib
import urllib.request

YOLO_URL = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m-pose.pt"
YOLO_SHA256 = "dbe539ea268db2534390942cfdf206e521f376f19e5415967a57f6a2ddfa3c90"

class AssetError(RuntimeError):
    pass

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _download_checked(url: str, destination: Path, expected_sha256: str | None = None) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and (expected_sha256 is None or _sha256(destination) == expected_sha256):
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    urllib.request.urlretrieve(url, temporary)
    if expected_sha256 is not None and _sha256(temporary) != expected_sha256:
        temporary.unlink(missing_ok=True)
        raise AssetError(f"SHA-256 mismatch for {destination.name}")
    temporary.replace(destination)
    return destination

def validate_yolo_model(path: str | Path | None = None) -> Path:
    candidate = Path(path or "runtime_assets/model/yolov8m-pose.pt")
    if not candidate.is_file():
        raise FileNotFoundError(f"Pose model is missing: {candidate}; rerun the download cell")
    if _sha256(candidate) != YOLO_SHA256:
        raise AssetError(f"Pose model checksum mismatch: {candidate}; remove it and rerun the download cell")
    return candidate

def require_rocm(expected_gfx: str | None = None):
    import re
    import torch
    if not torch.cuda.is_available() or not getattr(torch.version, "hip", None):
        raise RuntimeError("A PyTorch ROCm GPU is required; CPU fallback is not supported")
    properties = torch.cuda.get_device_properties(0)
    match = re.search(r"gfx[0-9a-f]+", str(getattr(properties, "gcnArchName", "")).lower())
    gfx = match.group(0) if match else "unknown"
    if expected_gfx and gfx != expected_gfx:
        raise RuntimeError(f"ROCm device mismatch: expected {expected_gfx}, found {gfx}")
    print(f"ROCm ready: {torch.cuda.get_device_name(0)}, gfx={gfx}, torch={torch.__version__}, hip={torch.version.hip}")
    return {"gfx": gfx}

import math
from pathlib import Path
from typing import Any
import numpy as np
LEFT_WRIST_INDEX = 9
RIGHT_WRIST_INDEX = 10
DEFAULT_HAND_BOX_SCALE = 0.18
DEFAULT_HAND_BOX_MIN_PX = 36
DEFAULT_MAX_PLAYERS = 2

class PoseResultError(ValueError):
    """Raised when an Ultralytics pose result has an invalid public shape."""

class PoseModelError(RuntimeError):
    """Raised when the pinned Fruit Ninja model is unsafe or altered."""

def _public_value(owner: Any, name: str, owner_name: str) -> Any:
    try:
        value = getattr(owner, name)
    except AttributeError as exc:
        raise PoseResultError(f'Malformed pose result: missing public {owner_name}.{name}') from exc
    if value is None:
        raise PoseResultError(f'Malformed pose result: public {owner_name}.{name} is None')
    return value

def _as_float_array(value: Any, label: str) -> np.ndarray:
    """Materialize a public Ultralytics tensor/array on the host for parsing."""
    try:
        detach = getattr(value, 'detach', None)
        if callable(detach):
            value = detach()
        cpu = getattr(value, 'cpu', None)
        if callable(cpu):
            value = cpu()
        to_numpy = getattr(value, 'numpy', None)
        if callable(to_numpy):
            value = to_numpy()
        array = np.asarray(value, dtype=np.float64)
    except (TypeError, ValueError, RuntimeError) as exc:
        raise PoseResultError(f'Malformed pose result: public {label} is not a numeric array') from exc
    if not np.isfinite(array).all():
        raise PoseResultError(f'Malformed pose result: public {label} contains non-finite values')
    return array

def _canvas_size(canvas_shape: tuple[int, int] | Any) -> tuple[int, int]:
    try:
        if len(canvas_shape) < 2:
            raise ValueError
        height = int(canvas_shape[0])
        width = int(canvas_shape[1])
    except (TypeError, ValueError, IndexError) as exc:
        raise ValueError('canvas_shape must contain positive height and width') from exc
    if height <= 0 or width <= 0:
        raise ValueError('canvas_shape must contain positive height and width')
    return (height, width)

def _validate_result_arrays(result: Any) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    try:
        boxes = getattr(result, 'boxes')
    except AttributeError as exc:
        raise PoseResultError('Malformed pose result: missing public boxes result') from exc
    try:
        keypoints = getattr(result, 'keypoints')
    except AttributeError as exc:
        raise PoseResultError('Malformed pose result: missing public keypoints result') from exc
    boxes_xyxy = _as_float_array(_public_value(boxes, 'xyxy', 'boxes'), 'boxes.xyxy')
    boxes_conf = _as_float_array(_public_value(boxes, 'conf', 'boxes'), 'boxes.conf')
    keypoints_xy = _as_float_array(_public_value(keypoints, 'xy', 'keypoints'), 'keypoints.xy')
    keypoints_conf = _as_float_array(_public_value(keypoints, 'conf', 'keypoints'), 'keypoints.conf')
    if boxes_xyxy.ndim != 2 or boxes_xyxy.shape[1:] != (4,):
        raise PoseResultError(f'Malformed pose result: boxes.xyxy must have shape (N, 4), got {boxes_xyxy.shape}')
    if boxes_conf.ndim != 1:
        raise PoseResultError(f'Malformed pose result: boxes.conf must have shape (N,), got {boxes_conf.shape}')
    if keypoints_xy.ndim != 3 or keypoints_xy.shape[2] != 2:
        raise PoseResultError(f'Malformed pose result: keypoints.xy must have shape (N, K, 2), got {keypoints_xy.shape}')
    if keypoints_conf.ndim != 2:
        raise PoseResultError(f'Malformed pose result: keypoints.conf must have shape (N, K), got {keypoints_conf.shape}')
    detection_count = boxes_xyxy.shape[0]
    if boxes_conf.shape[0] != detection_count or keypoints_xy.shape[0] != detection_count or keypoints_conf.shape[0] != detection_count:
        raise PoseResultError('Malformed pose result: boxes.xyxy, boxes.conf, keypoints.xy, and keypoints.conf counts must be aligned')
    if keypoints_conf.shape[1] != keypoints_xy.shape[1]:
        raise PoseResultError('Malformed pose result: keypoints.xy and keypoints.conf keypoint counts must be aligned')
    if detection_count and np.any(boxes_xyxy[:, 2:] < boxes_xyxy[:, :2]):
        raise PoseResultError('Malformed pose result: boxes.xyxy contains a box with reversed coordinates')
    return (boxes_xyxy, boxes_conf, keypoints_xy, keypoints_conf)

def extract_hand_points(result: Any, canvas_shape: tuple[int, int], *, detection_threshold: float, keypoint_threshold: float, hand_box_scale: float, hand_box_min_px: int, max_players: int) -> list[dict]:
    """Convert valid COCO wrist keypoints from one Ultralytics result to hand detections."""
    height, width = _canvas_size(canvas_shape)
    try:
        detection_threshold = float(detection_threshold)
        keypoint_threshold = float(keypoint_threshold)
        hand_box_scale = float(hand_box_scale)
        hand_box_min_px = int(hand_box_min_px)
        max_players = int(max_players)
    except (TypeError, ValueError) as exc:
        raise ValueError('pose extraction thresholds and sizing parameters must be numeric') from exc
    if not math.isfinite(detection_threshold) or not math.isfinite(keypoint_threshold):
        raise ValueError('pose extraction thresholds must be finite')
    if not math.isfinite(hand_box_scale) or hand_box_scale < 0:
        raise ValueError('hand_box_scale must be finite and non-negative')
    if hand_box_min_px <= 0:
        raise ValueError('hand_box_min_px must be positive')
    if max_players <= 0:
        return []
    boxes_xyxy, boxes_conf, keypoints_xy, keypoints_conf = _validate_result_arrays(result)
    detection_count = boxes_xyxy.shape[0]
    if detection_count == 0:
        return []
    keypoint_count = keypoints_xy.shape[1]
    if keypoint_count <= RIGHT_WRIST_INDEX:
        raise PoseResultError('Malformed pose result: keypoints.xy/keypoints.conf must contain COCO wrist indices 9 and 10')
    canvas_center_x = width / 2.0
    canvas_center_y = height / 2.0
    nearest_people: list[tuple[float, int]] = []
    for detection_index in range(detection_count):
        person_confidence = float(boxes_conf[detection_index])
        if person_confidence < detection_threshold:
            continue
        x1, y1, x2, y2 = boxes_xyxy[detection_index]
        person_center_x = (x1 + x2) / 2.0
        person_center_y = (y1 + y2) / 2.0
        person_distance = math.hypot(person_center_x - canvas_center_x, person_center_y - canvas_center_y)
        nearest_people.append((person_distance, detection_index))
    nearest_people.sort(key=lambda item: (item[0], item[1]))
    selected_people = nearest_people[:max_players]
    hands: list[dict] = []
    for _, detection_index in selected_people:
        x1, y1, x2, y2 = boxes_xyxy[detection_index]
        person_confidence = float(boxes_conf[detection_index])
        person_side = max(x2 - x1, y2 - y1)
        hand_side = max(float(hand_box_min_px), person_side * hand_box_scale)
        half_side = hand_side / 2.0
        for keypoint_index in (LEFT_WRIST_INDEX, RIGHT_WRIST_INDEX):
            wrist_confidence = float(keypoints_conf[detection_index, keypoint_index])
            if wrist_confidence < keypoint_threshold:
                continue
            wrist_x = float(keypoints_xy[detection_index, keypoint_index, 0])
            wrist_y = float(keypoints_xy[detection_index, keypoint_index, 1])
            hand_x1 = max(0, min(width - 1, int(wrist_x - half_side)))
            hand_y1 = max(0, min(height - 1, int(wrist_y - half_side)))
            hand_x2 = max(0, min(width - 1, int(wrist_x + half_side)))
            hand_y2 = max(0, min(height - 1, int(wrist_y + half_side)))
            if hand_x2 <= hand_x1 or hand_y2 <= hand_y1:
                continue
            hands.append({'player_id': detection_index, 'center': (wrist_x, wrist_y), 'box': [hand_x1, hand_y1, hand_x2, hand_y2], 'confidence': float(min(person_confidence, wrist_confidence)), 'distance': math.hypot(wrist_x - canvas_center_x, wrist_y - canvas_center_y)})
    return hands

class PoseDetector:
    """Native Ultralytics YOLOv8 pose inference on the validated ROCm GPU."""

    def __init__(self, model_path: str | Path, expected_gfx: str | None=None, confidence: float=0.55, keypoint_confidence: float=0.35, image_size: int=640) -> None:
        require_rocm(expected_gfx)
        requested_path = Path(model_path)
        if not requested_path.is_absolute():
            requested_path = Path.cwd() / requested_path
        validation_error: PoseModelError | None = None
        try:
            validated_path = validate_yolo_model(requested_path)
        except FileNotFoundError:
            raise
        except (AssetError, OSError) as exc:
            message = str(exc)
            instruction = 'rerun the first notebook code cell'
            if instruction not in message:
                message = f'{message}; run `{instruction}`'
            validation_error = PoseModelError(f'Fruit Ninja pose model validation failed: {message}')
            del exc
        if validation_error is not None:
            raise validation_error
        self.model_path = Path(validated_path)
        from ultralytics import YOLO
        self._model = YOLO(str(self.model_path))
        self.confidence = float(confidence)
        self.keypoint_confidence = float(keypoint_confidence)
        self.image_size = int(image_size)

    def predict(self, frame: Any) -> list[dict]:
        """Run one frame through YOLO pose and return hand-only detections."""
        shape = getattr(frame, 'shape', None)
        if shape is None or len(shape) < 2:
            raise ValueError('frame must expose a two-dimensional image shape')
        results = self._model.predict(source=frame, device=0, quantize=16, imgsz=self.image_size, conf=self.confidence, verbose=False)
        if results is None:
            return []
        if isinstance(results, (list, tuple)):
            if not results:
                return []
            result = results[0]
        else:
            result = results
        return extract_hand_points(result, (int(shape[0]), int(shape[1])), detection_threshold=self.confidence, keypoint_threshold=self.keypoint_confidence, hand_box_scale=DEFAULT_HAND_BOX_SCALE, hand_box_min_px=DEFAULT_HAND_BOX_MIN_PX, max_players=DEFAULT_MAX_PLAYERS)

# ==============================
# Configurable Parameters (safe to tweak)
# ==============================
# The model is downloaded into the case directory.
model_path = Path("runtime_assets/model/yolov8m-pose.pt")

# Display and inference canvas resolution.
CANVAS_WIDTH = 640
CANVAS_HEIGHT = 640

# Camera index; use 0 for the default webcam unless a source override is set.
CAMERA_INDEX = 0

# Gameplay tuning
GAME_DURATION = 30.0                 # Duration of one round (seconds)
SPAWN_INTERVAL_RANGE = (0.7, 1.1)    # Time between spawn batches
MAX_ACTIVE_OBJECTS = 6               # Max simultaneous fruits/bombs
FRUIT_SCORE = 10                     # Score per fruit
BOMB_PENALTY = 20                    # Penalty per bomb
BOMB_PROBABILITY = 0.2               # Probability of spawning a bomb
GRAVITY = 900.0                      # Gravity acceleration (pixels/s^2)
TOUCH_RADIUS = 70.0                  # Effective hand radius for collisions
TOUCH_CONFIRM_SECONDS = 3.0          # Hold duration to confirm menu selections

# Pose-based hand detection tuning used by the public adapter.
HAND_CONFIDENCE = 0.55               # Minimum person detection confidence
HAND_KEYPOINT_CONFIDENCE = 0.35      # Minimum wrist keypoint confidence

# FPS overlay
SHOW_FPS = False
FPS_SMOOTH_SAMPLES = 20              # 0 to disable smoothing

# Touch zone labels
TOUCH_ZONE_LABEL_SCALE = 0.6
TOUCH_ZONE_LABEL_THICKNESS = 2
TOUCH_ZONE_PROGRESS_SCALE = 0.5

# Spawn presets and weights
BOTTOM_VERTICAL_SPEED_RANGE = (-1080.0, -880.0)
BOTTOM_HORIZONTAL_SPEED_RANGE = (-260.0, 260.0)
SIDE_VERTICAL_SPEED_RANGE = (-780.0, -540.0)
SIDE_HORIZONTAL_SPEED_RANGE = (360.0, 520.0)
TOP_VERTICAL_SPEED_RANGE = (360.0, 540.0)
TOP_HORIZONTAL_SPEED_RANGE = (-220.0, 220.0)
SPAWN_PRESET_CHOICES = [
    ("bottom_center", 0.24),
    ("bottom_left", 0.18),
    ("bottom_right", 0.18),
    ("left_edge", 0.16),
    ("right_edge", 0.16),
    ("top_edge", 0.08),
]

# Output scaling for visualization
DISPLAY_SCALE = 1.3
DISPLAY_INTERPOLATION = cv2.INTER_LINEAR

# Performance tuning
JPEG_QUALITY = 85
USE_FAST_ALPHA_BLEND = True
VECTORIZE_COLLISION = True

# ==============================
# Environment seams for normal use and finite smoke runs
# ==============================
def parse_video_source(value: Optional[str]) -> int | str:
    """Resolve the default camera, a numeric camera index, or a video path."""
    if value is None:
        return CAMERA_INDEX
    source = value.strip()
    if not source:
        raise ValueError(
            "FRUIT_NINJA_VIDEO_SOURCE must be a non-empty camera index or video path."
        )
    try:
        camera_index = int(source)
    except ValueError:
        source_path = Path(source).expanduser()
        if not source_path.is_file():
            raise ValueError(
                f"FRUIT_NINJA_VIDEO_SOURCE path does not exist or is not a file: {source_path}"
            )
        return str(source_path)
    if camera_index < 0:
        raise ValueError("FRUIT_NINJA_VIDEO_SOURCE camera index must be non-negative")
    return camera_index


def parse_max_frames(value: Optional[str]) -> int:
    """Parse zero/unset as unlimited, otherwise require a positive frame count."""
    if value is None:
        return 0
    text = value.strip()
    if not text:
        raise ValueError("FRUIT_NINJA_MAX_FRAMES must be an integer >= 0")
    try:
        max_frames = int(text)
    except ValueError as exc:
        raise ValueError("FRUIT_NINJA_MAX_FRAMES must be an integer >= 0") from exc
    if max_frames < 0:
        raise ValueError("FRUIT_NINJA_MAX_FRAMES must be an integer >= 0")
    return max_frames


def parse_autostart(value: Optional[str]) -> bool:
    """Allow only the documented 0/1 switch for remote finite runs."""
    if value is None:
        return False
    text = value.strip()
    if text not in {"0", "1"}:
        raise ValueError("FRUIT_NINJA_AUTOSTART must be exactly 0 or 1")
    return text == "1"


# Touch zones for hold-to-confirm UI
MENU_TOUCH_ZONE = ((CANVAS_WIDTH // 2 - 140, CANVAS_HEIGHT // 2 - 80),
                   (CANVAS_WIDTH // 2 + 140, CANVAS_HEIGHT // 2 + 80))
RESTART_TOUCH_ZONE = ((CANVAS_WIDTH // 2 - 200, CANVAS_HEIGHT // 2 - 60),
                      (CANVAS_WIDTH // 2 - 20, CANVAS_HEIGHT // 2 + 60))
HOME_TOUCH_ZONE = ((CANVAS_WIDTH // 2 + 20, CANVAS_HEIGHT // 2 - 60),
                   (CANVAS_WIDTH // 2 + 200, CANVAS_HEIGHT // 2 + 60))

# ==============================
# Derived / Constant Paths (fixed for this tutorial)
# ==============================
SPAWN_PRESETS, SPAWN_WEIGHTS = zip(*SPAWN_PRESET_CHOICES)
ASSET_ROOT = Path("runtime_assets/assets")
FRUIT_ASSET_DIR = ASSET_ROOT / "fruits"
BOMB_ASSET_DIR = ASSET_ROOT / "bombs"

# Download the pinned pose model and teaching sprites once.
_download_checked(YOLO_URL, model_path, YOLO_SHA256)
import base64
EMBEDDED_SPRITES = {'fruits/banana.png': 'iVBORw0KGgoAAAANSUhEUgAAADsAAABACAYAAACkwA+xAAARHUlEQVR42tWbaXBc1ZXHf+e+pXct3ZK8SbbBNl4Agy0wYLBlgxcgGAKJgAkQUknNVOBDJpO1UpPEparMTJIaJjU1maSSykxSk41Bk5AQUjCQYITBxoAN2HgBy9iyJcvW3i31+t67dz50C2QjA6kkE6mr9EHVr2/f3zvnnXPu/5wWpvZLAAPI5ReyyRjWOBbNM1Jcmc1xPD3Gjy69hm9///t4E65918WmMigtLcRCRX4xu45N82fDnHpIVcPwKOw5BAeP8owIt/3+RYYrsOcEVlPdqnqA2kiITbeuhxuuwrct9J7Xofs0ZuUSSqsvYW3R58uAbmnBercFrSkMa7ZuRf1XO+m6akq9Q7TMSKJePIAKNBRKSN8wYinMYJrGRovvdrxK6d28dSpblrY2NMBze/mn3n5ezxVQ8ShaKYiFIRJCjYwitkWTqWcpwNat0xR23EqNjSRrE9Tni5AvIgIc6oKuXriqORE0pCwKeS4GePrpczPZ0yEaJ2wS2pDsGwLHgjdPwuUXxVm9IsI16+McPHKSvW8ES99rsaluWQ1Iwzx6Mjn2d3bD3k50fdLlC5+pZ/F5LsUBj1S1hRiWADQ0TM9oTCX1WB0d+MC3hjLQOwAbroyhDRSLmgNHSqpxpoOBRcYg7e0E0xa2o6O8+Yuu4qH0GL23XhexNq6PG5MOmDPHpbOrJNVxi0SMBbdvpOnduKY8LGBaW7G+9S3y+QLbYxEx8YQKCnmNRIRklSWdJ0rBkvPt8NEeWgBpaZm+sPT1IYC4Lo/uOViQzGlfxvIakzdcsSJK53GPRNQiGuZj07WCOtuVzcwZ/G/PaT164rRvKRFjPE08abHhipj10oGirq2Sazev5uaODvzmZpxpCTvuyg8/SV96lKePnSwxnAmCU4MBxjPctDHB+sui9A4YHfj85weu5pLdu/FaW8+sEKcL7FuunC/wy1cPFbBt5HsPDSMKgpLhK59tUPdsiTO7wU7livz+pjWsbG8n2Lr1bcZpA9vRgQaMNjz50v5ivnGGYx3v9cxP/2cEK2lx4ECeQ0c9dePaWNC6KZ7qH+LRj7Yyp60NM845bWDHC4zD3fT0DuhXTvZ5fOKDNfrfH0yTPuWTz2k6j5fYvjtr3ffROv+KS2OzOg/xE2OgtbVcdk4nWCpHOCkW2H/kRIlrroubmXWKn/9qhOZ1Cf75c/XEoxYjQwX7S/df7NfUhNdtuFzd195O0NKCPa1g160ru7LlMEtEIKLk+jUxOl7KYUY1yRqbv70nRciCmXN9dfstq/RYTrfd95G5tR0dBNMJVtra0BuX05CsYs3CuS5kjbpqeZSB4YCB7hKzm1xmNThEoiHMSI9q3bJIn39esm7/geP3gRg1zVyYIY/bFs93quY2Oj45LQuaHFxH6DzhYQR8z4ARdMkjWj2gbtx8tSkU+eutW1dGpw3sW6cZw4YFc8OGiCIoaaJVFqkai9ePFRF7XKbQiBWCdKfasHYZdSln/vbf7r5u2sC2t6OVEpTi/PpUXECUNoArzEhZvNntVXTISk5VNiY3wMxGpS9atsBkRrlhusAKYP7jKyZsCfXJmjAYg6lk0FSNRXpMg54oQAla+2BOyoXLFonRXD6tovHTL1JjOdQma1zQRkTKUJGwQr1DeTIgFhR7ZNbMWmyHedMCdmvFYKfSpMKuFU1WKwi0jNvRUhANC6izjjxigz8gIdvDsqieFrAHKhVQNsus6oQrddWiCUxZeRMolgypWhuUTKA1iCgIsihGEYUzjc6zQqnE4vqUS0211gQVu2rDcEbT2GCfEaDKuAqMhy4NoM30KioINM1NsyIQ1gQBWBZQMPQN+SxocsE3k4jGhlLuFMZQmDaHd2O+qpTi8kXzooCvRAnGEgb6fbJ5zcK5LpQqrj0hRqHEpNMjaE1GTYPgpABz64a2hbGwWrL4vJBBabVt1xjdPUWO9njUVllU19sY/yzYcrA2w5mAQNM35WGfbkGB0N/PlsbZMXvZQjsoDpZ4cV+WuqTNy4cKNC8LgyvoCaHYmIqbl4zp7CpiK45Nedh1HWhjvqpKHveubm7AqfKV78GVl8SIRC2Odpe4dlUMCvrtXGtALMEHhk6XOHbSIxTm4JRuf7S2YrW1E+xc9bWb6lP2xbdsnBWQ67KiEZt166rpOVqiIWkz73wXXTDlzGNAHKG7p8hTO9MkYpaMZg0hi5enen8WYx6yhkaDf7hl0xIze75C5z2MEYyBw8dLrG2OnpVfy5Z95oUM/YMl89rhgiqWKITVFIZtbsZub5dg9fI7v7R0YXj5XXes1CbdY4nlIMpAQQOwfEmo7MLqzNtUnbAIuWLSWYOveWPlZt5UU/Tsau/ejbdptbvetXXbZ+7bEMTiWWWKo4goxBIymYBFc12csMLoM0FNYLh+fS1zZ4d032BgBJ5ra0OrqficdnTgf+yW2qbhoeLPP3n3Url09XIJBvaLsiu6tzYoJcxqsME7M90IICIUcwFdJz3JZBERHp9ygtvWraj2doJvfH5xYt+h4Ufu/fCcGXf+zR1aDx5Wlhqh3E42YCAeneSko8rPMrbwxrG8OX7Kt8byDNXV0DGlYFtbsdra0A88cGXkwV+//us7b6q/9N67Nga//tkT1nd/8Bin+8usphKIJlrTGMAShkd8fvjLATJDHoePFYKRUWP8gMfaf0e6tRVLTZVntL2d4PMfX5z4yfeff/QjW1LrP/epW/wf/vgZ67lnX8C1feIJBxOYc4+HWMLAsE9Ntc1Q2ufI8aIaTCMIP5oyc1AtLdgdHfjXtyQbh/qGHr73tvrL7v9Uq6+9avvhhx/H9o+wuaWacFhhvEnKQc60sIQUv31iQG97ISd7DulDa2Msb6s0xtRf8vkEVEcH/poV9trR4aEdf/eJeZfd/5m7fF2qshXCh25IccumOGHXekcgmhTUFXq7C3R2FUz/sBHP59ttHfjjyuRfooKSlhastjZ8yxIuXWS+GA/5X/vc/ZfY1265PtCjxhZRUDyKye4HnLJa+B4+aAwIwt7Xs7p/WKuTA6Y3GeXHgIx379X/pyVbWsrhtKND/DXNzsqVi8xTV13ifP3fvrHJvnbLTTrIBJYShehhyD2PiI0I7w2qQYWE3p4Ch48V9KlBJF/kXx7ZwWjFqubP+cwKIK2tSF8fUhkAAYSWVeHGzHD+s7MbuP+vPtjk3nn7hsCKNqogmxXLdkHnIfMEBGkQ571mL8uwAn4Av3isXx98syQ79gbdkmfpk3vJT5xnlD+mbt0KMq4PVfqnNDRgzpxYEYzRctkFskobPjKznrs3rU0lW29dxZzFywOyytJBEWVFyoCjT2H8YcQOlRt3wbvvUhtQIcWTTw2x742sv7fT2IdP6Lt3vMpPW1uxJu7lXMvI1q3IgQNly0xU5dvby82lcw+3Cg899KD1wFfuaSp4xRUirEnEuG5ekyy/9uomNq6/iMYLlgR4EaWLJVHKBlFQOoYZex4xeXBdxtI+YznNjHobOYdxtQYVUezbO8bvd44Eg2ms7S/7z3TsoeVs0PEAJa2tqL4+ZIJVTFubTAqkFDz44Ifc33zv8UT3yGhNJkOqpM0so2lSigVh1yz8ztdvv2DmDObOa3TDFy6u57IV57Fs6ULCDbN8Cq6ls74lolHKYLweTP4wyu9CHAu0w6nTJYZGAuY3uee0htagwoqTJwps25U22gj7DvvFQHMfwLJl7/T/d6wlAnMSJGNVzIm4NIrFXCXMA2YjzFCKetsi5TjUxMLEa6otuz4VZmZ9lMZZCZpm1zB7VpL6+jpq6xoC4jUG7Sryngq8HKLHED2MKZ1GBadBpcEBnbfpG/LxfUM4JNQlK+WSnjwgiStkMj4/e2QA39P+vs7Afq1Tf3HHPr45mVUB5KZmopFaVgaGVZlRVhaLLKuvc+bFYtFkOBQmEXdJxELE42FqqsPUJaMka+Ika+OkkglqqmMmkohoQq7BckAbwfeFUkFMaUwCLwPBCEqPIDqDMAZWsTz8W1L09Bne6CrhlTQL57s0zXRwbAF/8mrJGBBbKBY1D/6mn8yo758cMPaOvcHT2/ew/sNlUD1ZZLOLNgfSg8xLj8F5cxy++aVa5s2IEo3VEw7XBoSrDE4YLLesdYDgAKVAyBfAz0gwUrB0UMDoHGLyiClgSRGxPGzlg5jKAVvI54VTg9DV63Oq3wNg8XyHiy+IYbsCnplcOBt3XUcoeZpfPDbIcNrX2QL2noNBn9jcY0Da2889TW4bw+m9h5mbHsP/x0/XyIoNVYp+XzC9QqnbIg/kxif1BbThzaMFqhMWqUa3XJY65XtRfl/AB1OAdBoyWSGbh1xRkytoPE8TDgmNDTZXXBgiUl2Z3iloTEUKPSdoSJHLBjz8+AD9g572tfD8Ps/Ll2jd/jLd53Lft9z4tnVc3Z/mmRf3IzdcY8mS8xwWzgvRUGvx2ht5Vl4YZc7sigCtwPcNnceLOLaiqsommw3I5Q3ZvCaT1YzlNMYYImFFbZViZr3NrHqb+lqL2iqLSFSBXRGLPIP2y5FDybtURgKEFad6ivz2qWHGsr4OtLD9FV/1D5s7nnuVh8Zr7HfLl3bB49MCqlDC23x13Bka9dj2fJbBkQClYMerBSIhheMIloDtCLGIwrbKmwi7QjikSMQUyWqLCxeGmT/HYWadTTQ+ASwAAoPxzBkWVOrckFAu7LWn2bUrza6Xx7BtgnxJrJ37PDMwwt073ifoeOppd11uOH8OsVjE8tdf4dgtK8I01LnEI4pUrU0+H2ApIRZTb09PWVL+U5X/x30vMFAyZbCSwRTLgUYql7xX+fcWpFO+8NixPNtfzHC6v0Q8qvwTfdreczAYHS1w145X+c37BX0r9dy8hrWB5rtKqWV3b0loJYF0dhWl5Bk2XlNFJKR4dFua++9qIJm0MZ45Y2Nmwk6VkvJGA/N+Kr23Pm90xZXdMuSp3iK7Xhmls6uA6xD4vsihrkB1ntD78Ln7qZfZ+4eAUikorMqcUFhlaXNsvvChjQlWrwj72mAnq2yqYoojJ4osXRAmHCoLXJNaR6BYNPQPeTSkHFxXzglsJtwlsQBHgW84cbLIKweyHOnKI6CVJaanT1sHjwakR/lOQxVfbO9g7L2C0TktO/GDm6/k9tEs31m+2El9vLVWX74qagiw8MEU9TskkTMCiSP09ZV4bvcYLasSJJMO2nvbjScGHLFgfAwrN+pz5HiBA4fz9PYVjQhaEDk9rFXnCU3/kNmJxZe3vcBT4/2ftknLjfcBO/Gc2dGBf+9m5h85zdcE7lq9MsydH6jWl14UNihRFLToYEIrVM5aRAFOOV/qwKBcxVtSfUUrIjDksgHdp0ocOV6gu7dosjlfKxFT8MTuG9IcO6kZypidfsC/PvsK/10ZyztnwfCHwnK2lW9uYcPAMH8fCbHuikvC3NiSYNXysO/ELcGg0Ai6cvQwk6zsKg4fyPLa4RwXXRAlkbDo6S3Rc7po+gc9k80F2tdGfF+sTBZ6BzW9/XosV+QxDT94djdPmEn29Ue1GCZ5qdbW8o8LRODGa9g8OMInHYsbFs23Q0sXuMydaTOn3jYNKUsnogrXVaJExBgjXgD5gmasEPDiK2Omf8gjFlYGhcnntXgBquTBaA4GRgx9wzqfK7DTD/iVA49se5ku/oSQ70twO9tt7rieBd29bPEDbrAUzZEQqUQMqmJCIipEw4Lrlgc6MKCNwbEVBsgXDKM5QyYLmTGTyxfMkaLHrsCwLWLx7O92c3zi91KeffqTQP5B6uJkX/7xm5ndN0hz3ucyHbDcGM4HZliKKqUIKYUAJa0ZCzT9WnMs0Oz3A16yYM91t9I5/vOzcW9qaUFV5oo1f4bX/wEOfwoncYtxTQAAAABJRU5ErkJggg==', 'fruits/black-berry-dark.png': 'iVBORw0KGgoAAAANSUhEUgAAADQAAABACAYAAABVy1Q8AAAX3klEQVR42s2be5BdV5Xef2vvc849996+/VR3q/VWS7ZkIdsC4QeGcfMOA4Y8BmVgYJgCKsUUCZVMMikGxhPC1EAxKQh/UKSoGkMxSagMcciDKcCBsYPMGIyxMLawXpZa3a1Hq999+/Z9nMfeK3+ca1k2NhZjD3BVLam77t1nffvxfd9aezX8al8W4FVvYWDjdu7asJm5kS0kw8W/39x5He9WxXTfa65mQPkVg3HXvZTtSwv87303ceCWN0AphsVLMHMKTj4CjVW+X+njA5OP8dMuKP/rCMgAfmycbQL/79AHGf+tD5A3G9i5GSRN0FIZRdCvfxH70L0sVPr4jZMPc6obs/95A/8qwOiWLQwa+Nbvf5zxd/8B+ROPEXz/m8jpo3DuNPL4QxgU89b30RZhWFNuBnRi4ufH/MsGJIAc+u+YFL72vj9m35t+h/yh+wimjkMUQ6kCcQU2bMKdPQZf+Qxll/N/x17K3YA5fJj812fLTRDI/eS9Q3zhHR/iA++7k+zh+wjnz0MYQZaBMfhWAzl7HDn2MOniRf7TyB7ufOw7NLvx6q8HoENY7sbV+njvK36TL/3JXeQnHyE4dxqcg6VZWJzFzU5j5y/A2jJf9Y5Pzl/gsSti1V8Gy5nug/T5SGBoE3uGRvjxJ/8bpSzDnDiCdFpw7gl06iQsXESSFkcQPjxzinuvYEN/NWBerDP0fA8TQFQxeL78zn9FpacPTv64ADP5OP6xHyAXJpE8509vOcBtXTC2G5+7WjAvFJAA7LyO9+3eTe/P2WoGcP0b+Nc3v55bX/UW8kf+Fttpw4VJ/KlHMWur1KOYO84+zsfuvpvsSY16Ps15sQGZLqz3u4i3AsIEwc+85278juvY3r+Bf3/og/jJx7GNVViaRc+dhpVF1ipV3vTEo3zj4EHC7ufcCwvq70JYE8UKGcN91vARQCeeeS4PISLo6iKfet3bqfYP4y9MIuursDiHn53CWOFdJ37Mg/v2ER05gjt0CDMxQTAxQcAh7C+TuAzAzv3ccMNt6I49/CHA5VkugmHjOC/fexD/nx8m/w9fQz/yBfT3Pkx26z9AN+/iiwhM/B6xmBfRHL6Q1+o8S1t38a5Sld/2KY+ePsWxgwcJZwdQOY6GAV9463vZs3UX/tIMZn0VvTiFmTpOZ3gjb5mdojX9KDlKBGw7+GpesvcAbwgi7ugf5kDfKKur8yxd7UoFLwCLHjqEvftu8sYaD+05wHja4atq+J0jR/gaRzDD41w/NMCbD07gz50mEIHGKm7+PMHaEp+cfBw5+Fo+sGWc141u4cZSmWvTBM78lEuac5fmfLcScPZqNeiFAmJystCg1XnubazyzpfcjDVH+B+h5V9cnOHzSZ1/dus7sMaQJ21MmsDyPMHsNNmu63nl297Pv9mxl/5aP8xOw+MPcf7s4/zZsSN8Ccj+ztT7i2/TCYHD+RWWJt61yhM33MLmqISfPIa9NM3ng5A3/8tPs7PdRLMEWZyF799TeLU3v7tIFZbmyb5/D3LqEb508Sx/AjIvosVy6NNTjb/rCgkcMhPMy2Ggy1wcLv7WYuDiu73sHUqiZEgO92k6dvLLZ4+377z2Rth9PT7P+OcDw1Drh6VLSBhBqwFJC24t8h5/9iT+218x4fJUfOfiWusTxfQqqrCJvUMxkZ3ksfnimRPBRDeOp2I6rM8Udnm2pOvnzcA2rn2lxb5H0VcAW0BriASCtNzm6crY/hZ7bxSOPqjudW/H7NiLXJopnPSpn8CFSXjNP4Y8g/91F7r8o60ap/3nM9P5mHqZMdh3AreCbi5UQU6BfnySE994vmTxmYAM4Hezu9cR3K64A4qOCgaRaNWIecL5bK9BPiJiUBRVj6KIgKjFm5x08yRb9masLcPtb4Wx7YWTDkswNwN5DnsOCN/7lnL/F4eI1zai4kClO8MG7RoERbvZnMOR/zmIN0g/kCp+BsKHznHyARB9EpRcCWY7ez8o6B8ZW9pqgggJItTlZJ06SobBUqxxnntyY4nEEpPhcbSpbFyX4ZfUqQ5m5CkszUGewlvfC97B+TOQdiAqw4++bclPXENgBO+9FkGJIDjAoIiIENhYAxubwJQQsQiCqid3HZJ8jdx3jijBh2c4fi9gBT5m4ON+B3s/YSX4qFYqEMVOjNWsuYjJHAPVXQz07CQKBwSPzXwT79pcXHyMNXOG0RtWufbWFtv3wdAwJG04fRSiyPBQ1zPf9FrYuc9jLCxchFOPGI59p59kbpiyWBSP0xxPhmCe9LREtoc46tdyvMEFNka9o0sZeJ+ZdmfBNDvz3iNvm+HENwqDyd43CObbVGuZxGWLV5OtXWIg3sr4pt+kt7oVUUOpEjIyMkTashyd/htWh7/M7tvPcM31EMeQJUKnBYuzQmNVaXeUxx4szs9vfQCGN0KlKlR6IAiVqTPw9c9HnPjWJsqUKcc1NvTupxqPENgKqKeVLFJvTtNoXcCEMdXqGMYL4hURARPkjfVzQTOZW1JqN1iAfgbvMmF5h1SqCGKzxgIbKru4cdf7icJeWukCc2unGB8fZ/PGXTy8/BnsTZ/l9kMrbN1u6DSFpUtw6ZwycwbOTynnJ2Huse1sTn6XUrqVx36gnDza5vxUytIctNdhdKPhFXc4gqE6l05s4caRP2B0wx5q5U2EYRUnDg0spXgAicqst87T6SxjKr1gpDg5ignDisvSRlW17WWcfdsc+bGgZ7BKKVayXKS5zk17PkROxvn6EZr5IknWYtPgPvr3P8TmiQfYNCacn1RmZ2B5HhrL0Kpb8vUyA3Ibo9Et7Nn4GyRtz/zSRZrZHEudMyy7x3GVKaqjC4zuyBnfBy99pWF+1nL4c7czvPYeFjpHWVifJPedLkEI1kRYE9JcmUbVURncgThPmBsCCX2jdVHWO7NnZDt7XmPE3ie9/RpGvZKtzbGpdoCNIzdzcvEenGYEJiZNc4ZvuY8bXr/O7Gk4/ZOA9vIAeauHmtnJxvgAT0zew67RO9i16TXs3bcTEzlCW+IbD/wFJ8//iErUh2iAomTaILGXyEuT9G5qcPMbYdt18Mjd13P6u9ciNsVcIZNPnhtEWF98grhnI0G5F5yj7EokaZ2V9ck0MLgACUAMgYTkzlMuDzO5cj9eHYGJcTn07X+Avi3r3PuXNTqXdlJmMzEVBCGwMReWjjLSs5/NG26imc3ScQP021FWGvMsr81Rsj3d9DYFIJQqJa4jXRultTjFqWQLF088xP7XH2VpeZ6lH78KEzv0aQ5OEbHYuEbWqROW+/ACufGYggFt4JHzqMsjCQLRQgeW0mlSbWOMRV2AHzxCiyWOfesaysl19NsYrzlKjoqQuhVa7Qvs3XIHSs7p+R9w6v772DS0m9mlM6w1lwmDCFW9LH2Kx/k2QakH1iqM5rex8JPd3HPi6+yduER9Zgpd3QVBclmjQEDBhDF5sgy+0CtvVFWdgM7b3eyst0j+kbHRxrzT8C5riynXMMYiPsSVLpH0/ARmb6KaXocYxV9RGjMmJGmv0BuOsG301cw2jrLQPkmWZcytTOFcRhCEXTDP4rPEkuUNLsyfxEuMX9rO3FQIlYtIa8vPWBkRwfkUl6wTVfq7Ihr4vLVqOm7tG+YIRzIwX/XNNZLOvFcU1BcChiet/pRS/eWUkl146XT3sjzN+rm8QyUewWvGYus0Bos1ljiqYk3wnGAujyOGUlhGDASljLB5LbJ4ADHZs/pnrw4RUwitGMVldNJlVfxnzTjjfcA7PI7RgYPGmpBOtor6AFeaJUg2E7Z3o6b1czN2I4ZWtkLiGogET7NGPzepUo93OYgBpTgzNsVoFTT4mTRI8Xif4V2HpLWI8fissWgzbX3qPGcfMpj49yG/YbDvWr9t46sNKniXkfgV0KALpvMcYIq8ywQRnXSVTr5WzN5V5y6C0wx1KSYoXRG1PGt1TBByn+CzDiKWTuOSX186Y9fT+c/NMPlRwBrv3W8bCXV06GUYE1AKe9E8xWlG1o4uD/ScRTl1hFGNRvsSnXQZI/aqi2gKpOk64hUbllHvnzNFexJ86tpo2iGujVHu3azeJxiCqSItmxDjyHbF0ZDEUb8BoVrZiEtbiBicT2nndZzmXVDP8jD1BGGFXFIW68extnQV5TRBEFJtkydr2KCMsdFzfE4QDE4zEreO5gniwYYVosqgBFENT/bRYfb1HOZwbhStRFENa0K8z8hsDj5H0wQxFtWcTl4ndU1UXTeUp/6A4DRHg4DV1VN4l112xTxtEp76jKqj4xrkeRM6baLqUHHIn2VsVUfi1unk9cLptxoEpR6MDUHV2LimggxVYDdAIJAJBIjBuQ6JJNiwim/WMb1DYCyoJ3UtMt/BSoDpCjEoTnPydh1tNxExtOozhL2jGExX7MzlDebU4zQl9ykI+NY6qCNtLYG1IAKqlwnFaY7XDFWPmABNWohTSv0jxc/EICZQwSBo7ckU/EKWt3arOgUVxSOlMrq+imssYap9SBAiUjCOwxUr4jyapWjSLDxVqQcxIWlrmbQxh6n28QyZL+hbCu3R9jqatAnLA+TJOs2VKWzPAGKfpHl9al1tCFmCa9aLLS1XUL56o3gMZqW7QsH3kmR1d5Kt+1LQYwMiWs1VAhuDevzaEtgQsQEYA96hLi/uQAAblDBxXFh5lLDcR9au45HCFctTFSjpVhN9q4F2GgRxH8ZYwrhGnrbwa0tIqYoplYudIRQr1mnimnWCsAwIzaVJqoM7kSD2Lm2J4mdTBs8ABFFQ+sskb7x3rXFWNo3cSpQZVHOiyhDedfAuw7sMzTJUM2zYg40qiLGIDYsgBUQCvEvweYew3EvWruPyFFOqIEEEIqjL8Z11yLMCjA1R9RgbE0SCdyEuaeM6TbBdDfK+Szzl4rkISbtD2lwi6hnxrtMIBHPPeR5sTzAR2GU/P93HwK152rpmsG9PdnH+QRuGNeK+MVAPIthuQN7n9AyNE5b7CaIKJixjghImiLE2wgRxMaM+L8RSPWQpPmmhSQtN2xgJCOJejLGAEpT6COO+rrD6p8ZwCdaWsEGEjaqYoMhWjS0KFHmypj5t41xLPcF76iwsTjNdFMM3svWBtlt9V5Ks1hqdC35s4GUSlfro+HXCqIoJyrisjXglqg6hmj9D9J7a8yYsIyJknTrGhARxwUgmCLFhGRvGBX8ZS1gexIZlUI8NSgV9BxF52iwcebkXY4Lu+XMYExCWB/A48vaKilPr4d/OcPzrcMjCMW8AOcVjZ0OCNy03nphxmmBtSTeUx7lchwlKKEpkKlSCAby657ZB6gminiJQn9Nb3YYtVQmiGjYs9CYo1YgqGzBBdNk3enV4UUJbxmUtbFBCvcf7wggHUQ9hZQhjQ7xLvWByRf9wmhP/saj43O24opEhOMOJI1FQ/V2DMZlrUgkHGYx34HyKKBhjyV2bgdJWjAQo/jkchCJiCEo9JNog1JDNQ7dgShVs3EdUGSIo9RaU213k3Kegymh1L77ZQKVwDoh0BXSo2KZiyX3q1GVG8Y+e5dhnrrjle9r9kAdMGFSmPD5L0jVxmulw5Roq4RC5TwjCHhLfwGDZXDsA6nGa/YxogpDmLQZ6r6En2sTM0v2EzjI+eDuxreJ8Su7a5F098uroizexe+g1dBpzrDRPUe3fQVTdQKkyRFjuR0yIaGFkU9dUsgyBvwXsBBPmua5TdDi/tZ0x/y5UB4b696kRKz2lYdrZMqm2SdtLhKbC2ODLiG2t8Hu+g9cMj0fxGAkYirczVruegdo4jeYFLq0cYbC6i80DL6cnGiYOeikHffTGY4z1XM9AvI1zc/dzfvl79NR2Epb7Cw0Tc9kCO81J/Do+bau2141HPlRn8cI000+7mbgC0ESwwDezAdm4N3frN5ejIddTGTN4T19pDGMD2p1l6utnGajtolYapSfcQC0aoRIOUg0H6S2NMVK9hoHy9u4W7bC2PkXqmiyuHiNJV6lEg9SiUXqiYUqmwnrzHE9c+GtW1p+gVhunVBkCdU8lC5qT+Tapa+LV5dJsBs4n/2eGU58taoqH/XPdPnSrpy+5DrKjYVAxe7e/nWq8UZxLsDai1Vng6Nn/Qins5dqt/5BKabjYaGIuD+V8SitZYHH1GHPLj6BGKMWDeJfS6SwgWEJbxUiAcwmZNjEE2KBC2DdS6JX67pdD8SiCiKg2G+KTZl3hwDQnp5+t78c+nXsP2TrfnR9gNPE+ee3S2im1tmTCoIL3OVHUSxz2M7/yExZXjxc5UFan2b5EvTnNQv1xLiz8gAsLD1JvnSMs9RJGNbxPCeJe4nAQ71NMEOI1Q6wljPoIoh7UZaStZTw5aqTrawXEIOrVN9dU01ZTkLdPcfLHz9WZJc/WUwD4Hex9HHRfQsNH9JioW7VBhDxvgzE437lcXygIwWBNRBBWC9rVQuVtWCEsD1LRCktrJ7FhjPq8cNNocQJEUJfh0iZefWGzTOHAfZaqAfHIm6c5/q2uB82v9n7IH+KQ/SGP2Eo4xM7hN7LamqaRzIIa8mSNysBWbFgFVVzexucJqu6yGVU8qooYSxD2QRASBVXy9XWwAUFUI2ktPm1GCw6wRNXhomavIFlOJ1+j2rdF2msX8JrOPF/PXPAsSaS5m7vdNnbPeZ9fO9K/Xys9Y0zXf4g1JdYXTxcFc1FUHUFUhagHLu97f7k6I2LxXYnoC8c417qPsHcDJogpVTfgsgTVHMEgJsAEEWJCPDn95e3UF48TRgMEYYyqUyV88syYX6BPoeB1S/wXiavLYv2Ej20v4gXvM0o9wySNOfrCMcrhAM5n5L5DrmlB3VKUBDye3KeEpsKOgVdSXzlJbj1BWEU1x9iIMO4lKg8SlvuxpSoYi9ecWrQR0oT11iylypBmrWXv8csxvTNXTPzVXutPe8CsMv9oL303JEl9/+jggcyYwK62ZwijXpxLyJsrjI++jr54M5GtENkKoYkJTZk46KEabmC451rGevYzN/8Q8/XHGRy5AeeS7koWK/zk/w2G0JTpjccoB/3Mzv2AsGfIWyVLGnOhR/7nWR79qyc92y96aSyA7OG2apOL922o7nn5rq1vS1eSmXCheVLEWJqr05SI2bXlDqrxKKp51+MpRgJELK3OAtOX7mOpfQZLxFDvHvoGriHLWzifdidaMBJgJSS0hbG9eOkBWu15KuUh0nadTLNpi719kmPnnq9FU56vJWycV4ykXPpmLRw9ODJ0IwQlt5Ze0kwTmvVpS57IcP/19Nd2Uwp7AaWTrLC8dorlximSIEfKPYjzmPUmA7VdDA1dj7Wlp86cFFqWpnXm5h+mkywh2FRxJ4TgXof/9DlOXnwxGgC7964396Ysfcrj3l2ip2ZNSK4ZiuI1xeMwhN2KD2TawVlFSmVsUCYyZRLXQL1DWi1iqVKr7aRc3oDpFmearYvU62ewpR5nJTTt9sL9Mzzx6p8pAr4IfQqXaXJX/PKtrrN2m6L7BDus+PfFcX8Y9W6S1sq0dMIctUVxxNoSkalQMlWsCUldi2a2VBjaNEWSBFEhMFFR8LAhcc8IQVTLW4tngsx3PjHDqTu3MxFPczi52k6Sqy9yPsOmA2znmo+XbPXflQe3pZ3WathxdQmqA0RSJjQxIqYQSTyCRXEkrkmmHXKXMFTaTnN5krB3lCDqQQTfXJnWrLPiA8J9Zzh25vnOzAtpXtIC1IQ9yAY7wECwmY2HF3X5pXRa+4w1Qp773tpOVL14XPdaXq7o4ixWxGDZ0neQ2EV+LbukcW3M+7zjO/WL+M66BfPRsxz/6+djtBe751QADnIwWGT9zwy83+OGRgcPMji4T5dbZyVx60Xtuiu2RiyBKbGheg1l06vHJr8iplTDUKTszieAfGKKE3f+Iu0wL2YT7eWDOs4NIzmd/YK/a8+Wf7JzaGBf3kwWgtx3cL5IBAMTUYk2qPe5Hp/6K9PoXJyxhJnHzVuChxX5yhTHf3g1vxLw99kV/LTztZ3tL0XK3x0fe2Pv8MANuYBVvAhGRcTXm9N26sLfUE9nPllh/59GzOgxjqXPRkK/KkBXjHUwgCPZNnYdVPivG3r27B3o20MU1XB5h5W10yzWjzVyWn98nnOfe3rP0YTtJmv+hc7u38tvnWzjVQM5039kCf+pIdzkyZccyXcM1U+f5+TRK/qx+UXamJ/v9f8BLqjTAC7kIm8AAAAASUVORK5CYII=', 'fruits/black-berry-light.png': 'iVBORw0KGgoAAAANSUhEUgAAADQAAABACAYAAABVy1Q8AAAXZ0lEQVR42s2beXRl1XXmf+fce997ek9P8zxLNUiqWaigBgrEEKAMlLsbR44rpkOA1SEhdqc7jpvYxp3GHbzsrDi2w3LH6cbuJLYb2xUcPAENgbIYihqkKqoKzfOs0lRPT2+80+k/3pUoMEPZ0LbvWau0tOoOe5+9z7e/PQh+vZcGcOA28stqebSokgslVaSLMz+frG/mTqWQ3r3ycl4ofs3KOM0t1C4t8MSWK9m15ybwB2BxDiYGoP8MrEY4FszlvpFzvOYp5f4mKiQBt7yBGgFH2++n4UP3YcdX0S5MIMw0yp+FQqB+9A20k8+xEMzlmv5OBjyZ3Xd68a9DGVVVRYGEp/7wIRru/M/Yg+fQjz2JGDoPk0OI7pNIFPLQPSSFoFiZXAWotrZ3lvlXrZAARPv3kSY8fs9n2HLwd7FPPo8+1gu+APiDEAhCUQXOaA9850tkOTb/t7yFI4Ds6MD+zXG5NnTxAnZOIV//yMe5754HsTqfx5ifAsMHlgVS4iZWEaO9iJ5OzMUZ/kdJIw+ee5a4J6/6zVCoHY0jOOFc7t73Ab752Uex+8+gTw6B48DSLCzO4syOo81PQ3SZ77kOn5+f5twlsqpfBcpJ70Pq3UCgsILGwhJOf/4x/JaF7OtCpBIwOYga64eFGUQ6QReCByYGeO4SNHQvR5n36wy928cEIJRC4vIPh/8Twexc6D+dUWakG/fcK4jpEYRt87k9u9jvKaN58jmXq8x7VUgA1Ddzz8aN5LyDq0nAySviT6/6LfYeuA37zEtoqSRMj+AOnEVGI6z4Atw+2s1fHDmCtRaj3i3mvN8KSU+tex0fhwBBG/rP3XMEt66Z2rwi/lv7/bgj3WirEViaRU0OwcVFosEQBwfP8tPWVgzvOee9CfXLAFZbxkJS8rwm+RSg2t58LtsRQqAii3zhxt8mlFeMOz2CiEVg8QLu7BhSE3y07zTHt2zB19WF096ObGtDb2tDpx3tVwlcEqB+Gzt27EfVNfJnAOu7nBGGsgZ2N7Xi/lMn9l89jvrU11F3PYC19xZU5Qa+gYC2uwgI+T6Sw/dyReZZqt7AR/0hfsc1OTs0QE9rK8ZsPkr0ogydrx+6m8bqDbhzE8hYBDUzhhzrJVVcxm2zYyTGz2Kj8AE1rdextWkXN+k+bs8rZlduKZHIPEuXayn9Peii2tvRjhzBXo1ysnEXDWaK7ynJ73Z18ThdyOIGthfmc2trG+7kELoQsBrBmZ9Cjy7x+ZFuROsN3FfVwI2lVez0Z7HZTMPwa8wpm0eVzc+COqOXG4Peq0KMjGRiUGSe51YjHN56FZrs4p8NjY/NTPC19Ar/Ye9H0KTETieRZhqW59Fnx7E2bOfqD97LJ+qayAvnwew4dJ9karSbv+zp4puA9UtD7y/upm0COuxLKE1gQ4TBHXuo9PlxR3rQ5sb5mm5w65/8NfXJOMpKIxZn4djTGa52652ZVGFpHuvY04iBM3xzZpTPgpgXQmXMod6YavyyFhLQLtuYFx2Ah1x0ZP5VmRdnfmuiqTDtSxeKjlxllvf/w2hv8sHNO2Hjdlzb4o/ziyGcB0tzCMMHiVVIJ2BvJu9xR/txn/mONJbHAg8uRhMPZ7ZXoRRU0FQYwKeNcG4+8802vc2T43WZOtSbA7v+VkkXHHE63qDI61cNm6/W0H5PofaloArTF1YipetzdYllbZzxUEJr2ikQQjmt1yFTiYyYQkI6BcWVUFoN0YvIF36ESPRVunlm3h+EZGpauWJCoh0G9oKqdDFlA80DoD00QsdP306mSy0o3sy3NnJVToKFa12cXRJRqmHgJxjRkYMmVhPwKemFLxcHV7get9FwpY1ZOUJVk0V0Ga49BOW1GSZt+OHCBNg2NO4SvPiU4oVvFBKIlqGEA0p47iFRHkFQ3rKxsDC/CLgaMk+B6aIm/BgnJxh+WSDUmlLiUmVq2Xy/Qv15iJzqMPlkk0uKODOMkSCG4YUYC8u2MGWWCIqACmHi4pAkWBYTxVtXCBVY2CYsXQDbhEN3g+vA1DCYKfBlwalnNOy+TehS4LquAqEEQgikA0oqlJBoZJOjssmTWQTRMZBo2FjEWGGJOWKsdvnxPTBC73OAJuAvJDzk1rD5YQPfpyuop5Ayx8BQg5wjziq72EcLByinXugY2opYYkFN8q/8hCF/JxXbV9i0N07tFigshnQShs+Dz5CcfE4gEOy+UVG3xUVqioUZGDgj6Xk2j/SFYrKEhsLFVGlSpNDQkN7Kp4RSqlUNG51s8jBJojxLpojLMfrkGP2ujvzgCH0/zRBMmm5S8Ewtm61iKjUbS57jFTaxkz/iszTThA9IkvGMsIJjsovv7voYoYPHadoKgSyw0oJUApZnNaIRh0hace4E6FnQfh+UF0M4JAhmCzTDZWwYfvQ1H/1PVeDDR5ms5Fr3dupoIodCbCymGOQ8JxjgHHkUsZlduDg42GjoGATs1ziuj9G3lE3uDg0gm7xHCyipq6AegdD6OM029vI3/G8KKWKKBV6jG11kIZTBIyV/you/fy8th6doqNFIxwSLczA3qZgYhoFJl8lRCJ7awz2jX6U6sp3zrzic644yNJFkYV6RikFZmWT/bS5uSQTt9BV8Nf0M18n9bFCbyKMKEAQIUcsG8iilm1NMM0oVG9HRcbBRODKfYmeBmVCCVVc0sKXGwurZxI5QPiUqTlRMMsijPI9FkqM8xQwjJEWcctXIwpX/jO/ep6nJE0yMKKanYHkeYkuQjmi4qyFuXLmH3dFDtKRvIAtYwWaCAbo5wZms55gu7MKpGiG/wWRTM1xxtWR+1sfyQ3dzaOBz9MpOut1O4qzi4iIQBAgSIMgJnsXGYi83Y2FikcZHwO3njBjg7LCoZMP1AbKe38xOVUyF6KGTA9zGHdzNt3gEkwRZIkxKmczd8vdU3zrPfC+Mn/GjXajAWC1mY7qV/dYH+LL1GT7m/hX/joPEgFUcAkLyYx6jm1OEVR4+ArgoonKBuawBxgpewV+9xO7boLwJJr+6B+2FW/ALhVQ6wgNi10M+DY2j/AtNtFJJPWmSCAQXmOQ0L5i6i6traGgYGPhJk6SeZn7It7FIExRhbKWY3vtPqLp5Tv19EXmjV7I9vZVsNx+BJEQOT/IDDnA7hzjIuIijlEshYZZVjGXmySYHgSAtEoAg281jW/x6ChK1nF74V5b7G5lvO8qGu04wbi9TeuxuEGlc9XpkUSg0dMqpZZZRqmhAIJDeaZJITfehT1mYtoGhr8WBIV4jSRyf8CGUn6HKx5kOjFNy5GquWL6eEGEsTGxhIhHMM0G/Os0f8yRJXH6mnmCBGaqpY5pJolzEwEChEOr1GBYTEYpUBaFUETtTHyT2g+vo7P878m4aZGmok/L5PVgivv7MmqVyKGCRWWwsT2KpbCzh4M7LInKGbKzuiyzQS5ebYJUYK2hCoqkAC8FeRgpO0NjZzq7lf4uBQUrEcbFBgaECXFCTbOUqNlPJSToZ5BxJ4rzGaWJE1pV5M8MSKgPNIcI8x/dxsdnSfTfOt27nQs55HGG+QZlLrZTBOXstGLsRlnCwOmQXXZbE970J+hngVVehcLARSqKEw1DZs+yc+BB1sb2kRRSF631EeBFZEmOFWhqxgLO8hIEPDY0sgmjob6HMGwWUaN7ZclAiRcPSNWwYux2H1PoZuvSyMZFIfPjRMVSSGLOMKR3ty7KBhlyB+xEHh0PcLf1kscA0Eh/zoV5KV7ZSs7KPpIgg1NtnxBKNBeJEWETHwPWWepc0xsUlTRJtrZygBKaIE7ILkMr3hucFAheXFEmiXGSEblwct48z2iqRL0wxelJKwn+YIrHjam5zf5//IgGSxFlmFt31U7O0D5MYUmlvK1CQMAtMs8QcDvZlZyUSSZokcaKEyVsXXiiZcek3bYbHDlglgoGPbk66HTyhTTH0yBwTnwY03ST1OwFC6jbuxCBAEWXEiWKSQiYC6zvzVjstENhYFFPBMK8xyzg6xmWX0QSCJeawscmnOOPq65sh3kL5FDFWuMg829iLi6vO0IGBfyyTlrUJaZLcUMNGUUatFAg2sZNFZpFopEmwyCyW57Nv5c/OJcK8zJOEyMF5x1xMIbwVJ8ocE+RSQJDwWz63BsspkqywxCoRHGwKKKaWzaKMWixSn95CW3YHHbZ0cIJFlOMnC5MUuRSQIM4Ki/jwY5FmiTmiLGNjZ0DSW2uCWVgECfMSPyVJHB1j/f/euCQSDQebCItEWGKRGeppRqJdcs/r77exWGGJZeYQCKYZppgKsgjjYMsyqhWIQovIRgBdoSwQukTLwDU6tWxmlF62cCUGfmwsVokQZ9VDlgyKgcAizSSDzDONDz9dHGUre5CIdbqfOcwZ9DRJkiCGhs4Mo6RJMUYfWWQjkTg4rCGtRRqTNA42frJYZIY0KRppwcZCQyNAUGVSCjMMoEvE9ApLG21MpVBCARXUMckgfZymniZC5HiJ19qHLGxMVlhmnikcbOpoIptcznKMPjqppQkbC3HJeXC9bNlPFtOMssQc29nHOH28zJNsYgdZZONgexwuAwQBgh6q9ZBNrueImQNgYUkXB53ARQDdT+DFWcY2XmTBzadYUyjO8jIlVOLg0M0pssklQBADHyZp0iQ9DiUpooxSajyLQRMt9NIFCKrZiIaG4/Eww8v4JxhkhlEaaSFINpvYyQQD9NBJKVUUUo6fAAKJg80FphijlzJqMfDxEj9hPx8ghwJ3mQvCxZ0tITg8Augh8v9xibm7z/CiuIO7WGKWBDF2cwMmaVZYJMpFEqyyyCxFlFNCFdnkECYfHR8KBw2DFAkAGmmhj9OscpFiKgmTh0CSJsEM4ySIspld5FKIRRo/QepoJpciZhhlgRl8ZAEuJiYONpU0UE4dOjrLXGCEbprY7c4ypusYTx/neLKNNl0IBCVUP9XAloNf4HvWA7QbBn6u4FoWmMnAN5IFZhinnxv5bQKEPLfI+Psa+ikUUZZJEKOP0zge1zJJe/cociikmo34ycLBIpciQoSJEyVBDIViimEuMEkpVRj4yaWQLEKkSREim2XmmWBQ5ZDvzjJOgPCOIV7tBYSuUKKUsvvH6Tv5CJ8qmmXcbed+WUIlKyx550cwxyQBQuj4SHndwbXzodb7XYJcCtE8phAkh1oaSXh5TYAsAgRxvZbPmmu5uITIwU8WLi5j9BEmjzqasbFwcUiTQscgl0KSJIgTVSZpzcD4xBCv9kC7BkccCYhznBzNInzwBM9MxIkSIqy2sQfhVXcyu+kQJpcSKrGxkT/XGBBeXHLIIZ9cCogRoY4miiijxHM9DYMQuRRR7vE3F4FcZ84+/CwyQyFlWJiYpHBxCXvP6PhIk3Ql0pbwZ6P0/c2aMlwyyKAPc76riOJ/L9HkCsuUUkUTV5AmgfA+FCPKRrZj4MPBRr5NN0aiUUQ5S8xhYXI9d+DDTzb5FFHmWVFft2qKBArYzfVMMIiGQRHlgCJEDkWUk0MBGhomKSdOVDq4Z4fp+VJGhyPOm/tDbqbVExpTuNY808IkrVo4QBm1JIhRQAnLXEAiuZZDKFzSpNajuUCuWzTKMldxI4208BhfwcHhDu4jj0LSpEiwSpIEKRLYmDTQzIe5n15OcZJnuZIbKKaCQsrIowgD37r1V1hSq0RQqJcArY02+XbtFHWA1uQUFz6qcPKv4XalYYhK6plnigSrjNBNDgXcTDu5FJEmSYJVTNLrvm7gp4kruJpbaeEa+jnDk3yLXRzgIIepoI58iimmnDqa2M9BNrOLb/MlHuNv2cPNVLFhPXCubZhJmlWWibCkZhiTPrSPX2RxepzxN3QmLlGoTe/hSauUqqYl5q6qZpOziR3SxqSORgIEmWGUTn7GlfwWdTRSTh3VbKCUasqooY5mdnE1zVyBgZ84K7zKy8SJ8iI/YYYxKqinlkYqqCebXHo4xZf5BJ0cZS8308BWLCyklypYpImxwioXcXHtSQb1BLEfjjPw5UxNscN9u+6DBNxNtDSniJ3PIV8+yP+kga0iQYwAWUwwyJ/zYYqo4JP8LTVsQiDQPHqTST0STDJIBz/iaR4jTC6NtBAnyqu8hECSSyE+/MSJEmWZAEFyKGAre9DRsbG8fNTG9Ri4RFeTDIoLTK0ECe7q5+z4W839vIk+Z9Cige0PmCQezqGAw/yJtp29+PATJp9jPM1X+AQhctjHQepoxMBPigTzTDJMN6P04uCwkW0UUkaSOAWUEmGBaUbJIZ9VVjA8GA6SzRQjLDJLKdUUUoqPABKBAkxSaooRtcxcQkO7Y5S+Z99uMku81UwB4NbT3G1jbbnIgltAicyj2KP+cj3BShIjSXwdFAx85FJACdXkU4xCYWESIkwR5Rj4OMXz1LCJNCkE0gvOLho6US56hDWJgQ/dA4MYEeXgCB3t1hF6n/K6Jvbl9ofcdr6vHeMBrYIG/oj/znmOM0a/V/+aoJXrKKAEhSLGCmmSuLjrlD+TfDtIJAWUoGOQQwFTDOHDT5Awq0TW8XEtr8omh+3sXS8gRrnINKPU0yTG6MciPfFuM3P6z2VfII/wYaeShgtpkpvb+DeqiSt4mscIEuYFfojpFS9M0uRSuK7EGh1ayzAlOjYmAkkNm3iG79JIC36yKKaSNIn1HMvI1KnxeelKM60c5V8oppIQYWwsZWCsnRn5C8wpZHA9RPh/LTAtXuYpN49iFC4pEmxkB710Uc1GSqkiRYIEq15MsS7p6dikSJBNHrdzFy/yYzR0CijFwsRPgBwKKaCUfIoIkYeOjoVJLZtZYYkeOimjWi0w7bq4y5KciUs2/nLb+uMuIJeZPxskvGOO8W23cNjy4df6OU0xlSRYZZQ+fo9PsoFthMglTC4hwoTIIZdCyqnlCq5lPwd5nL/jJX7CB7jT43XO+lo7QxKNEDnU00whZXyfr1FAqWuANcGwoeAHo5z7bga4etxftGksALGfD4YG6Xx+Dzft/gRfMQc5a3TyM2Hgo5Oj+AjwH/ki9TR7fm+iUPjwY+BjjD6+yecZ5Cw6Bq1cxzUcIsKil2ooz90MfATW04zH+CojnKeCWuaZxcIa19CuHaFn8t1GNMW7jYTt46aSYXqerKWx9RYOk02eM8WwSpPgFEe1KEviBj5EK9dRRDkKxQyjHOcZOjlKNjlsYBsJ4nRzkhau4RYOEyR7nV0IJDoG80zxBI8yyRAGPtPC7NPxPefg/vUk/TPvxwCgBNyDfDTnLC9+wcK6M5+isJ8sUl7VP0EMG4ssgmSTB0CaJAGCVFBPHkVeBSmJSYoBzuLz8q06mskiSIoE/ZzhOM9STq1j4JNDvPbCLGPXXVI+e98GANdhcjc3Vy8ytt/C2WKgF9tY99Sw0Whmj3iVo6KAEnyE8XngnOlnZHk5VGKNvrDANHNMkCaFnwAKRYAgjbRQTKXdwRN6jOjDUww9WEtbYJyO9OUW+y538EJcMoy3flWy4aECCv7rXm4yRxgwTOKijm1o6PjwI73CisJFoOHikCKO7VV0CijlGE+xk6sppAyBdE/wjJpkyM0ld0sPZ4bf7cy8l+EllVGqTWulSMsnX6+mvGOSuZYFprZkERDLLLlb2I2FKdaKhmss4vXAZ2Bjc4BbkWjuNMOqmVZ3lYvuWV5mhjHNwPj0AOd//G6I9n7PnAqAVv5An+foX0rEvSbpwtu5i+u5Q53nuIhy0WMRjrd7OkHC7OJq8ihSn+GwKKUGgWCGMeKZuuDDY/Q9+IuMw7yfQ7TrB3UH+0oiLG2zsR79JI/UX8dt9gSjeqZOngZYAwqVJqE+xz1ygLMTfrIsC3Nex9cZIPCdQc6duJw/Cfj/PVy+7rpFVLTU0LjyMP9HncCyjmO6L5NUr2C6p1D2N3hZ7eVmlU/xwx/n4/4ttPvetK/yvQrzPirWqkOXVU5Nq4Jv7+Xmpn0cpIhyYqxwiud4kR+vrrLymYssPKJeN4KANs1L1n5tlnlHoNnOgfwcCr5YQtVoLY3pcupmcin8xyoat19yn3i/Z0r/HwQaGf+jcNN8AAAAAElFTkSuQmCC', 'fruits/black-cherry.png': 'iVBORw0KGgoAAAANSUhEUgAAAEQAAABACAYAAACjgtGkAAAebUlEQVR42tWceZRk113fP3d5r/au3qe7Z5/uWTWLRvvYMo2QwIqOwAdMY7BNgrFjDBwgmBB0ThwSJ+FwMJD4JCSAMU7MQXZMG7AlWwgLLLc2S6PRaJld09MzPb1vVV171Xvv3ps/qnpmLFuyZzxSyKvzTp3uU8u93/f7/n7f3/JK8NYeYngYNTZGBHDv+zrbLpwK7g6r7l5rxW1CslUKMs5RAl5FuEd0Sn7+zJHSmdb7FSPAKGbt81rP7rot8C2DYgS1tpEDt2e31Gv2I1HgRuJJua17QNK7XpHtFMTiAhPB3MWQ5TlHfsmWqkX3ZS8t//vZl4uH32AfYmQEsbjY3NPYGBaw/xQBESMjyNFRzM33dGSrK9EDJrS/1NYp24b2afbc5LvOddIq6YS1CCEQQuDCwLm5qcDVKkrNTDhOvxhSq7i/SaTdJ195rvyUlLgr7cO5170I7mqAebMBkWuL2X1j+l1R5P4gmdJD23bH2bpbR42aVY0aIhZTpDIKzxcIZVHaEktYtGcplRquLattFCr15CMNzh2LsM6dsAbjnPNAWKAhBKtSiZxWbtr3xQtepx478rX8FMDICGr0Ms3+3wCytoiREfwT4+lPCCl+bWBTgu03pKJkWqswcMJE0GgYKiVD2LD4cUk6o4klJEKC1pZY0lKpVNm8A8aPWfPoF+py+14tpAatBc6CtdCoO+pVR7XUfA4CV/J9/jaZ5Hdf+Gb59JWUfcsBGR5Gj40RHTzUvrlaDh/0YurtnT2+2bYzJZJpJbUWKN30hAIQUmCMo1Y21GtN606kFL4vCQJHFDgKhSqVYsDk2Yif/WjShmFEW7uHH2vSxTrnTIirlR25RScmzxp17nhEsWDL8Ti/cexI+VNr63qjtas3C4z9t3fcWatEj/b0x244cFtb1NkT09WKEZViRK1miEKQSqC1xFlHFDadQBg6CrmQhdkG5WJEtRxRXA3JZGKUVqG4GrD31phwzoncciiEjYvcghblvJaNqpaeL2R3v5DbblBucLc2uSUXzy+5Hx3Y4tcOPxM8OTyMnpx8fZ8i3gwwbril7T31qvnsxm2J2MFDWSOlVAKHVAJroV41lAoRhXyEiSxSCZyDes2SyaombeKSWEJRrxqsdfQOxDh3qsLLzxX4mV9J0d0neP7xkLCeYt1AjCi67FX9mCHTGdDRawDsV/+ybi+cMTqe5sdeebb08Bv5FH29wdh+Q+afY91nN25LuLZ2bes1o5Ry1CqGWtXSqBmMafI/mZJ4MU0ypYhCR6kQsWFrAmuam4siR7Vi6OzxMZHFiQbWgG1txRrF/FSDtnaJ1rJ5hSUEDUluPkG9EtG7MZD3/2ycL/5p3c5Nmc/cOpzcNzpaXbzS4V93yoyMoB55BLNzf/rd2hOf37k/bQd3p8EhC7mIStEQhQ7fl7S1azp7PDq6fTLtmkRS4cckCzMNOrt9pIYoAqUEK4sBiVTzNcKr8sTf5ZFCs/tmDz/mKOUFWseYmqjixxS1qqFatlQrEdVSRH7RMXsBpDZiYKs1F065dFATm5fmg78aGUGePPntgu56ACJPnsTuuyl7s3Pu4fVb4nSti9laxYhEUotshyeynR7ZDo9kuhVaRZMitrXx/FJIZBzdvT4mAs8TlIoR9Sp09igGBgOe+ccVJk5GZLI+Ow94KM9SryjasklKhRCADVsTxJOSZFqTSCmSKYnvK0p5j55+KZMZYy68avdu3Br/5tcfa4yPjKBeC4q6Hkr3xhvTPSHusOfJTBg5tboUqExWyXSbFn5cWs8XRspmUBEIIVUzXArRDJlzU3X61sdbNBAYA8XVBkM3CLbsCVlebPDVB4scuNMjP68Y2qfR2iCkImrESGc10xfqSClIt2msbQKtlMCPCeIJRa3ssWGbdEvzgVxZsPvuviv49GhTtF13QFx3t5dBqlPW8elyMXq4UbMX5qYa7uK5WnZhtpHILwWyXIxko26ENc6tLgfWOivbspq56QbpjKa7z0NpR6bTOunV6N8SuljC2HMnIvHQX5RE54Dhxnd4nDmi2LpbEUtakimP8qrG9yVR6JidrNPe5eH7EmuaO3UtFesshIEnU23WzJwPB3KFxPHFucaJ10adN1Wp7t2bWlcL3EGkeDvSHVJS7tOe6BUCUhmF0pJaxdLepYkncF7MgrCiXLQEdUFp1VEpGpYWK/z4LyrauzVf+ZTPbT+k2XFAkMn6zJ5LYI0itxSwuhIShpYde9Mo/e1bcxY8H/PcN3JydjI4+t6fLN/28Y83DfN6RxkxPIzq7cVdkVyZ48crC8CjrZM778x2LFfsLs9ze2enar+faZdtmXbJylJIrYwAgbMEQhFIKHhxOVGvB+v6t4odG7crd+qIEWHDufyyEdrz0R74CUu9rHE0Q/PKYsD5M1V6+n3CwCHlGlFbIdmXqr0rZhdmgpu/9JXsMBQevzIMXy9A3OsoQDEyglxcRIyNYZ96qpAHvpnK8s6OHjI/8JPa9W6QPPklI08860rZbv3e7o7E8SAMAtnZvvrCV+eqqTaeP3i3Qmrc6eeNsM6I8qp0nicFQDJtqJebPiMILFt2JDnxQonZiw2613nUqq7lmxymdVrrbCqjZKVoPgQ8/qbokNcD6goBpAHbtY5f1jH+/T3v0Xb9NsmzjxpefMLUdYx3jb8SPD5O0Hp5jY5eDsUS3LJlj7RL004uXHSltiyPFnKMVMu4RAoRTxm0Z1FK0qhHKCXYujPJ+TNVEilF30YPE7qWtwPrwPOEUgqOHSndf+hQqnd0tLK45g/lW1QNUUC0bgu3G8MnD92nzLZ9khPPGvvNR4z0Y7xvZYbHAa8lmHxABDXu690o6B0Q0cUzllqVV4d2xn+jUrKN0y9GKC2c0pZ4OkIK0bSCyJFKKwY2xZg4XaVSjLCt1CCKHM46GnUjevr8qK1dt+XL4l1rlF9Lz682qkgYUZdP5HdxzgJg59vIVAo8uPeQ1PvvVGJmwtonv2w0jt9amuFvWmCELQcXCoETkrvXb5MgcDMTFgHPfO3L+Snf53OnjkZiftoYPyZIZkK03wzXjubGu9b5ZLKaiTPV5iIEl/wICOo1syaHfqJFefu9AiJaG1etmG1h1Fw+sa3/i+HhYf0dQrkCzOxpfn9gixg8dJ+KGhXnnvhbo0urfHF1mU+06BReGcp7t9LrxdnXt0VQzDm9MucQkr8HRCJtPtaouqWxhwIVBthkxpLOGkzEZX1joL3LQynB5LkaSotLRSQhoKc/ITduSwiHu/PW4Z6+1j7kd9EhIwpOWjjpALdnz3C6t71vsC+7eXdPe/+u3q4NG/s6BjObE0NmtnS+Ojk5aS8LnREFJxUQ9fTzI8rjk3e/R0frNkr5zCNGnj7sprcd4L75CwSt97grAHRYDmW7+NCt9yiTm3fq2NO26qd4oFqgPD8VldYPxk4Xltx7C3lnduzXQntOzJyDdJtGSoGUgnrN0tbuUSpENGqW9i4PY1rWghCZdmXmpxvxcqHx4spieGx4GKXfqEYJo2bz5s3xhFr3406an6oHy7dGMhxwMSfWVi8cSCVz27bvO+NZf8wj9tDxc898s2U9Ys8w6alX+JNb7pZu6x4pzp+w7pWnrUy08cGXx1hds6DXUsxZbsx2CtLtROMvOxUGjK/MsyDEWqG69PC+29K/dfaV6PekJHrnT8dUZ58T9Rpksk0xprWgWjEM7kpy8qUy8aSiq8cjDJvRxo8r19XruUI+eifwudeLMmtZoNsxdPD9Av5tXRd3WSxtYSc91QHaw24bswmssFRVSeb9xc4Vf+FQxS8ewvLA9u0HnsWKP37v+9/1l5/8o48/0LNebD14l4qqZScOP2ZUo86nSzkea33/dyzYCMfebLdAa8HqksNEvCoEDppV+2Z2Xf7Evlsy6TMvRv8uDDH7bheiUbLSGI2UDj8uyS0FSCnYtjPJxJkqsbggldaEzfqL7F4XExfO1n7w3nuHYo8+Ot7Q3wmMzZs3t8e97k9FKhgJCdhZPGhuL97DttoemTXd+PhStlyFIaJOzeW8RXsx/qp7MfOkejX90h148o4HH/z8R61j6Ka7lGvvFPLwY0ZMnXULW27g3xx7Cvkay1g7rBCAYltbZ5P3pVWHg7NXWtDYGFFTUJV+e++tmdWJE9EfrszB7ffUo0Q8o4O6Q+tm/aVSNmSyio1b40yO19i+J4XSAhM62d7luWRKbbowM3sDcFS/FoyhoX0bFN5XG151f3u1O/rx5Q/J/ZW3KYWmQY06FUrkCEQDhyPmEvjERH+4WWXDNvaVb2UxPmv/uvt/uKPVbxzYuV+y46BkZd7ZV562Uio+duwp8q9jHQKwX/gC6oMfoT+VhTBwolYGBFOvRa5Vs1Wjo6X/sv+OzNniKp8ae7jSt2WnM1u2p0UyqaRUgkoxIpVWZDs9wsBx/tUqQ3tSOAfJtDLtXZ6enjB3XgmIANzArl1dKvS+VvMquweLN4Q/N/+A12PW06CGw6HQnOBJlphBtCTMcmqOOGlW5QWOph8lZmK01fpkvjrN1j1Ve+e7PKG0syeetWplzj1VLfHpN6IKwG/+Dhmt6EimBWGAbNQAy8J3akqNjmKa9Ck9fMMt8duM8T756iuVn5ifarD7QCZKZZQq5CLRMwBR4Ojp94lCx+R4jS3bk0iJ6OjymDlf+wHgv7U0xIgESIeJz9f92u7B0p7wF2Y/7nWaXkIRoPGRSOKk2csdSBQKzbnu45zofZ58OMtzmb9m8G0Vbn1fji0/d5If/EiJ+97vyUQGsTznxPHDxnV3buncseP2LS0w5OtpluIibVKT9hMQNhBh4JCKwusBuEafE0fqUyePlt7tx9VIKR+eOvLkqj57vCwKuSAK6tbG4oIwcHT2apQWTF+oIZUQnT0+2pe33nxzf1KOjIxIGDXbB2/8qPXMD2drXeG/mH/Ai7skERHSyTWZQUSDLvrYxA5qlJnLTuIHcUpRDuVBf59HOuWxa3eMzUMeYQBSCiaOGxkVMs5LZPZYU390165dXa0P/Y46SIXElcbzfIgihI1ASqpvJBBaKYIA5LHDhS+mNrXdIiX/Or8UTC3NNfTLz63K+Zm68RPSzl2s097ZJMfMhbpUHs6PiY3levlGOTo6anfuPDgA/IfA1u1PLP5L3WvWc5znaFBDXtJjzQtosXSyDoHAi2JUUgW0H8crdHDqWEjYEFw4Y1ietUgFWknmVx23Vd8p37n4vqjiF3e6MPl/WlpFfCeVawwxKZFKC5xBWAdSXUpy3jB3AuzICOqFr8xVj79Q+kOvTR0AfnNprjH+whOr6vDjOTk1UXOzF2vR0J6UiyLHymJoPV8KpLurefkj8auhX0/vLd1ub63eJcbFMaY5R40yCoXF4nA4LBKBExaNT7bWRagCcl0rDJXfxvwpnxMvBPiepLAC85OOcsHS0y552Xuafflb9C35u8KaX75n97abf7OpVUbkayljJDEpQQjcpRZleOmqfNfG9pq1DA+jjz1VyJ84WvqD7ZvbDmhffGB+uv5McdWImQt1/fwTeaEVxvOlzXZ4xjl+Wu3cuTNjrPyUxWTes/Arwo/iIs8yu7mJM7zIJnbgNXMtPHwMhpd5mjoV0mEb8+mL1BIlPJdkQ347E6uzLKzW6eiQ+DFBuQhtac1EfpWJ4jg/tvoRebrtBVtRpTv72zY8uLz6j8U9e/Z4e/bsEZ7n6X379om5hckNfowP7b5NoRScPGxFo8KfNxrMtmj2PXX7W5UwMTyMfuyxcmNxNnhpZTH8TN9G/xuRIcgtBgOLs0FbuRCpWsVIIUSv6mrf8s9C3fjw+tog9+XeL2c5z2Z2kqWbkzzPDOeIkcAQkWOBF3mSZWaRKBImRTpoYyk1SymdR5NkQ3EnpdWA8ekchZLFWEuj7lhacLSvbKEz2CDabLu9kD0dk07FV/LzDy8tLZnJyUmby+XM5OSkjWmSfpJf2X2rwovhTj1vRb3EZ4KAuaGhIZ3L5a6qq78GTKuozOJscGF5LvjKntuSnwnqHG1Ubd5au+wsX9ASd28kQrencou1WJUWWVKujYA6Gs0iM6wwj0ITESIQaDwcjpCAdZWN3DJ9F8fXPcdC10XKqSzr8rvpWt1MPj/LxIkckQzIhuvoVtvIyXmShYxqb+92JT//0/u3vf3BiigMhCrYGWHWOxmlbBB2Y2ZxFqE0zvMESnndEJjx8XHTbH2MqNHRUXsVsyGXazMjqBFgdLSQB/6qdTaLNtbZ26STYlN9hwiok3XdSCQ5FimSJ0YCWh6kSZ3mX03CC0ICOuo93DH1w0x0nuRi+1ku9p8lEaRJ1vtoC9cjjaCWqXNRT7CxtIOSXxDOOmqy2t6Q9SeTLk276SZhUnSafpbcOc7HZ/F9gQChtHN+pvN/DQwkn04GyS8O5Q48NDr6YPFyEjlqrqqIMYoZ/faKnhsZATE4uG9ZSNH1a5OfcNmgW2TpwsNnjC+xxCwe/iUAXt+1OyQSjUfJW2UuM8lieppKrIiRFmU17bUu1he2spJc4Fz7CZImza7KTeyu3Ow21XfYnnDAdbh1FNQs/8n7eeHf9ZK6990+1sKXPtsgcfpWBpO38VLyCapUJpMm88fJUu8fvbLwWOWaQHm9rHbb0A1B3Ca9fzX5B1hj2cpuXuIpTnMUn9h3BeO1wKiWaIsIqesqRkbEogSL6RlO9hyhoavcmb+fd6z+KOuDbfgtR70sZngs9Rd8Xf8V6V057n+Ph46BHxO8/FzES5/rcR8JftfGRSdHOh5XL2WfpEHttF+P//qrky8+er1AEduGbjAJm5a/PvmHjJtjgGOZOeQ1t2wcToBwAolEojjZc4RT3S8wUNvCz8z/GrvrN+NwlFhlkWku+uM8kXyQc+uP8LY7YO/NPtJzmKhZu5AKvvFQyMsvwP7azdyX/1VXiOfNU31f0UvxWfww9rEz5178nesBiuro7P0thfIOFu5kzl4gx9IlX3GtvSuBwImmtZzsfZ7T3UfZUT7AL878ZzaHO6mJMufFKU5zlG9m/54Ff4q56AybD1bYuc9nZclQKUK9AtUSFPPQ26/oX6c5OjNNstop0kGn7C1ssKvesq2myvd0tw1kc/mvPzo8PKxbhaprA6Srs+/DVrn2weJeVzUlIa5D78oJh+9iTGbPcqL3eTbUtvGLM/+RrOlkWczxsnuai5whJODEhsOIusKvKy6YKZyJ8H0JVhCFgigQhIGgUnJMXgwJp1Jk5AYkmniUFF3lPrGs56IoHby9K7MuOPrS4Sda1Tp3bYB09N9ndDjYVu1wsSAhjIj4fkBp2oek4hV5pf8ZQPCB2QdYHw5iMDSoUSRHjTISyVzmIrm2BbJBH14hzsxUnZnpGvNzlvlZy+yU5eK45cIrjvp4D5uqNzHXM0vKpOio96CEFIlKWi4lZo2Lux/uyvQ+nlt9/MK1gqK6O/q3OmXvCk1g+yobpcVcMyACQU4sssI8F7vPspiZ4R25+/mh4rsxIsInToYOtnEDCVIsMo0XxZjNTlJJl0ioTrrDLWRWN6GWumGhE7nYQzy3kc7GEFl/A8u9y/j47Fq6GeGa1Iy5hGjUa66UXZUId2tfX/efLy2N2WuZX1XdHf0Vh/1goBuqv7AZ7Twc9qpBEUgC6iwwTUPXWOidwrdxfmrhl+mwvZc+z2Kago4NGCIqYYlUmCGXWKSUyFNOF4lSFpmIoZMJRMonanOUsmUKmRXa613cNPsDJKIkpnXxpJDYKJIlb9WYVNRnG/pULj9/7Fr8iVrJz893dfTfH/lhv2tgexoD0ommELtqSxFQp0IxkyfXvsCO8gHuLry7VTwQV1AKDJY0bVzgNG2NDvpKm0iGaTwTwwpD5DUIdYCTFt/E6Kr2sn15HzuXDuLZGIYmtdc0UIM6DVNz1baScLjuXG7hs5OTk1c95awB67B/qvH+bLpjnFQpQ5/biMbHYrg8K/jdw612mj42sZCcwhAyVNuHwsMKi3DN0kFzE7S2spbIhiSiFIP5vVgshggjo+ZFcQLtPFSrHh4RYnmtn2v6rVg9IVWgRaiDO4aG9m0YHz82/XqjU280WCsajdSDNrTnXcLKmc4JO8sFiiKHaOmIy3L9e1CsQhLEGijr0d/YQkiDY+4ZREvJNr9UEifJRc5Qp9oqMRgC0SBq9au09fCs36JwM28KWyUR8S0j7perAspp4TViRigRl07eckXN5XsfhxoZGZHT08/WHOaj0mpR7MzZ1dQKy26OGTFBhWKre6SRqDekkcYjUAE1VcW3MbKmkzo1FpjmCI9ToQRAQMArPMMpjlyRGgiEE5c+313xuEw18S3KWbbWBBCJVuIZ+g4B1rldzWHAxasCRI+OjhoYUefOjX5pcHDfZ5Tn/fxi33SoZ7d41GBWXCDmEiRJEyeFTwyF/pYFOhyGiIAGebWIlRHKKWIuQZlVDvA2GjT4Gp+njQ7qVKlSvmoBKBAta2oWqwqsoFCkaSeg0SSObU0jCtF3LZGyVXUftTCihHjxl2yY2iG1unNm4HzYt7DJy5SbX1YXVYRrlpdlK2NZA8S2mG9x1Klc8hAWi0aj8RhiP8vMMsEJ4iSvOk8SCAwRBQokSRMSMsckGo8BFCHBJYXcOmLXOpzfIuOoGx8fb4QRP2qNfUpp7c31T0aL3TMWCb6LXyonRiKgTo0aVerUCAlaIZAmYE4SiZCqLLUsyiMipI9NaDQCeVVgrFFonikWmGaG81gMSdKkyNCgTkTQrPnKsPnsXO37AWRtzkpOTr68Wq+v/Ig17n9r4elC17Kc2jge5bNL1iiDQqOdh0aj0C2LaTpfiUQHPspoQhmy7M0RI4kU8lJf59LkylVSxRDRoNb0Uy16rGMjnayjSglc0ypDP2j2mxFz3y8gl0CZnp6ujY+/9AFro/dJoy+4uNXLfbNyatNZO7/uYrTatmxq8YoNvYaLdOhCHbhGrOaKqVWb61qKnGjOGk8mzqDxWuEbQoKrtow169D4dNCDQNBGBynaqFJiiZlLkSpSIYEfCKzACnum2bPpddfgQ74NFAHI8fFjn9u8+cAjHvyCRP+889yOSmdJVlwRmgODTfkMOOlANnN+ZRS+jbnx5DHREDUC10DjUSR3KaJw1ZSxdNBLG52tyNIEpkAOJyzSedQTFWd0pISRDbAvtvyju9pRpzfocYyoQuHrtVxu4enOzvSfCWJjIhLzwrmgeT+TE044nHCBwOWxclwY/s44N+bhH8rrRTdY3yvSYZbz4iSzTLQU5jXn0a2w3FSni0xTo3JJDqx0zVuTiIQwHD43cfy/Xq0o+x6G7kZb/Y1hNTY21gD+oXWyZ88ePwj8rHNhXGvfGOOVx88dLq69c/vQje+WQm0Ya/+y/WDlY/JZ/r6lJ9V1uE1Ls8wsJVZbYEhqsQrVVMlp64lIRJ9rapBhOTY2Zq+XhVxRxp+0a6NVw8M9anJykqWlpSiXm6/m80vFlZX5Ui430wAYGro3lsuN2672Hh2TyR+Z9s6ZjcGg7A02sChm8Vpq9Vr8iEQiECwxwyrLTXCb00As9s3YKB4KZ9xKGPLhQmGxMTk56b5fp/pd2oSjZmxsLLrCz7SG8C4P3o2PPxo058DE/4xMOBkXCfVQ72es1Ya4S2Cv8kbJNSes0IQEzHKeAitIdLMq5zSrHUtUUkWrnScd/N7Fi8fyzZ711XvwN2m0u1nb3LZt732e9r9ac5VooL5FH5y5E2vdFcmZeF0IxKVSkyQiokiOPEsYDBLVbKc6TSlVYG5g0mi0MtaeFKJ60/j4ePiaROf6Uubqj5MORlQ+/41X29u7UgmZekdOL4a55ILqrawn5hItLNZguXL78lIUCQkosHLJXwgu5zoaj1J6lfn+KSuFFDgROuz9586dnroWZ/omAwJwkhYoX2vv6Nodk8n9Zb0aLqSnpagLoSLVkvvmkvSPCKlTo0yBPEvkWaRM8VIWvZYpA+Q6F1nsnbVSSCGR0lj7/omJY4+1Sofmn+qd3Wu8EIOD+z/rKe99gWtY56xrX+1S7as9+FG8OUfVejgsTjTDq3CyBcDlRLKWLJPrXKSaLEXa+RrnIof5wPj48b8cHh7WLR/3fY1cvxWHy+cX/ibb3i200HcpqWUtUTblTJHQawgcSCtRrtnk0q2UYA2EyAsoZwqs9My7XOeiifxQ+sSkdfa8c9FPnjt34qHrAcZbee//mqXYwcEDd0nJ70uhbratCitOGB15wgt9oSMtpFPNbFkZF+rAhV7gjDJCCim187DWRFbwqVql8duzs6dXYFjD9w/GW/tjCN/amFaDg/vfK4X4MHBISKncJcpcDg5XOlocWMuMwH3ZOPsnExPHjl05Ov7/369DvGb2/ZKi3b7/dufkvcLZtzvEENADLtl6TUEIcdHBUSndP0gZ+4fTp59fuQJcez1/KgPg/wJxl4w6tJfRXgAAAABJRU5ErkJggg==', 'fruits/coconut.png': 'iVBORw0KGgoAAAANSUhEUgAAAE8AAABACAYAAABbYipTAAAa3klEQVR42u2ceXBkx33fP93vvRkMgMF97IG9sNgTu+RSS4oUFRGUWaQkx7IVxSsriR3lkByrIrmiRHE5qaSsqjiWI8UqucquStmWI0VOXCLtyK6kdDl0BMmSbNHkkntggT1wLHaxu7gx98x7rzt//PrtDLAXeIpJ5VXNHjNvXnd/+3d8f0ePx/+/ADzAAhzewyf62llbWGMR0Mn7t7v0m3hB+gR4IyP4J2RxGlDudbvPXtE1Aj4QA9mjQ3wjFZA+O8O5kRF8wBwfpH14Dw8l476ZJUC5Sb+i6wR4Dkx1m9fNMW4CfwIPoLedY285wJXHHuAZAGtRyWeHB/nS0X0M3k7Y1JsFtBOgnxEJAODQXoaV5UEN+4AuC2ms7L5SRBZyKOawTBo43TfJ1ChE9wK3cQyAgT4+PLid3+1u51t/Msq7+vvpvX6dKpAbHuSrseU7geVrNU1qYpLTDkDzZgHPc2rDgV3sDjw+YuGdQKjgjLWcVB4XDCyYGis2QyGo0WwVQ9bysNL8DSzHlGLKwrM2ZsHT/GVkyJmYEMAL8JViIIqY62qnVCxwZK3EA20tvHP/Lv5mWwvfvbrA+8am+Ds97fyr4UH+auYaO+cWGOtq53+Vq/xzW+OJsSusOhtof9TgqRHwRiE6uI1uv4lfV5p3YRm1it87c5HRjV84fJiUF/IYMQ+gyFpLqDQFa4mcVB618NPAdxSsoWhGESiR2lYDbWGNnUFA+66tsHcAFlZY/d6L/Le2Vn5s304ODm6H1Tx87yX+Yt9OTpbKfPzyDZ66OMuf3bKANwIkQI+MyFh9o9hnROwtwPBe3qXhM1hGy5p/c/EiueSLDx2k2xh6I8VWE/Pj1vJBrRiIDXMWlqwlsJYsljbPI6sUFlixljSAsShrSKPEfmkFW3vgwG5Meyvm1HnU+DRq3070wT2QSRNNTKPPXOTG8UMslGvcd2aSX754mf8wNERbS4h+aYbc66m26gTo+RHU6CjxnVz9I4+QKS7wcxY+oxWf05rnYssRDIdiy4AxbMPSZyydWuMrBdaABaM12vcgeQU+rBWgFmGcd1Ceh0qnIJOGlgw224zt6YStPah8EfXCOBTKcPwg9HVj1vJwchy9nIe330dtcZXUi+d5JtPE543hoLZcsZZTYzPMJ6r7WoLnnQA2GuSHDrMljBnEsC+2HLDy2m0MvSh2RBE1a1GeRwCgtQCS8iEVCDC+j/E1JgjQ6QCdCjCpQO7zPFTgoeZXYCUHLRloSkE6hQ18SAeolgy0NIMxMHsdJq/KfQ8cBKXg6g04OQFNaXjsLcSTV/BOXSBqSvOfbMzJ2PLsuWlmXnO1dR7sphoeH6Q91LzDwpMm5tHYsN9C203ypgSgwId0CtOSQWeboTVD3JLBNqXB94ReWIuKDRiDsk5+ra2LslL1BSgFtRBKFQgjAb4jC80ZAXklB9NzUK7Czi2wox8qNZiZg9OXYPdWePgo9uwl1AvjfKu1mf9aq/Jn45e51kBTbKMm+a/WSyaSdmyIkVjxc+WI95iQbRYBqL0FOlqxHVni1mbINqNaMqiWJggCtFIQx1Ct4ZWrsqBqDaIYTMNUlQNdO8CiWO4tlGCtCKWy3NORhS3d0NUu0luswJUbsLQG3R1wcI/Mq1CCC7MwdRWOH4JDuzEvXUC9MM7vtaZ5NoxojlIUAe/oTnadvsz0ayF5CVE0gL5vHx+wlo9HEY8aK6Lf34nZsQXT14Vub0U1pVC+J18KI9n9UlkWVqlBFDmgnGQqVQfM0/J3bOR7+SLkimLjylUBtDMrjqC/G1qa5FnVGiyswI1lmdPOLZBthkpV5jA+DSt5ePR+6OvCnpuEF8Y5G3gsGihQ40NjV1g+tJf3mYgXJ2bWqa0dGqLNfwUqGgPcN8R7LfzbMOQhC/R0YPftIB7ox2tvRQe+gBwbmXCpIga6XBWwrHVql0hU4+44CYtikarVvLwKJVFN7UFHqwCypRvaWuU55SrkSzLW/LII7e5tIoVhCGt5eW9sUsZ46m2QDuDiZdSpC0TpgCPG8vVMiZ95/hqlw4N8CEvTxMw6qVMAmYit6uVGAAd2sTud4rNRxE8bA1t6iI8MoXb0oX1fFmytjFCpiaQkduimnXKvRjumtUiZMQLycg6W1wSMKBa71d4KfZ3Q3yPg+V7dzkWRPHM5J1LZ2yXAWgvFspgGgIkZ6G6Ho/tkTgvL8NfnMFGEroX8KR5f8BUtMeQ9+NjZSd6zaxcdGZ/B8UucdJoXH9nDk2qTPA3AHhniZ5Xl87WI7o4s8fGDqN3b0VrJRLQSlckVYHFNqIXSLrh0tiqxtonkeVqAq1TFLt1Yke9HsUhFeyv0dkJfl9gz3xcpKlegEgoogXvv2hKkUrB7CwQBFEtQixzRVM7OqLokFitwchy7WoAo5OSjx/j+9ByPzdzgP7am+fxyhf072hkqh3ymFvOR8SkuODzM8CAf9Tdh3wzg3TfEbxrDPw1juG+I+IGDeOmU2BbtdG5xFeZXBMgmx7Es9V1v9JK+J1K2WoAbSyIxUSzf2d4HPR3yam0WgBMJq9bqz/O0OIXlnEjr9j7oahNJK5TroFkrS84EMnaxLBt7fgaWcyhriR4+SndLho/NL/NbrWl+qxbzoe0dvFtp/sBU+ej4FOcTszU0RBuGnepewA0O0p7VfCWMeVcqIHrHA3h7tqOqtToQq3m4Mi+gtTULACgBBxpU2Y0WRQLack4koCUjqtTZJmAFgdxfq4kdq4XuWaouwYm0L+ehuUlUNI4FmGReiTnwPXEy5ao4FqWEtkxMy4Y+eFioy9e/x/VaRBbL05k0xWwLH1tY5Y9NxMdTPt2nJjkDqMODvB3LCf9uwB3cRndK841axIOdbYRPvJWgIyuT8LRIwdV5WMpBW4vsOriFNsQVnhYAo1i+W6nKgoYGxNinA7k9cp44UduNTqXxqoYC3rZe+X6+KOM23pcKZI6zN2S8wQG5Z2EZLl4WE/PQMOzZBv/7rzHVkC1acz4V8EBbK8eWVlmJyvxCkOFnbcSXEnqm4P0opvw7AGeHhmhLwTdqIQ9u6SV86mEC35dJK4R0Xl+SBWztFoBicwej6RYfG6ELfZ1iu4wRqUoMeqJeqsGxNF7W/RH4Qj+0EnByjrJoLc9Inn1lXuY50C8A5YuizmNTEMYC3K6t8Pw5WFxFexrja3Z0ZMms5qFa4xeamuiKLHtOX2YF4PAutgA/Q8hj/m28qnoGyFj+qBbz4NZewqceIUjIbBzD0qp40sSgG3Nn4Bqlz0vJvys1iMsCRmOEoO5iRKyVZ6SdSajVBABFHTTPk2csrYkdbUrD/fvF0SytykaNTcr4jxwVFZ6+Cpevy0YEPrq3k6blHFSqfPP8LE8P7+GLFv5gZAR/dpYuBV+18Bdjs1xaN92REfzRUaKje/lNY/jFtlbCn3iHSJy1ziOuirhrVbdryeTvdcVxHWS1SZKUqG46kHFqkaj3neza1QWZ0/Y+GOiTz1Zy8l6lKp/v3iabnivC6YtCiZqbYFsvZmEFCiVWmtLcV7TUgog/t5Z/p2HVKn5ewU/WfLafP8+ybiTAo6NEw4O838Iveh7RE28lCAIZuFyFBQecUi7IszLpak3U9154aO/eEtYIWmK3Mi5qaORzyTMCXzZk8ipMzQkow3th1xaZ4/KabJrnIpxtvZBtkWdduSEMoblJ1Hc1jylV0IHm509OMBeEfMBTdCrNIoqntOZvW/ix8+dZPNGwDA2wfz9dqZizUUTPOx+CoR3oWijiPr9yazznaQlxlBKHsRnpSxzBPWNAlzywVsZv9NY3VRQJv+ZXxPEM9EK2FZqco8gV6/fWQphbgKEdAvZKDl4YF9Ue3Aa5ItH8Mn4Mnx27xC8ND/JpT/ORasw/8TUHPc2vxoZ3n73ENxPK4jupU89AnI751TCmb/9Oon078Cs12fFFl3xulBhPw9yiLLC34942r9GubQY8hdg1Y9d/LwF1rSCe3vfhwC6hOYEvnxfLQpBpkNBLV8TbWitacm5KpHrvdqjUMEtr+LHhm57iy8OD/BXwh3HMDwPNr2HpjiIeHpvihyPgP+NqJcq5X3P/Xg6HhhfTKfRPPY7KpFDGwuIKlGvOxjmP52tx/0oJP4riTUiSqgfymwHQbpByCwSe2Ly5BTH6A45MGwPplEhYvii2rVGtx6YknBtwc52YFqZwcDeEMfb6IuSLzGqPP7GGR5Xlw56ibBSngG9HFf7e+BxLI+A3Fpn0CTe3GP51bPCP7MVkmwW4QsktuBE4TwYuVsR+bAY4T0uMGhvZbezLiAkbsiuLq0Jusy1w3z7JppQqIn1aS+CfAGcdpbkwK88a6BeJW3JR0MHdMvdcAQolrNZoaxiymn+gNB/AY8LCfzkzybvH51g+fpxgVJIiat0cjw+ys2SZaE6T/qnHJfsaxuLubwbvCQhFmJwTcW9tFkDUXaTH15IZCSPoaZf7XYJzc/kvLVK2vCaRx5Yu2cBSxaWjHDFfzYtjSIBLBTBzXdZw/JC8Z4xQld5OxzEjuLYItRCrFQULzyrFQxa+heWcsfzluSm+e6e5+QA1zQdtSNPgAFFLBr8WitTFcZ2xa0c1ZueFN2WbIdoEcIWyGO7tvQ00ZZPZCItsVhhDf5cY91oo0tWUhvasbMpavu5QrJV4d25BUu7373fjKZlHT6fMI4plbtWaCAWglCJr4R+fvcQ3jwzyKwr6j+xmxGg8aygqj6UgYvGlGfI4h6GM4W/5PnbPdlRCeIvlOnCJ1F1fFKrQ31lPO93NxiVeeqDv5adgYycZCRFP8oIWSa23tUpmJV9c71ACXzzwhVkJ/1ozLjZ2tCqK6pu4JkzBorDWMmUNS0rxuSN7OZzgbTUXsJxVmnPKcCH2mBkc5MLkJDn/2CGGykWO9XahurLohEyukzolqrO4KumcTPruUpdcc4uiIil//f2bkTyLZGaUSwAkDqC5SWxeoVRPvSf0JfAle3xuup5ssFY2MHmOMQLicg7CCOsIvvY8jmrF0TBkDvhjY5n1FZ8+dYn5O6qtCXkH0NTXKdWpJPWjGpyE55KMFrFbxt5bXa8vC/fLNteJ7WYjCq1F4sLY5QRdnrC5SZ6XK2xwZAlwq+JZM2nhc81NMH1NaE1H1kVDSp6bK2K1RjnCXzaGr0fwJTJ8Iy7xNq0YODXJ/Aj4fWDnQY3WC0BGwINHlILeTmyyO7WwvthksJWceLd7SV3iIDwtUhrFtwFO3QE0JU6h6qQ8KSDFVsZtbRbHUA3XA5cKxPCPT0uRZ2hA7r22KMmBXVvrNlRpKOSxcYzSimvWoi383bOT/Hkyl+FBPoLiC9QBi2+fejIMex60tUp5r1qre0LrwCuUBMBEDdRdSHA1FMC62533uwsNWReG+QLS9BxcvCLgt7eKxKUDUcPVnKthqPVpp5nrcOGytE8c3iMSt7giKfetPWL3Ekk1BlMoYYF5LO+2lvE44DnXXcXwHn4Cy2DPAN91U70jGfNjw05XWVexEdu2ccVrBZlAc9M9qIlj7+0td+dytpH4utj08g3hj51ZkZxUIM/yPZGitUI91Z9sqlZwcVY+O7Zf7kPJ/8emJPW1tUccSEJ78kVsHOMpxUfjGE9r2iYmyE8AR4f4BJbP2ZjHR0eJXBh2x8u30OlLdV5F0fqdVYiKlqrCr9Qm7FVi5O/Gg62pV8gWVsQuaQ37dghvi2IHlBaJyxXFbibAeVqk+tKcgHvsQF3yc0U4c0ls3OCAmJtyxcXJhrhYxotivn1umv9+eJDfxfKtAzvYlkrxRd/jyWrEx8emGb1dO9qt4FmafJfSKVdu9bKlivydbRF1VveIXTdzBb7YxckrstgdW4TOKOr2Vimxc8XKeofjaZnn/IrE1F3tbsO1vDd1TdR9aIfct7wmn2fSUCyjKpIFbzm0h3+k4EkUK37AJwEqVf7+uWm+3Bi/3hU86hxSVLYxLNIS1zY3yYJfjte8bTLTpYVmb4hd68zCWw5Cc9p51oZ0l+8iizheXwQvutrv9j6ZUzWUjc8XpS6ytbtubxfXxDO3ZQGFLVbQkaHkKQ76mi+4pIOy8Nux5TfOTXMZ1/a2mTX5CsLYkg7j9V420bswEnt3S6S+WdDc1wJf4tu5BZGmA7vEJiVkuDFHl5gLtUGaKy6x0NMhQCeqbIzYO2PrOcYbK+JgLG5zasTVGr5WfK6pwqerGb6CJTozyfsbzLDHPVR1fSCgyLleERsb1jHZxBM2pe4dUdwtKQCSwZ29ISp1ZFDIc7ihqpbUOpJEQKPUJkmFTErAavyep0X9K1XZ6IVVSUnlizKeUthCBa9aI/QDfp9thBYeRfNZwB4+TCqJQF9W34mCa5Ejjdj1UcDNwrS33gHYzSVGUI7mJOmroQHpJzFOatRtVPvSFXjpvCxc6/VZlYQsb9ycXEnG6WoXu1eqSB0j2yJZ6GqIKZRAKX5w6jxTlRX+mbVMn73E90+ANzZGLSG+Lw88zUQkqRqrblOL2Kg6Sok98vW9bVy+JDZpaw9s65FNCKP1UnOTT7ok5tyCEOFrSw0F6w2csDHjUiiJU+huF8BqoXzW1VYvha5IQV1pzW87Cftlz/IJXLHrFZ910B7PGSsDaHWrwco21xehXIy7nHPhmr2zA0lsTU+7a4dwoN1OZJWr9TY3SfYk2yL20BjXjqHqXQaNwBXLImkdWVfUdmFl4AvFQUqTplzBM4bZM5d4+vAgT1vL/zw9zXdOvAJVXTfv+/bxcC3k++2t6LcO30byGoBbK0gEolS9VWvXVqfWdwJlg5+5Ww0j2YgENBAVnJyT7+3sF9XEitOZX5ENSryw59UD/6QJcn6ZuFRBWcsnteKYgbfTxOHhMeLGpsxX1KDYvYUFHfPBao3uphSmsw0Vb4hHtZtcNZRQyfNkZ6uheMq2hAOqe4diZpO1DuMSBOdnb6oduaJQkWJFYt8kXq2FrsPKedrmjLRxlCqwnMMqRUFrjlnYGZV5eHyK/Nj6vqNXprZjY9SU5Wuehplr0tjse7famkqtPsEkDGtpur0Nu9Nl7Oazx0kt2LiNTDpCG01GFNc3LSkUdbU5rYhhcRXjaTxP047ldBRw//lrN8+UGV7lpREy/GUr/R96ak6kLCn73VS3aL0Yea4YY83mttByb6lLNiBXFGnyPSkwNUufMi0uwE9aagN/fTGpv1s+DyNYyWHjGK1g1Fred2aS905MkH+tgEtIoZ5fYa63kycU7FIKkyui0ykhnkmPXalSD3OUknj3+pJ4uaTTfKNXXid18eYAPj8DM9fqxeieTol3PS08r+Z681oyIp3JuNt6ZTNXC1CtYlbyKKWYSnfy8ItnGWtoB7a8RpceGZGHeopfsxYVxeK9pufkVXOBSne7a+nKSZZi9rqQ58629SFUAly4oSXibiqbhG6reaEbgS+O6fpyPY41RmxdqQytLa5xyHUCbO+t99AoYGkNoxRKwSeff57QkWDzWgIH4M3MYH4F9NMrXOjr5PEwYrCvk3jfTnSl6jqYjEtMtsrOR0bqoDv619OH5FpeE9UrlKRQg73HrB0VieJ6+ggkMZoQ6rWCSG971t0biWPY0iWqOr8iJcilNeJKFR/407OTfOoEeKMLm4tVX8nRppsx3fBu7kfzPMBbDqJ7O0USE+KZnLZJ+GBk1icLXL7s5mGSclUWv63n3jWPRPquL0kngOfKirVQJLI5I1496Qptb5VQbCUvVbDAh1IZc2MZUOS04ujpi1xtYEy8HicOAewJ8L69yrX+LgKleHxpjbilSZq0U0GdqJarEm7Nr8ismjN1g61ceXIlLxX8BaduPZ11h6Nu02+HC7+SeonviRSt5kR6O9oEqKT/uCMrz1lcrWdV4hiuLxErjWct//BMEnq9TsA1gseY65SyK4xWOnkyMuwqV4mb0mgT11tdL1yuZ3Tnl4X3NWdcsI6ABmL01/Kwd4csOGncbuyG91zcrBtydNeXBPy1vGxaUn2LXa0iFdSPJdCQQLi+RGQMvoHfOTvJr4+A/7VXET28HLVd1047vJcdGp6LDf19XcT93XhJK9laXhaUZG2jGPbv3FBadAkBpQTMG0vieFrdUaYk2RrFwslWcvKs2FW30mkJC9PBhkJ5Q59zo6mYXyYqlvGBHzR1MjL4POaZ18FBbOYEkAfER3bxNnyejWMyW3sxW3rQOVdH6HYBd7EiAGztEVphXBtFFNclIilTFiv1UzsJibWmzhkTqQocuAkZvxeZXlwlzhfxlGI6CnnUnRXTvI7qeovaNpqiEfCfW+NyTxfP+5oP5Ar41hJ3d6DL7kBKbER92lqdV02tb8JOFm9svRnI9wTUILh5lJPWZgE+FdRTUJvt31taJc4V8LRmKYanxqeZTLq+4I05SHzbK2mxHd7NU9rjjyJDtjNLNNCPX3WHR5Jiz8w18YTJEQLVGMu6lzsrezPUetkTbYgklILFVaJ8EV9rliN4avwSz2+maPOGgLehR/m4hWesZU8qINq5BS+dQpWrAsq1RfGOfZ23tpypV3kotfHsRvLvhRWiSg1fKa7EET95boaTG3vneIN+HOGO18wMZgT8H65wtaeNr3geh2LDweU1VLVGnArkxHWScMw214s8jV61Mb1+y6uBJK+73xWCwsh5d2mNtfPLxFGMrzQv1ELeM3GZcz8K4F7OkdGbhZEje/kXWD4FtHoepiOLbW3Guzovdqu/y/XfWVHtKK6fsbANmWmt3Lk0VeeQjfm82J23TWxhrkCUK+K7xOyXdZGPnrpB8Y1W1Vd63jb5cRdzZBcHlc+/t/B+K+0QNvAlodDfjco2RAJJDi82DWAm4Ji6g0ikM3AEOZ0CT2OLFcxaHh3HwosV/MvTk3xxw9k43uzg3fz5oERFhod4Sll+yVqe0HUPG/d0QEsG7WnUxqhiY8tF/TRf3SmEEaZQwhRK+M6GWq34zwY+dfYSsw0e1fIjvNSryQMmuz68l8c1fNhY3utp2ow7rRP4kEoRp3z53QDPQ7lsx02grMXGBhtG2GoNVa2hkwSrsYSe4qva4zdeusAPucOv8vzfBl6jLbwpAQcOsM2PeY+y/LgxPAxsb6zJbnQUtoELJjTEWEKtOK0V/wOfPzw9wcTtfgji/wXw7vizIEd30qkDjhnFQ9ZyzFiGsGyz0A5kHH4VFHkFV5ViQsMPLHz/9CVe2FDFtz9K23an6/8AKJJUoMhDcxwAAAAASUVORK5CYII=', 'fruits/green-apple.png': 'iVBORw0KGgoAAAANSUhEUgAAADIAAABACAYAAABY1SR7AAAYOUlEQVR42sWaeZDdV5XfP3f5vd9b+/WmbkmtXZYt2fIKxja2aVQGzLAMDJ6GZKZSMJDMTDIzRahsU0VA8RQQUhWSTM1kGWb4I2RzpicMSzAmxmBhY8nYsmxZasnae1W3env77/2We2/++L1uSZZkQDGZW6UqvX7dv9/9nvM953zPuRf+5pYYAQWIkREUDnHxqxv8TdvKD912T/e/eeA9fafu29PzdoCREdS1Hqb/pkAAbhQMwOgoBgFr1/a9de1W99HeNQsfGNqS27VlR4lXX2jy2qutn7lPfbUXvP5ne/cixsYQFy6kVhsYwI2OYq/yuz83iDJ076Znd3Rz16klL7i3tyv59K677Tt7+zPccW8fPb1Fps6G8cmjbW3C+BcG4q6CyT366DU2vBc5/DRy3z7MzwNqL8h/AW432Y0P7ur/0ea1me2HX1uu+r9K+V2f0qwbGGTsBZFEzYw8PhFIIZyS4lLKvQGQ4WH0vn0k972r/29rj884a6VJbA7nhFAk1oh2HNGII7kQBmIyDuWpQsEdjXXm6PFHZxb3ge08S3Lx/1ddT4MUkHx0qPy1t9/Rt71ZPxv30SzvvHWzHRzoc4uTGZUvtvWh/RWGNmcpljVx7EiECH8mkIGB1JJR20yUujJ3P/jeftZu9MgVLDMTAXEcsfEGS6MR0awnLJxPOPaiproYLmbv6T/YbvFX3Tuz//XA6FRwDWoCMAx6HyTvKnV/+t5b+94dtufiE0drXmXDOtd726BcnLQUuizjZyPmpkK0J5ifjWSjlqAzrg5w883X9roaG8MB4vxEe3rNev9jW3YUewXK9PQVyWZ816p7dvtNg66rXLQbthTs7rdJG0URj/xeWNh5l9uutfrg7GvxhzbckHt6+kww3/GMez2lvg7mdgq33H/rmr/cutGJk0cmVKtvUPgPFUX3gGTt+ixjL9VZuBCxY3eRfEG57l5PnB5rBjMn2v+qVkvq+/Zd21ASYHgYBZigbr4/fz6USjmOHKzI8dMt2WomyiRC1Ra0vjBR0pNHh/T8RI+0rSF35/3a/MMvNaNHfsftjhrir377z/AuCbDVYBsD4UDu3tz1tdtuKmbPT0wwU8+Lt/6zzXSvVTQrhrGX60yeDdh6Yx4APytdHDvCwMxOTQUXrozhqwBZoZczcnTidAsQsrvPY2Eu5NihOmeON/DzgkLJkckZWo2EuXNlce7QDjU7PZi58c4o7h2UN7/8WP/fAWzHMACMgBwF83Cp9w/uvqX3niicT86dS9S2j29l6HYPXyleeq7K3HSbtzzQTdAyaE/g55QLmsa1GuYoEHdqyBsDGR3FAOKmbcv7ly6EY8deqYvxUy0rhGD7rgJzMyHPPLHAqbEmxS6PXF6xvNDC8wXV8+uZOT6k3rIndPm8+9y6devy+/ZhO16Rf+mwN5Fbf+PW4qODa5w9eXROtbYMsOvXStg2NBoJZ443yeYUP923zJnjTcZPtpg6G7iJ0y0RRvYpgJXU/4ZAVug1Ooppt9xXlxdisePmot1yY57BDT7rNmTZtrPA+MmA7/y387QahgvnI3IFiRCGZiUvS10Fe/dDYku+HHx6xSsjIwghcLvWFD9/x87u8oXp83bBFQW3ZfE1vPhshaMv1XjrO7pZmo8odinKPR7dfZ4D9OljTed7fAugY5xrrlUKjI+nQT94g3+8XbOfWr8xW8jmlItDK+LYopTkhpuLFEqasZdrnDvRZM1an8GhLFJZXj1gxD17ut3xlxtv619b+LqvgmbhEyBG/W333NL3FxsHrXjp4Jzc8lvbRWaDYPK1Nod/WuWjf3eIvkGfrrKmWNZ092ryJWmPH264qfH2H774TOW7w8Po8fFUBfxMIB2v6Of3ha2eNRmL4OEtO/JGKimthfnZkCMHa8xMBGQyklxRc+Jwg8M/rbJ+c4656bbYvL1stCdzYy9Xeg4+H31zbBR3/5rez99/Z+/9FyZnzGsmp4b/yXr2P7HAi89WeO/IIM1awvJChOdLnBWcOR65H35nWZw51mr29qn/eO5k6+T4OHZkZDXD/mwg4+O4vXuRtcWNB2fOVz9S7NKDrYYxk2cCaRJH30CGtUM+azdm2Xlbibse6MFZ2P/UEtPn2uCcfMev9NsTh5u396/N9AwN5N2ubPGzOzap4vOvLEt7b7cw2jB2sM6tbysxPxsyNxUzNx3z6gt1Th+vofxF8e5HmiLfpfzxE/o3hzblbr59j3j2W/89rnc8Y6+ley5bIyOo0VHc2x/u22Fjt//Wt5bLd9zXJUCIJHYksV39S5M4tJYcebHGyaMNALbemEcpyfFjdbwleF9XP2u6Qr798hLVW/NEzZjh9/VjEs3ZEw2MbZMrJmzYJrnpdsct97Qp5xtY4MX9vn3sT4pybkKM9w6a33zyG8s/WVEib+gRgFtuQY6NYW/YlXtL0BC/WlmOi5UFh9ZSaC3wfIE1cGG6zfipgOlzAaVujwcf7mPXHSVe2l9h6myAUyTlRcXdm8tiebHCa21HssaiheDcqRb9m2a5a7jNHW/PsX5jGUUBYXsJ60MsV7oRuRY33hCIu/aEyakxr3f6lP6NXbdnDj/9ZPv41TwjXh8j+/aRPPDe3s8GNfWFBz/UZs/7A/fD72TEyVcymEShVIZm3RC1oXdNhmxOkCSWVjNBSghaCSZxZLs0a09r3rerh+NjEzzvZ/jQl5v8r3+b59d/v829w3l+MFritVcEhaJm+84SQ9sdmWxEq1pGKUPv1tMMDs7TbEnzlX/Upc4c0Um+x77vx99ZerLDHHMFkBUQ9z/c+7k40H/0gU82zCMfb0kLwgMWGxmOPnMjS7MKJ+t4hSpBLQvOI5fXZP0CubzP//nGAq++UKXUp8g14YaqIjAhpwcE+A6s4G3vkhx9PsOGLVnue6iL9Vt8/NIy7UaGqJVPdYATWKPo3XKGdetnWVhU9ku/U5a1RVkrdHP3U99cOLF3L/LRR1PPiEviwrzzAz0fa1S8xz7wyUbyt36rpeoG4RCYKEN9fh1+NqbQV6Wg2uRIEAgMECCIkFjrszDjM3aoQd+AJdsF2kmUJ4hCOH/a49BPNBOv5fj1Tw6yfbeiUdEo7ejfPEV1Pk99fg1SXWSNNYr+bScZWjvPCwcz5k//cVnpjHl5fXnpHsCs9EWCvUgexX3oYz0bJifVkTvfERc/8y+rBAaJBCEgCbMIFZHVFg1MTElOviqpLwu6+wU7dityeZ+wlcWZDErmiNsZrBFInRAFWbASL+OIYku7ocn4HkLXOD8uOXG4zZr1gpvv6MYYQRI5hLykSbKSgZuOsa63wn/616Xk2W/ndLZovvjM9xb/+YoT1MhAGtzdg4U/L3aJu/7+l2omX3QqAWTnYUIn+NJRXVZ89YsFHvvjAvPneilk11HMDSGidbQWB2ku9dGqlAnqHsYYgnqeOJRErQyokKChidsZlIIksZgoS7HLo9St+PHjFV74cZVtN+UpljQmcQgBKScEraUecv1LbNvdFj/5rm/jiPs23Zgf/f63g4W9e5FqbAz30CPlOxvL+o/f+ZG2e8eeUDUNqE4+cw6Ug2Zd8sXfLTFzqsjHfnsd7/pwH1t3Zunqkan1hEVpg5COYv8SXYPzxEEeE+ZQXkLXulmwGhP56eaEQAiHtVAqa+68r8zRgzWeeWKBm+8qkS8qjAGB6MSMx8SpHDtumRfNAHv8haznZ+3gxKlgdGCgY/Ogqn+v2IV44IOBjRwoCUnoYxOFA7ISHvvTHI2lLL//6Hq27yrSajoaNUMUWpxLredc2pe2a13UZ9eSRBmEMjgrqc2sJ2wWEPLyeiYEhG1HHDs++vc2oLTkm18/n+olsWJMgc4Y2tUCB55ay/CvBcovGGsS8ZEHfqX/xtFRjHz/b5R72gEf3LAjZmiTUZEA5yStSjdCWTICZpcUh36s+dg/KFEo+DQbCVI5pBRYo7Gxh4k7rYhw2NijXS+Bu5jdbaIv+3yZcpWQRJZcXvHwIwOcHGtw/JU6flZibTonskbQ1ec4+WIfrUpZ3PFg28ahUsK5jwPIVl3twcqBHbfFzu8YobnUSxzkkMKRAY69oFi3UbF9V4lW3aI1OKNwTpDvrtC3eZzuDVMpBTpgXm/51e+uJcOVoB0Ybrq1xNDmHC8fqHY8JkCAcw7tSfy85cgza9m43ZNCGpzhQ8PDaB3HYo/SuC23xNZ2Kn1tbhC/0MR1dP7UacnOO3I4k0VIg0k0ua4afZvHyZbqaeMB6EzEwtmtSJ1c0/pvtKx15PKK295WZv9TizTrhowvMSY1glICPwutusSYfunnlkhisVNk+2+TNuauTM6JgSErHBC3MwTV8irfYwStumDD1gJxCNZqugbnGLrlKLlSHQcYJzBOUOxfQHkxWHl9UzshiGPLDTcXcA5qlRipLqegVBKcResM2axnhEQl1j4gjWFTruAo9ThhgXatjE00NtEIIHQOL6Po7skRtSXF3iUGtp/GCYftWF0IhxMOqQxKJ9c1tVsJ/BWVXe71CJoGKcVl3ysF1oKXgWK3whmQgrulMaKcK1iyeSssEFTLCOGIQx+AKIJS2SPj+QiZ0Lf1XCeViCt475zAXQelVjbpHERRSq+BdT7WXiKiOmlMStExlKBQ0MI6B46bpHNO+zmHpx3GCaJWAaEMSZjBujSXl3s9nM2Q7ariZ9vpZl8HQnQyk000QrjriA84P9nm3Mkm9WrC2o0+Sok3nF/6uRVUYr3EobWXRnkSeSRhBqkMNvZIIh+lU5VrjUT71xj4deIpCX2sUb+4NyTEsaXZSIgjR6OW0Dfgpy2Ddddsojwv5Z2DsnROaKnSrJNuRHcqriJs5MhmoHeNxiTgzNVnySuvajeKOCt/YY84C74vWbPWp7vPo9zr0dWtyRdURwGszDDdRWDuMj2WlUKkbnVAEnsXaeMEYbOABKTIgDQEtS4So9LgdgI6MbEy/mss9COkva44cQ56+jyGNuXwMgIpYfFCxOxUG2tS5WAdWJOCcis1ptMgSuecMXFKNbsCpFPAola+A1AjpcVEGeZPb8dZiRYOKRyq829pciNho4hQ5rqDXcjUJGFgWV6IaQeWxQsR87MRQgiscSTJxRMhZ90KG4yWkjCOhLaAs+oi56UlCnIkl1he6YTGQj9RkKPUt4iXC3BW0lzqpbnUi1TmugqhEBBHjmY9ptilSZJUTGYyaVsdBgbnHMbQUcUpC4xxrpP9W1pImlFbFGKLc241tyGEw4Q+cZC7LL1KnRC38iw2iqsUpFNDrncZ45iZDGg3LZWlmA1bcuQLkmbdgICe7gxKCeLQpFVepOjjuOMdQU1L6SphIAfC4GL/sSr+Ek3YKF6ZoaRFKpPydPU4SPw/FcG47VBaEIUWYxzrN+eoVxOUFuQLqQoP2xZrXJqWgSi0rqPiFqVUYi4MBK06Tqo06C5d7XrpGsF5kXLXC2IlyDO+pHcgtXrvmgx+NrVoqawplvTqnloNc8lhmiNqGyckCJjUSrvJuC2oLwtXWGuusHzYLHQ86fhlLeegd8Cju89DqnSfUdvi0uOFNGMZR6tpVlOutdBq2XS47Ny4VNKdShKYn1HojLnMI0JaknY2bZCE/aUe8zrbkSk2BRa0LNoTOJcWzHZgabcuaq8kcZ3PIIUYk3gcswYuTCqhvOSK0xTnBGEgQfD/ZQkBUWiJY4v2JM6BlIJGNZ2XpSpYELYNYWClA6zgmPR8joJNps9o6Ug7v0uRSCmIQkMc2csmG78sikklqFUSvIy82OpaqC7Hq5+lhFbDuCSxAkdFCXFcZnuyp7XH9NyER9hOrPaurAVaSyqLMVKIK5LBm+0NEzsa1YRCUWGSNEO1moZm3SDVCtUE1eXY4UAId/KZ7y+cl//7q+dbnueOLF9QLM9bl8klq33GSteWzUtaTUO7bS9P0W+yN5QWVJZjEJDJyjRuJCwvRBfHQ52KXl2KrVQCkAdXT6xUlmeDhmLiBC5biHFGXPGCjC+pLEQo/Uv0ioPFuYhSl15VxVHbsjwfr3pDSmi3LdXlRMhU5v9oFYiv3NPGOE68IpXKhJf1G6KT6rq6PZYXYoy5aJk3E4DSgnotIWgaSj1eSistWLwQEYUpE5xL+/bqUuzagVHWuFBr+9wqEL1l6ZDSbuLMES2CVmiVvqSh6eTwQinlbH05Qak31ysrM6wLMyF+VuL7sjPvsizMRqveSL0kWJwLrRAgBD/d98TSFHuRcngY/cSfEPpZvrc4q93Umcj6OYuzl/NXa0GhpFicj9707KWUoNUwVBZiyr3eqhKemwpXvXFxmGe4MBs67Ukc4psAw08j5coZu8q5x+JIiFeeS6TyzBU9hXNpy1tbTgjfxKBfKXgXZkKcS2WJkFCvJCzOXYxJ50B7guWF2DVrRjnrQkX81ysnvnLljJ07Fp9Vyr124rAVlYXEau8ivdLmy1HsSnXP8nz85tCrw/lm3bA0H5EvKnIFRRI7ps8FpPn18j+ZHg862Yqnnnmyenbv3vQyz+oVjn2PkmTy4mv1ihVHXmraTFZiL9mps+D5kmKXZm4mTBsc8ebExtx0myR2lHs0flYyda5Nq2EuDh9WkkEl4cJMiPaEsII/B3j66TTOZcc1BiDr6/8sBNVX9ldVq2Hcily+9KXdfZpWwzB/PlzVQtdLKa0FlaWYymKM9gRr1vnMzYQsvO7ZrhNH46db1hgnbWJfixYXvwuIlYPRFaa74WH0k389d8Hz5V8szUfi0IGKyebVarOfCjpHoaTJ5iRTZwOC5sW0eD1VPEkcMxOpN7p7PZLEMXEyuKxWrdSxWiVm6mzgfF8Kh/h3Bw8SX3rnZTVkV+6PCPiKUNSfe3JRnh5ruowvL5s9+VlJrqAwCZw82iDjy4tg3C9WxafPBrRbNo2/subs8eZVD82VhFNjTWuMk0niJhtC/5eON8wVQAA7MoJ85vsL57Xmy0ns5L7vzZuZ8fYlHkldnPEl2bxkYTbk4E8qq8WrU2mv7aHOd14mTa3LizFhYDCJS4cNbZvOei/xhpcRzE6HTI8HNpuTQjj3ucNPzjVHRi6/F3bZNG1sDPbuRc5PDz0fuvZHGjUzaA3WzypRKCm0l8r5dstQXUoolj1OHG4Qtl3agmqBn1VcVlAvKaxSCZROQcyMt5ESTh5tsmadn6ZhIa5QwnFoOfiTikGgTcKPn3tq6TOvP5q+6oWBgQHkt7+9FG/enjvohPjk8kJkiyUtotAJPycpFBVB01JZjPEyAi8jOXuyRbGkWZyLaTYM1qbUUUogtUB0ZhpByzB1NkgreE5x6ECF7j6PgXXZK7JgOrAWHDpQcdXFGKVFK3Hi/dNnWktjI8C+y4l8BZCxsTTwn/lhMLnphnwLJ967MBclvWsyqlk36eg/sjSqaRNW7NJUFmIqSzFrN2SpVxKqyzHL8zHVpZjacsLyYsz8TMjcdETQNOTyikP7K0Sh5da3lolCe8kxWwrCy0iOHKy5yTOByRe0SkLziQM/XNo3MoIa+w9X3ke56qB2fBzbAfPsxm25zcbwlrnpdtw/4KuobalVEoRMR5nGOPoHfU6ONdBK0jeQwZh0Q3Fk0xY1sMRR2lt4GclLz1W4MBNy754+LnWDsyC1QCvB0Zdq7uyJZlIsaS9sm70HfrT874eH0Y8/fvXrTtecOI+P40ZGUE98J/jWpu25W41h9/S5IM4Vlezu9YQUIh21diRGb1+GIwdr5Iuaco+3qtVE5ygg40ucxR16rupmp9ri3j19FErpMG6FShlfErUtLx+o2skzLZcvah1H5sv7n1r6/LUu0/xMICvBD4iJ08Hopm25rVKKuybPBjRqiS0UtcwX0wQggEJJo5Tg4LPLxjlcoazws1JoJRECt7wQ2cM/rcnZ6bYY2pozG7fmpVQCz5Np3bCO6fHAvXygaipLscrmlEwi90+fe2rpj0ZGUI8//sY36H4ekbE6ar3/PX1/KBBfMMYpwA2s811PvyeswVWWI9eoJiqOHUmcdpW9/Rm8jHSNWiKqyzFRYBd1RkRSinVCYPvX+qLco0WSOGanQltdiqWfkzjHpDX87nM/WHj8ahnqeoGs3o9/9FHsfQ/13a21+KJzvNuYNO2unMqKdOz4fSXd/4gTPuUcD3beUvG0/K7nJZ9tx34gpfkz4fhwHNvOpF2gPYFUNHB8LQjDLxzc11j4eUH8IkAuu0EE8MB7+j4M4hPATnAtIXjRCf7ns08sPrXy++/4wMB9wrqyzKpjP/rG7Pilz7r/3f2fktL9gXPsAtEC963E2q8c+MHyq5de9Pl5N/d/AQDYuwARHxieAAAAAElFTkSuQmCC', 'fruits/green-grape.png': 'iVBORw0KGgoAAAANSUhEUgAAAEMAAABACAYAAABBXsrdAAAZmklEQVR42t2beXBd133fP+fce9/+ADwQC0GQICiABFdREkXKjGiRkilLGWls2RIk27FdtzOpM+6kbf7oZNpOh2U8TTrteJyOYjtW4jhyFUcRI8m1I1WNFpKOKMkUF1GkwA1cABL78gC8/S7n9I9zHwFSpMxNrqZ3BvMwD/fhnfM9v+X7+/5+V/Abvrq7sXbsIHj4YRJ5i99Wgk9JQTOaSuBx2g/Ym2rgnVefoVD9zIYHqSHgm6lauhNJa7kQIum6arpUUh94JV6SNj/d8w/0A2zbhty+HXU9axMf895FdzdydNR8T74Lsf8pvM2P8HA0xn+rqWdFOgO2DVpBEMBMFvJT9JaL/Lyi+EFEk4km+evbNqZXdq5MkK5XxOI+lZLg/GmLsyeKnD5aHJuZ0j9YsIz/uuO7lKqAf2LAuNIJbfkiT6RqeHbJCkjVoYRGR2IQS4LtgPKx8jMwPiTpP0mpUtTi8/9sXqyhRQZaF8WqOwrSRlNQQmeH09qrRPWZo1hH3s1z7FDpoB3jidf+lpPXA4j9MQMR3/QwD0QTrPZdAgGDjsOTHasgVUvgVrASKYglAA1exbzWN6FuWanVxJCIr7kzRTxdVCd7ipZlgxWF+nlw+oQWmaYZsagNtLC0FYn6kbhz++G93u4Hf4etO/6Gnmt1GfFxAXHX/dzVuoQfNy1iRSwO0TicOQrSgs7V6EoJEU1Aus64SPVSAaTqoLYeXnhK6k8/5OC6FSEtA5TvgQhXrTUk07DmUzA+Av3H8M8cxz52gLMNtWzY8WMmqv/2atYuPwYg9O2b6Wxbxisr7mRFfTN+Ywt+8yL8QBHUzUMrhRACojGzwYtOR0KlBHtfB8vSQuuKkHL2Pidi3Ml2zO/FAux9A8YHIVDYnavx2pbSPprlB4Datu3q139TwejpQQB6/iL+pHM1dYGPF41iO1HsUh7bK2NF4witjE3atrGUDy1KgmVBEOhLsULri39s29w/PWn+HgQ4izoIEike3drNpu3bUd3dWL9ZMDRixw6CleuZn2niISHRCOxozGQJIQ0AKjRYacH4MAydNZu5dMPROFTKUC6Gn/3w9124F8CyjfuoACIxdEML2vP4BkA1m10pVGzejM0Vvub6+MPjyO5urNpaVqVriQuJjiUQkThEopCuhZoM5LJm4SqY3cSJQzA1bqwhPF1SdeaeyZEwRlxiItK+JOLNAcd1kalahJTcDrB79xWzigD07t34gLZvVqzYtQuxezf+xs8wGQRQmIFK0fiy5xofR8NQP8xvC10hCjX1JihGouEpaxNQM43QvBAGzkBDC9iRORtWkMtBPAGWMxuApYRS0fw/21hKfA5kchtAGEN27ULu3o3f3b0yUp4Y+wMZuD+0bwaj3L6dAFBbH+eBwON3+46jbQdhO2ZRkZiFbdvGxx2fnn0Bq9abZc5koaU9tAgPnKgB7/h7MJ2FmUkYG4R58yESBs/JMeg7Dou7TDYSwgDhVqCYA8tBeS5SBZyZY1N6O8D22cSlNeKhLaM/bayPPDo1Q6+4GdR6azdrlOJPE2nuq2+C+qZw8dHQAmxBMuUQicQRIsY7rxU4dSzP0tVQ32zuiSXMz3A/HN0vSKYi3L4pwcj5Mof3lbjjHnAciMTM/UqZe6MxY2nFnIkvGrBtguOHkJOjfKkttfjn7oSbmilWogg7qj0/WvGCmNb2wojD79+2sub+riWJYOfbk0+LG+UT93VzvyV5vmUx6cYFBOk6SNVgKWV8XljQdRvUN4CWYEtJcSbF3jcCDrxZoFI27tC4ALLjhousXCfY8Bmbtk5Nfkbx9H9XFGZg1frZgGk74JZno54KTFB2InD6OAx8YKtUKXNCQEoIEkISty3hWFLYti1IxC2WtSdpbY74Y1nfPnBk+pC4ET7x0FdYUvF5r72LdPMifK2x07WXhGVhTlRaJkM0NIMVgfZlArcIb7wY54N3A7ITFbSGDfdBXSM0tsD0BORnzMbf+Uco5qHrdkikIAhQaAQYziItCHzo6wV3NE5nsglLWBdcyLYEjiMRAi2F0LaFQqCnZnzb87U4e740ZV0PGE1NyJ4eVGsn31vQzrqFnXhuGSeRBOcyRCrwTRwoFWD0POSnoW0ZpKOw/FYbpWp4f2+R1RugaSG4JWMllbKJB1rBwg4ol6D3sHmNxBDSQgSBsZDxIeg9AuXRKEsSDdjCwfMVnq8olRW5QsDklMd41hNjk64YGXflxJRnKYUolIJgasaPiOusdPX9X6VJVTi9ej2JRBqUQtTWm1Sp9RU+KMxPEJi40twK5ZLFL/5aks953L4JYmH8n0u5q5cThewo+vCvEDOTnI0lKGtNV7mIkBJqMpJEwmZqXOniuBRJL31UajmloQF0RAiiGuEIUBqdBwIpREvEkUnP1559HUFT7thBUC6wOpMhGUmgVICsJjClLg/CXPYohDnJkfMQjQUoHVBbb0DSVbQvA6hbgroGiCfwx4f5mhR8w4nRtWQ5qm0ZMpnWOBGIxVKcP6vU3p0TqlThq3te5PTXv744FgRlS2WjERHkg2TbklJLy37ds6++WQXiXoQeu/7Uqsg4EbQl0UKGfhnWDNUNI0D54Ptz3mM2AFq2yQ7JtKlHlG8q12hilpTNBdKOoIf6EKMDnI1E+HamiS1LVqCbFiLRhr6Xii6VsiduvbNBNzU1rPpf/3P85a1bWff0031VsSh8Dfk7kwPAMzdUwkvNtO8jNIjCFExNGLJTPVrLhngS5jWbNFvlAZdaie9C4wLJycMaITXFvPlsNGb+lVImZkgJM1Po3iMIpViUaaSz81ZUpgHp++Z7q8FSKc3pk2NyzboF3vot6a43fpH7t8B/WbcOZ/9+/MvQcdnUhBbXU4Mg4HP/glQxz/l4khoVoNMZEzNsB8DCtmKUi4KJkQqe79GxClqXmEBapc/VGmRiOMrbr1VoX2GyiFc2f4vFjehjWSaT9B6G0QFm4klqlq9DNTQjLfvDLum50LkGli631ZF368UzT471ndmnl/f24l5M3m9Q3Nn2nxHbQU1l+VptLdHmhaimhcj6JnMqWkPL4oC6+hJaJSjl5tF7JOCd16cZG3RZu9G4jZAQcQw9f++tCo1OmrrpOMPTU+QqAY4tKZd9PFeTrkW5Ljo7xk+kZOPCDmpStZeveKtWVzsP4o4vW2+p6IamSPtoa2UFvbxXpQU3DEaVet/zOe7N1PO9zjUQT6ITaeMi1dLcmLoi05AnkijyqdYEa+9O85PvFOnZX2L1BvRMFnHsJJw9Dp3JRvJ5RX0yRiVbS/vGHA0NinweclNw7iRyYhQXqE3V0lXfCEIjLWvW3S5yYcuQt2QCInGtaustSwgWA++FMsONy34rVxpEbYdvL+5CR2IoIbGqi6qSreF+GAxL87oGxcbP5qmvL/LY7zby9HddDu4JRHYMylmHW5sa8fI2I+ocwycmaGmDRXHI543JZxpNGm4dInJkL1/MNIblupwNzPacdK61caupMZicAIc40sohIPbryvmrByOk3xvuZ0mqhrsSKQh8ZCL14VudSDU/mkLszDFY0KI4dyavywUtCtMia+WS1sqW+pqEjrB/bIhl98TY+kiC6alJRs7rCy7ge+a1vhlshyCRMlTfss2Gx4dg6VqQYk5Fpk2WalwAk+cDSkWFFORCwqhvWNzpDs1LSjqStdjCQqMNFb6cpqCVeZUSjh2At3cJ3j9QEItTGb5wR0cyVUciEbd0/3Ch7NqVUteaOMtX51h1l75Q7l8gatKAohSWEw05R8W4Q13DxRFRCMN403XGckYHlTU94SscTs+17pundIkLB08QBkMhjeAiLmOEtgO+p2nvlNjNFfYcHYm01qXsXN4XxVIQiziWO2+BIlCenhoLTf4SkGXIZapp1ivDxJAJlJeyXq0MrZ8YEmp00NfTWf807ZwCxJWC5zWBUUU08Okv5lBKmc+6LgycNjVH/wlDni4HiOdBY7ND2a7g521dJ9IMjbgDUlt/FJ+pcY7+n7geHIgwPBDykUuyQyRmCrRi3gCvgZYl5n0VXMxfnIhZS/8Jofp7S6JU5Gf7n8LbvBnrSmn1msAI+w+ivZ5ThWl63DI4DkHgG4KUrgtZaGRWvpsr3CaTcPq4z/gxi875dfr0+DRTQe74QGpoXzmVP/j+P/ri+f9Uowffj+FEw5Xp2YA4OWII2tT4rLV4lQ8z26rLlAroyVElens8N5rizwG2bPnoloF1PWJOawcl4JFMA7pUQBZmzIk5EVNBhqIsTnTW5wfOwOE9giRxRirTohCfpnmpt6SxUXw5cFSb01ihlEOc3l1DfsymoaOCsGZdpFQ0YtHkiIkVNfUGDN81blLNMCJUdm0b/4N3safG+B+7nufZ7m6s73//o8EQ16tubfkir6druU9rAtvBssJ+qe+Zgst2oGG+aQYN9qFG+mVZIhKJTMDCTliwCBJpdGEGdb7XsqIpi+RgA3UdRQ6+btO0okj7bxVxC6EmEjEpOzsGQ33QscrEC7dycT/FskBaeKd7cE4f5ejCJazvSFEKY4W+ae3FkL0F93yeb9XUcWdzG6quHmlHQjOzjCX4nkmpI+dRfSeQbpkjsbiONi5UXe1d6Lp6hDIZQURiWItXBBSmAz44PEE0qLD03ghjx2PoYNYNfA8yTRBPmY33HoHWWwzgtmNqGM+FfAk13I8z3MdAPMkjz3yHwrZtyF8HxDVZRtUi7n6YTfOa+Keu20x/IvBNek2kQ0VazrqGVsbHjx4wJn3b3ZCs+TCNVgG0dkBmXornvhdQ21CiOBpl6cYK8bhJkSow4lBhJqxnRuD4QROromH7slxCF2cQ5RKvyijffPUZzlxLv/Was0kkxrcXd4ETw3crCKVMdVqYMacjhDlNFZ5qfTNsfADiKaHPnzaLnpsGRdhYalkMy25xmb/Qpq8XOpPNjH2Q5MQRmB43Wapa0WptXGXFnbA2BNh3CeJxR5SK/OHrO/jstQJx9W5SbSY/QHs6zd3JNNqvYEcipq3nlo2kH40bE67WKFUGKS3YcK8j3nk1YGYyIFk7q1dUr8GzELOjTE8G5LIC0SI5d0IwJSwG4wIn6rN0rdE8evYZDaS9ywBz290w1I/ODgviSemCEt3dONu3X6hSbx4Y3T2IHSAswcpUBkdaqCBAODFYEPY8mlrNppW62PmkNMSspk7T2BJl+FyRrnlQ8Wetwo7A4qXQ3xtjbCjLA48lOX6kD9mo+eqXMiRTUf7hmRnef7tIJGpY58p15v9qbSymrh5yWVcXc8QAPTp67dM78mqCZi6HDWgpTKUoQAsxq2xVU9vlLt8z4Ayc8ZFS4lYulgYDD25ZCZVilF+9Viae9knU5lm1TrPxfsHy9Vk6bx3jn/9hnHmNCewI3L7JxJELSpghZVJKRG6Cfb+uBrl2MDSi6nOvvEJl2zZsTxEpF0GFZy8uIVaXjhZoZXqm81uhUrJRynCPaMxkHss2/j49Dr/8BZw6mmf5HUYVVwEUcprR82ATEOgZPvtYBrdkcf7UrLyIBstGFXKIM8c4NXaWNwGxY8e1W4Z9xcwhCLaD/vRDbFCCb7zZw9bGFmvR9ITGLSnpRMzpGGXrYuqMNgEvXQef2gopR2DJGn76ZJaaebDnFSgXQmZaYxrSZ45VWL0BUjWz8qBlm4zRfxzalnu03JJnzfpaet6bZH6bcRM02vcJTvfgTI3x7/v6KF/vTJf9ES3DNs/jT+rm2V9ZujpOW0ecJSs0+9+aYu9rijvuCcXb2KxViJAcORETSxK1EHMEQ6NxDvzSxfMUXhlqWwSLl0aJRiNkxzVHD+SxHE1Dy8WTOdW2wsQoJGogs3aaxcsyfLDfZvicT7qWwPdgqB/nVA/fffMldlwvEB/iGZs3Y+/ejb/lUT5vWTy1bLXdtPLOqF65TgWRWCBLZU8satLiyT820X/prRekezzPWMmyteZ0bdvCLdZw/JDmlWfzVCo+q9bD/EWzJm5ZFvNb6wjcCM//1RiJGp/bN10sHFddzqvAhq0wPmDz6nM2flCmbRmc6YGhM/zxzp/xH0Mg1NUQrI+sTbq7sV5+mWDzI/xeMsUzt36K5LLblL90rWfNTPqy94NADJ1FVFzDHY4egP6TRpZTwWzWyI7YTAwmGDyd5tBbitdemGbefMW6zaZtqALjXkEAnqsZHymRaUjSsTzF268VyDQa15lbiYpQKnAipkcxfC7QQ2dxJ0d4u+8E/3LPy/zFtm3IX1d7XJVlVE3rvi/QHUvz3O13E2SaEEKa7DGTNWnTdswJHfmVKZxa2gwFnhqfVaQsS2I7gplsQDEPK9YZPmDZs6RsbnlpObDsVpt4tIkdP5yiWCqybrPpqF860RP2bdXkGPKt/83BnS9wx9z13+h4hV3NGJ99gkWB4i87VqMyTQjPNY0ZjYkLVbXp4C/NolbcAZkGU50GgQHJdcGyFDNZeO9NA8TCDgPkpaW2EOb+W1ZBR4dPrjRNa3uMA28X0WpW17xUtAl8U706UTIPPkg0nca/nsxxWTB27UICvuvyHxZ1UjOvEd93sasdsWr6FBr27TSpsXNNOKknTBqsjiMlQvbZs8/oj9X48FFXNG4Aj8crJGuiaA2FHAyegQVLwr+riwKcCAGKlUo4r7xC5WaNcMrdu/Hv/ypJ2+ELmUZ0oGb7pnOHzU6+b06lc415L540J15tJitluMPR/eZzi5eFUqC8fN/UTPGYxtDZPpgcS1IpiQtdsYtCoJ7j1AJdzIEOKOe7qNzMaUUJUJmkLRKjORpD+C6yuviq7jg2aLLHkhXGVWJxoytMjc0u3LJNbBk4YxhlVYP4KHZaHWaLp6CUizI96aND5TudgezIbNyogl7MoUt5CBQD+5/C4yrL86sGQ0UItDJ+FwRQyptTLxVMD/XkITNilKyZrT1msuaHsEq1bJNdEmlDolRgLGVsIJzYkx9uAwa+EX/StSZgjgy4uiZjKmDfg2gyrD+Y1UgqFXQhB57LHoDNu27e+KYEsGrpr5QZLJVQtoOq6gaTo8Zf6xpNoKzGhuF+4zqLu2bVcd81PYzGljDGhJ2t7JgB40pW0XoLZEcFw+cUY0MVkWlCVWk72mSq6cmwse2jfQ+Gz4Ht8JOr0TWvCYzNm7F3P01ZKf5y7DxSB2jbMZVgtST33Tnagx921ptDLhC6U7lo0myqNkyh0mykpt6AdpEhh1aRaTSU/dxJW589Uda5aVUaPY8MfJTtEGiN1hqtFdqJEAjwB05hF2f47s7n+SBsd940MKy+PjTbkJuSvDt8jo2VMkuStehYHGFZZuGlghkralxgTlNaxs91mHot22SA4X6TQWSodk1PhE8JhMBdCMzhaFI0BqMDkB211OFfubJc0Q8VcwyXC2yyHaRtI4RAqACRm0L2ncAaHeSvGr/Mv14F8mal1ItIV5WGP/h1bnULvGPZRBOpcGYqlNw8DzpWGh2yUjY9jGSN2VSVmO19w0zkVXsZ1RaCusKSQ87g9+zH7j3Mz//pF3we4J7P8QU7wjcjUdZaFqkgIO9WOOQH/PCXL/LiHFj1zQSjSrr83/4ay8p5ftaymNjiZVAqIurmGX+tazBB9dBbxiLiSVN1Br4JrNqBdAYViaLy01gNCYQKDGhO0dxzKSBaQyKFOnsSefY42WQN/4ptyM27kLt/zovAixsepMaOkPJd8ntfYaaqurH95lrEBTfZsgXR1Ekml2Nny2I61v4WOpFGNsw3WSGeDFNdrbGAYwcNP0jWmPPxXBMvpiYQ2TFkuYhomG8eo4DZ4RTLnnWfMPvo0fPonnexlOaJV59lX3cT8uWXCbq7sbq7ET/9EeVzx8kN9FLZtg3Z1ITs+f7HA8QFN7n3UZ6pb+R3Vt+Fl6zBqcppWoXEKrTHSNRwjhOHTKoLm0TaLSPyMxzU8GOh+Xfty1nU0o4feNhqTgPassN2giQYG0SePYaoFPn9N17gz6qu+qH1bUNwFT2PmwLGZ55giyXZ2bWWoGkRlvMR9FlrM+CqtZnfrpgehRg8w/FvfYXVjz9OcP/j3I3gpQXt1Da1EjgxtNCI6ox8pYQcH0IO9eEGAd96Ywc/ugIQv/FLBi7/pr4xfKTBv0zf8hJZ3/NMCT6vGeYvQvkeQlg89/jjBN3dxF99jj2B4J5zp9hz7CDW+V7ssUGs8UHz+4mDyPOn2KUUm97YwY+6u7E+CUAYpUvw6XSdAeaCsj0HgGr1WA2AVQE28NEnDyPHBsnVNfDnoe5oJLdneR/Y9JluHpyZ4kEEnUKjNRy3HF564+/ZeTNL75vmJvc9StB1G7Ku0bxRN292JElKM24QS5hRoiCYrRHGBvCOvYdTLvIHrz3Hn87d2FU0b8S2bYjt2z++YHhdYGz5IlNLV1Pb1Ip2XUQ8fKJwOmvSaWHaZI6WdpMZlHkQ1zv5Ps7UOC/tfIGHH3vssnLbhQd8q7J99fdPkjVcKgjvzU2xtWkhSgiscjFszORMxmhfYfhEYcZYRnEG78xxnKkJ9rUs4CtaX2CCH3q27pO66SsG0GiU70yOIbLjaCdiHlool0xNkkxDdtTonOUyamKEoPcITnaMXXYtD/zNk8yEjzlp/j+4rFNHONW2nPlukQ1OFJWoCWccJEIItLTRbgU9eg55/hSyVOQvOrfy5ef+yLT6d3/C/P5GH9y36IbJv+fPbIvfyzSZmGGG0oy7TE1AucAhabP91Wd58eOmxf/PwJhb9Gz9Ep9THt8C1iOpRTGpNfukzbMrm/m7J5+kcqO9iU/y9X8B7DQjAMSFftIAAAAASUVORK5CYII=', 'fruits/lemon.png': 'iVBORw0KGgoAAAANSUhEUgAAAGAAAABACAYAAADlNHIOAAAduElEQVR42t2ceZRcV33nP/e+pbbeW4slGcsSxjaywcaCYLPJtmyGZNhs0yQEE4eQmcw5A8M6mRySnD6a4ZCEgYEzyYFzSHICk0li02AH4rDY8iICxjARYNBiyZIlWbLU3VJvVd21vXfvnT9+t7qqq6tbLVkGz9Q5JXVXvaq+97d8f9/f8q7ihfVQQ0Po8XHUrl2k7W+++030uJSNieIyrbgCxWYFG4D1SjHoHAUgC2QAB1jAOJhTMIViCssJHE9bxb4A9sY1Dn55F9Otf2fbNsI1a3AjI5jnfcMvFKEDtG946I1chuVVgeaVKK4BrnCwVkHkLJgUUgPGgLXgnIjdtW9MgVagNYShPHUIzuG04iSw18L3rOGRSpYf338/5da1jYxgvUL//1HAMOhHt6FbLX3oFnoDzQ0OflUpbkLx0kARmhSqNUjqsuI4g+vuUrZvEHr7UP2D0NOLKnRBvoCKIhG2tVCtQrWKK81AcRo3cQY3cQamJ52qzKGdgzgD2az/jOOoc9xvDPeM7OR782sbIng+PEL9MgS/bwjV2MzQ9eR0D9uVYkgpbtGK9SaFcgXSBOIsZtUa5TZuQl36YvTGzbDhEtTgKujuBR20fvvKjLRaUUyMw/GjisMHcU/ucfboYdzkhAuCANXVBUqDNXzPWb5g+7lnZAQzjHjqDoG2/+cU0HBnEfytbNJwlwp4d6C5LK3D7BwohV1zkbKXX4neci3qpS9DbbgEcvnOQnaNp2v9RcmzdYvKoZRDBa7jpicnYe+P4Yf/ouwTu7FnThMU8k7lC5AafuwMn7j7Qe670N7wC1FA64Jvv4WXRJqPaM27A013qQipwa6/WLnrXo1+1WtQV1wFXT0LBd6IqMoCJkSlMSrJQD2GJAYTQRqCDcDqNiWIAlAOtMFpA0GKCxJcVEdnaqhCDYI6YJmYgO8/Ao98C3NgryKXJcgXHCblG2mNj47s4pDf03OODeoXZfVv20ZfJuYPgoD3B5rC9DREIek1r1T61regr30V5AtNoVtv1copVNKFqsRQKUA1LwJPoxYhO78T17KrJeTS6hnO/68cThmcTiBbRffOQf8cplbm+/+S8o2vYvc/gevuUkEQuumkzsdGHuavW+TnXnAKGB5G79ghWPnOm3m7jvhMHLF5agoCTfqaG1Xwtl9HXX7VQqEDKBOhamuhvB6qa6Ceh8oxcMZbsvWu4PmOUi1iWMmW3MLLGp9zCpzGWbDOEeTqsGoWF0zz6GNF/uYLxhSnCXq7oVLj8yM7+QBgh0Gfb1xQzyfkvOlNZHoMn44j3l+rwWyJdOv1KnjX+1BbXu737hxOeUuf64LyS6B6KSQFQIPyQk+OgJ0Fp0FFEK6V5aej4JILuBWvHO9dNtUoDWpNlVPjk/zJn467p4+kZtWgCssVdz+zvHPkcSqy2HNXwgVXwLZthLt2kQ69nk06y99lM9xw+gxm1WrUnf9O6VvfItdZJxvVaQgzg/KsxBBugHgVUPHL0/KB2mFwdRFQtAGCQXndTED9BKjwLEjgzm+7ShILk2iCQsBMscqOzxziwNFKsrpfReWq21kKefOrX01y6n6CA124NWtkIVtGcDua1OD5V0BD+HfcyPY4w98pzdqpCdLX3KjC3/swrL4IHA4H6CSGydUi+HoGtPXWHkB8Keh8I+xCMgbpOBCASyG6yHsAkI5BMnoWBTj5LOb8t6zAGofOBBRLCTv+x1Psf7qSrBlQUbnqvjayk3cshwjj46gbb8Q2YPmCK2Be+DdzZybD39TrhLUa5q7fU8Edd4oQjIPAKJhcA5NrIc2ANiL4eQtVIqywDwjBzgn0oFsErCAc8AqYXEbwSoQeroFwFSQnIZ0WJZ9n3LRWlFCaTfjknx/ip/vLZqCXwDq+7+DHwGFlecZaTjjHSQYZbaOsC+i4upDCf8ctfCib4bOlEq7QhfvIHym99QaBGwWombUwdY0EVTcqMOMUqBiitWDmwEz6gOjXrBpKcU04gpb3g7MgrBHICtdAcgKS8RXA1QqUkA2YnU749F8d4Cf7axaDzsSKIHAoPZ+blJXmWQX7neP7xvHQyAPsbo2T6kIJ//bt/OeuHJ+anMKsW6/0H/0p6pLN3uprORjdAPWrIeoRBmMrUDviIWUVxJvE2qsH2/DaChypCEyxg/O6swdVAtBZsOULBrfOgXVw+Kn9VFyWR36emKePzrm5EswVUdWK0mki5hPFjmzO25Xlu3XLJ+/dyXeGhgjUBbH8m/loPs+nJydIN71EBcP/HTW4xmEsBNOrYexiSY5yG0Flmylr7ZAoQIViobYMZqoFihALjjdD0APVfXL9eTmuXehBFyKHVTV+/sReLntRSO6SS7HaUA1PU6xVmJo2jI/jTh7HHdqPO7AXN3WGoK8PhYZ6jf/y1Yf5lHrOmH8Tv5PP89fTUyL8HZ9Rqm/QYhOFHt0IM6shMEAKQT9E6z19HIf0tA+OFpzn9TrrTazexH2VEw+wpRY4+mU+HI4QxWkO7j9GGMKlFweo8EpUpgDZOcjXIF+FbAV0mTPjFf7563Df32PiEBVF6MRwe/CchL+dN2cz3F0sYjZcooJPfM4Lv5JFj26D8kWgppp00lUERswU2GKLRWofe/sgs1ECrEvAVb2C6uBq/nvsC6SSrlDpcYrFOrNlxUWrrMQvXYCkG1fK4ia7seN9MLGKQjDIta9V9GTL+gc/wGUyYA1XBueTZH3zm5jbtnF1JsM3q1Xi3j7FJz6HXnWRwxZ70ccvA7cWQgNmtkVYPihiOsCB5/cq4xUStcCRbsKS7vIB2C6hBHv+nH/FiVoIpogyoxRnYbII6waQAJxOobRDBQEqsOjQoTDY0hyMTbIml/DDPajiHCrQDITnqvYtW3C/+Tr6Tcx91tKNwnz8kyq46GKLmRwgGN0ki0yPLEykFgRP1zkdsXMi4MbPC653EF8sXmJKUH+mw/c4UHlhTrb8PClBIin1E+LUVpEkjtkK9GXApRZVexb0SaHRaJwzaJ0yV4WT47B6AEYnIAxR5xKV1LZtBDt2YJMMX4pjLivNkH7wD3Rw+VUWc2YVwanNzarjvNW2C0h7K3dtTEVLTEiegeS4jw+6KXwViHs751lRuDAvwEI4CJnNErSjdRc4Vrgm7a0fFRaHIknl9ZkFjq6o1RzWJGBrKJWSJIojz8JcFcYmsHGEw3FMnwPuB57x/H4+z1vHx0iH3qPDN7zRYiYGCcY2gq5LsO24cW854UWQvUK4f6fr0imfXLV/1kjG66qS/bp6GxXVEAz4n1PxlKX+xjkL3hsASqhzMolSCoyjUoc4gpOnYWpSjN45x8ws1BIggiSFA0cdSsHhEzA6gcvEKAcjwbng/tBNbI0z3D0zg3v5dSr48LBTlHrRJzejtJNNB70+YHbA6PlfE4GRTsJRmsWO6THdViTQ2bm27/YeEBSEMaFlDfNZsjo/a0eJ4FUgsaz2NKRFgTjlqCfw7Gn5tVqH09Nw6SUS4rryEMXSZHrqGeldpwa+8iA2NWjrOBUm3BWutGB31zay1YgvGUOQyyv7n/7QqSDNY49vRqlU3D/eCLYKZsZ7QttmCKSsYGe9RQWdkynXSTFqiay3xQuS0xB5VpWe8a+3lh3cCrbaWt42YijJaU8IZB3OOVQgcJKkEGjIZ2HPYbjuSshloVaFiSKcnpBkP5+Bf/gOjE1i+rqIynU+/A+7OBOuwPr1yAhm6BY+kYu5enyM9CN/TLhuQ4h5cjOB0oL5piQWYsstHL5FaM5Dg0s9xWw8U89qTDMXoHW8QS0Uzrxlas+WGv/7v5M8638OfZzQbdeqzrEJ11ybrcg+bEkMaoETNZU4XZJtOQe5jMSBh38EV2wSy7cOwgByMXx9F/zkAMlAD1Glxhe+tpOvbNtGGK6krj90K9dGIR+bmMDc8AYV3PoWhz22kSDNQpj62nki+K08hQSwNanv2FkwFYEFlzSFu5JCuXuOXQ7t/3ENJTSYmW4TfsMIzEKnna9HuQXOaFNRQNCClvksHDkFV1wqSokjef9rD8MP95D0i/Dv3/Ja3j/UL7JdVgFbtkhXcMjyBWcJC3ll3vchpyiuhZkBiJOFLT4dixWnk/I0pWbRrF0wSi0rwDSVTURRmxJUG6q55bJVmJuF1FgKOSvftZKWiWrXvluAjiqE6SmoVGXGyFqx9t6CBNkklddxcPcD8NMDJAO9RLU6D+UThnbsaLp5uJz179iB2bud38rnuH5sVJn3vt8FGy7OY36+niBKvVUZwVmlROjJKJhyC2qoJcZG3JLCr9VhYlo2298D+ZzfuJJNz1Xk92xGgp1a4nuKJSjOyfvVmvDvIFjYukyNT6A4h/EWB2MTCw3KWOjugvFJqNQgDuF/fwv2HyEZ7CWq1vm2K3H7lx+n2trCXEoBamQEd+etFOqK/zY3h7v0xai3/Qa4YxvQKoDMBvl4/bhAS/W4KEC1W1DnzRi70H1bBZck8j4Ilcvn5CtrNZgqNfdd9Llad6FzTlarQ+DDRWrEq4LAK1M7yhX5vkIW+nqWZ89K+88FUCoJ/IT+NZR4QVcW6gmMnoEf7YV9h0lX9RNVa9x3eIpf372bpL1/rJfi/ICtGv5DNuaSuVll3/W7Tmf1RlxxAypMheCqrGB65YAX/tkF74DJGbGgyZkOV1kReCErga073/yqSr2FEPlxw2ptacHlszIYaozg8TycqaXJVvujOAvjUzBXbir+xOmF/X/lvysMZc1ffQj2HSEd7COs1Lj7Kzu5Y/du0k7N+04eoHbtwrz1rXTrCh8tFnFXXYt63U0h7ujl6HwX1CakFOAMVPaCrS8KVEvBS7Ui9C30NC6XgVxu4Ue1goG+xYig1TJw3UHTXXmxeGNEGVovnGAp5CAbN19vX6tJPdwBpTIUCjA1A1NFWX+rAp1nPHEEp6exvV2E1Tr3jOzkXSwzUaeXsH6XKXNXJsO6ek3Zd97ldFBej5udlukEFYrlV/evTPgtVQntiYi13or18glo69fmswJbjaAHIuTl8qxcFrq6OghZeQq5lBKdjD1mM82/7SwcO9W0+CVszGRjdD3hMfp5N8PzMu4Y/hd5wK5dmH+/lWha8f7ZIm7LtU698ldyuKcG0XHqVxxIMcqWm4mJXloHxZJYehRBJoaBHihXZVOZzAqppnfxVb1QrvkgHEMcs5g2ug6KbI9BKUzMQD2V7xno9R6mFk6n9HdDVw6iLDxzQjwiChcrQPtseK6GCwNIDY+PjGC2biXaDclytrmA+QBuup+b44grajXl3vpOtK6sxSaBL7T5tDwdIzWK01OO0QnByqUssX2x+Rys6pf/z4nnOwhCCbo9XW3CR5KfmdIKvlIL/NUSCaSVmo8lGup1UU5rnIgyUJyBZ8cXQ888NVVCP9PUxyiYA1RX1/LL0Uu40W8nCW7txc5uvTYHowPoyDZ3ljwLOIqzwjScE0ZSq7ckmi3K6O1u4/OOs0zL+FlQ1zlpdR0+b61kojMlYVHLln88XjfYmEK8q1yGM9NwZkYE2ShNpXU4dLwJnx0FqeXvV2oeYuEIK7OFBdTT/Obr6CfgjXOzSr18K0E+XI2tgFJ1sf50BlJpDTrvp62jmNbC6UlvUWcnRZ27t07ygLEJschFNT0lgmtVkA6gr1vyhkXJG22G4QT++rs9JPaKN1U9y0qNWHPj+sPHRbD1RF7vVMgINIxNQpoSWKGtPwO48cblUz/dWvMBqGV4QxQwYJwzr7gmUpT6cFEXhOvl8mR0/sPdBY+HPhhmMr4uaeV53gNQVrwp8XfAtLv7lKexYxNCDxtCLeQlGVIdvnOuDOMTAlONC7oKC6GwOy8spisnikHDsWelqJYaYT+lcmfdWgdHnsWGAcoYjlNkH4DPes+ugPFxWVagucUa5XoHcFs29sNcjI77pVNlir6SKZKII1jTD2sHoa9Xbqio1CSpOWd8bwu2g31imbnswiA7W5Fnw9umWyHHLU0np0ui0OnZDtf7ZxSJQnq7QEcwOi64H4cQeYoZh4sNIgzFY58ZxeayOKf4zsjjVLZt46wDSGEr+/ELvr5SdeolL1V6dfcAruRQ+qiUG9LTLbxNOJxSwrUrZZgsNetshZy4+FlrNkswlwb9a/9ckjRxWEtPhNR42On0nb4JFwSigDj0+O86K79R6zlzBo48K9jeUHhPV2f+H4ew52mYraB7CigFXwJozIiuRAEKcG/fzqBGvaSWOK66PK9QeawzBAS+2jnTcfbeGZiZ89zZj93PlYV6ZrNigUuVHUpzYsm9XUv0RNoemVgYTMNwtRIYXFDI86WMTNwMkKv6xDu18goLO7M1FcLEpARdpWRfiQ/IQQ36umQvjeWFgXjVT/ZjCjl0kvKDkZ08Bs3xw7NC0PCwiDUOuBjoDUJ4+WUDilSj5qnnpK9sqo6Y7WxT+K3MpFqVAtVUqfMCKjUpsLkVwlMhJ8oKAxHiQG+z8tioI52ehtNTotyG94WhrLHomc7MbGfhn5mEg8dEaQ1aqX3Zw1ofqFVzf5kYfvhz8f4oQCnYAbihoZW14UKAffsk3jtLXyCbsYGJ9YJ+7HyhzS2q1AaBbLBWl4U3eHEc+5vtjAim8XqrQFf1Lb5fYhFzacP2ni7oKXSGr0bgVr6Q190ogdQESrTPysu+DBL7CroKYew0PH2ic8ljvjHaJvxnRuHxPZieAkEt4VsjfuRwpfeQ6ZYA7KzjTi1ZnBs9XZc+Bj7xshUcwrPnyotbsv09TZcPAvk9DIRR9HXLs9MMhl6iHOGcT6yKHXi9x+p6vY1tOQna3XlZS29Xh15ChwRRBXD8pNDNhoIaDZVM3Ox6xaFkzc7JdamBb34f54T1VQx8sDG6s1LOEXptpbfdwr/NRPxutYq1huCZU5UWGjEh/DhVzJQcUehLxG2Jzep+X19vdP/8QrsL597dmiyKlSok/V/d3+T3xuca9QQG+wWWWu+n6OtZXIrIxCK8BtfPxpDJigIPH5OKZ6cSQ3e+KfQ4auYp+Sz803fh+BipbzP+/r07earRR1mxArZswb1nO4N1zV9WyrhNl8moxf5DZUgcWtVlph6IQsdgn1i4Up1rMPN4vILeS0cG5HG8WpcyQSNbLVehN2pems0Kj897tlT2TZrCEvRXIV5Zq/nP56FSgUPPSBbfSfiNR0PwzonCCnl47An40R7S/m6iSpX7Rh7iL7ZtIxwZWXzEwrIQtGMHtgyfzcSsm53F3Ple9FtvgYNPVyhOpyhVwrUE30LeU8SWxdZqzY2t2Mq11I+mZhZDg9b+hivX1GUYNAcWglCsvKsxBOeTpMmib+QsU6LO5iBbkGC75ymYLS8v/Ibg54Wfg58egG89hukqENYS9pYifnt4GD1P5c9FAUO3sC2OeM/EGez1NxK+4lfg+osVuZzjZ0+WIJhahLMNITsnjOK0f07MnOVmKL3Q6uOohe+zMKj3djdjQyEnkGeNxKDZ2ZYBCiXnPvR1S94RLFGVbbCcNIXDR+HgUd+VC1bWmGkIf+/T8I+PYnMxgbVMuDpv//a3KZ7vCEFw1Wb+XMEVmRz2D/8runsaJkbBKnjqaJXXXleBxC0uQmlx+1LZY74STI4CGUha1OetSVk60GLBDbiKlsgVo0i6YvmceJ3yHtMo+sWR/J1qTSArn5MzHzqNGCk/tTI1LRRzqtQ5Dzib8Pcdga8+hA0DtA6Yqxl+9d5H+NnQEMHnP39+t6lqpdhWrsIlGwjWpzB2EE5OwQ0vA02F8RMGnelsJY1K4oIbgmznCuZ0SZS1gH+7DiylxUMap5s0rgt97FFafi6Xpa05VfTlcNeB4YRiGIeOwZNHJLacDXI6Cf9nh2SqLdDoUFOvJdz2jw/xA4/7531sQQjktcaVplHH9sNMuZktvuJKOD4Gq9d2GPDzw0iz5SYV1LqtdtMi2FxWlLMIctpGURRND2nPNwq5phKiWBTa8MxavaWUoETw1sDYmNRz6kmTPKxE+I1rCjn44R745+9hMjGBVpTrCbfd9wgPNu6T4Dk8gpdu4teyGV40WSQ5NUGwYbUIKw6FgvV1S/rt7OJaeBBAJmoyhb6lSsG+hFDILQ7gDeHXE8ley1WhfUHQoYzs6/aN97T2A7CeLkZRc0BuYkrKCWP+nr+KT8SCoNkLWK4cHmjZ00M/ggceJy3kCJVisp7wlnsf4eELIXyA0Gk+lBoe7sqTf+Ig7sAxzNoB1Kpe1GAfat1qVH8PdA/68X290AviRltwOcrZkhMsFabmS9htFpqmgvuBluxXtSl1lW/eB7GMKE1Oywx+cdZ7StgsIaRGFJ2JloecTCwe9fVd8LODpH3dhEnKoXqd2+7bxZ4LJfx5Jj60nRsCzZ+heH2jBpKk87M5Zu0A+r1vR71mq0wYBvrcAlip7BOhzBKZqWvWhZTn+A3Mm5j28OJLEF15z4Bojnw6I2xsbKI5K9RatVR+2KuetExHLAE5+SycGIevP4o7eQY70ENQT3hoLuXd//QwYxdS+ABqwaEa/4brtOMNOK5H8XLn2BxFZObKkBj4kw/A1S+TCe0GL8fP3QRBZ8ufKUoAjiO4aHDhrK3xSm4kOosSMwXTRal+4hvk+Vyzi1GvieDHJ4WRNUrjnUZP1TKzQNYKfAUa/nUfPPg4xjiCfBbqdT77lZ18DLDPx6lZquVkkwWcZHgY/dT3uDRRvDYI+Gxq6I8C+Pj70Fe/zE+Z+z5skoqr93W3eUfLNFs2lvdpKY5NlwRvs7FkqR1nlq1MQUShlA5MKvByxlc8G5CyUj7fjvVaiWdOTMMDj+P2Po3tLhDgOJ2mfGDkIe4B1DCoC3lSVsfy1PAw+tFH0e0nBt6+nTvyGb46V8EoUO+7Hf1rN0O9CKOTErCNlUDY06EA1imAn5kWxWnf3x3oaY6ozN8cr5v9htKcJHrTJSlfF8vieUqJN/YUlp/3XZTZ+nqQMbD7SfjubsxclaCnAPWUb1Zr/Mev7+LohTqY6XwO61DDoPYNEY6MUB+6mQ/GGT6XpFCaJX3ddYS33wTrVoklzVWk8tnbfdaBBCm2zUhgDHwPYXU/hI0YYeS9Rt5QnJXfG9Nn1brQ39bScD7r44M7u+Az/lC/w8fh0d3Yo6egO48GJqzj4/c8wBdbb8eFX/KxlQ3su+Mmbo8i/iKKWDc1A3GMefXV6Ju2ol66Gbr65B4MmzYtueP9FVruhZgpNeErikSJpbJYeLnanEDQeuHQVKksENaK63Ek5eeOGO/7DbEX/IkxeOwJ7P4juDAkiCOwlr+tJ/zxvY9wbHgYzY4Lezjfcz6upqGE217PujjHsFb8jtZEPsCmr7gc/bqt6OuuhN7+BocUQRvTHJJNjXD3eiJCbAi6loi1p6lYcyHXOXA2GM3MXLNxYq1Yfz7bvLZh7Q0+bywcH4X/sxf75DGc80HWWh6vW4a/9iAPPJ/HU16Q84JaF3fHdl4RBXwYxVCoyRald2rXDOAuvwT14ovR69f4HqpviNcTEbCxzYxVa3lWak0mA/K5cImSgQLmGtNs3rIL2Sa8BVq8SmuJHYdPwBNPYY6elLe94Pellk+NPMj/8i3EwB+wZOEXeb/9czx+8vY3cmUEv6UUvxFqNqVGrDo12HwWu3YAdck69KXrUKv7xVLjqDlLY4zvflVEWMrHhP5uz+WXWbh1zSAc6CYDm6vKbaMHj2GfOo6dnCGMIgm61rLHOf7nqRp/u2sX1V+G1V+QA5uGh9H79jUPYH3zm8kX6twMvEMrbtaaFzVqNPUEtBaF9PfA6j70qj7UYC+qr1uqnbEvmhkrTZZsWwFwQS2qORVDasQTZmSO3x0/hXtmDDsxg3YOncvMQ9fD1vHF3gnu/eJuGZb9ZQr+gp2Y1fEI4m106YhXo9mu4A3A1VrT2zr2Zzw1jUNsPoMr5KErj+rKQjaLymUgjlANy27Q1STFVetQqeKKZZgt44qzMpOTpCitmyzHWQ5b+Iay/P3dO/nXNii1L4BjVy7oYQpLHsJ9262siRRXAddouMbBVTg2ORjUSuYsrPXznrbZgVqyfamaoyK6AT3y/rSC/Q4edfBgrsYPvuxh5hdxEPcL6dzQ1mPoTacN33kra+qWjSpgM5YXO81GDS9yjrUo+nH0OkVGOTIt814GRQVHGcUZHGNKcRTLkwr2W83P73mA47QdrdPpsLwXyuP/Avy2nXHVtQt7AAAAAElFTkSuQmCC', 'fruits/lime.png': 'iVBORw0KGgoAAAANSUhEUgAAAEkAAABACAYAAABWfFoUAAAZLUlEQVR42s2caZBc13Xff+fet3RPzwwwIAmABCjuK7iYQkTLdijQ4mJFli3R0oiKJVdRKUt2HCWpiuOUHZWDMEnZrsoXV6Wskisurx9U1IjaLCu2BFGaEuVKSpYtKeJwJwiAAIEBBrP0+t67Sz7c93qbHhAAyZS7qmfpfv3euf93zv/8z7n3tvD/5yHz86jlZWRxETP8xv0f4grx3Inndjy3ebge2AVc4qEmoMtDW0BT4JSHF1D8APiuz/j+N77ISnW+AweI7r0X9+ijuDfM+DcVmoOoA99CDQNzYJ7pVHgbwgPO8Q6EW7Rih9KlNR68B+/AQ/kDRJXGqoHR1oD3nAD+t8CXMs1XFz/DGYD5efTCAm5whn9kIB08iFpaQhYWsNXdre3iXg/zHt6tI/bqCJwFU0CR443BOQt4BEFEQMrfJWi+BM2LwusIohgdJ6CjAEVRcFrgMw4+fegxnh4Cy/7jAekgan4InHf/IlcZy4eAj6iI27SCPIOsiysMTgkyNY2anUN27IS5nbD9EpjeDmkdkhSUFkzhyTPotmBjRVg94znzKqyehuZaADiKoTaFTlIoMnoe/tQ7fvfQAkcB4SDCRYagvJGcU4Hz4Dz7ET6B8P44ZabIoNPCWYebnkFdfhXqmpvh6ptg99Ww/VKopxCFaOo//JCRw4ZaIAM2zsKrLwsvLcGzP4Djh73vtrG1OtHUDJiCs9byXw99lt9/PV4lb0RoVSR5/8PcIvBJgX8eJ6hOC3o9zMws6tpbUbfdDTfcIVx2BaQhcnDloH35dCUfOSc4K2WoSR82UaC0J4o9sUBCYPYecPwI/MOT8PffFn/iqLdpStSYhSLnb2zOr33j87x04ADRePJ4U0Gq7syt8yR74JMo/n2SMtVpQq+LveJq1N0/jbz1Hth1uaABg8dUoHgwuSLvabKOJu9q8kxhcsFahbfgvZRkFKwV8YiA1h4VeaLYEdcs9WnLzHZLI3K0C88PvgPf+jL+paexjRmiKOasLfjlry/whQsl9YsGqbojDzzEbST8cZzwtiKH1jp25170fe+DH78PZmqQQx8YnGC622muwsZ6Qd6JMUbwJVuUhA3ig3ETLPSl23mknw1DBgygTW83XLrboFTBP3zH8fXPY5dfQU/PgrH81qHH+L0LAUpeD0Dv/CDv08KfxTGz3RbGOvQ970be/YswNwPd0gIBil6D3tpuso1LsdksvXyNZvY8WikQh4gqAbBjzHRu62UMPO/BWcE5SGqeS3cX6DjniS/n7sm/wk/PovOcPzj0WT5RUoV/LaD0xQJ0/wf4qNZ8RivS5jp22yVEH/0N5MGfA0kDsQrQXo84dWSW7PRd5K3L8C5CxKIjIbdnAYuSlG31m0ijHeR2o2Spi3PyEIpBFjgH62cjOs2Em+6IpTbl5dnvW1Of4u1X3cyVf/IpvjQ/j15aOvc59UUB9H4+phP+SMA11/DX3Y7+V/8Zrrke2g6UQLcZceKlBmeOTdHreOJ4iiSuAwUiEdb16BVn8N6RRLPU411oleJ9QWE3EJlsmseV8Ml5ARZF4e/2mmbP1YnEKfqFH5miPs0/ecvN7PnKAl/e/3HiG6dRRx4BFjd7lb5QgO6b5+Eo5s8A21xD/dg9qF/5JDRmA/c4AydfbnDycIO8q1HaoZTH2DZa1dFSx2FoZ8ewroeIxnlDrGdQKqJXnC5fVxMhitQUguAv0NuUhiKHK6+L0Rr94lOmmGrwtmtuRj/553zjyBEci/iDB1H33ossDoElF5LF7n+It0vColJEzTW46x2oX/4N8Boc0D5b5+SRWXqdDB2FGkNJHFK7yxDRaFXH+wLrc0K+C0JASYKSGOs6W5jlEdHMTd2OdRnr3acvKiS9g9qU8N3Frv8/T3RdWkMrJX8hyCHv3Hf++jFeHJc2+nxU9NKn8AceYm8UcyiK2N5cw++7G/Xx3wSJwDo4c3SW9sk7qUd7UNpQmCYiitmpG6nFl5Lb1RAsPgccgsJ7F7IZGrA4nyNbkrYAjkhPAZCZ1YvmLFPAldfE0m5atXLK+aLwdzr8Q8DHbriD/ddczw//5A85ffAganERr19TSe9ELS3hb7idryQ19rXWsXuvR//L/wRpDTKjOPpsnfbKZcxM7cRjiVSDzK7ivSWN5tCqTlacKfkkgBPraWbr11LYZqmcBDkPz8jNGpk5O5Q3L7JEUIJ1BTfefLls396wznnf7djEe3+LTvjIW27kb//007x88CAqeo0wUwsL2Hd+gINJjXt6bczUDNEj/wFmGrDehePPzdBtKZKkhXUFWqUYH8pzBJrZYVRJ1OFO6n74aFUr38tCJXtBsu71FQtKBBUVmCzj3nddq3vdgjPLbf+/vvCCaW6YbXHMwoPz3PLoo6zq1+KhB+e5W2n+wjpc1kF/+N+J3L4P1nsRrz63m27bEcVgncGUfNItTmJctwTE4VxOGu+gFl9aHuNxvqBbLOO8KUn6ze3ajIccQJ71OHk855JL53DWs31HXXZdqfQzP1wvokhmvWftxad4ciuQZN8+ZN8+VAe+mCTsaa7if/Kfod77AVjtCiee30Vd9mE4g3UFIgrrM3KzEki5n508WiXM1K+nFu1AVExWnB3xqCAg/RbeJG8KgM458iJj+YRl+/ZZGtM1ej3L3I4azc5ZOX7YojTX7r+V/6nOFWZnhV9LU/Z3Wpide9APPQJdC8dfmKbTymnlT2Nt0SdbQSESl0S8uZp3OGI1VQLky89oZmrXUE924b2bUIIYvDdvRO9syJOEojD9OvDM6TWUEkQszqTccPN2pSMQJdeeVTw4yZPU0hL+vvdxiWgeB9JeB/Xwvxa58Xp48YUGrdWUKHYYu5lLPA78sFcovDc4b4hUnV6xTGFD5vPeksQ7aNTeQqymKWxzEz9NpVeQxpdiXeeCtdG5Hp1OG/CsnYFex7Bn706UxBS2h49Oc+xw7nptRIFRE0SjArzX/HqasqO1gdt3t8hP/JTn2MmUjeUUHZvAyyOqWPDeMhXvZrp2Fb5/5z0iisyssNZ5im5xqgxFD6KwroMrSd0PdZC8tyR6G41kD/V4J430qjfIixS9Xg9jLJQlTLuVcXL5ZVr5y6xsPE2n22J2O8oaBPgpNa6JFhex93+IK5TmE3mOj2P0z/6Sp5tpTh6eQsceLY0yxCaEgGhiPb3pfg+H5OA1wdgu693nWe89h7GdUaUtggesz4l1A5H4dYWdiJBlPbrdLiJV3yq899ILKzTbp0EMeReSFCmvdMUISAe+VY7c8KtpjZn2BvaudyA3XAuHn2vgnaOR7uWSxo+RxpeUhCsjiribn2Ct80yfcEXO/VRKY10XY1sopcpQC+cqzDqFWUdLQm43cL64qHCrwrfb7dJut/vZzbswmaBjaK0KGytCkoZJhzjp349kWCfJ4iL2XR9m1uR8rMjwaQ11/wfg9FpKey0mSg3Gdmjnx8jNajhbRcB9HhG89+XT9f8OWYwxT5DS4FJIihv8L4LHsdF7kVg3MK6DkgvPdM458jyj18uw1vYBEgFjwYaoQ2nPKy/B9Bx02yNSIYuGuEgvLmJMzsNpjd2rZ7Bveyd675XCj75bR8cOvKJXnMYX4U6rMjSccxRFgbUGay3OOZxzJTiDptj5axgpm2/ViFZRohFRZRaqnqr/9yAbhptjrRuyx4/oo5IOybNQkIsKkw7LJ+DIMzCzHawp521guQ/S4r04FgH4qHN4reGe98DKmRp5T5Okrk/WqrxaURRkWYYxRd+QccE2/vfw/+OO1X/d+7KB5vsH2rItXc3Jqej86Knf6RyTJKIg6wRO0ir8rzW01sPERNbFKUELLKmq4uVR3IMf5E6l+fH2Blx3G/qqGxWvHk6JYoMrNYyIYK2l1WrSbDbJ8xzvfd+YSUaNP1rr4Tksi7yHXhey7oSyrDov0GmGz+ZdUGr0mkptfm3LboCHdmtwHe+DN62tBLvaGwE4D99WAN8KhI11YZajKLB33xe6inlXMzt1LfV4F4inKAqazQ3yvDinIZPeEwlEWeTB1YtscEy3PXi2mzA+QyYCxoQKHqDXCeeSoUF2WuHcWwn3fvSqcJ5uK4BakXhaD9dvbUC7ifYeo+BvFMDiIvbgQZQI78l7MHcZ6pb9itXjO4hiiNUs9WQnRVHQajX7nnPOuxQuNMJH3oe2apyE7mU1tV3N5KrS7Y2BfMJglQ6vVb3s/gRAeY5eJ3ijnwCQM+HG4MN1Ok3Ie0Mg+WCXKDj6As6ExsSP3n4L31fz86Hz9bdL3KE1t3Sa+Fv2o9K0hmTXE0WK9e6zrLdfpNNunxcH5BlkvRA6vc7mwTZmYWYOorgMuQmAW7MZeK1hajr8TusB8AosHUG9EUJmQr8ueOhGsEtJmNj0Y+cXBXEMy6/gtAaBrzz6KC5aXg4meviZKEYhmDt+kmjttGKj9xw6sngK2u0Wzp07zofTayl3cFvMlw4Tr9bhLmblna0GPclDkxrE6cCjhh9T01Xanxxuvnwv60FzbXCt4VK61EnaGAzwWQC1c2cwVYR78gwu2Y3svUaxfjpGdA7isdZRFKbvmpOAcq505/JCaS0Akda3isnRwdemwmdEwu8kHQ2nkTm3CbLC+0Do4yFefb5enr/eCORc5INQGzvWKYU4y8k5eBYgWljA7n8PUwhv7bXhtrcjUZTQ6wpxGvDNsl6fGK0JF4qT0UF4F3glTsq7OlO6sJyfThIJnjDpM0UePOtcSaLXCR4y7lUVgErD9GwQj2dPhf8n2hVmGUCRZD1SIFcA2+pcrYTd1sCV16E6G3EQaYC1FmMKTB7SrjUhK1ShVw1IR5sNO5eQnKhfxj5TZcON1cAp5wz1oXOW65ZobUC3M8Q5AmeXA1cqtbVdpX5Leykx1VSpguuVRnSM3bFT0WlqkBwAYwzOhYUKVRq15QqHPCslvHDByrq1MfrZSbxWeUCSbhka/eOSekgEEMKqXPdEVt5YrYO9K6+ew4tK2ihFf97KKPogIdzggXoDX6tril7CTP1KlNTIiwzvgwFJbSC6qsxi8gsszCUYUvRCiPgxYu+2A6n2OgMgprcFzvKjZR9FPjhOqXDc7I5gZzWLO6yNlo/310eR9yabV+T4clXd2gN30YawJAiB3c6GcBHReFejVttBZs5ijOlfaGoaamXqtSYgPjV7/rxTEbbSgbOcK2vkIenQa4db12mVJF6ffG7vgtapQqveGGipPgeV10jSAPzqcrA964asm9RGvdJ5yHp4pcA7jj76KG4wWyLsrEDSktAxTVY7P6IoCvxY2q8Aaq0PBtmYGSXyraSBH/Th+llv+DPODinjErThgWw1eWKLzVmz0j2RDuL0xMuD19OpzdJMhIp3vdJ46/h+VY2o8lozzkFjRkk/AkVh7WSRk2cDgLwLITKsgMcvXmSDsBhfATL8qDhlUtE78pmyTpuaDiAm9cmzTL5U168eCcWsMaXS7o6WKRWg3TYUedUIK8v9kpPEgwSX1Xg/YEhjzGuGTlUSWBsMaG9sHrwxIYzOFZEV79UbAxBqU2Olzjq01kKJUZURaS3cgNZ6sGN44FEMK6dCRhM1qBWdCyE3bufGKl4ptCk4m2d8uyrZqq6ZVSpI9apodM71m1TjqbfSQtUCquoY56AoGFHm3gcem5mb7OKixo6dgtnt4fgqhPs6zJR1XRY+51xI8ZVG67VHa8TmWggzrSekeTcASakAWmsdG4d1il9d/BJrVcmmDswz7R37RWBl2Uqv49BasMbivQ/8szGQ+tUdmpoJhioFtUYwpN4Y1FbjmWi8zKiq+m4r/B4GSunNuknpALbSZViOnV/p4M3WluTcgWPPD5VBalArVmHYlxkKVs+AKVBlOP/hSAkVw8/riKsB11x3avW0JYqFogw1a8MdMvnoQNJauNszc+HvYfffKpxG9IgN4dNthxAyxYAnel1YOx1+D/eGag2YnYMoGQwuScuGvi2BiIK8OPJsAH+4yq/4K04GiUOpcJ3V09ikhjKGxUMLPHnw4GA1sRKR9xYG9uza6XfN7eTwC120FowxfU00uyMUleO11LCY24qMJ03dV17k3CBlZ91yQ0AZXkltENZFNgricAZL68Grk3roLhQFvPxsKFH0BNGYpOEzogZAnz4RMqSEtPVfAJaWBhYr4O3OgrFW3XTbDEefN+SZK5fnD048zEt5trUY27x6I3hL1hlooqryHxnwEIBpPQhDHQ26hO3m1teIE5iZDUAeXgpEXondSR5dPaMoFLtrZ7C1KXSR85dfX+CJ8fXeCs/lzoP2dbnjzsvoZXDiaE4UuU2hIhIG21o/t+Eyll6HtzYME2slAEUNwrTXCSm6WmErGmrTW/SJhvpMrSa8tFR6UPQa4rb0oKwHJ4/idQzG0BPHrwNy662jiVghKGdhZk7R5RSXvwVeeKrLpNV43g86hlVKNfnmpn/WHZICJUjjYrMKlZm5kM2SNJQZnVZ4Vo6c98oWx/QYWfuBtlpbgZefDrZofR7qv7T31SNQFNgkRXvDbx/6PM/Pzw9WuPW9/rp9fFzBjNXr6Ok1qdegteH7+zv8mLAr8lKPlOIhiodEYGlApzXghElKfATUoSxDqXorAHu9kP2sDR1DxjKgEjh1LKR5CZPHr1lHVtn55LFA1vUpoiLjW4cW+NX5pf7a7tEGoff8dVJD1k6LWz0ppDXYcRl0NiZnpbQ+EJAiYwCV46jVIUkmdxchcMdwiTLcw57eNuCjak7MFGHJITIYZJHBy8/AqVfK3rc6f4BWTsGZV3FJDW0Mpx18BIGFWyev6dZX3chTCI8oRf3saYyzqDgJfDE7t7kVWm6hQpW6KIo3A6nLjuT43Fi1XrG5Fgh5vMCUMiVX4q8CSEeQpuF1pUKheuz5kLrxgSeVHuircwG0fhaOH8brCC+Acbz3mwv8cH4evfSpybuY9MvPsHrdPp5Smnnvic68il05hV85idIRXLJrUHgOZ6Y4mdyXea1pJl+1ecs0X7VOOmVYRdGoZ1WaJooDqR9/Kcy0QqmJMjClAh/36hGAImiuw7EX8Eph44SoKPj4Nz/H4wcOEH31q1vvXtLz8+i/XOCZt9zEN7TizrTGXhFUtwVHnsV5h+y5dqAjRgY8AaCs7BPFyeRZ3GoxQrUwYXh2xZowmCo7VQM3BSy/AicOh3NXQFbcVIW9Uq8NkICNa0RZj9984nH+x/nsWtJLS/j5efRffY6jL97KH1/rWfKe7VpzTVJDnTiCm96G7Nw7mPjzZY1WzZON9JrbpU6JS7Id6lO7cnpa69E+UvV+JV6jOBxjC1g5CccPhxZu2Ma1OZlE8WTvrUJs7Sy88iIewSU1oizj4BOf43fOd1uXTNq3BnDfh7hLOT6tNHfnPezd96Gvvy2k9+b6IDQas6MK3BQBpHpjcJd73UGPut4YlDHDveeqsI7TAPTaSiDYXju8dl6pndFzag1nTsLJo7gydFWR81uHFvi9C9n3JufadX3fL3Ctjvk7YC7vYfbdTXTLW8vp5DJDTU2PDrraOTQsQNsb4fhqvr4xW6ZsNdiAbEw4bu1MOdffK6v9kuPSqZKYzyODaR149OQxWDkVdBCCM4ZfeeJz/NGFbgzccv5h/37i732P4v55/qnWfEUU2zpNisv2EF1/O1IRepKWBOwHKXrTjG4POu0AUJyEFF/1xzutkO3aG2WPpyTkrDfo/1Rhs6nPPcl7otDXOnEEui1MrU5kLcuF4Ze++Thfe8N3TlY1zE8/xJ1xwp/HCXd0WuA9dteV6BvugL3XBG+oes2VJw2vt/LlNHOvEwRgnofKP+uWHukGWqcyKOuOLn4QNShjtgota0N4nXkVV/axVFHwpMl45Jtf5MWLAei8ltVXQD3wAA0/x0Gl+Ddak3ZboCLsrr2oq29CLr18sMfMFMFLqmmdvCxf8rxcVVJmsygK7Y/x6eZqgUO3PQjdKAkhxxiXKR1Abq7B8gl8r42NUyIR8Jb/np3iPy4uYl7P1ne54M3I89wuit8W4f1Ko3ptcB5Tn0JNz6Ia28KUcqVZhjuA1aCybll+EEh5hNOGLLNFALTSVGqwaBdVdiZb64HgW+tYpdFpDYqCZzD8268/ztcAOXgQeT3fOCGva1u75hN4fiGKma28RgQTp0hjBtWYRWpT1f7+oXRfTgx4N2iCbTWJMNLmHSqyW2uwtoLvNHGiAjjW0sXz+5njdxYXaJXhZV/vSnm5mC3uANWdefBhrgQe9vBBgbdF8WCxlXdYHeGjBElrqNoU1OpI2UcOncXakGYaK+eqpX/Vwq9eB9pNfLuJMzleaaIkBWOweB7zwn97I79Z4nVv9Rn/qo3Ku7zi3R5+Rjx3RTFTVavWmP73jzgleJ1ApBGlkSgC0YgqvcWV66udxZsCbwy+BF0phURJIGpTsO7hcev5gyc+y9+/0d9R8sZ9w4RHDtwbVu4Ov/yuD7PXG/Y7z08Ab/Vws3h2q4i4v9DcDzhrsI29Pxc/ug5SVTu1WUP4OzxfsJ4vPbHA8T44t+J5A7/t5k35rpL5eXQpRDfxwM//C2Za61yjFDcDNwM3Alci7MIzBzQEUk8Qfnh6QAvhFJ4jIvxfL3xPwXe/9hjHhq956634R98EcKrH/wOCdUq2aaYUHwAAAABJRU5ErkJggg==', 'fruits/orange.png': 'iVBORw0KGgoAAAANSUhEUgAAAD0AAABACAYAAACp3n/2AAAWeUlEQVR42tWbeZBcV3XGf/e+19t09+waLdZma7NGloxxvCALj20MhrC5oAYRYjBVkEAWF0lIIJWkMhFJJalUhQBJSGICLmIcQpSUHSUYBxvLg8GOhGXLEh5tHlkzkmbTLD29d7/37s0f9/bMm1aPNHJIiryqVz39ut/r+91zzne+c+4dh5/So68Pmd+Cu+0a/vKa7VxTTXM0ew4foLcXZ2AA/Xqf7fy0gu7qQj71MP6aLSRb23ioPc29q9bjD53k8MAAwRKBi54e3PXrkR/9KPT3v/6J+j+1NsA77+fQ730FvfsB9Ns/xIFb3sotNYtfCvD/O0vXxjc0hEqn0Rt38O61m6nkZ1kXT3F/5womn3iMg4tYXAD6/vvXxZujyY9v3ZjaveWaROHUmeJZQPxfghaA6APZBXJb6OwF0d/AOkNDBozwGWpq5Rev30lqfBi/rRM3FuddrcsQ33mU/XXAZV8fopWWlnxGPfW+OyufePMbqjtPnIn0dqzs+Lvh4dny/yZo0QvOsh6cjw5BP2ZQ/aAH6s7aZwB9IOnBuXkIMQC6txfnwAEKK9Zx04ZtdMcS6LGziJXrCIC7WjuZCFmcnh6cr3+dYNvm5r3LlyXuevN1Fyo/c21Z738xIcfORf56dHI25/6kgfaAcweoPaD2QkA/9AP79+M++od0FHN0VnyalU8TgCspC5dsaxOTW7cy+YkH8ehH1R44MUGkFxjK8uToEO+/eiv6zHFEpYSzYi1BPsvnb7yT7+7dy2BvL9G9e6m+ZWf7n266OvmubZub/RdOKWcyO+0WS+KlQ8eHx/r6kOInQjggB0DshaB27ZNvoiuX46ZSkZ1emRuqVTapgOUI0lKAEIA2JlYahKDgRhiLRjkRbeJgxKE/Gefgg4coAlx3E2+69e08130TeuQ1RHYa0m34xSzu8Rf58u2P8cAeUHfd2vGJjesSf9tzS6ufLyJefKXknB8r5Fyp7nnsexPP9/biiJ8k2F+5iY5MgXcV8ry/XOQ2Ae3JJLQthxVroGs1tHVBqhkdjaGVgnIJMpPIC6MwNgwXzkFuBvwAIgnOJJLsEy5f/accw/f1MnLj7SROHUVXS4h0G0prxKmXOffth1l7986O93S1Rx+7e1e7CnyhDh4pRCam8pNSBO9+9Knp/+rrQ+7ZgxL/g3iVNbA//wa6qwV+oZjjg9pnRXsXbLkBrt+F2nYzatU6RLzDENklUommiL4wgj51BH1wP/Lws8jJEZBRlI7x3cQ97HrHfaROHIZXj8LajWgEYugkmalXUp/pSia/ePeu9oQjHO/AkXxkYrJwWhO8e9/T0wM9Pbj9/UbcXDHoXnBqYHfvYLMu8Zl8lvsiktjGHXDX+wluvgc61yLnnl8EXQJdAe3BXMQKmzRdEFGQcaBpflTZMdQPvo16/CHc00PQ0Qu33wuVEhx6BprbTP4Zfjnq65E29223t+NK6R18OR8Zny6+ECe4d+/T0+fDgK8YdA+4/eD3ribhpvlsbpZPy4DUdTvh/Z/Ef+NdOLgIcqDGQM+AyIEog6hipkrBgqwq7ekCUdAJUEmgBZwuoBU8H/35P0INuji33Q0XRuDEYUikYeZ0jOpQGz23tGmJCF44mncnZkqPi5T64L59U7neXpy9e+e5BvtTS3LnPhB7wP+5bnZ5Ff46e4Eda6+Fj3wW/9Z34aBwOQXBOZCzIKsWnLCgRN3f805tzipQBpHBMI2EQIDfBLEtCJpxVnaZSZudhiCA8y8lSOTaeMvOViolrX80kHOnsuWvrdx44ZMzM6hGgJdk6T6Qe6xD7t7K75Ry/KHQyPd9En/3b+DEYgj9MughkOWQy8oQqCs5JJTLUPGg5So4cgCOvAKDq0HcBGtXw8HvCYYPNrEm0cFtN7UyPlJVh45nmSmVH3zimelfDv9qI+BiKYBvXU1ifZKHChl2L1uN/rW/QG9/M5JjoAZAVqzPOHVAtY1Rz1oybGVxEY2htXmG0vDFb8C2TfCWm+FTfwy5NbD1fogGDvu/0swbr+5kw4YYLx2f4fTMDEG8oqLJYFgF6HKRHxdL/OMP9vEtQPf1IfcA7DHGE5cD3LuadpHg3woz7LrhdrxP/w1uaxoR9IOcBhGxYMJARejvuI3jAEjZa5NABfDt9cBed8zr8Sl47hg8/CR87F44OQjPjMO1vQ7esU42r2rDixV5YXicSHuF5VdBUxIiERDSEN3YMEyMcCCX5XMHnuDxsNXFpQDfu4qOaIon89PccHcv3q99iYgYBfU8SAVEGpCSa61ai1/PAiwZUkIDwxaoCDmGAxNZ+Mf/gnPT5rNsBU6MQCoGyTbIrEqwobUL1Zmh3DrL5u3Q3i5AaKQDQqJLeXS5iI4mIDuJc/IojLzGo16F33ruCQYXEyeiD8Rzy0m0tfB0boqb3/0R/F/6PK4+BrwUsq6qs2oT0AzMAlkgZ9IVngUZsd+vv89O2IU8ZMqwqg2aHCh78NIwPPwcnBmHyGYHdTWs3xrQ2QXZacjNgnRMaGgFpQKs2wydKyHditIahk4gX/oB01OjfPjAUzzuNNLOX4fgxqv418I0d92zG/9Xv4SrfgziMIh4rXCbd8f5GsqCHAEu2IlJ2dO1FteLBJU2g0/EDIkJDU0RWLMc7rwWhi/AibOalbdq7rpXIAVE4zD8KqxYC+1d4LgQi0MuAwMvwPBJRBAglq3EX7OBZD7L7vZODjsN8/AW/ric5WM37ML77a8S0YMgXrKAaykmAaStFWvAJ4HzFngKaAvl4XgoNYnFi3tfQdSBhE2myoNYBG7bAsdPwysDUE3D9p2w9hpYtR5mJmD9tdC5ApZdBVddbV5zGehYji7mkUDQtgwxPcG9ol5pfWAL91SLPNHehf+Fx3HSGqH3g3AbjNC1bpu1gL1QqmozYmOBJ/jA1BJTl5r3HqVARmFsGj71CMw0QdttsOE22PoGcLRRep0rwPeMmzsRk/q8KqRbIJuBUy/Ds98mU4MiukH3XkOLX+UrSqE//Xlkuh2h/gOkDLl0LQ59O7Bp67YJ5mssvUh+1pdw77pcTdKIFXzz+0EVVnTBB26FL3wHPlyG6CD88HmothrPirdAxIZfFKhWUIUM5KaZKRZ4rZzjgFfmIddaWe6B4AMunytPs+aDD+B3344bPAtOqc5iNVf1bOxmLCu7dYAKQKxOpBSWYGFtCa/NPjtrniml0e7v3AHfPACZLDzwRtidgYkxmJiCmSmoaMhX0H/fD5UyxXHNzx59jVeseYwMtW6tfr6b7uwsv7RmI8EHPoWjzoI8Xwe4NqC0jd2MfZ+rExzCxu6U9QBhrVZZgpXD94biX2CkZzwF79gGB4YgyJnJ6EpBV1gvxBH5Ufxv/YjU9ji7jsKzfT24zwB33IGqubeuePyBqhK579fxYy2I4Idc3GIQ1sJn7Ly5i7irDsVwNvReXBy2ZStM4k7o92peIUALKPmhzz14y1Z4+hRUc5AQoAp1VKDgPTfifPc4zJb5rQ9fy4N7+o2l+/vRzgDo3dvYUJzlrzZeh/z455CcQcjTi4iPcXuqOpIKx2wtHwd1xUbdkfegosDXEGiIOfVtxFB2lLbboqClCTqTsDYBjm+u104pQAeQSCMKBfwjZ0m6DmdfmeZgD7hDoCSACvgQHpF7PkggYwh1MhSL4fJvxlouCiwDVtY1kSWwHFgNrAE6Lx26vja3OxjQSjfQ40BULuRRAfRcB44zX5ov4EHrEW/bhkjF0eUqH+8D2W+pVvb1IStF3tu6DG59O5IRUxrOuW4NTMnGmbCftVkCS4Ss3mGv1Sap9l41Dt2onJfeEVkXTtowslgOQcK8FyFDVEqQqRpvuajpLUD50NWBs2M1VAOuH9zETYDuBUee3seWSontm6+HttVIfaqBJ3pWYYXjesrGdSmks5tCaQv7d3Jx8kq6kLJn0q0D7IJOwavn4flhODgMmfy8B0pxaU7U2vBBzyYCR0BF8T6AiR6E9CrsEoro9p0ofFATDaycbcC8Uza2g1AO13UDX0I9HZPmrDcyAsbGYXAM/ArMFuDYqGFwrGe0RCEdaQxeShAeXLca2doEFZ97NIj+fgLpVbglGoFNN6AZA1Gpi2c763QsEuciRJvZ0DXHvs7WTcBSmwoa8hNGJ8gyxFyo+OB587/pXM7aAbSmEes6oOrT/bGtrAW09D26m5phxWoE5xbGzVxpqKwgSTSOz7nvZqz1y1Z/jwH5UP5M2lMvDXRH0vy+50HJg5YExGKN79eha+UAZqtQ8AEXsXk5gYBIRXE9gOsHrE61QTqOYLIu91ZtzLoh6SkuIyxqZWXNsuFJjC9NmdVSU3sarl8LoxmIR2B953wkXSQhXHOP1lAKzHfKAaQUbOpCOxI8xQ5gn6t82ppaIOYhKISW9KQFkLcVk2ctGM7JDtBur+dChNaoGSVsygtd9xRUlREe9ZW90qACWNZszrk6XF8MWGkYmYS2JkjGTVao2GoNDV1pEx6+xxYAqRWJaMIMWgchIeFZwIF12+LidfCiLr9YwVEzuG+sUfAv/mopgFnPANcKtD9/b1WZe5Q1uRfAwAhMFcykNznQEoG09dBkFBF1wDcKAheBlDW3FCGXLFp3dhaJwdrMT76eDvo8EQWahv2buANuOHeLeasWfHMfQFIYK96+GaK21BXCDttOstSIqAsFj3YBuELie0UiZOtILL/EFu4SwPq20xmRC29LRUzTIHw9PCGNJkMIcIVNKqHP49HFy1nfh4jpsjb9fjdRVzgUKhlaVQ4tHcQcgZXr4rPuCLTNTou5vFjIpkpfDE7QGPDl5jgVMc9zRANd0ECZVapGu2uNm23GkY7DVD4PxaIBqQmlqUXw5HzIepCxBcOCD+XFIZF0jYj4SW5puNx6q7ZkXChB1TeeJgTKb0VJx2G4WIZMHj0nKKJWMzc4yoEhktpR9EOFgmvZvGNhhSaWEAUVS1wF/wr0i178uwJDgNmC8UpPgRCUv/QdqtKRHCtXYPSCAa2DULNPN3Zr0Yi8dUiJOZcOjUbPLATmtaxsjb0Ecwt3nodq2SBfIzkJ2TxUPfAUuuKBFMwIgZZRlxd8BafOInBAV4GJ+TZN/RGVC8G6NcKppbmMzceX6HpqvVBBNeJErS8N2PPhxHkYmzHvS4GZsEpgQi7wYSprCCxbQld8cAQjADIiOYjAO34GiQ/St4Nf5IhKE6MRAXFpKiQRHnXZnpezkLuQqRPOfLkZd+znYhG5Kc1qyPExODFuuqVOqOCWAiZnDGu7DozljXtLwSkAWbqO49EIJ0+fg8wMStSsIC5dGaUjBnzDlpJcHHDFM5XT0IR1Q3t/wpmvmoIAXjkLBasAdQOJ2pGC9R1wdaepqGLWGKkolIuQK9ouLjA8g7BsfwRA7t1LEI3wvdkc+shJFMKA9tVCwrqSoxKY2KrvhCgNR87B4ISx0qmxhaBEbaVDwooWKzYazH+gTF6+fh2s7ZzvsMQjoKowlTHPkMIsDb02jaMhiMChmsLGcXhUg/j+i0jKZjYri8jDpRzVwDTzlF5YH1eqkCubAcdcmC4aq4kG7Z6ONETci8nU16Zj4tfkqbIcate1RyfnhWXEgbEsajwLEcnxDYOctl9FtKT4oRtl8Mgp5Pg5lIhad4ssXVLXd0Rao0ZGhi0Yj0J7EspVkzu70iFtUP9M1Th7OAKaI7ZRWPMQCZWKAVwLTa0N6KNjqIpRZI/vAdXTgyN7enAePIQXd/mHQhn2n0ARMTctdZdZvSqTorHSEgK2XwXdq+D6NbCx62JL15hdLFLQ1FTcXIkpoVSCkQlDaLWxSAGFKvxoGEcINPAtgK5+tLzD7s5z4WuOpPDUMZxiDi3dy6SN0CCLlaVXWY6E1Z2wfBHxI5xQeVvr2qQW+a6EXN5Y2G7Am+OORASOjBCMmrR1YO8gh7DbwMxuA3AeeZVzsQgPj88ivvsygXAurYxqWyWGp2H/cZjMm/dLmijfiqAGYTI8CWcnTVMPbEt5jV33Vgt74tMZGJ++2NukMK2lp06AKxFS8ud2VVbOEVm39ZREjD8RkuJ/HkOOjaLl5fSjhnQcVrZCU3SeVJYSDuFB6lA6OzZqmL3qhbo31ZB2kEaYjFyA6dmLQzBQkIzCD04TDM0gHcnRiVd5zPa9/bn93v22H/zNC2S2d5LIluhJxwhWNyMjEbt4Vt8IsUGViMHKNvM+VzVqSF9h9VR7ruOYNLUsBW0pe71o1aE3Ly3Hp6BqK6f6lBiPwGgOHjqAciWOFHz08RlOdpktnXrBJvcBu9ckt4YDpTK7Jwt07liJqlYRibhZ5Ueb3J3zFta7yl6reZ+nGzTvw5WHWDjQYmCLMwEtKbNsI+p2OFSqcGEaMrYtVf9srefKR/7mh/gzJVwheORfTvNn4V2O1Dd7B0B84wiFlgQfH5mFZ15FuQI9PAqFUOlZvwQjxKW1c006zhTg2ZPzSqtWbBT9+RJ1Lt7FvCtfmIbzE6ZMlLJxj0wK0xN76CDBa1O4rmRQwq/0GfJSi/47wwDoHnD/fYrT3R045zPcuaIZf20bzmTWaNlUHJKx+RxMqMvkWaARAfE6TS6Eub9QNfnZdRamt5hdoBNmlxBVz8Ts5AyUKhfzQLhCizrGyl87QHBwGCcZJVfR3PPoawzfYf5rQF+u2SN6wPk++O+9mn9LSN7z9m6Ct27BKVsXbm+GdBJcN9Tot91NHcqjDYNXmj6WrqWYkLurwADMFaBYttsu6hbvFqRv0/QjWzaAfzyKk45RKvu8c98Q++vd+nIdLgGI+3aQKOf5nu9zS/dK1M+9EbksZaSk40AqAakmiEXNzqCGEk0v0g4O9a/KVSiVDVDPDy3LXAJsVBrSOjoKjxzCnyrgJiJMlQPet+8M369tGrqitt7cBrpr6YhUebkpyipHwJ2bEG+62pCOFxj3dF2rpyNm117EsYMOLbsoywNBYIBVPZOiqt78+tRiLly7v7aGFY/AZAGeOIZ6ZhBiDtKRHM77fOjxIY5dCvBle5k197hrJft3v4Geng0E/YO46RhsX2lys6prCNQWxucWymtNGL3wrP26FAt5YUGLPFQfx1wTt5N5eP4M6plB9GwJp8l0Qb+cKfCZJ8cpLObSXOHWZ9ES58tPn+KOazoQH/kZ/KkiTtWHRBSlFUIIpK9MTAfKWkXXzaqdgIbCRC+kVyEMUUbslotCBY6No188hzo8gpgtIpui0BTlR77idx99jSdD3hnwP+1a19z8nWv5bMzhT3euh20rzH7NTAkOj8CKZvT2FQRdKUQ8ghQCoZRNbcquxtQ18eY4zOZ7KefzfjWATAl9NoM+cQF1fBwxlsPRdouGI3lJa7649zTfAILaZqGlromKpf6Dyh5Q713HHSWP33QdbnYlouTxgoC1rkN3Kmp2+axqgataYHka1Z5EJ6MQcxCORFi3F9rGt6+g4qOLVZgtoScK6JFZODeLnMgh8raQcSVIyZQj+E80j/zzaZ6o5YyluPPrAl3/8AdupnlsErH3NLP6n3Hu/11uLATcWfG43dds13BVzEHGXUM6cdfIy1oNHAI8d1Z9Exq16kkKCq7kVUfwnNY86bo8+82Tc4tIXKl1Xxfo2g91g95ziVn+7I20jOfYXFDs8H26fc0mpVitoQtNWkNCCLP4KwWeFBSlJOMIxiUMCckxITiK5uVvvcrpBbv0rZh6vWBrx38DCIvtcVKxRxcAAAAASUVORK5CYII=', 'fruits/peach.png': 'iVBORw0KGgoAAAANSUhEUgAAAD4AAABACAYAAABC6cT1AAAYoUlEQVR42sWbeZBd1X3nP79z7tu7Wy2pu6UWWpEQQhpJRCw2NnYHbLzHEMedcSZxhZo4k3gyM65xQsaplIuyZ8oznkpNxeOxK1VOYjLjTBIr2Dg4MF6wETYIBAK00BKgXQipF/X+1nvP+c0f577u14tkEDjpqlf93n333XO/5/f9fX/LOVf4Of3dC+bRPsyePSTNYx+6obdYcPFNHvN2I3ojcK1XVqlKO6gKUhXRiyKcRuUQsBcve3cfHDzZvEZ/P3brbvRz4N/I/cmbDbgf7FZmb6x/a3dbYuV2Y80vC/oO53Sjc4Y4Bte8ddHwXwUFrEAUQSajGKOTYnS/qnmg3mg88N2BsTOLjfPPBry/H7t7Nx5QgDu392yPjNwthn6DrKnXYqp1IV+MWN0bu6vXx6xe5UzXUkexoCJAtS6MTRg9f8HqyTORnjkXydi4NQahVFQwfhq4X9V/+YFDI/sB+vqIWln1RoE3j+trsfBu5gB+V8bIv0f4oHdE0+UGUUb8ps09+ra35eTmbafN2t4JbEnBpL/SltEEsOF9Y0J48VhGn9yf8888n9WRERtFxhB771H+OnH6+QcHho8Bci/I67H+YsAt4NL3hktc7N7wHc3BfnlH9/siY+9BuL1WTajWHb0rc8ktt642fbdfZa67ehiRg1CbhgZ4D5oCFgEjQDbcUXlcOHfBMjhsmZwyTFeF8xci9h/I6sSU8dkMNmOEONFJp/pfHjg0/CeApqxzVwLcAN5IaojF7S39/ZjmAHft7H5Pxph7UHl3pRzjVP3WbZ36nvevNrfcuk46uhsw8RSMn8J5ECMYo2FkCZZVhWpDOH4qw4HDWU6eyTA1bXCJ4Bw0YiFOoF6HiUmhUhUEkkKBqJQXYucfrTf0tx8cGD72Wqkv8yjrtm3iQ5UK/9VaXFuOf/X8MY6k5/nmOQB37ei+NTLmsyLynko5waP+5rd26y/dtdbecONKKORh7Ahu/ElEY4yVIGIC6iGOodYIoKOMpVzOMDllaSspS5couaJCpDOTQ0Mol4XBEeHYSctzBzM8f9jq8Ihx7SWJMhl/MXZy9wMHB7/7WsBLK+hd1/COqTqPfuBtmEoNfvgU9508z2/19QVa79lD8r5NK7uLRf2CjeQT9aqj3nD+Lbd060c/ttFu37UcbB6mp3CTT2LiU4gNXFan1GKo1qDeCCMXc4ZSMUNkDdgUqAecgJsrMCIpHyOFTHDA0SHDo49nePChrDt3wdol7aDq/939B4a/8rPAW8AMgO/bzuoL43zvpm20f/YTNE69ijn4MpNjU3y9qwu7dy/Jh7f3vLtQkAetkdtHx+q6cWOb/9Q9O+xv/NYWs2JVCa1GaPksTP0I44cQGyg6VYbxKZicBq8QWShkMizpzGBMIJ16IBFwEqwhAWzz1TxHneDrArFQLMF1OxPe/Y7E4PEHB6xGxn5w26pC4+GnKo/19RGdPr24RtnUZzMvH+d763vZ8vu/jh+dJHpkH/LqMKWNvfz4mUPyyl3bez4VWfnres0tTZxLPn73NfbTn7nerN28BC0rmhhM7QBSeRyROrETxieV8alA6cRBqQDeCedfLdK7xoTBWyzaCnJResqsEErqMr4m5HOw65ZYtmzw7NtvXaNu79h2VaH88FOVn14KvAVUarw1E/HZT34Ut+s67LNHkfEpmKpQGpnk7nVdmXd2FQu/l9SRVasL/o8/d4O9/c4N2IbD1w1GEqT8E6gfwStMTAtjk0ojDjfoFdpLMD5uePypdt76Nkcp78HLZYH+TIESMCadgKpw1UYnN1+fyN59kSuX7ft2rM29/NDe6oH+fuzAwNzQbAASR9aaQEFroBFDZzvy3lvQt+8kGxXiOw4MXfT5btU//crbzNZbV9EYrqCaxTABkw9D4xTVhmFwFCanwxjWgnPQ3ganzlj+/BvLuP02R0eHR+MrA626MNqIpGNNCGvWO/n8H5dtW8m7Rs3+5a/e0LVr925cfzDyXKpvvJaLU+N8YnCUtrYSjfFJxPsQcLo70at68IlXe/RcVZ7cP8S6rgJrt/Qg9XP4sR+An2a8HKjtfbACBNAd7XDkxYgvfrmbf/vb7Vxz3Sh+2mLsleWZYlN3WCTUGgOuJnT2etnQ6/XRn2YzivzijSvavu7eWU4GBlqA94P9wTlqq7p4cXiMD+07TPHiJJI4yGeFbEakmMesWQGdbfDs4TL/99vHGT4/ys51Byi1N7gwJFSqijGzPup8AP3coRyf/2IbH/uXV/O+O0dxF8vY6ApQmxACL44rLoFcbnG6GBP8vrdHTblC8sLRTI/kfNvfP1J5qJXydiDMnRka4+j6Hv6ukXB2aJTOV4dl1aHjyqlXwiQU8sJVPcLm9aGQePjRcf5xj6eUg83rwkUSF4THeehogyf25fmTL5fYvqOHP/zMCvTiCxibQX52JjxLbUAMlCtw+JhnaAyGxoJfdy4RVFuSkTT9FSNMTQjrVnt5fsD60XH7luvXFb67+4eVV/tTzLblJ+bVUUaHxtn7rk0r3pn3mZ07ty73vRs6zTMHK+x/wTM0Css7hevWw8ou4fQF+NaPYGwSbtkRVLtahyUd8Ojjef7sviXkc/Cfv/AWlpROoNUhjImuSMROnFGmqpDNhM8TZVjWLmRzIf1FQVRAwzTUa4J4kfYSuu/5jBWjG48OVv5P/73Inj2zwOnvxwwMwEd29tzlEr6wrDPn/ueX3mF/pX8LH7ljPRvWdfDiiQo/2lulXBO2Xq2sXhFm+/GDsGc/9HbB1o3ww0fz3Pc3nVSrdX73d6/j5nf14Aefxlh93QWhCHgHZ4d0Tm7vPLQVoK1NEA2vmVJJBZ8I4xPC6hVqnj9q/Pik3bhzbfGRr/5N5XR/P9bO5N8D0LN1a0ZtZXe95rru+cPtumHHcpOM1unozLNj10p+7cObWb/K848/HOTQccO2q5Ul7VDMwYWL8J3H4MIrefY+0cnkVINbb+3id/7jDfiJM9j4GEh0xV2DoVENrmTC1DkHK5YaspFw+qyiDgp5g6YJkI+FqWlDWx4aMe7A0YzJWDoHLpS/uW1bqr99fdjPgacw+mv1qr/ullu7/c23r7bxWANjBY09OlHHOOVXPnY93/1aFz2dnvsfgVI++PM16+HalTkOH1hCte5YsSLLf/j0DlDBNM5zpQ0T1aDk3Z0hC/QeGg0oZITODmF8VDl2Vjn9KpCA+DCUpFVvrSbsvMbbUtFrnPCB/p3Lrtq9G9fMwV1/P7ZRd3+QyYp+/DeugYwl05nFFIOV4kRp1GKqFzyrt9zB333lJtra8jx5GJZ3WGpjRYpJJ+0liGPHH9yzk+WrO9BqGXFDzAujr4vqOFjVI6xfKeSssLTNcO1ag1HoKBo29hrW9oSSUn1Ie5vgG3Xo6fSycY1z6k3em+iXAKJmDZs5u+q2yWrtX7z7jl6/7i0r7KmnR3hy7yAvHh3n4kidet0FBRSIIsuSpUW2Ll/G4HCd00csEhvqsaNcifn072/nF/p6cRMeq6PgJkAyr6Wvcdlqas0qYXVXmsurQAwZA1dfJeBDYSMpscSHlzqwCNet83LwRTTr9X3An804XaUc391WjPT9H1jr/+KLB813vnWSWjWh0fCoDyFFJPUfVZJkmFzWEkWG0WpCJgOdnTn+zSe38MFf3YibaGBtDiovviHAC2hv0mKmqZEutCnx6SHfUtD4MGVJDBtWehNFKonKDf1bu9sE4NdvXtYxOiHHN25o62pfktN9Tw5JvhBhBDZvWcKWrUvpXVmgvT1DZAXnlCRR4tjjVcllLcuX57h60xLaugv46TrGFqB+HKYff8PWnp/ITF0Upiagq1PIRsHSKC0NMGjUhMGLBtGQhk+V4XP35XS6IkSRviMCqDSit5Ty0vXKuap3ZyomX4joXVngk5/axq4buiEfzcsoFrkhr1BP8FN1TFSEZAjKTwdlehNAN0Vuuqy88IonnrJcHFe2rZNQcDRB+7TMdbPWTxJoy8OydvXTFWuN+rdGAD7RvkxkVFCvqmbFijz/7X+8leVr2tCJGF9vzNz8JdLkUDKaDCbKQvwKTD8BmrxpwJuDV2sQO8jmPZWywcVgoiZwmQHrnaRUD3lAPgdd7crJ86CR7orCTcuNiooPbsxn/uh6ll/VRjJaI8oYLGn7Q8wiXatm88xBMgq1l6B+Mj385oFuqvuyJULPUpiYhNVtQsamPq8tdE/jvLb4vUVZ2qbivaKwMerf2t3WULYhQnm6IR/96AauubELN9ogyuYDKI1B6+DjlEPpKOrCcTcB8SAkw+FcyV6GG2/sz1q4dr3gErCTAo25oJu5u0/SSZD0v4P2wozo90YN2CRCb5x42tsz5s4716J1i5g6NM5AfCFY0pcDqNZGuPoW5zIhM5PcvGb5m/yXXtZGs4BmbkVnwbqkJWdKFb6Y1SZNl0cmMlsiK3ZisuFvuaXbrNy4HD9+HFM7AG569kpi59G7SeVo3l39nAAvNgGpgOkccUupHrcAT8/NmJnPuch53ZyNhCTxeuNNq0DPo5OPBU7h0vw6ukzK+U8EdH42k4DGrYUJc967WMKSXApaPUSGtCQWGxlhvfdKPh+xeXMRJp9BxAQ/zV8D+euCsox9J4z2Ziy3aUtYlHmX1NcIvGqg0bLW0wrahRA2f1LUKRq0x0VAb5J4OjoLsmL5NNTGEfFQuhHabg2/qp8IIvZmJCKS9s1rMveY1UAsO3eVZc5wmqb8CeiYmcncWmkuQBIHH5fmcpAPzIhnu+y1yBi647qnd1mBjrYaVBpItAwK28FPB7GKz6cxOfvGgWu6KBABNTMjPs3EA2kuGmhYS4t0rrQ0BB1Mrc1CqwLEDUEdiCjqw3ucUmuIErLuyUihK3HKkiUZkUyMOkVKm8G0BSUXSSdA3lxhyik0FGJpoX/qs3WZ0VS1CllFItAGMGWYWR/xC30bhVq9KXppbzut1KYqaNqSH4pEpMN7paMjA6aKIkh2Q8uCqYKvv/lCJkBeA8gm6BYQ2qRxLFAOvbWZuKyXAO0V56FeTwEze45PYKwsGlZu/KkI1aJXyOdE0DKYPERLSLkyWwJdStT0CvVOCXQ2gb6tFtR5FpwBKwuFbPY3ikGo14OPS8u54pU4gZFpwQogeiRCJKMqZDMefBmVXBq+9PJS20pNq1cG3gBZharMrKIutOLlLBzES31aj6NUatKSqgbqWKBchZGyEWsUEXkuElWPGBOZanAijUFrIO3zpnxeKJpDTQnWu0Jf18sCuzStZzouXhENxUu1JpgWmqsLK0QjU6KTVTEivibCc0ZF64JBk8lAaV+H6kEwhWB5sWmq2lL2peEhhJKwcjnjf1ei8GkBMhOafMv7xf67uWrdFK9yVXCJzk5SM3HBc2bUaMMbjPDS1ueGT0QCYyJSqlZqineCyUH52fDj4k6Iz4VwRma24G+leXOQmoGivzK6W52N65fIxMKE6Ey3JcTn1qRFma62ZGst56iDl4ett4KIyuOfAx8Br1orq6emjeJTORMLlaehemjWBGpnab0ghEgaVw2U/OsT/5aqdka8FtA7BaAyQ/HWyTDARDW0mAwtk+KDf09UhWMjVjIWUeRhAKMqRyILI6OiWpfQcFbSKqtlD1CT1r6Fjq20V4JITZu5mdfl/lpaRYvTXIOVm00Fpy10D58l3VYyWRaMhvObv/cesqIcHzE6UjUWdRdtpvFYAO51XyZShkYME+MCJuw6CJhlNsVMZHHAfl7aOG1gzIZj9mdsKGvWPwlzAfl5ftwChpnPqW97GC+HPmCrX2uamxuvPH8+ciBqjDy0e//YRH8/NvLIT6xoMj5pohOnLL/QlcyG8MTMVEKLqzlzlm1m3pcFqlHw+YLOxuvW3XNJGr/LAlOyMDy10trPVfLmhBtguh5CldHZHVThGpBBGakIBwYjk7UqXuW+5rxH2S1DA/7lnsPq5fr9z0d+102J8V7m+pteyrfn9bq0JetKgDEDY6nlrc7Suwk6bvFtbQlPrcJ1CdBCSErGpkzonzfjtp8Fnssoj53PuPGaNQXrDl88NPwYILt348zu3Tiv+p1SAZ54JuMvnrXYOaXe/PDVSktZ4Pfael4TaB2YFphMX+UUdJPy/jK0drrAt8UrmsDIpMG5dNKczrqBB+PDFtHHzmTJRYgx8r/2QNLXF5Z0wlaQmG+I8fH5YWP27M1oZVpwcapQrWBnQC/0cU3DxqIi5RcRuqYfOxYKV4uwzZ3o2WMjU0ItBlGd/S5lik+gYJT95yN/ZtIao/5cXuUbgOzZE9TL9PdjHxwYPuZV/76QFfPIE5EbHTUMjRjq1dabXETUUrA6H+QlE49W4WqKqC48ZxH1biq4pKDLdUKG1pwMnT3fKFRj+H+nspqPRETkv3/j4GA5tbYCmK27Q/CSWO+NIl87dsbKT/dHmouUC4OGqQmZpXzLzc9Y2C8C2C1i8ZbvwsJeC4UXsejsdXRmTcw7GJoSpmuBygvPDd2XkvE8djbrz0xljMGdMMXM1wDTtDaA3ZNufr1/T2Xk2p5ippA1tx09Ke6GLd50FGFy2tBoGDJGQxtOWwDra8iv52dd2ipCLPzsZ89vfjaE7snwlFBrgEEXHUe9kgVGqoY/Hyi6yBjrVX/n/mcvHEg3PvjWXU8MDIT95iMnuvcWco2P1BpmxdkLxr19uzeiEDeaeXDoVDa3pc5v+yyadfl54Wnedwt/r3NYIkC5BsPTQpzovOUiZncbpyzMGeXrRwvubDkTWfz3v31o+I8W29VsWuvOPadP1xLvf7OY0/jwccv/fiijhbTqUg9TU8KFYcP4ZEgPhXkT4OaLU+rHXhf32wWfZ2ktGhqGw5PB0t5piNUL3Cv8xjlot55HzmX16eEseeMqsdPfAyR15wU7G0mtrv392G8/Wj23ubs02F6QD79wUpI4EXvDJkejMVsA1Gph67RLwgWi5sZkbQlpiyYgLIzRLbQ26bE4gckKjJaFepp/yyXK0maBUjLKS5OWrx0puWLGRInqp/7h8PD3+/uxXx1YfEsnreD7+oi+t6/89MauQq6jYPuePyZxtS5218awTu6SQHX1UKuHCajVQy5t0ps3BAVGZxfoW2+0aVGT1tGSilI1DgXFeFmoxi2TuQitm++9g5x4RuuGL73QliRqI+/dNx84NPyf+vqIHnpo8Y37C/ZnnD7dBF/5wabuQmd7wb794AlJzl0U2bHWSykbtl1L2tnAB0pWa8J0VajWw5ZQlwjeMZuUtKi3d+ASpRELlbowWRXGq8JUVcL+V9XFLeznToBLQVcS4U8H2t1ILRNlxL1gHR/uH64kf3X60htvFt2Ycvp0oP23flx5+JqugmkrmNuOnzdy4IR1a5arWb3U4xIJu5BU0/1lOtPMrzeEak2o1KBck5nXdA2mq2GCpmrCdE2oNoQ4XYs0TcCXorWfTW0TpxSMpxwbvnSk3Z0tZ2zB+vN1zLu/fXho6BdB9lymPrzkjpym0t//48qPNnUVTxezvHeqarM/GTBJtSFydY+XjlzYoexdc4Nd86Wzwpf6vPdhn6t3s7010TQpabWwn5efz8vVvVecU9qtcr5i+dLR9uSVSiYqRn60keh7/+Hw0NFL+fVrAj7X5yvPblpaeCibYVc2MmsOnjby7EmbRIKsXuqllFV8EsRO54NgtjUkvmX3xDy/v6QAajMHCK3jSJU243lmNMtXXmxPJuIoKlh/vlr373/wyMhzl/Pr1ww8pb3v6yP6/jOVVwury/d1JMV6PsuN9YYt7Dth5cBZk3gP3SWVJXmPJUxAeMpoFrTqvBiteon4P1e8mmwRVdpSf959puj/9nTJR8ZGkfhnpxP9wENHRgZezzNo8noemWw+anXXtu6N1pp7RPTjzptiNVaWt3m/a43zN1yVmPVLnSlFoQBpuKAFvhm3WejHkmZdrYv7moKN0CBgsbDvYk4ferXgBmtR1JFRnOpfTKp86gcHB8uv59GrK3nScM6jVx+5vusaq+Zfq+jHnDPrG06wRlnR7nRLt3Nbupxc1eHMsqxKzihCmAzvwaXtJE2tOqMNqlhVDEriYbhq9MBoVh8fzvkzlUxUjMDiTiSez3z70PDu+Ub5uT5ieS+Ygf5Q0APcsWNFqQNuM8Kdzvt3JWo2eDWIKMWMp6voWdXmfG/Ja3fBy5Ksl1Kk5EQlI2ClyQ7VSiyM1kTPliN9aSKSk+WMKSeWnFGsuEFR+erYdPylH54ILaTWxzr/yZ4tXeyJ4d/sW5dvTFdu9E5uc55bY8+OxMtKTZejBIiMkjFKZJTIQCSKJ7hE3QsNb3Bpt8ria1nL0yr6t7VK9M3vvnR+pOVZVsc/89PE0t+PGRpC5ovLx29atbwRx9cqXK8q25yySZU1Hpaq1w6PhB1GSiJCxYiOWiOvWNGXRHjWqdlz/4HBF5rXSwXMvdEVzP8PU3cFoA8NnC8AAAAASUVORK5CYII=', 'fruits/pear.png': 'iVBORw0KGgoAAAANSUhEUgAAACQAAABACAYAAAByZdXUAAARDklEQVR42qWaeYxd5XnGf9/3nXPuNvfe2c1gjHds7GAam5BCIGMawIZQtmRMoCFtIhU1SlM1kaiUf2o5UdW9qbqkSlIlURMpiwNJaKpEZB2yQYmh4HhwbGxs42WYfe52tm/pH+dej1dszCddXZ1Zjp77vO/7vM/7nqu4xCNE9hoZQRUKeD034eVqBFvext/cdTOPr13KIxq+NzHNzHaQo+Au5r7qIv5GbrprqHBiXyMPFKGahy4flgNTZmwMd+IEdnIMs/oy3n7jW/nPVUtQ49P0zNc5/Oo4v2QYdfgw9nyf7dQL7/VIANzWrfjzYTx69/sXLc0XkUYbGUU29XPHwiQqzzdq8nirpvbNT8rnXj6SvPCb/fP/PF/nw76HLOW5DfiHzZuxo6On33xkBLVuHeKTn0Q7dx50Z56REdTOnZjrh3u/sfW9gyOr1hfpHVTsfbHGynUKFcQIr8X+3QlHDxjmZ0T47M+CJ5eo6WtufItesWsv9b0HuGrsMOOABNzICPKxxzD2FL6Gt+ON7kBfMGQDAxnVXVVle/qDbbmc0rOTTjTmoa+36pzucr7stvVZ316x2rm7P5gG1w3Ha3f9Ot9jpmOzqI/85DwvHX2N50dG8F/ai9mzB+ccLF2Zv+GW3+/+6NprK/808Wv5zPir8fGREdTrhYzRUQzA+Fzjx0dfKc2s31jprc0nbv+ehvADJ1av70J5Pq2aYvxwN72VxW7oqlft2+9oqaf+I+Cmy2KqXbwP+NLOnSQw7F1/y3Mf2HiT9+He/sp169/ax4vPNNi9qxlcTA7Rpljt3MnszLLkO/t2Nz7YPeCbpauKntHw3C/m6VsUUCwrThxt4WxFvPL8UhV4J1h2fUvWJ3D9PbyzCtXi0NCK4Xtf+Ny73lPeuOqqIfY8a9yBPZGOQqsUIj1ZQRdb5jMT8b9PjsfuimV52b8oR7nb48qVBdLYsveFOkdeDoljTe+gIpwvsviqXpECA1Vyq96S+7c//fvwhw9+tLhRNxbp6eOBPXY4JIqM36hrUWtEDYB163AXLPuxsYylnz+VHusqexsHF+fWdvf6WqdO1uc0SeKImobjRyJe+W0L5xylssfMhIeeq4s8hkYud82dH6rmw+nFdv+epmo2U1GpegwM5cSxQ9HMz743/VdAPDp64ZDRQQ6IyQn9Fy8+Pb8ln1fe1ERsdeqk70tKFY/b7x+kUFKMPVfn+JEQoQR9xqOLhL7F0r76Uo/o6UH+Ztc8a64p4/nCNGpGTpyI9wDz27cjd+zAXowwMjqasfTLp5LJUpc3V5/X716zoSyWrSrq/kWBLXd7Tkhc2DQUikpMHk9IU0F9okU1bygsHxSenxNH9kesWNtFsSSp9vry2OFQ7H5m/tHxY/HY4CBybOwiQnZm6H70ZPRMtdevz04lN8SRLaSJk3Fo5asHI3n0YCTmZjT9Qw6hmozP1JnQMDXTZO/uKbRJqFR9lyTKvfLb5tzu/53/xLM/m/vC9u3Iz3wmU3JxCW1MAvbylfklvd3BnYUutcHzxBKjRb7SIxYJv7HBErmlK32x9KouiiUPawytpubQvsge2JsIjH9w7Ol088wMRzuhuiilvpCCn/qzR/+W8q+e4kfL1njX3Xr3Za5a6ZVJ5OOcQHmGIJ+Q7wo5dKBp/uerk7JZs//yw2/ysW3bkKfeS3DpR27ahBoYQK6+A57/AT++6prcjXc/uNi25ntk2BBI5XAOcAIQOBIuW17j0P6me/xLE0JKbvrBN/jFqR9QvglAdsUK7Pe/T7z7p/x576C8cct9/WlUr8iwIRDCksQWKUFIi5AGgc/up2H5moJZvibvGg3eDzAxsUDMmwEkdu7E3PcnDKYJn7juHWXbVe7yGvMefs4RR5ZGTQOgU0cSO4I8hE3F5AnEijV54SzXAoz+dCFklwxoeDhrzDMn+MDgkKquv67L1md9IRVYA/mioqfPB8APJLm8xFnwfcHUcWSl18f3WPqxf6SAyHTuTQEaHcUIAWnCttVvKbpyJRBRy0OKBUepvIUU7XgeqaBVdyipUD7VsefoPrOE3/jZnnmbOx9kRZDn2lXriiKJpLRGgXBngegA7LzrFIwGJclpR6F9z0tnaPinSEDMznL75Vf6weVLAx02lRCvU7RpYrE2A+kcpInDOYRKT8dwSYA2b8YCzlruX7a6QL4gRRJ65x0GktgyM5XQamqcsyAEaeKwDh3kSd4coLay3vUwVxaK3LxsdYE0QRrtn1PVnMtyqVjy8DxBmmRSEEcWawi9Ck0AdmRTibzEcDE7zbahK/380NKcjkMprFYIce5JR3mCao+PVJAkBs8XLgoNzjF/7TD1N8OQGN2M/exn8Z3jkZVXFykUhUwihbPynHNbRlP2lsQGox1KQRRqnGVqxzaSzoTzhgEND6PYgd35JO/t7hOr12zoMklipU5yZ4HR2mG0Q4gsbEJAFKZY5xASF7YM1nGi3RvlpTAkBgdx279IPgz51NprS25wyBdJKNBxcDJcWVk75qZT5mdTjFkAFbZSBAIpnItaBgFHz2wd3sWi2fQI3s7Pkd4c8XeDl6uVN/xet0lTo3SSw6QKId1ZLHGq9mhLFGmkFDgcUbgA6GIn1wUwm/B3fY701vewzQk+ftv9fabc7ak4MsTN4mnV5Rx4vqC7N6s6z5MgIG6m6NQgpcRaQxxZhGL8XGbrQnnj7dpFuuUB7klSvnzrPb125dUlGccaHedJouCs6uqUulKCRk2jU0ccaax1CAnGWJHEFqWYAhgcXFhEyAsxMzqKvuMhHghDvnXz1qq/8R0VEbW0wCqac13n1eZO2+hUWhzp9rVAp1boFFA0LpqhTZvwd+0i3foHvLdZ56tvGy67d27tcVFohZDQmKlg0tN717lAFcsKzxMkiUYIkSl34rAWPJnN8xcE1AnTnQ9xT7PG1669oeRuu7ePKLRSSktrrkwS5s5K5HOCsmBMJgGdk8TtnibwLwhoeBhvdBR95/vZ2miwc/3GorxzpJ8kdlJKS1jvIqwVLwpMxyQ75zL9aYcwjdsiKOh5Xcc4MoJqg7mxWePxlVfnvbseGnBaIxGGuFXI8uZiwZxp2gU4HFJJJxU4w5XnZWj79sz9v/uPWN1q8J3Fy4PCPQ8POBzSOUMa5WlMV87br87X6Tsq3UluZyHIKTxPYl1mYc9VZQLgQ49Sbs7z7Z5+1X/vwwMmCKS01qCTHPWp6htqeh3PAyClQCnZ9kKOIFAyn/exhpseeQR/586FuUx2esmOHdhDB/l8viDW3flAv670+CpNDSYNqE1Vs1FGXDwzzbpmdiohCg1KSXxfnWyyUghZ7Sla5bH8aIMbOukCIDsz0R0P8gHneODGd1X1stVFL4401vjUJruzTv4GQtWxHJ4vkCr7FPmCh2sjMsYxMFhyhaIkjHiUUza0amwMce9H6K1N89/LVufyt9/fL5LYCGc9apM9WKveUN50jh9ICkWFUgu01uYjhBA4B0FOSZ1YU6vFa6+6lqef+Dr7R0ZQErDNST5eKNJ/0+09VkqkMTIDYxRC2guG59xJdMpEaR25vIcfKJzLur8xlsuuqFAsKpKYv25vZZ289yP0RS3+eOmqgluyIqei0NGY7s46uHA4o8678nYusxoXznBQniSf92lbaqyFIFDqssurRnm8tSa5a8cOrAxnuM8LGLj6d7qclIjGdAUd+wjpENJRKDeRnj0LVOZ7LLPTyUkj9jp4EAhy+YU86liSwaEuyhXfxTF/BiCThLtLZemuWO672kyBJMwjlcVZQVCIqS6aIShEOCdxToATOCfaNkNS6fGRSpzeTMXZoXQ4fF8i2mPqSaviSbVoqCKE4OYt72O9NCkbuypK5PKBbM4V2xuL7K5xK8/ca70krTzgyJVCKpdNExRinMsaZS4nTxsCk9gyO5nSqJsFg9amSarMG3WuOyz19hdNuar8JOE+6SyDxZJEOF8YLdv8OnLFEM/XhLUStm3gg0JCrhTi55KTIXRnhLLZsCSJpVU3C566U9JKIAUY7ZBq4f+DQIlqTwGjeZcHeMoH5zIwQjnK/XP4+WwYCGulk/2rNddFGvskrfN3+nxBolNLkJenlTxt439gb0TYtPQO+ixeGnQmWdFVziFk42oJaGuyHuOsJCjE+PkUZyXOCvKlFkoZcAKrFVGtdNbIc2rVFYqSvsGAStUDkYlgliuC8VdjZiY1zgleO5rQqBmUAuuc8AMPpeiTQjIXhRZjMn+Q5UYnQ11789Uem4S7oC51mmnnzM/qzNALAcJlVsRmoZSyXQwuc5IClJSSA82aJo6M83xLEuaIGoUMjJPUZ0tYLc9p5YS4sP5Uuj3yeYnWlu4+xdCSgEJRsmRlnlJFIWXWXow2WEvd8wN+3my4352Zjl13WZBEjsZ0FVUvIlDUZg1eoMmp0zXE2kwU/ZzAnYM0IbM08LyMBWchSTRLVuTbaz5BElumxlMWLc7ZONZSGw7KwPeeiEPc4X1Nqbx2RQiHTnysUVhniFrpAr1kYF492OLlPQ3mpk//HSJrFVHLnly9QHsuCzXOOozp2FqYPJGSxM41GxHO8nPZWKV/JSQvHtxXJwpTK9srMCGynFFK0mjEC+zITKEbNU2SWBrz+vTQOajNaeZmUurz+qQfisKUNDEIKVBeVv5BTrDh+i6ENHJmMsL3+ZYc3YEuFOW/zk5rsf+lWZvLKax1bXfnCAJF2EwJWylKCayBICcZWpKndyBgYCh3MknPHIHcSfmG2lyEcyAl1GY1jZptr/iEmZlqErbci3YDT0m2I+mzX7GGfXueP6FazcQqT+Jcu1x9hVKS2anWySx2DvoGcyxdVcyWme5031zp9qh0e5QrXsZOK6Vei/E8gU4d+/eEHNwbIiTEkXYTJ+oCxadHd6DVyCDyiS+Srt2o9jWb+uGolZoVq/tkB5BUkihKMdrR01fAddhzWS6d05wpgZ+T7TlMcOJYjTjOSl8qgecJyt2KandgDr8y4712Iv6/Wzbw0c2bM4PmRkZQT3zNvbzqGipzs+E7wmaa9g+WlFICz5MoKZl8rUmzkVAqB1kS20xwxHm2Zp3mOzneYHYmOk21uyqK7t7AjR+ru0Mvz9lAcfcXPsOxwcF2RxkbyzztpnU8eWSC62emm2vClk49TymdWkrlAK0ttbkIEPT0FfB8edJ80bYXrrMOVgIpBZPjDaZea2ZWtl0szoLypJuZCs3BfVOeSfnwDx/jux0rLc4coe56hEI0y3c9n1sWX9mTDi2ueH4ghUAwNdmkWU/oquToHyzR01ekt6+Ac669BxJZyYcpUxNN6rWYIJDU5jRHDsSsvLpAV8Wzr403OLR/SoZN95c/+Raf6gyn53r4IgE7/Ifkgxb/JRQjfQNdLF3RbRr1RM1Ohxht0doSRxqjHf2LSmzYOESpy6fZrsawleKcQ8ospI2aYeKYdktWBmZqsu4dPjiHMXz8J4/x6VPBnOu5vQPk4RdID46xc+V64vpc8s7Z2aZfrhRsoejZifGWEAJRKPiUugLiSPPbPZOkqcE5RxJphMxClL2kyxWkUX6qDr8yLY8faR4Sggd//E2+ciaY13s8JbZvR+zYgd2yjY3G8VcOtpZKCpCkqbNaG+cMolQOxPLVPUJKQbU7j3U4h3NWOxeFWtZroZiZajI9GUfO8tlqF5/69peZPheYi/6qBcCWh7hVpzwi4LYgJ7s9T7YFUdI/WCTIeWhtscaSJJowTGnWU5LIHXSCxws+n//e19l3vgeAF/0Ab/t25I5ssW0BtjzAMmO41cEwcA2Cy42m4hw5AYmDWSE4KBzP+jl+kC8x+sQXsl10G4jldb668/8Lzc7OrBQrlgAAAABJRU5ErkJggg==', 'fruits/plum.png': 'iVBORw0KGgoAAAANSUhEUgAAAD8AAABACAYAAACtK6/LAAAb2ElEQVR42r2be5BdV3Xmf2ufc5+tfkrqllpP621Zli0Jy7IxtMMrGEggTHqGQAhQU6mZQKaSmcqkkkolLirJZEKqJjNMiplUpWrIQBKSJiSQYDLYBtqALWPLD2TLtiTr2VKr1ernfZ+z917zxz73dreNY9lArupUt+6j7/n2Wt+3vrX2OcK/wGN0lGhsDAdwyxE2qPCvvOfdznKz96wRIQaqCrNRxJnI8EyU49su4pHj40z8oL/zzzwiEZzqq5+X/Jhxy+goZmwMt/c21kURv+YcH1bPYJyHYgnyRXAWVMPRaoJNwnPAbC7HQxLzud3DfHlsDMcoEWN44GXwRiEaAwfkd2zkZ3MxG2YbfGZqilqGVf+lwBvAA+y/gw87xx+pMlQowead+DXDUOpC4hxiE5i/Bv2DaNJC6xV0bhozcwWpzIFzEEU8ZYTf+f5R/mHZuXfAjIwQj49jh3r5id07ip++affAPps6vvvY1H0nzvFT2dv8ihT58eQ5ESfwe/eSX7+TzzjL75mYVXsOYDdtR3oGMD0DiPeIdxDF4WPVBaR3NVJahVmzHhneiq5eh1dFawsMO8/PrdvC0KYhvj45iWsHrw1841o+cWBf918dObhuqK87nwz0l5PpmfreVtN+u9LgzChEJ5YtWPTj4PeJMdz+/Qyyin9wKe/v7scdHEHWbSKK88jVCSitApFweA/FMiRNaNagUIY0CdEtr8Ks34zpH8QvzuKbdQ77mMOl7XzxHYfRtWuJxsexW9dx78Gbez91+NYhTVJvJ6/W86fPzufmFhqo6oVrC3xz7QjR+fNL0Zcfh7DdeIAtcZGv2ZQb128mvel2clEUAOUKMHMlcHpoU3hOJPA9iuHqRejuD4vj7VKY4lxYpGeOkk5dICeGLz7zKKMAm4f4vUM39/7WwZsH01NnF6JTZ+bN7HzytHVcLBZYQPkfz57lseVU/JGCb6feTYfYbnLcn7S4Yese7E2HidMU1AeQ7ceFk7BhO8RxAA7hdZvC1EXYsC083/6MKoiBKIInHyKdvkwuXzK/dO24Lx060PPfDu4bTI4+OZV78VxFHPzGhz7GH33yk+J/gC7+aAWvDXzvEXYYuN9btm7Zg+1bTbxmw0rgqpDLw9SFIGTDN4ToGxNei3MBvAisWZ+lfyZtquF9rQYcGxdfuaJmx9pVHDm0zn73scno3MWaKxb54PHTjGVRllFgLKyAf1lN/GGBHzpE7pFHsHsPs9fA/c6xZectuL1vIK5VoFmFVX3g3RLHVQOvpydCiotk5c6Ho1CG+WkY3ADFLiivglW90Lsaevphy41w9Qwi0yXufuN698ixqejshVpSLkXv//5p/ftDh8hNTuIBnwncDwx//EOVslHk2BjpzXfwVu/5gnOsufENuK17iBpVWD0El8+GSOULS7Xc2pDuURxK3MbtkMsFnhdLUO6GQgGGt8LAUHbqAs06JAmcewYqp/O85c3D/tjxaTlzvmq7ytHPPHXSfe3QIXLHjpFeD4DXA15GRoLCMgb77+Q/OcungOjmI/iN24mSZuCnGOhdA9cuw7qtwbxEsVAsQs8AdHUrM1Ow+wBolhmKUCgG81NdUBCYnYLKXBC8xVl4fCziJ988rCfPzvL86UUplaIPvFbgr5XzS6CBW29npxP+2HverR49OIKu24xpNpbKl3dB3SdOweZdwvqtyqr+kMJxLmTEI/cJNx0WyqtA8TSbUK/AlQvh5HI5oViG3tVK9wB86Q9hW/dGaknLHj02HRcKfOz4KT77WoFfD3gZHcUAdLz5CH0k/LK1/Gfn6VmzDrdtH6ZRQdYMB0ESE+p2Vw8MrBVmriq1Khx6M0ye6mLyuQEmT/YyP1lmdjJGEHIFMJHDkRKX6+y9a55dR+bo39hECBT58qdBzw3Rv9bY8UcmYxF+58QZfnfvXvInTpCOjmKuXkXGxwPfXy94MzKCaUcZ4Ja3sIEGP59YPi6wOV+Azbtxm7YTxTmoV6G6APvvCCldKIWyVOiCqxcN3/jsevz8BiZOdFGdFwQhX8hRKucwJka9Yq3FuhTvHarQ1efYcOMCB95zmdn5q7zwD6vYs7uP+781AUU9J6v1t5/5Dp9vi+iKx70YPvlyS/vK4Ecz9c+iPDJCcSHlJ51jNLW8W4S+YgnWbcVt2YXpW4MUS1mEh+DCKXAprN8iNJtK2oKz3xvm9Ph2Lp8qoF4ZXLeajRtuYGjtZnq615DPlzFi8OpwNmF+fo7JycssVCe5NjvJ4lwdk7P0bXycI4f63be/dylaWHBfWH+EX5y5wJe944wqtzpPIsKFOOZRE/O14w/zwqt1grKs8+p0SjfdyXaxfMg5PuQ9u6IIugdg807c5l2YgUGkUAypuHzVTx8HY4RdB5XZiRKPfG4fZx7rJ847tm/fwU17jrB+aDuFQhn1Hucs3nvEGIyJEAQxsDDbpLJQpd6cY2bhMsfPf4Yjt1c5dbauExMqXfqmh23vs0cb+YlfWdUjUaGk5PKQtKBRhVaDVhTxpVyJe596iFNZJ+j+2cjfdJifEMO/U+U9cUxXsQyDG/Bbb0TXbcKUexCRrB5n7stktnXyHKgKB+5WTj88yP2f3s+1ScemLUPcfvBdbB7ei6qQpM2QiaJEkcE7ZX5xhtm5K1SqszhvKRa6iLWfgd7NXJx6nEX5rxTyMQ8/cY3u6ABFs55k6Jus2dJg/WZs71rM3BSay6ONOsxeIb52BVp1FqMcv/7MUf40o8GKmi97b2OdGN4v8IkoZi8EixnncBt3wLqN0LMaLWVmo9yDlEM9Fu+RhRmkUYPuXmFwi/LwX27lwf91I54Ghw6+kTsO/hRxXMD5Jj09eVZ1F8jlDLlcRKWyyLe+8w1Ovfgci5U56vUG3luStEkcx2wZvh2b/39s33mJ8Udm2D74QSrNAmevPeD7t130H/114qP3w53vhOefhOnLQWuMQZs13IVTxAszIMJ/f+ZR/uNLZwGxwC2E46JNqAADKqxJE3pePE506umlCEdxsKbFUjAk+SKsHkKHbxC6+lQe/ottfP1PdlPoSnjLm/4NN9/4JhqNOsWSZ3jDAPm86VhcgN7eVbz7ne8iMj8NwMTEHBMTM9SbCzz7/MOceOG7bNnSy/T0BEVzAzvWv5/YlCBeMPXSRbN22NDV7Zm6CLe/DR7/JkxegFweyRWId9yMXjqDu3KBX917G/kTY3yibcV/oNofOkSuZugRw0AxYo3CoFOGVFmnyrB3bPCOIe8ZRBlEpORVKTQ3Ec3sIy6mvOttH2XXtoNUaov0dBfYcsMqVMF77TQw4cszv4sGPgEnnp2i1fJEkfDQI3/H86eeIl8AI3kik6eU76dZPEN5z0P8298wnHvBc/r78Ob3Bov83LHQG5hoqVeYPEt66Qw55/itZ7/Hf2mLYNwua4OD6NgYPjMKM9lx6pXKxDv/A4WFZ4rrUtMcdNe2fjRu7vl4Wmj5d771o2bntoPUGhXiKGZwqBjqjdcVXR2AtrsVwHtPFBmiCJKkSS4fc+u+Ec5ffJ5m0iAyliifB2O5Wn+CI5vAWmVwA5x6GmoLWeMjMH0pmKxcPixCuZu41I2tzPL7N9/Bo2NjPDg6SiT/TPkTRkNXdPXqUoaMD6LLebOfP+rKb3zoiVa6uPPuN47qoVt+wtQbVYSIUiliyw1dr+q0VCGODdVKi5Mnr+F9+PPWJXzpq39CX/dGdmw6TDHXx6I7zfjEb/LhX/MMbw0LevY5OPl0MFbrt4Q+4tnHljpF7wCDu/gC0eIcL5jt7D8xRvqDvL12fo7B2CtZXUbiccZtbvMDY2mS7Nq35w538OaRqN6oEkUx3kGhEGGM4J2+op1ShSgSatWEs2fmoJMhhjRNGBzYxp6td4NARIHnLv8dm3Y7Nu+IaNQcJoJ1m+CZR+Ht/zrokjGhAp19LiyE9xDliIY242oVdpvz3AN82by+Ed2oGWfcHtp0zy+o03v6+4fsXbe/N2olTYwYBDAiGGNAXnmc0AbebFrOvDiPtR5jBDR8KIoibtl7NyiIi5lrPseV5ve48x2CqsMrxHmYuRr4vjgLaSu4zQ3bQiY4l2WAD+K8qhdNHD/TnrC+5lZ2jDG9bfitq1H/Ke+dv+v295lCoYT3DjEm0FihUXM4q530eylwMUELJs5XsveZIIMCqp5CvkS53IV6Ry7O8ezVv2HzXseOvUKzHuxz2grm6tDdwWGWugLovjWhJa4tQmU+dIazV4OXEuVWwte/5qgL4DXO/aa1dmjX9gN+y8Y9ptVsYEyEt0ocC6u6Y6xVpi4nWYSXQKuCiQQBLp2vUq9ZokiWxg4avD+qqPfkTBfXmseZbD7GXe8QPB7vA9DjR6F/LWy7KfiTi6fhiW/BP/1lSPuZqTAzyLyLlIIErb3zp1kVv46o+0Pb3rPZJ8m/zxcK/sD+t0TWpogIzip9/TnWbygTRUKj7rh8IeXS+YSBtTHlLoPJNLbVcExeqlNdTIhi6ZTBlQP5kDKRiTl17ets3KVs2xvRqDvyxWBqLp2Bez4Uhp35Anzt83DLXbDnIAxuDEI4eS6UPBMFSqhSShfoek2RH2HEACrW/aqztmvPjkN+Tf96cTZBvVIuRwxvLHdErlgy5PLQrHsmLyRcOtfi2pUWl8/XOXe6Sq2SRTy4XUSXS262BD4icXNM1Z7kwJ1gIh/mBHl4+J/gptsCMAVmr8KWPfCm98DQ5pBtA2tD1NMk/NQw0vRxHv9awMs44/a24fetds5+JJ8v6o07D0epDRPGKBI2bOrqADexUFmwtOqWyCiCp1GzzEy1WJxPAxVMSHVZjlkDHcJiKOJiFluTSHmWrbugWVe6++DJh4KQ7bo1DEWSZhiF9a+FMycCyLQVuF/uzgaWcRBAgUa+j7p5DVGPACRufcA7P7B1041uoG9IrE3wThka7qJYinBOMREkLc/0ZBNE8epRFJHwmhFBFDQTgCCQGgAvqw2qirfQTCqUe5SubiGKAo+f+HaIcKsZQDXrAdj2fXB1Alb1hClSrhB6kmYdWnW01QCEmW/9DbXr5vw44x7Aqf+wMaI7tt0iqh5nPf39RQZWF7E2Y6oR5q41SVqeOCdom8+a6ZlqAJ4pXFiIrDfOnhYN9dmmjpwphuluFr37Pge33BmiWlsMPUa9EuR7zbqwJ3Df54O56VsDi3NB7eMcWq+AEc6LoNcJ/l4Dn/RHtt1zc6PRum11/zqGVm+OklaLyAhDw0suTiTYzupiGljrXlriMnTtvrjT6LSt7pL/d4mSJi0GSpuon+2hslDlwvOhmXnjPbAwq51ZfqMW1D/Oh0W4OgE/+XPQ3QfVxY5v1blp8PDsddf5Eb5lAKzjPShm04bdrpAvYlNLb1+Rrq48zgVnZoyQtBxJ04J4VFceqM8iT2crQVShkx1LApg0Lc6nlKLV7Op9L1/4U8/EOcfug8rpZ5S0JfT0SyBxHbq6M04L7Nwf+I9AoRgEslZB0ibEwveve3TdTnl17l1RFLNh3TZx2bf0D5RWGheBZt3inCOOoyzSssI8dxqabBG07YB0KfreKUndYTC0bJV9Az8LM8r5xx/ihUfnsLZF3JVw+K3w9p81RMYT5UKmxbkQ8ctng9fXXOD91EVMmoIUeO56wQvg79j2M4P1xvyBnu419PcMmjRNyMWGYjnudGxtDI26DWNDn0VYdHnHsIz8dHi/nO9GhHo9waUuGx0pzln293+I3f69tNwCXlNmmqf5ztifc+b5a9w+Yli3xZOE3V0GN8HcVXA+LEbfGjRNEG+p93Zz5brSfpRRA+Bd8gZUugb6Bn0hXxJnbSfNl/PUe6VZSzN3loHyyw59CfD2//3SOninNKtpmOm1BRBouUXwSoFeCqxmU+ku7tnyB7RePMxX/sIzc0UolQPYYimYnq//FTz9Xbh4ChZmQAyLjYjqdYG/ylUJKu/eAMLagY1eNPjvNHG0mhaRJYeWJp5mPQ2Z0AHNMutKZxFWLE720wCtaoptuUCW5RmhUbZeFtWUllsk9t0c7v8lVjcO87k/Vo6NC6uHQiDmpoPPP/hm2LkfNQYUkt19YXPjVcEPMhhi5NlnjKGve03ot1XxzjN9uYKJhDhnyBciKvNN0pbLRhW6VNZe6fBLAAVwVqkvJhnwpbKX1ciO+xMVjEZ4ddRbs2yQu7gh/xb+7s/gye8I3X1hnrd2GCoLweqWVoF3SGVd+POvyvkxxoLYqd8Wx3m6yr3ivUV9cGiz0zXMc0J3X4lmwzJ3tY7Joi4rZjbSsa8rBG6ZDogItbkGaj0iklWHkCnSFlRd1iEqiAhpWifVBmu4Fc15xr/yLUplYectyvx02OdbOxz2GLynxCkKQCu+DrHTkS0fKc4nE4OlYhfFXFG886FP13DC16aqzFwJFzxFcYSR9j5JBrq9ACor5yXLeG+MUFto0arZDmWkXRmWpz/ZQvhMib0ltXVEIix1SukNrM9XeeCLjzM3Ley7XTv7gn1rQD29FUsfsHhddT6JFnsU7cvFeYxEot4t2VKUyAhRLMSxBHvaVvqM87pC7HRZCofXRIRWNaU+3wo8XJbm4pfSXhSkTRMUwZDaBta1QqYAKTUKdiurc/v43jeV7z8sfmAIGjWkfwgfReTqCRuuh/MhVF5KqC/EcYHIZLW7fRI+s6t+2YEuG168HHBb3NSHiCd1S3WmufSezmJlH3crtaEtoKJCK6mg2WaMqsd5j5omcW0bPW4HTz2i5tg3Rbt6odyFL3ZB0gz7E68C/t6A3ZBXiCOTvd2/xKR4fRmwJYBt/Mtey8BEIjQXUyrTjZDGK4CF340RCqV4GZWWe39LK10EERTFqcWrCxlgm0QLuynawUv3/63KqafFD6wNZsemHLluexsZ70FCMfPLurFldVtfVsuX0r+9Yawus7OZ+FXnWiHiyyyuqIZv8oqg5AsRudiEEdeyhRUMSVojcfXMFXusWjwuZEAKSKrlxTvfHsV85cEvYuZn0IFBcI43jY6SfxXwYY9XvE0FnLMWVY8YWcZZ30lRXVa/1Wclzi+9x2R7FGnTsjDVoDHf6vTtHS/gl5c4wQh451HrV3oFhXoyh+Lx4lE8zieoKNZ6Jz5SRI89ev7vnxsa5iONmr7w7X8kZ3K4OGbHyUvsfbXIhz0tFy1KJNVWq8nMlQVa1TQIlQmlZqV9XUmBtgsUIG1YKtNNFqca2KbFIMtMUBbxZQugXmlVU5rVFO/8kggipLZBI10AURSHx2E1NFO2pWHZ1HweMONflvnuEu+bv8bCxZNIvkCUWA5dV2MzeD5fWRg2c6lLVteqdfUtkSgWCsUchUKOOB8RGxBjVoxnvVesVVzLkzYcNgkZICYbUHbqt3bmdpIpebv1dU5XlLswFDHUWjM4Erx4PB6nFqcpTr36lEg1rUZx/NeAP3RIc8ce4fn9R/iATfmqifAGbrmOyN9rxhhzxshZ5xNSW1MjBps6GosJC9N15idrzF2pMTdZY2GqzsKVOvOTdRYm6yxONajPtbBNh6gSSUSk0QotaIubLFNxwWSuT7NSp5nFNSS2TjWdASFwXDypb+HFYlveGWIB/dJj5++7MspodOwY6cgI8feP8k+ifCKKMAq7zPX28kL8rPOOSmNOJZuqSmefUfGpxyWWtOmwTYtLHeoyActqptGIWjLHTGMCo2Yp1duCly1CYhs00kW8T5G2wHbqu7DYmsLSwovDY/HYAN57bEuNV+ud549XtOXj2JER4mce438nLT4jyh3XPcMT4Sgos/VJ6YicX17OllKzLQNLdXnpvRExeQqI16VIZwug3lFL56jbRVq2SiOtZJUl1HDB0EjnqdprIOCw+CzqqbZIms4ZjY2iX3r68v1PjTIajTHmli2AY5ToxGN8QoXPxtc7yIjVfMeqNuYbU6XE1jWWnLQ9uqywq0sefIVX0iBMJenCxN0451ZcMC8KDbeI9Qkh6aPAY28Rae/rO2aSizixqPhO1JuujrVWbRME14q8+y1AxtirL6Nx2GTlxPf4leuJvId7zdGJr14yJvpu01Z1tnrZR8ShhOmyzu2lLaxf5tiy/6v3eGcxHZ4r4gXn0w7wcEWex0i4TkfVIxoxk0zQ8IuoKI4UFUfL1Ul9i1bNu1hykaJ/8NjlB0+GOcQn/StUMHnNM7wI81kBubhwMpvJrRxW6Cu0rNIBmXl1L53GpV3nOx2chlF3RETZdGelLaLqZpi1E2ACcC+OxLdouhqteurExbHV9PHu7a3fz9Ldv1oJN9c5w3OAlKPG34KcnW1cMtdql3wsuU5vrysywC8zOdnAMhMuUWG2eYVqc45ITSdTwmI6vHpi8pRNTyYjhqavMZmeRNsCJw6rKQ1XodVMvW2KUdyiV//B8fFxm6W7Xs99MNfz0BFGovHz401D9LsGIydnHvcuuxtA22PoZQPJl25ItMXPe4dRQ8EUUfV4DSmOGiAiL2VKUTc+O3erCRPps6Q0Qj0Xh9OUhl2k2WxpUvdqjBHUffCpSw+cGmU0eoV053Vfcn6e8/5e7jV/tfhnTw337HxH01a3qKobKm8xgaumo/ANVyEiDuWs09EvucBS1EVkYhwOzfitQCyFzvMgWCzn7dM0dCHIaga87irUmw1t1ZyPTByppr947NIDfz3CSHwf9zlewx1Q1/34ZHuy7vVjsYlrZ+afkouVF3zelHDeoiipb5G4BqlvBWDqsyP87jSkrNUUpw6nHpfZUw0VO/TptDjrjlHVzMyII9UWNbtItVrzrZojkihK1X782MQDf9a+UuS14HmNNxuM6yij0QOVr0wP9W57MSf50Su1M76UWyX9hSFxGhofqymxyRMRd1r0pZmMLvvnUdFO9D1KRJ6qzvKiO0adBSKJURypb1JLKtSqTesSIoFEsb/w5MT9/+f1AH9dd1qc4ISOMBIfXfzG8fU922cjE7/7cu1FUPWri+tMLHmMmA5w6cCl49m1U8x8B7Qhxosyqac5p09iaRFJjCWhkdap1Cu+Xk288XGsoqe9t+974tKDX3u9wF/3bSbnOe9HGIkfWXzw6FDPDRdiid99rXkpnmlO2mLczapcnwTwvrND22a27/ym2ZV4MYoyxyRn9BjTnMVIOK2WrVNtVHyt0vAuITJijKr78wY6evzSA6d+GOA/9A1G7S8/uOFtRyKT/4zDHVD1rC4Muw3lHTpQGDLFqGwiiTPHJx0KOCwtaizIFa5xnqrMIGowLiZJE201E580neCMMWLw+KdU/G8/ceHr/5htpqywrv/i4JefxN69e/Plyg0fR+0vO/x2gIIpUo576cr1unK+TJQzosZJahpSlwXqOkdLGyo2wtic2sSrTbxRK2KI2vR4GuV/MjH9f49xLF1mYPSHPfcf0X1193as5JGNoyWV+tst/r3e27s9bpt0rjdtW7owBDEaIWqyvflAgowiE4I8aNAvPDZxx9fbf/tHEe0f143EMsJItJyDbx/6+a5KYeFWj32jUw6gugdlSFX7EM1nlb9m4BoiL2LkmDj5disyR49f+OrcS+jlfhTRXv74/4tz+Dibo2kwAAAAAElFTkSuQmCC', 'fruits/raspberry.png': 'iVBORw0KGgoAAAANSUhEUgAAADEAAABACAYAAACz4p94AAATIElEQVR42s2ae5RdVX3HP3ufc+5j7rwzmUyekwckmUkIj0RcBWV4+EBAQOgACsjCLmS1YGttratam6YurVXXQoXlsmC1IrQmE1MFq8FUyCAREfIgj5k8Jo9J5n1nJvO4M3Pvuefs3T/O7yaXGDAJKL1rnXXundl7/17f3/f32/sceHs+GmD1OmJ19XyjZjZ9tXPITZ9FT+0c1i5ewdVns5h6GwxQAPfcQ/xnm9iw5BI+cPl1EItDugcO7YF927F5n++sWMInNm7El3n2/4sRimb0umZ44JP85Pp7uP6ez5DPTeL2dqLyPrakDDM6hHryIXTnPv7rsku4u6URyxrM6y3q/lFNaMJR6wnu38y/3/6XXH/vZ8nv3Yp3YOcJjyo/h3PFjQTvuJLwwA6uq2okzhomxeH27Y1EEy6tBMkUn7z5Ph76q6+S3/NbvCPtEEtEmjgO1osRDnTjbloLg938WddhvkszDi2EbzecHCBMJGhadQ3P/fMThF0HcQ7uQrkxCHzQmnDsOM7+V6H9FXoy4/x93xEeFxIwb3diK0BVVFBRPYtXv/Akc1PlmB0voE0IQ30w0EXQcwS37xj58REedhJ8ua+D9JkY8FbkhBacvi5z0IxW6wnx+GbzA8ydWU+w5ee4+Rx0HcQe3IMd6MLNTtKqHD6V7mJbcfTORIk3a8Qbe0mwXFrJDRddzl3vvYNgx69wsxk41oHd9RLq+ADK8Vh9dD9fUAoruROeqQFvBk4aMPOXc1tM8/L+nRw5DZcrQK14L8l0B7s+/xjzSyuxe7ei0z3YXS9ie44QxEq4s2MH6wsF8Ezgc9rKedZE0xTNU5Zr8iH3ArapCedUOgXMka389RU3sGDOeYQHd6Ezo9juw5j+LpQX57aOHaxfuRJPHGDORZ9zglNrbeRxBc8ozSMrV/KF1lpMEZdrWgnrG6lT8On33YE5tAdnagKGBzC9h3H8HJ/tPshPGhuJbd1K0NyMHhiIkNHaijkbg84pErREAnSO/00kqT2e5cu0EK5cGTmluTky5ngfn7riBspLyjBDvaiJMcJ0N3pogJ111/C15tXE2tvxAdPSQtjaStDaSnC2ETlnil29Gr1mDeaSK9nkOLzncDu3D/awrrGRWFsb/qylTEsoOv72YSomxmB4AHXsAOGul3BG09x0ZB9PyVLJWIx5l17LXGVZ3n2IWhQHXJ+n9+9n6Pf1TW+KnTZvjuh1dJDfNLyDq7OT/EBDtq2NpwA93stHL7+TylQ5Qd8xXD9LONSPM9THlq4OfnXRu7mnfgnX181lZbKUhZPjsG8HbWGeR7w4m/bvZ/hMDHjTFLsa1CNpnhsZ4B8aVuJZywblcGf/MdaWTePelVdhB3tRWsHYcXTPYUimSN79d2xf2EB9VQ30dMKrv+Zgx3Y+e7CddX+MxFZN4LQSYbcVoJ9nu46yo2YWFzauQh2I8cMgzzWLlrGkpg7VdRDHGBjsRflZ+NB9XFI3D0aHyT/1BOx7ice7j/AZYOgEuC1YUCqC+9lX7GZwBkDVgm2UULaBGgDVCkErBAB3Qe24x4xUHmdXPy2H2rmw4WLskktgcpz7GlZCdjJaM+/D8TTMXwp18zDpXsKNj+ON7+bR7lHuL9T9eyyJLEx3IVDQC9im0+jYGhVCezoj1GpQa96gUjZDnYIHget8WBjLUxEoWJYm2OOiDlfgLF6KTVVgGlbhjI+AluqhNSxoBBOif/1LVGwb5t1T3HaR5qAxtHtw1xS808L0AMI74AUL96+FY6+ji9MiuqoilrIAt8O7A/hgCI0KKjwIXEiH0GHgI0moL/QEAVHMYxamFGydC9ULIfThypuhsgYSJWAtHN0Pi1dATw9s+go0doJR4ETQwZH1Co1YDMjCAQsbFNTKkLSC7Vl45qcwuBr0mmiZyJBmKFfwqIbbyoEyIAEMAgNSUFQkKJiKHKwqQeWAEQX+XIhdAG4l+JNwrAPq6iNjug9F3WoyBXt3w7TNMBuYBKssVta2FpQF5QAeGA8ct8jLIZAHpqA/hM//CB5rBketjvTTe2BTCq6shXAa2ByovcB0x+GyRMIujcUod12dxerjQciLY1P8WGdRK2D2FTC3EabXwGgaujrA5GHLRljUAJe+H6rrIDMKh9qg4xlIbIcZQF5DzkTKeUW9twZKwFSCsRKlQl/jg+tHkXpgPXyrEIUHYvDILPCnQ2wcaAduLS3l1tJSPMeJYpwZhRCOxhJ8dV6G/vdlWX4xpBLgZ2EqA4PdMJqBrgFofwmWXwnXXA9VJVBZAYkyOD4Omx6Hrn+DWXmYG3NpiieZ73qUOopJC4fzebbmchzwfaZZSwwYk4hoMDkgA3kXLlIfBy8NO2phaX2ER/0qcHdZGTdUVGDyeYZ9n3zeZ2ZmgrUufP8WWHYDzC2D0eMwNAQjw5AehN4hMGnNDd0lvD8fZ5OTZVtlnvEFeSoWWhbUw/kLYeZ8eGEL9H/R5XujNcQrXAgNvp8n7ftkgoBJY3gll2OzMcwCKgTa2Sgq4QQ4E/CYugNW+LBtETjTJQccz+NLtbX0ZzL0ZDJMhSEp4LkaeOVBWLoMug7BwcNwvB+CQfBHoHpcc0dYyrUTivNyBuJxjLUczeXYFubYnMizsw7GFkH9Emhqighh8l8S3P9qkj4mOTyVizhcIFUmOm0EzgNSQD8wATYARuGYuhVuisGPF4CZB3oX8KdVVVwUhuwaG8MByoGnZsP2j4OdgqMvwIxjMG8cGgKXCxOlfGdojM+VV7HYcWAwTYglALYKBEqAeCSUnTH47wrYOxMueh+c9ycw/jBcvhnKJMMLn1AU3w7sBxoluQeA8QhieddEFE5MBFmtmRGGHBwbw1VQZuGlSnj6PWA3w4pX4O4JmG+iIlNKwFPjY1yRTLA4lWRiIoNHhOFh8aYDZBVMElHqu3xYkYZvjEDJEdizG6Z9BLZk4LpXIK8imiqwkk/EZm0CJU8cmxFWc104mgMbA50Tjs5kMhgxLO3AQ+fBtJfgwX2wwsIUEQyUKLoLwxdLSwmDgI7MBFa8Pl60CVf2JLscF5gsyUNDHmIb4clO6LoCzt8HS8ej9bU92XckT9YOYpEhVoMy0KeBNgt7e4HnwYTGYI3BKkhZeGwW1A7B1/bCUgtDCnKiVMJCj4U5Xozpnkf3+DiZfB4fGDmDTUwFsAVIKbi3HS5YD3vKX39/oF678TYmctAW3QJTGp4fjBLIBFExodTC9iQMevCvnZEXM4BrT5Z3R5Sd5XkQhvRPTeGKsN/XWVoxJgbkpeJfOwRNfZF8bU+OU5IHoczxokKpMpEOD+vbYRFwqwU+nEo5VikOyuQ9KfjzboibyPvO63g1BEw+j2/MGZzhnHTCmCikJLKTCirD342EEmj6QCeQhnA4gv9DLfCiduATPtRcXVIS3l1WpkJZfLeGlRlYkIOJ0xhQCG0VMBSGZILgrE4npoRhKosM1haC04zNR4oXZIaHwBmGp38Mf9MMjs7DtTGwHygpUShFpeMwCkwZUNloUfc0ni2wRh2QzufpCYITRxa/D0YO0CXOmsFrFVenbOeUGDsk7LkMlMiZ2wyJFjA6B3PqHEfVOY5GKRZ7Hl2C1V7gkHjcLVq8cBUiMS0MeT6bJcXJRt+eorgt6v2HgL3C/1VFRpw6DjF2VHKvNJqjq6Pm8CINVwJWh5CochxcrcEYlgQBAzIxIRS6V8JpxIuu3HURrp8NQ4aFWov/r+V3IZpdch0CFoiM4jGOjBuRMVLQyALVEv0KMA5YA8sLBk8pa0sBAmOYE4YslUp7hSjlA0fke0qiVMD1LhEWA9YDV8uiyaI8CqI2gWFR8KgouV/qRVLGhCJrSpQuOKETmCkyAnGaAyqMpuJqODZqTAPWGkArCdsu4KUIg5QWMco4EVMNAodF6BIZ81vgBZnjF2G6cBJWIvN2y5wRYBOwWGpGWESphbb8QESnTIkB+qTB1okAghuHF9Nh2NAXBHaG59GmNS8awwKBVKsISIoihQUh2m7NLtoDXADslN9Li/6uipJ5KzBLjE5I1DqAabJeXMZPSfRzwDul5Tgs62bA8aNN5XYAtwSeGIKP/SabVTfHYmxRiunA+cIKE6J4r0BpvihQIQoY8dqECG8QQ9LAPOlxQqBbYDFHlPWFXl2JUK/sYQpr5mXuxXJfJOuOgRmN1jzkwDZAud+HzTfCc89PTV21Kh4POoPAvQRYKN4bE69PSNhXiVGmKPSIYcPiuYQo0V40Li7zy8SAmULPo6J4hRg5WZTwhRqSEypOAAchVOA58PUW8JvA1Sqy9MFjYTj50MiIG1prZ8vCdRKReaJoiQjxC1X6lHuNwCKQe6MotFiupIyrF6XyYtQCkeFLlOaLAYVkrxQUqKhf8nx4dgC+vRp0K4S6GfST0KbhlqNheNyC9cCmiva1MVE0UXRgUHxx8hCBGhE4KZuY2aJYGZyAadUpSeoJG43KmEkxMC6/K6IcMTpipGct3NIK4RopLboFwpXgrYNnPPicCzoLYUqUDosU9yUa4RtUY08St1v+1igRrRcIxWW+KjKkHHhFjK0SFFSLAQkZNxjNUyH8UwuMNp0sKVFXvBDMatAuHAgj6tNKwljoSEtlW1gpC5vXaTHywEqJxs9FwPSiGpAvotwygdELsvblEslakVeI8BDYoci5Y+VR7VWtRb7U4i27Jlp3v4FcF+ggghXVAqmZQnFTIqS8qAoXLlcUmwt8WBJyrfxvoUSoRq56cchPpR5dJXPDkycaIKw1COF4tM7G70G6+eRe68QTSlrBrgb9TRi5AK7NwJx5YKtBF3ZVyaL9coPcE0VXSrBbLl7uFEhNCTXmZY24wOMQ8LRU72skmf2T50qMAD1RrtghsAPRgd29e6BnGei2U40ofO8EsyLa+9zWBXYx6FIxuU7uraL0HFEoWXQI4Eu0fiHVu7qIrfZIwXpVnNFWxE7VsuEakeZwWDoDJ7qH/dFh2bfWy4lfyylpqU553qDXALfBBgM3VUL4LnBmFBWhp6SSzhNIxEWZ40CfKJIUyBXqyHKl2A441p44b00U9vAyp04MThSRxDCYPtA+/HImXN8HQctp0lGd7qT5NrjQgR0ZsBOgCvjXRQwUE8WDom1mqXjWk6jEBXqzHYfngWQYMn5KLmkppAOSD85Jx5h81FocLoFl349Y+LQvqZx2K5yE5AhwWTxOYzzO1kyG0Bj6gKWex9x8vhBqjgsUcmJQ4Rx1mkQqBUxozXgQsALYV9QFUFTt5xXad63ZCxhjSIA6BhNvZMDvGLFOjsmroDMNWRfiN1ZU2Jm+r3qzWbqAfUHAYvFeufB64cA3KEo0T5K6ynV5GSi3ljKJzIgUtAI5lIv3HWB2LMY+32e25IeN0KrO+BGwAtsMztehNwE/2u37atz3wxnxOMNCnTlr6XBd5nkeoVJkJQqmqP8v9DvzYjGynsfefJ5lRXuEQmtRL99dyav6ZJKt1jJpDHEwI1GV7pAnRs4ZP8dukXysgU+nre1/YmzMnZdKhbM9j1HgXcC2MGTAdbm4pIS5iQS1sRhVnkeF61LlecyMx7kwlcK4Lv+RzXKB1sxynBP7Bb+o8KEU5Z7HspISMkrx61yOejCdUU5oBzacyXtIv9M9NIPzAxhbBS+3B8FdnrXOdRUVZjIIdBCGzAB+EQTEtWax51HpupQ7DlWuS43nEdeal4OA9dksLjCoFMvicRbH48RclzLXPWFwtecxMxYjA3w3myVnLTlQE+Dk4X+Af2wD1ck5vLRVxFS3TsHai2Mx57JEwuR834zkcnRY6+wAtdRxWOG61GiNAbrCkJ1BQL8xVEtijwBZpbglHmeF677mkagDHAhD1mezjFiLGz1zafPgRwF8tUUCds4vMjaB2wrBR+CqcfgKsKpcKYzwPYL9wql34RQ7KS3FLMHrMUnyYWCR47DcdalWCh9oDwK2BQEVEOjoPOmpDXDzW/YcuxWCZnD+E55T8I574LIJa1dpmG3gY6VQ3QjqKKgSUTghjDWjqIGMSxsSA46FIXvD8ITgmFR/oh5JOfCsODDRGvnIviXvdhSeUp4Ct/sS8Ohs8EfB06CWSbFLFdEuRdvXtNwrtWantTjWUhUluemPopXxYPEPo9p3Rq9EvF5iny4iFlDN0dMk51JwW+Dl86EhBxcGoPIQrlJK6ej7iZajcNxZeKawSGscx7E7w9DWgMmCGYyOYrWFB9fCr5rBaTuLN22cs8FeG9hOMCJAL4cNIZSEsHwKkq7jqGWua4OoXXgNjyulSDkOJa5rfhoEOhdtcPQ46HwUtdUt8PXTNXh/yNesT7QBd8KcAC7NwXdvdt2K5Y4TjlnrGGtP4CGhFC6EPwsC51Vj+uMwHMKgCy8DP1wHL58Otn/oFxkLENNPyunkhyDYGAQ/yYNzgdYhWjtygGyGjDHPh6G725jdc+G902GgWOFzNeCtfC9WfRzcRyF/E9zqwKPnaV29QGviQL+1tIchI/DMLPjot6PEVcJCzpVgztWAt/zTLDl2OzTcCGtvhOGbYOqDsO1D8BeqyOtvpeD/A7ac1sdU3ofYAAAAAElFTkSuQmCC', 'fruits/red-apple.png': 'iVBORw0KGgoAAAANSUhEUgAAADIAAABACAYAAABY1SR7AAAUQElEQVR42sVaaZBcV3X+zr1v632W1sxoJFmStcuSbMkWsvEyuDBgIARjGBMCFCQmK0U5VGWhigLFhFBJpUglRUGKxT9CCAYPFHbAC5UIPInBxtZiy9JoZMmWZkaapWef7n7d/d679+RH3x4/tWYsG9nwqqZmquf1vfc75zvL/e4FfnsP9QISAPX2QoJBL/9ro3vFlbm37trX8s83vb399A23tr4ZAHp7IZcbzPptgQDAfYACgL4+KBDQ1dV+Xdd6vqttxdTvrFqX2LZuUwbPP1PGyef9S67TWmqC5s/27wcNDIAKhbrVOjrAfX3QS7z7qkHkgJYdaN0RbM+enrEr17dlo3u27dVvacs7uOb6drS2pXHuTC08dbxqqVr4moHwEpj43nuXWfB+iJ7HIfr7oV4NqP2A+FuAd8Bbc/O2/M/Xdjkbjp6cnXd/F7nb7rawsqMTA89QFJQdMThcEUQsBcUp9wpAenpg9fcjuuG2/IcsG59mrYWKdALMRBKRVlQNA5TCQEzVKjQS1sTpVIqPh5ZzfPDe0el+QJuxBF7+e8nncUAQEN21Knffm69p31AungnbUc5t3blWd3a08/SII5PpqnXkyTmsWushnbMQhoyIqHZJIB0ddUsGVTWcyTp7b749j641NhIpjdHhCsIwwJqNGqVSgHIxwtRYhBMHLcxP16a9fflDVR8/aNnqfeepvnOVZagJAOgBrH4gui3Tcs/1O9vfVqtOhC8cX7DnVq/ktl2dYnpEI5XVGDoTYOJcDZZNmBwPRGkhguVwEQC2b1/e63JgAAyAxoar51d0ux9ctyndRpCqtT0Nz3HZL9p6w5ZOzubSevW6lN7xJqGDIMD7P1lLbd3DGyxLvmf8ZPje1RsTj59/qTJpPMPNlPo2oK5G6qobd654YP0aplPHhqXf3knuW9PU0iHQ1e1h4HARU4UAm3akkUxJbmmz6cWBcmX0heo/LixExf7+5Q0lAKCnBxKAqhTVTyfHakJKxrFDc2LoRV/45UiqiOTClGUVhjPWyPFV1uRwq9D+Kt59o6X+4kvl4P1/wjuCEv3gj78OOxZgi8E2ABADYsfa7H27tqS9seFhjBaTdN3frEVLl0R5TmHg2SJGzlSwfnMSAOB6gsOQUauo8XPnKoWLY3gJIA16sRJ9wy/6AEi0tNuYmqjhxJEiXhoswU0SUhmGk1DwSxEmzubo7JFNcvx8p7N5dxC2dYrtz34v/1EA2hgGANALiD5AvSPT9qm9V7XtC2qT0dmzkbzyY+ux6mobrpQ4/Mt5TJyv4tqbWlDxFSyb4CYkV8qK/ZI6DiA0NeSVgfT1QQGgLVfOPjlTqA2ceK5IQ6d9TUTYsC2FidEa/u+xKZweKCOdtZFISsxO+bBdwvxYN0YHV8lrb61xMsmfW7lyZbK/H9p4RTzA0FuQ6N68Pn1v5wrWp45PSH9dB7a9LwNdBUqlCC8NluElJJ7un8VLg2UMnfJx7kyFh1/0qRboAwDQSP2vCKRBr74+qKrP35idCmnT9rRetzmJztUuVq72cOXWFIZOVfDj/xyDX1IojAVIpASIFMpzSZHJpvTet9K6ZK5yT8Mrvb0gIvC2FenPX7O1JVc4P6anOE3Y5cG1gINPzOH44QVcd0sLZiYDpLMSuVYbLe02g2G9NFBm18ZDAGCMs+yzSIGhoXrQd250B6sL+u7uNV7KS0gOa5rCUENKgY3b00hlLAw8u4CzL5SxostF5yoPQmo8/5Sifbe28OCzpTflu1Lfdp+olFNXATTgXrnvqvZvrenUdPjQhFj3BxvIWU0YOVnF0afncdcnVqG900U2ZyGds9CSs5BMCX3iWImHRqufOdw/93BPD6yhoXoXcEkgxivWr/prfusKR4PwjnWbkkpIIbQGJsdrOHZoAaPDFTiOQCJt4YWjJRx9eh7daxOYOF+ltRtyyrJFYvDIXOvBieDBgQHwjSvaPn/j7rYbCyOj6qRKyJ6/6saTj03h4BNzuL23E+WFCLOTAWxLgJnw4liND/xsjsafrJa3vOT82zG/dGpoCLoXkAOvECMXABkaAu/fD7EwvebQ6Nj8nZms1emXlBp5qSJUxGjvcNC1ykXXGg9bd2Ww56ZWsAaePDCD82erALO45Z15PXisfPWGite6PUjzFWuTn920RqZ/9dys0PtaSNkKAweL2Lkng6nRGsbHAkzMhHjuBR8vHi8hcWCaPvBMifJz0j3jWh/enE5vvyGReOK7vl/sAayhZYruRQHU2wu5vQ/8cG/7Jjmjn9x1bS63e2+OWBBFkUYU8eI3VcSwLIFjBxdw6ngJALB+cxLCJYwcKCNzhrBvXxs6kjX815EZzO9JIiiFuOVdeUQpC2dP+1CTAbJnK9gwVMHuiQDXlBSyACIAz7iu/n4uJ8aJhlq1/vB3Jyd/0SislwYCyD5AfTi74u3jXfh2sBEd3TvT2LDWo1xKwnMEokBj4nwNk+MBwlCjNW9j484MNICH7x/D/HwIckXUdVKK917XKWYnJvDTyRDBSoKlBLQmvGm4hKuKITpKGqESKJKFNCvkVQQHjBYG8gAmhIi+2tpqnbasajqK7npgevrHS4GhpdqIO/P5z1Zt+4u3FH3crMv8k5UOPd/tImy3IVpslCsKYaDR2uHAdQWiqkZ1OoQMGLUaQHMayUAhFzh45+48TpwcxtEJBx8tl/FDmcEH54q4Mazif5wEBl0HKVbYGtTQEUZgAJLrXs+hDqZMpP6ptVWedZwoGQTv6puZ+e+GwS8C0gDxvnz+c5Ftf+HdCwvqA+WyAEA2gElBOOEQpmyBiitQcgTsUMMLGJ5iZJVGOgQeTWbwnJdETiuwZHS0Oaj6VcyVLAjBYAKur1ZwzHGxNgxwk++jO4xQAcBEsJgvaMWzADoATAqhv9TeLuaFWEgotff7U1Mv7AfEvSZmKE6n3nz+g75tf+/dCwvR75fLsggQUx12hQEHQAqAZzicMRNWANTMxFNEeD7hoV1pJLWGCDUsQQgkYdSycNh1MWJZ+ND8PNaGIQIzRpUICoDHfNG+IgdgJYBnHEd9pa1NSq2ftfL5fRgYUH11ICz3A+JrAH+8tXXNrG0/enW1av9RsUhlUywbHaADIAnABjBkSRzxPDzjuhiWEjbX91gl8/7qMEJaKXhawwLDZg1Pa3RHEbZVq9jj+2jTGiUiFCwLhxMJVITAmiiCMEaiGGWqZn+wRSkxThSd9bxuWSqJH/j+gUZalh2AGAD0FZnMNzNEe/58bk4lmWUUK/sEwAWwIAS+nsvh/kwWk0IgGym0hhGk0qhyfcKq8Y4AUAZQM5auEqFKBG1yfmR+Z7VGhhmPp1J4KpnEpiBAmvkiMBXDhCujiH7heTokumGX5/X1VSpT+wEhBwD+vXx+d9my/rXH97mnWpXlWIFhs40sCYG/a2vDeSnx0bk53F4qYWMYIqv1BRMCQJvhdcV8nwBUhYBlxuLYu9qAubZaxVHPw8/SaeysVpFirjeAsV7qLIANzFQSQp/0PNtm7hzw/b4OQAjUJ/lkGqCbfF+H5kvKTMLGG/dnMigR4S+nprA5COADKMZiI549igAmgEX+A0BSa1jMFwsCZowQwEfm5iCZ8UAud4FhGsasAXgKwFsqFekopSOiO+/I5zf3AUr8WS7XWgPeszoIsFopuRh8ZiAHwLgQOOy6+Mj8PFL8ciyImFV1bGGhAcOvVLCaOtcIQIIZ71lYwEnXxXHXhRcbVwFoAXASgK8UXRMEOpJSWkQfAwAx4zi3shAdG4OAXTNfxSymAWTAcbAyihY90aBHw1stJk3GFy1eo7QijPG21WpYE4Y4mEwuUpBiXkkAOAZgTbUqqE6/9/YAloiYb7WI+Mow1A0LlpsmGLFtXFWtLn6mDMAO85MzcZG7lPpwiUcbw+yuVDBi2ygRLe6mGkAcs75iGApHa2iirW35/C6hhNjjaE0rlCJlXByXLCIAPhGuCEOE5rM0gE6TRdgsQJn0LC9TtYsAbKnVwAAWpLxgPBFjg6UUvChSQggJ4CahgCsSWiOrFLEB0eB8w902M9qiCIFxbVsMAJagFF8mkHwUoUUp+HShqEXGUNrUs0wUgYkAor1CEeU8ZnjMpGPeaHRkARGyWsM1GaflEothXN4TGU93RRFUDAg3xR4BSCpFur6uLYKZLVdr2IYe8fTLABQRWlW9N3ONJXgZa+qmGvFan6oQ8IWAAtAVhkum67jBXMMiAroFiCwrlgJVbFGN6p6PIijDT7qENfVlBHpkFhgCyCu1yILlHpuZwAwmygkGLMF8QRFsAAkAeFqjXalLirtk3v91qSUAuMywmSEBtJhejZuUaL1EbSJmT6Cp6MQXEpjgEsyLFThaRhfVAPzLoJWxMBKm+3XrqRUVogvopGLzk3mXiaQgQCmiJbNQGEutZH7PmUFkrLoLAAsGOF1m+kVsPmJGjQg1E/SqqTNmA5KYlUXMtZDIUsukwmZX+gZgMpbTK+ZHXAYIBhCajRWbsRbpRASYJjKKrUURMYgIgG8RUA6IUiERMzPFU10jizUDDI1n4p/RZYKoCAFtwCS0XozZBuWEoXa8Iw6NR7i+w8BcYNzXXMx0UwfbXPhEzHK4TCDclMITWsMzMWObGK02AQmEqFueaFpIYKImBHwiXooawW/gQFEAcMxiHWN9mCQT38OXmtdmgBDziCDmkZAIC0KwtUTrHbwO1frVZqyk1otAguZ4MM1ifOtQkbLOIuYhYQGnIyJMSbnkyahqcudv4mGz6IZhhUkmfgxIBKAqZT0+iQYEMZ/QAMYti8Qyg1Z/w+fWjR2jHQMyH/OQAFCTElUhBDODhDghLCGOs9bRqGUJvYTlG9kiuMz0+mo9IU1GdJr29bNN+3dfSo6EINZ6jpUaFGkpX7SYz09YFmqAtpY5w55+pZPO19EbobF+Oqa0lM3WWcay27xlMdUlh1M/mpoaE98YG/Mt5mNzUmLasthpKoLaFL8yEapvoFcaO8CG5b3YnmiquaIDmLdtLeufHVo8sXKAJypCYMiy2G2yemMClxlTRLDeQK+wUV+yTfv4yZg3hNHIFmybBDOY+ecvAyF6XDHjpONIsUxD2GIs80ZksIaxFgyNWo0HLACFmODXiKF52+aKlFIpVSOtf7kIpK1QOCK0Hj7juuQTXRAnjRScaTSNMUHg9Y6PUUMpNyaVjse80Xhv2nG0qMfH0z+amTm3HxCiB7C+AtRc5kdnLIuHbVu7TXHSsFiaGYU3IE6kqdpTRg9otEDnYt5YTM1EKLgu2/UN1YONqyGiw4C1tf5eyExHPE8sV0/aAMwyv65B3+D9aEx5FyYFTzRpaBaAWdvmsmVJrVTNJvoRALwF0MIcltDW6eknSOuTL3gezQhxEb20mYQBTL5O9GpwvmgCOm2OLUKj8y6lUI56npZCgIEDfRMTZxpnJMIc8sh7gSjBfF9RSnredS+ilzZFKmus93oFPQE4bxbfRgTPgCg1xYYEUJQSE54HS2vSRN9s0Gox2PtN6+8K8e+s1PyhZFKWl+iGydCrBGDMqOt8mZlqxhRbmwgrjZHGmtSahueGkkmthRCR1ifPFQoPA6DGWWJjrdwDWN+ZmCi4zN+acRw66Hkq2SROa+ORBDPOmv20uIxrdBGA4YZqIiVCZpyKxUUcxIKUOOd5dX2a6F8OAWFPTNhcNLq5QEaa+ctC62J/KiUGidhtopfX4LHWOGHbcGM5nl+jN842BAvbRpYZg8vEhQRwOpXSWkoRKTVSJvoP4w211IUB7gXkD31/YVsiQbDt2yYAlQ8CkY15xDK9UAhgSkrMCoEVRuCjJg8up0I6JrWOm1Yctg3dpPQ33rcBTDgOBjMZlQCkYr7nkULhGXPkppe8+TBgLomd7e7+lVWp3Fl23c4wCLSjFGXNoA0BYhZARimcSCTgK4WIGZbxWPNi4rqtZUAMARBEOJVOI1+pQMd2hvG0HBLhcEuLElJakdb/++Dk5Kebj6axRDngAYAeO326JoT4QzDrU7mcHpKSB02atI1FG1/urtVwKpVCwZxbnDCWrsU654ZCWQYwaCjlAngum0UmDJGMCXHNgvXRbJZ9yyIoVRbMnwCA7Uuw+KJTgAET+I+WyyPbkkmfLOv2SduO2mo1uWD21YGhFwCktcacbWPecdAVhpg33po0GWnWZKVRk2bLRtE/ks0ikBI7i8UL9LBGm24DOJbJ8LlEQiUBWQM+/tDkZH8vIL+2hDK75HHGEKB7AOsx339iq+et1Y5z7bjjhO1BICvMmI+pJwpAPgxxKp2GxXyBvBrENK+GamkDOJzLoeB5uH52Fmi6INA4AzmeTvOZVCrKAHaFef+PC4Wv9gDWI1j6utOy5zJDLwf/Q9uSyZ1sWTvOuW6YVEq0KEWiSX1vC0Mcy2aRVAo5pRZ935CMnLoyyEdyOR73PLp+ZgYppS4QGJy6MoJns1k9kkxyGrAC5n94qFD4/HKXaS4JpBH8AOhEudy3NZlcL6XcM+J5KFmWTiklklov8j/FDKk1Dra0KACcrqvpZDY/PGvb+mguJ8YTCVrt+2pNtSqE8VCjgp/3PH42m1XzrisTgAi0/usHJye/0AvIRy4h9NNrkGT5jhUrPiOJvqiklKwUr6jVuC0MSQM8b9tctG0ZSYlICHhKobVahc3MJcuiouuiovW0wxwIKVciinQ+CCgXRRQRYdx19bzjiAQArfVIJMSfPjQ+/shSGerXBVK/Hw/QvYB+X3v7XiHE37MQb9PmVgOZVAqtAeafSub7Q6K7WYib6ybgORt4GLXaZ9l1K0Lrr0PKO8JYz2YDIK1LRHRfTYgv/mRsbOrVgsBr7fviPL2zo+MOwfxxTbQVgC+YDzLR939YKBxovP+hjo4bmDknwvDEd+fmhuJj3dnZeTcxf4qBbQT4DDykgC8/VCg8D7x80efVLu7/AUx2wmyJHcv8AAAAAElFTkSuQmCC', 'fruits/red-cherry.png': 'iVBORw0KGgoAAAANSUhEUgAAAEQAAABACAYAAACjgtGkAAAa2UlEQVR42s2cebxdVZXnv2vvc86d3/wyJyQhA5mBABKJPBRUilIpLB+gOJRapV1WV9mtbWt/PnZRqe7q+rRDt1a33VVqU9otDjwcEEQULXwQGUIShsxkIMlL3nt583t3PsPe/cc5NzxCAiYEqu7ncz7n3Zt7z9nrt9f6rd9ae58Ir+9LurrQvb2EANff1tZ0eI9/bVCx1xsjV4hikRIK1lIEnkPs/U5OfW/f1uK+5PeabqCHqHG95GzP2wBfNyi60Q1D1r2heWGtav5V6NvudFYt7pijmDFX09wmpNJCFMLA0YCRAcv4sClWpuw9bl79j/3PTG15GTukuxsZGopt6u3FAOZfIiDS3Y3q6SFaf11rc2U0/FwUmE80tammJWscVl7q2baZymhlxRhEBBHBBr61A32+rZa1Pn7IsvepgGrZ/iiTt1959onSZqWw0/3D2jNOgj0bYF5rQFRjMCsuzt8YhvZL2ZyzZPGKNItWOGG9anS9iqRSmlxB43qCaIN2DKmMwXENxWLdNjU7Jgy0fuT+Ogd3hBhrd5mIyFrrghigLsKE0jLmaHvM82Sb2+b0bv3leB9Adze654Uw++cBpDGI7m68XQfyXxAln5yzIMPSVbkwm3d04FuJQqjXI8rFiKBu8NKKfMEhlVGIAscxpLKGcrnCBcvgwA4TPfCDmlq62hHlgOMI1oAxUK9ZahVLpRiffd8WPY8fZ7P87bbHSnunh+zrDkhXF05vL+ElG1ouqJSCO92Uvqqt04sWL89JNq+V4wjaiZlQAFFCFFmqpYhaNfbuTE7jeQrft4S+ZXKyQnnK58j+kA98KmuCIKSpxcVLxeFirLVRgK2WLGNDVo7sj/TBnSFTk6aUTvPpHVtLX2+M6+XGrl8rMNa+oXVjtRw+0Dk7tWrdFU1hW2fKqZQjKU+FVKsRYQBKC46jsMYSBjEJBIFlcizgRH+d0lRIpRQyNRFQKKQoTsDUhM/qy1NirZWxkUDEpGXshCOlcUfVK45yPVEds0UtXqXthSucaGzYpseH7TvnLPSqWx71H+nqwjly5MycIq8FGKsua7qlVom+PX9xJnXJhuZIKaUFi9KCMVCrRBQnQybHQ6LQoLRgLdSqhkKzjsMmrUhlNLVKhDGWGXNSHNxT5pknJnnvn+fomCU8+VBAUMsxc06KMHyBVb1URKHNp3VGBGB+9p2aObwvctJ53vXs48V7X45TnPMNxtJVhQ9i7LfnL87YphbH1KqR1tpSLUdUK4Z6NSKK4vjP5hRuyiGb04SBpTgZMm9RBhPFxoWhpVKOaOv0iEKDlTomApOYYiLNYF+dphaF46h4hhX4dcXYYIZaOWTGfF+94wNp7v6Hmhnoi+64vCu7pqenMjSd8M97yHR3o++/n2j52vwfOq58b/navLlwRR4sanIspDwVEQYWz1M0tTi0dbq0dngUWhwyWY2XUpw4Xqetw0M5EIagtTA65JPJxd8Rt8LDPx9HicOK9S5eylIcFxwnRd+hCl5KU61EVEqGSjmkUgwZH7L0HwblRDJnkYkO77F5vyoXDA/6d3V3o3bvfqmgOx+AqN27MWsubV5vrb137sI07TNTplqOJJN1pLnVleY2l+ZWl2w+Sa0Sh4hJDB8fDggjS8cMjygE1xWKUyG1CrR1auZc6PPor0c5tDuk0OyxfJ2Ldg21sqapOUtxMgBg3qIM6awim3fI5DTZnMLzNMVxl87ZSmULUXT4ObN6/qL0Y//0YP1Adzf6VFD0+VC6F1+c7wywW1xXFYLQ6olhXxeatco3OeKllXE9iZSKk4ogonScLkXilDnQV2PW3HQSBkIUwdREnSWrhIUrA0aG6vzszinWbXQZH9QsWePgOBGiNGE9Rb7Z4djhGkoJ+SYHY2KgtRa8lJDOaKoll3mLlR0e9NXoCbPm2jf73+yJRdt5B8R2dLgFlN5jLN8sTYX31qvm8EBf3R49WG0+0V/PjA/7qjQVqnotEhNZOzHiG2ONamp2GDhWJ19w6Jjloh1Loc1Y5VaZvTCwqUxkDu4K5af/tyhtcyIufpPLvq2aRSs0qawhm3MpTTh4niIMLP1HarS0u3iewkSxpTZRsdZA4Lsq12Si488Hc8YmMzuHBuq7Ts06r6lSXb06N7Pq20tQchXKbtBKrXFcmSECuYJGO4pq2dDS7pDOYN2UATFSmjL4NaE4YSlPRQwPlbnpTzUtHQ73fd3jirc4LFsnFJo9+g9mMJFmbNhnYjQgCAzLVufRzktNswZcj+iJ34yp/iP+9ve9p3TFpk2xY57vLCNdXegZM7DTiqto587yCeCB5GDjxubWkbK5yHXt6v6+6hcLLaqp0KIYHQ6olhAQrMEXja9g0k2rQ7WaP3P2Ilk2f6m2e7ZGEtStHR+JxHE9HBe8jKFWcrDEqXl0yOf5fRU6Z3sEvkWpRqAmKdlTuqU9ZU4c99f/5L7mLph8aHoaPl+A2DMoQOnuRg0NIb29mM2bJ8eBx3LNvL21k8LV73HsjHmKR34SqV2P22Jzh/O+jtbMTj/wfdXWMrHtZwOVXBNPXnKtRjnYvU9GYmwkpQllXVcJQDYfUSvFnOH7hoXLsuzaVqT/aJ2OmS7Vik24yRIlhzHW5ApalaeiPwYeek10yJmAmiaAHMC0z+TPnBS3X3eLY+YuVjz+QMRTD0c1J8WNB571HzqAn3y9SusMNqQyXLZwpTLDx6w6cdQWm5p5YHKM7koJm8kh6VyE4xq0VtRrIVoLi5ZneX5fhUxOM2u+SxTYhO3AWHBd0VrDjq3Fd2zYkJvR01MeavChep26IRoIZy7kDVHEVzbcoKPFaxS7Ho/MY/dHyktx2+hxHgLcRDB5gPhVbpgxX5gxR8Kj+wzVCs8tWZ7+dLlo6nufCtGOWO0Y0vkQJRJ7QWjJ5TVzFqQ4tLdCeSrEJKVBGFqssdRrkXTO8sKmFqdpvCQ3NkK+UZ6fFVfcDqobdOO4Pb6GvFJqXv5GCuVJ7ly9QTlrN2o5fsiYR+6JHCyfHT7OjxIwgoTgAhGsKK6du1iBYI8fMgg8+st7xvs8j+/u2R7K4LEo8lJCthDgeHG6tsSGt8/0KDQ7HNpXiQchnOQREGrVqCGH3p2EvPldAZGG8YDdBKYHosaxKTbAAtIFTvdLU7kGov69fHHOQrlwww06rJetffjHkVOc4O6JEb6QhFMwPZXPWMQMN82aWQuFqTHrjA5YRPELQDL56PP1ih3u/amvAx+TLRjyzRFRyAv6JoKWdhethSMHq2hHTjaRRKBzdkbNX5wRi914eVfnrGQilHr5rl8MQsP4T0D+o7Dij+BNH4S3fQiu+Ris+TNoB2wvhD3EnJH81gHCztm8LZXm42/8fR3mm0U9+atIHXvO9i1ey59Y+5KaQgGURlmbbybfPkuisUGry5NU0k1sA+xTj1b70wU+euJoJL/+cd1oB9s+OwAbh0zDaIDZ89MnNYrrvQCKNciCCzNRruDkp8Yqb0nqMeW8TGjIJog+BOkQbrJw8wm43Ic500PExhaMdcO+FPR68NM74LEEGFnZRb7vWf7+smuVXbRSyfO7jH32t0ZlmvjoM71MNDzo1BCzhoub24R8C+GBZ6wOfA6MDnJCpNGoLt675or8Z/c/G/5XpQjffmtKt82yUqtCoTkWY44jVMoRF16UZffTJdJZTXunSxDE2cZLa9s+w7WT4+Hbge+eNmQSTrCbwNwK76/DU1X4bgX+oABzV4O8Bczvg/k9MBuBJdCWgg1F+NwIPHoLPHYzfNCCHN/J5zrnyqJL3qyjSsmy5cFI12t8c6iPBxMPOm0ZLpbVzR2C4wgTw5Yo5DmRGP/eXsKuLpwdW0pfSKX5T/ueCp37/l/dtM0KTDZviCLBWouXVtQqEUoJi5dn6T9ao1QM405b7CmqY2ZKlOKa669fkurtJXROBWMTmBuhJQdfD6HbF2EdRNdZy2oR1QJgX5Kd7Akwz4F9GPSzcKWGK9+V4lMhLFl/jbItbaK2PBhJ3357YuEq/v2OzagzgGFEAM3iprZ44MUJi4X90z2ot5cwFlTFv1x9eWHi0K7wy6MD8IbramEmXXD8msVx4v5LuRRRaNbMX5TmyIEqS1fm0I4QBVa1tLs2m9MLDh/vXwVsV6eCcRPMy0FvDbpbIPwP1pq/slZvdBzdYq2E1koVmAImEia0IDNBX6FxPq2Q/6wwsz2iLYp1y9ap3LJLlYwOWrvjt5E4ms/v2Mx44p32NBnJ/OAHaKWZnWuGwLdSLQFC36nIJT1bvfPJ4n9L53nX1ASDvfeWnb3PTkXGYtIZjdISp94ImttcZsxO8fxzceaxFrJ5HbW0u/i+bJwuzGQT2D+A9gz8sgIrVkPwGRG3aelSzIIF2FwOMzXFvq1bGS8WkcSiA0Ae2J6DOwrQFEGbjxrwoH2FmKvfqUU5mF2PR3pokM3VIt9skO2ZyPwzf0PB0bRm80Lgo+pVwHDidItSPT1EcXOqeO+qy9JXRJH7leeeLb97sK/OinWFMFfQenIslM45EPqWztkeYWA5cqDKwqVZlEJa212OP1+9Gvg7laRVBZCB79VgxUqR4D+C23TxxURXXYWaOROdyeDOn8/iq69GXBcP6AV+DvRp+EYzXPBGzer3O7R+UHPVnzi88zZHZQrI6ICVHVuMvbJG28cNC4Hw9tOnfAGYGqJJOeS9DAR1JPAtSjN5JgAb4bNra61v9/biH3pp3V0cD/ZsfWTC2b+zJJNjfujXjEmlhcC3tM1w0I5w7HAVpUXaOj0cT12+fv3srOoG1QPRzfApC29theCz1rqe1kQLFqBrNfB9sBZbrZJtb2fW/PmUgJ1KyAMnFEQuLJgtNOVg9QrFoqVC4IMW2L/bqtQIdmbAygl44APQvulkgjqNrA1IawfX9SAMEROCUlReTiIkJYIAaseWybtzC5ouU4p/Nz7s9w0P1J1nnphQg8drkZdRZuBojZa2ODiOH64p7WK9lMwv1UoXq544m8wR+KsKmD8Fpwl43hh8Y3hRUk86Ok2dnQiQtTAE5COYOQVPP2sI6/D8Pstwv0VpcDUcG7PcGKBuM4RFWB7A9xOtIqdTuVFESimUdgQbIcaC0ieLnJetnQDT3Y3edt9AZee24pfdJr0O+MzwQP3Atocn9JaHxlTfoartP1oNl6zM2TC0jA4FxvWUoOybG8T2FwHkrwSzHmRYhP50mnqxCI6DMSb2EGshyX1pYkFSBY4Y6C7BxF7D09sMaRcmR2HwiGVqCma1CA+68EYH5xoIanDdrfCZHoi6X+wlAhApUkrFtzq5RBmc5I5XXNhueEtXF86OzZPju7YXv7T0gqZ1jicfHjxWe3RqIpLjh2vOkw+Pi6OJXE+Z5lY3spZb9UegUIevR1D4CxFpBhlbtowL1q7l6I4dzFy0CPG8uKfgxT58YOtW/EqFNhH2AqNAwcAbQ3h03HJ43NLeKqRTQmkKWvPCU+OWo1PwWR/1CJgKbLwE7jQwtQrcThAXnDUgIxnm6RR/vOIKjdawe4uRepn/U6/Tf4bsdNpX0gmTri6cBx8s1Yf6/adHh4I7Zs33fhNG+GND/pyhfr+pNBnqajlSIjJDr4Dfq8PHFovwXmvVUDbLjDe9iWw+z5FnnmGkrw8vlcJEEcXhYQ5s2cLEyAgAzUBHkmkGiDNMVwAjRdh+zDIyZQmNUKtC/wnL2jG4LEBSYHZBSkH6Lrh3N0RHwIwl52or2YLmz1dcrnFT2D1PGinVuOOmKgM+OGNnuarfACZpKjPU7x8eGfDvW3lF9g6/xvZ6xYwbY0as4QeOhetDsJfFN9Hu6tV46TSmWkU5DmMjI0z09r5IXzuJL9eAi4D3Aj8DdgKdIVwzBZfUYc+4ZXBPRL+GywJYV4U+YBHomWCH4dYPw50lmOPD8jrMFchNDtNxaE5cb2gH67pCU2A7knLgZK3U80JheXa9mW50N9DTMzkO3JUcsW0GrlAicpG1YjIZcvPng7VMjYxQKZXwlDrJH05Cro3AVgmHzAc+BDwKbAMeB9rqsLQOKysxF5cNPA1cBgwnXFGGljo8UgA6Ez0zE9gH7PfA80CBiIe9oMo/Xg6/zcDdK+Gnm2Jt2AAmOqsmRg9Rz0s7era7G5wILkjFA5KgowM3k4Eo4siOHTH0DTKdBsSp+x3qSTPjbcDFwO7EqEEgtODZmIDXAvcBTwAFoAtYD3Y5mDlgHaDswg0OMmeJ0rkCGIOQh7kOHW8PufGf4MYtcOQW+N/z4H9+GcrnBMpLO3r09IC8G/ysiPtVa5ELLqD1uus4vHkzh/ftw30Rzb/yyyTAuAlIRcBPjN8P/AooAe8C3gHMm/bbCQVfzcK3HOAi4aZbHHQKvBQ8/YRh7/cj+606prMOD4HeDFRgbx7+7XfggVcByouV4bshyoP6O+CY5xG2tjJ54gRK5Azbcl5ZCDS2OTjJ8Yukk7sE+CSw8mQmhTHgMPDVAvxwgXDtlYq16xXiQhTG4aY1/PqnEdu3GW6qw9+OYQcs0d3g9AEp+Pz34G/OByh6JXzWAfcaYCSKmCqXY66w9pxXrmRaV/kB4DcJd/y1CPNFsCL0A0cSMu4DHk1B5zrFqjWakWFLeQpqZagUYWocZs0W5s0UfnHMsqqMXGhQi8EMgJmE69ZA013wQBc4R85hb9lJQFbDxyNovhRsBGJFEPvqNvUZIJsQ7IPAMhE2JWm6DOwBjifhdE8SYikXHvYtRELKE8RCFEDox0e5CM8dsWT6LUtLoC20gswH2R+vj29cCf798HA36N3nuDPRUbDPhwX7wF4IYq19Vct5khg4BDycAPNvrKXJdTEXXUSQTqMGB9F9fajk33cBqyrw1gF4ZiziYAs4BcFJxdcMaxBOWton4cYKPGNiXpob85NcDc4DEAr8l5vht3cloJxL+DgKNit46w6wi8/DDru+pFfyHDAiQre1LM1mia65Bt3ZSUsU0bJiBUMHDnD48ce5ylp6rGWnhbkleE8FKlMwpixllXibgTYD+RCet3HZsDLxMAssALkw9hQMfK0b1q/k5bdOvVzIlA18tBzzCanE5eUcwKgAh5JMsgfIiPBJEfIbN8Ls2UilAsZggoD87NkYY/AGB2kW4WiidoctRAZyERRCyIVxLI+aGOw5wM1A0zSh6MZ6SA3FP52lYM/XYMe58IlaDtt13De1j0OkzqZYOCW7OEkIjCWgrLKW2Z4H7e2oej1OFyIopbC1GrMWLcI6Dpday0eBG4DVyTVKwHhy9hJFfGsiAFuSDCXTJiMHtMWf2Qg+BnDNOZCrswnMzfAPafjGM4lmWJNUs8FZbBGwyUwtS4RZkAgx6/vYUglpb4/7KonynbZqRC0h3I2Jd/rTwqGxjOcl360nniGnWfzJgdIgAVx5G8zbBMcardGz8XSxcGcAzwNqJ5jtQH8y4+4p+uKVQJEkk7jAwiRjHdy+PQbATa6mFJJOM3jwIH4YokUIk5CrJ9dKxx08Usl1q8lx6gb3xj0TD5UURBrSQZzp2X2W0d/omFWBTzkgY3FuZw+wNckWJDPUWHg90x1S0xRqGmizFiPCaKnEvkceoVYuY4HA9+nbto3DO3eeLBTllGubxBPMtJlTp0xMQ/jZ5L4Sj9MmWugi4vGfFSBO0qTRd8FP3gN3pOAjRyDwwBXgqSSM2oHWJL690wzeT+L9SNI9doC0CBVrWXL55YTlMlvvu49MPk9QrVLz/VfceiCnee8k9zPAieR9R+KVMq0naWHWOemQuPjDdIMuwicULHNh434IFoHbntxsKjHWSQBxprlqlHBGlHhH43OTzLz2PDqWLmXixAn6+/pibzvLOkmSewwmpOonBWQq4Y/KS5NB6lylA8n6rX0A6nV4ZwSbPXAPQngYjCTx3JidxrrMZHKuJF6hpoVVkIDjiqAdBxsEtM+bh2487nCWatgmjahDCWmHCRG3JmOoJfedJj6qrwYQAHM7qHtgwsDbAvhWGpxhUHsgHIx3UZ70EG9aZesks6SSaXGTGRwAPGuRBkdofU41UsM7ysl9a8lnS5OKebixwjWNSyS+/asChE0JKD1QvQs+7MNtTlyMOsdiYMwhCIcgmgJTA1tPjjLY0ZiQQ5Nopn2NhlK1ilWKyPfPqcCwCdBzE2M7E8+YTBTxZDIpfgyIJLy2D2DGWUqql/BakrMlyT7fvRHuT8PHHfgIsKwIavKUynYa84sQbylIgX0m1gTYgQFkxQrKExOvqmCck4DR4K924OC0PkwJbABaQV3F+YCVZwnIGfep7gbbDfpHUN0Fv50H38hCr4FBAd/G+kWSpqYPjCs4YODnEfSmYMMQ2NUgMyYnOR4EjBw6RBSG51w82mnE6SRF4XhihMTayQTxn1vugv8OqN6zVKvyO225BN17SrHUHYdzs4K0xGKodGfS5ySuN47WYd5asH8dD+xFafFcAWlU03uTxpKTXLMcfxamwfHhX/fA17rA6T3LIu932sl85IUw0p2g/wj4XxDuhsoumNoJxR2JyLweUgfArAQnD287ANFiUAtFGJqmfM8FjIYn7AGOTruWxOCYIPbYURc+tgPqR86hJ/JqWx/cnpw3vVhJy/sh48MuAwsKYD8HajTJFuocvaKYgDGWvG9wx0AMUJgDpx6vCH7pXPsh8to8kRoP5t1wQxZ+NgXhYnDeM02jyGkGYE/jvjpJpceTEAmm6SE3AecARCnQIeyeBZe2QbDpdyu/XvtHzKYT8g/huaWQa4Y3HYegD/SSZP2FafK/cTQAaGSRCnAs4YvBU3RCA4xDYOJlYAIL7/hH6JsBavc59lXlNX1eN9lq0Q3fz8AtkxC0gLMeZF6iLZxpRoaJN5SS3UmTibbQ077TyDInYrCMjteExYf33g3ff7Wd99f6uV25HSQpwb+dhtvK8RZSOxP0rKRYnF7dRtN4Q5/iRTbhkf5YroepZCdSAB/+IXznXLLKP8uT3Q2K6Ia/cuB2iQuNyANpAdWSAOOcUkU3CsdGJT0Gthh/5GRiPnk+hI/8EH5zPsB4PZ/9b+x7NTfDmwW+6ML66IWGUOTFzR1xQfQLYNg6WD9WoKJBJZ28EPh6Hf7yJzB6vsB4ff8zhGnZpzuW1+8j7n1u0KCnN4TstMHpacxv4mRzj4G/74Ed57zY/S8FkNMZcCu8wcD1wFXJamenjSMoEpiUWINtB36l4Nffg5Fz3A7xO73+P/KPb6VpHi2DAAAAAElFTkSuQmCC', 'fruits/red-grape.png': 'iVBORw0KGgoAAAANSUhEUgAAAEMAAABACAYAAABBXsrdAAAdcUlEQVR42t2caXCd13nff2d53/fuF+sFQIAgSHERKYmruJi0TMWVYztWZMUx7XHiZNzMNM0yaSczmc60H6qq7YdOZzpJGid2k7iJUzt2TNmyVctWE6ciLSsSTYr7ToIASADEegHc/d3O6QdcQCQl2aJEp56cGcz98uLiPf/zLP/n/zwHgn+cJZqf9gCogxA/DikLPydht4AuC34MgyH8sAuOfEmIKtYC8CHIxfAbOfh4yvPuF1KmoyBYqMbxuTp8Owl//S0hbmAtT4F8Gsy7ecmfyDoAahPYpZfbD/owRD8Pjyfgv3a0tW1sa21Fuy7WGCLfpzg/z9zCwtWKtc9J+JyF1pQQf7njoYc2rV+/nnxLC67n4TcajI2Ocm1oiEvXrk2Xouhz6/v6/svvj47WlwD/aQJDIIQFOGCtat2xQ/7p8ePhR639ZN51v/rgli20FQoGIWxCKdKuiyMlURSp0sICo9evc+H69XojDMUnH3880dvXF8dxLFatXSuV1jTqdVtZWLBRFNmR4WF17OhRzo6MnEi57ie/EgRX3gkg4ifmEkLYJ639BPCrGtbFIC0MOUrt3b1rV7qtoyNuNBoq47pkXRdrLbb5BYlEwuTzefPXzzyj165Zw/bNm83k5KTUStGzciXZXI6JsTEyuRzthQKVhQU7MzkZvXb8uHPk3Lmbjus+djAIzt+ty+h7DcQBkF+z1jwpxBe6stlf61u1ikQ6TdJxuHDu3FqdStHW2WnrtZpKuS45z8NYC0IgAGMMUkpZ8X1ZrdftwMqVzM7OSsdxABi7fn0RbSGYn5tjZnKSgbVrRTqXcx7ctCmKo6jn2KVLz//zTGbX05XKLCB5m4DIexwj5EEh4o8K8e9Xd3T82vZ9+8IVAwNxf0+PWdPXZ4wQcWd3tzVxLISUJLRetoZlNIUgiiJOnzyJ6zhCSymEeN2AtdYorVFKobXG930unz/PwtwcQRjqbVu3hht6egamK5XPI4Q5cBfWf8/AeArkQYgft7Y/47r/dt3mzTFC6ASohJSyUq3Khu+rVDIprLUIwJES9WYvJQRCCGJj3gCWtfa2H6kUQghq1SoAURw7Gx56KM5p/bED1r73IMQHeNM/85MD4xBIhMDAR1d2dXleKmWFMSKhNdGi6QMQx4sxTQnBzUqF4YUFpLj98Iy1pBIJ6r5P1feRQrwRlNfRWdyIlIsARhGJdNr2dndbHz6DEEz9aOsQ+/ejAXFv3eTjH1cKtubyeauUIpVIkEwk8DyPlnye9tZWisUiSmviZpzQUnJycpKZWg19C2AtLS3EYchEsYiQcnnTy+7SjDG8CUhBFMmWjg4hYRvA4bfOKgKwhw8TAVbfKxf5NojDzzwTPSFENYoisTA3R7VcZtQYgiDAc10EMDw0xMDq1Uil8IyhPZUiMAZPa0wzoxhj6OrooH/FCgYHB1nR1YV3i3VYa5nz/eV0HDfdTgpBNYoI4hjHcRCQtMYsxRz5VPNlAQ4dQh4+TPTrv77DuXn1+u/KOPhTfS+I1dMQI4T5mLUfjOHhcxcvWj04KB3HwdUa13URUmIBqRRHXn2VPXv2ILSmWK+zOp8HIIxjPM8jDAJeO3OGYqnEbLnM2NQUKzo6cAFXKaZqNS7OzrKhvZ1Wz0NIiQT8OKbs+zieZ/xaTcYwJIRcxvBpgKdv80bxkUdHvtLZ5v7ifIlB8W6BaAaoh2L4g4zrvr/Q08OKnh562ttJeB5KKZSUeIkEUilia3nxpZe4duMGW7ZsodDVhQZSTbcauXGDI6dOkU6n2b1tGzfGxzl78SKPvv/9uEqRlBIpJcZaRhYWSGpNfy5HOQypBQEW0I6OX3vpB/JmqfSplqf2f6v295eyNTdwY6sSMo5dP5QJa6M+19G/s3VT9gMbVqfjF1+Z/aJ4N67xNJgDSn2AOP766lWrsivvuy9uzWZF3nVlbAzGGIQQ9K5aRSabXU6NjXqdV44c4ciJEwD09fXRu2IFU+PjnL1wgT3bt7P74Yfp7O6mVq3yZ3/xF1QaDfbu27d4xHGMoxSNOF4OvrExKClxEx7njp3mQnHYiL2FK8Q2LQUpIUlqJRwlhdZakEoo1q9O09vlRjNzkX7t7MIp8S6AsL/keasD3z+5aePG7MC6dZEJQ93iuovR/5YAp7RGSonjOORaWpBS0tPbS8P3efHQIc6cO8dcqYSNIj746KMU2ttJ5XJUKxWCeh2hFC+8+CKVRoOHd+wgl88TxbGx1gqsFUJKtFKEYcjFcxcohjfp+bkVSE8immnC0RJHC4TASoHVShgEdr4U6TCyYni0Pq/eCRgFkOeFMKuj6I/X9PTs2PDQQ2GjXncyjkPCcbB38HwTx8RxTNBoMF8s0qjX6ejqIpVO88D99yOs5fiZM+zdvZuV3d3UGw2qlQpRGC6H/HWrV1OtVjl15gzVWo2EUkILISJjaNRqjI2NcvLECYrxLIUP9iCzDpFvCENDvWEoVyOK8yEzc6GYLgZiciaQs/OhMgZRrcfx3ELkiXdYe9hfgIIU4tre97wnlW1pwUSRaE+l0G/CCe5kmMYYsrkcLe3tREHAV595hmoQ8DP795NsPiOEwDbTqbUWIQSe6zJZLNqXjxwRs6XScNpxGsba+2tRhATaslkS6QTFctkuJAIh72+9YD05LwwdYF0EnkA4gLFQARtLKbtdLTJhZEP9jig3xBE82JFIpFOZjImjSIomATJvAcCtDFIIQWlhgblikUQigRSC9vZ2YmOgyTXsLbxi6ffrjQadbW2kPS8a0/pXRBh+JqHUhgc3bLAbBgZkLpNBKYUQ0g4OXrMvv3zUVrGffl6Ia7/6vv5E3NdQqpxwIr9s0v2r6z09r9nzx9q6bKwfRdiZuwZjarE0x1rb6jqOlVJaKSVKKVzPw2laxtIG4jgmjKJlEJaWUur1OJLJULeWyFr8OCblOIvA3PI91lo817VDN26IG9PTw0n4T135/KMP7dhhV3Z0SGsMcRwTxxFSKrlzz05TKLRvOvjdF77zWKGw44uHR6q376S49DkGfPkdVa2FJhO2sBBGkcBaMVcsMj0zQ1ivY5t023Ecctks3Z2ddHV2oqTED4LbNmetJYoiVvT0cOriBYQUlBs+WimSjoO1FtPMSlJKirWaPXX2rDDQV8jn127fs8d0ZjIyDIJl4ISQGGMYvzEq79/0QLhvdGzDC6dP/y5S/ucdxjivQfQmdFwWClj1DtxEHAY2trdPR6XSb09NTCQmpqasq5Roz+VoyefJpFI4WlOqVrkyMsKVoSFcx6HQ3r5cm9gmIAnPZb5c4dSFi3jCJd/Swny1ytxCGT/0QSnQmoVGg9eOHRMTxWIp7brp7bt2mc5cTspm+l4CWQhBHMf09PVR6OkRSkounDu3xjfm8+cgvLO0AezICOb8+XdOx62Znf3VVCLh9a9ZY/r7+mR3LreYUq2lraODVDpNEATMzc1xeXCQH548ydjEBPt27iQKQ6SQuKkk10ZucPiVV8lvyhCumODi8Ayl+TLakwQLhiA0tLa1maBSsZPz838l4D3r163LteZyqFtizB2lLelsFtfzZHdfny20tg4UZmY2XhXi5FPWyqd58xiv75p6CxE/Ye3PdKRSn922e7dNZ7PkpCTwfUwz1VRrNQyQzefpSiTo7ulh25YtfPHLX+boiRPs2bXLFkslcfHsWc5fuUz3+zuoBNDSk2KiNseuBzaxsn0l1emI+YUSF4eG5ESpFFjIt2UyG7r6+rBxLJXn3aaQLdu9lEyOjeElEniua1qyWeXOzKwCTp7/ERWsumsXEcJugi9t37JlZaatzeg4lqklbtE011q1ysL8PMWpKRqNBn0DA7S0tdHZ2sJLr75KsVQSp0+f4UZ5gvueWEHU7jH7wiRjJ+dps+3c3/0AUcUipCCbyTCwciWd3d1qanx8Y/eKFaKzSeFVk5arO6xDCIHv+ySSSeIoslcuXZIjc3PfHJbyXKe1auQtlC99V6xTCPMBa1fnM5nduY4OoiBQ2UTiDSejtV52zHq1ys3RUTq7upicmrF13xeXBwfnwnUptWbPqpzT7nDpG2Ns3d7BY596hPhMjvniHEIJsBA2iVd3ayuu68a5fF4ZY9DNgu1muczWrq43pG/HcWhpbaU0P0+t0UBAGWuXE8C7EneWzMvAfflMRiul7HIKXSJHd6hRWIuQktHhYS6dPcPQ5SHRtT3Jz/7ehnQemUolpb0+UmvYqUZ9455VrE/vYmX/WqSSy18mhEBKSRgExHGsvEQC20zBWgg6Uqk7SQ3GGBKpFMYYirOzarZUMgauAWy6F2DcIp6I5eht7bKKJYV4S8FFaU0YRqxdO0C60sLR58fcrg1ZXa5GolaNE05KBu25NuKqtJV66Q2va61licsYY5BC0IhjxisV2pNJdFMeWKbHxtDS1kZlYcFMTUzY+Url2oYdOwaxlqfvBRhLiAoYKddqJo5jiRD4ccxgsciNUolLs7PUo+hNAQnDkI5CO+G0wM8Im9yY42YxHBNp/R/Ntjbn6Px1O10doTIxRxiFcIfZJzyPbCpFuVRCKYW1ltUtLSQdZ1ncWXpWOw7TExNMT06awatXRc3ab/3p8ePh/sWw8O7BeBoM1oo2GFwolc7XSyUcx4kja0kmErSm0yjHwXXd12PGLcJtJpPm4vkrjIlJVu0s2BuvzVI9NXOp/uzYMXlh4cSRk+Pic6e/Z85HY7hSL7+YbQbKyWqVMJ1m6ubNJutbdBVr7Rs0VGstURjaqakpeX54OMjD57CWR39My+Cussl+0N8RIh6wtk69/mRhxQpbKZfl/OwspWIRJwio1+sYY0h6Hp7nLfq8VlwdHOLlU6/h9rrMHZ8XXDOsTXSu7mlr+ZQp2/70bEQjqMvj0RAztsZqpwPJIrWXQlANAhKZDDevX0cpRUdHB77vExiDlhLddNUlcVhpHb36yit6olb7w2eF+OoBUH/yY8AQ70jdEiL+qLV/3+J570fr2HFdpbXGGEMYhkRRhKs1K7q66CgUuDY8bEbGxxooUi2JHBvWrWdg5QqyyaQt+765uDCv1GSRaPUYhexavr8wwQN0sDN5HzUbIBC4UnK9UmFyZobB06fZsmMHnYUCjUYDC3hK4TR1DaV1ePrECefs0NCFtV1dOzOTk/VmrLD3rKPWFHXix639rbZk8uGBdetMR6EgXc8DIVBCIK0lCAKK8/Ncv3HDXrhyRdThbMpRif7u3vWbHnrQdmTSwkYxvu8LD9SD+RbKUnHolcvM2Cvsf2QrQ2aGGMNSjyU0hs5EgvSKFbjWcvLEcdatv58Vvb24jsYAAZaFatUOXbjgDI2PjyVc98n/NjlZfWoxHNh71mtdsoift3ZfRyr1g4d37yaRydgoDIUAcq67bKpCCKRSGGC6WOSHx4/j1+s8+thj5LRe9M1b/NzEMYWebkDx1a8/w0BPD1Oywp5VW0ilPKIoIo5jqmHIgu+TSiYZHxvn6GvHaG1Jk0m3kPA86tW6nZ+bjWtB8KLyvH950PeH7qbfqu6WfT4Af7V9y5aBbHt75DcaiiYQpSAgoTWqWcLbOAZjyGcyrBsY4Pr4uK2Vy2Jtfz9RHL9Okpq8oHfVKvoH+rly6SJXR66z6lGXybFZpsZLSCXAdSk1Gk29QnDm1Ckeuv8+Nt1/H7VSnbrfiHWiLmdnq3/3DSE+eD6K5n8ijecl9rnf2oF8JrMv19Fhfd9XblPq9+OY2XqdlOPgKrWowTc364chWin+2SOPiO9+73vMzs2Rz+WIomiRvi+pCjMzCGuZX1hsD6BauTgyhH9TkE4lSGVzbNuyhSiOefWVV8ilEjy47gGMsfTu7uV6qSKGKiPoGwsFIiueAnG3Qyv6bbNPY4QjxKbWXM7RWpvI92VCa1a3tADQm82ihFgE4tbcvThzQT6Xo6uri5HRUXZs3kwULcoKxhiUUhS6u7l29Sozs7N87EM/y/nvXaYl7OIjv/wYmXSaZ7/9bV76wQ/wtKaztZXd27cTxiHWWCrVgDZHM11xCSPrNHsk9m6Tg3w7VlEGjZRWKiW1UliwUggcKRcjeJN9vtkyUYSSkqGREQTQaDQw5vUDi+OY7t5eIt/n1aNHaWtpobutnb27t/PhRx9h3fr1rBwY4F985jO0ZrN4SvEze/cugrlIh7EIkp5nnEbDBnARIThwl7ThR4JhQSy5xwtS+k+9733axLFbrdfBWrFMfZsl9JuNFlhrSaRStLa3U67VMNbiJRIkEgm01mityaTTBPU6L/3gB1wbHmbn1q3UGw2iKKZWrzPX7M1GccxHP/xhKvU6V4aGcJpKmAW0kszXaly8dk1I+JtlefIul36rzCEWW4b2I9bustZ+5uzhw4/1JBIrZ4tFatWq9JJJImNwlLqtKbwk5IZBQDKVYsODD5LJZGgEAX/xla/Q0d7O//7bv6VSrYIQtGSztLe2cvrSJfbu3Ekum12WB6WUjI2MMD0xQWd3N109PWzbvJnT58+zqq9vUb+zllCp6MyZM3q2Vntt2/7933zu8GF5+I3y3t1nk/2gvwPxgUSif20Ufa49lfrDXRs37nzv7t3tH/zAB7SUktdOnGDNffcRG4N3a5EkBFprHNeltaODXEsLHZ2dTE9N8X8PH2ZsYgJHSjra2li9ahUrenqIjOHk+fNox2H39u2LWegWCc9aS71WQ2lNvrUVLQRnzp8nnc3iJJM0rI3Onz+vL1+9WnYd5/E/vnZt8qlFadK+qwG3pWm8J+AJF/7soXXrCtu2b7dr162LldbSDwLRWSiI//7ZzzJbKrFl82YK+TwprQmjCKkUvf39JFMplFKEQcDFS5f41gsvYIBdW7Yw0NOD0+y4SylpaW2l2mjwla9/nfZcjv179+L7/m36xNI0z/oHHqBaKvHsc88hPM/et26dvXz6tBweG5tRWj/5tSh6+d2MPqpbXeM7ED8Bv5HV+kuP7NyZ3rplS9Tb368q5bIcHx0Vs1NTIgpDVq1YwYmTJ7l4+TJzpRJxs+dp4pjS/DyzxSLj4+McP3mS57/3PfpWruT9e/dSWGwLEjRJVBiGzM/P09LSwppVqzj08st0dXSQy+WIb+EiSyKvVgopBNPT01wZGhKTIyNivFh8Qbjux58Jw5Nvp/74sTFjqZv+C3Ag7Tife3Tfvri7UBD1RkNfHxqiXq0ipMTRmuvDw7x89ChCKbZs3kxQqXDxyhWW5frmc/OlEtU4ZufWrWxauxYdxwRhuBgLbjl1z/Pwkkn6W1u5b2CAS4OD9Pb0LD57a1muNbPT0ziOYztaW2k0GmMN+OVvCvF9goB3Ovt5GxhLZvXJRGJl1Gh8YcvWraarq0s0Gg25dDLacVBSEkQRf/f97yNcl1179tCZzZJsRnrf9/GDAK0Us3NzvPjKK+zcsoX1q1ejomixEX2H6cdRRE9fHytWrsSv11nZ18fxkycxxuA09U17R2lu4tgmPU9qKf2vGfN9rBVLXb53O2sim7NYttZo/Ls1vb3Z7t5eE/i+XJqRWm76AH936BCJdJo9e/aQ8zxEFFFvNAjDEK012UwGz/N49fhxVvb3MzAwgGPMbXXInctxHLAWz/NIp9NYYyg1GpydnqYeRW/UKkA0gU1/CHKAPfguXOM2yzgM0aetTTeE+FhXf781USRFc+Zy6TRSySSvHjtGBOzZsQMThqSb2mPcTKXGGFzH4aVjx0BKNj7wACKOUY6z3EJ4g5SnFOOjowgpSSaT+L6/KCE2M5S4ReARr1uULfu+MMY0pqF+L2fSZLPr2J9w3UIynRZ+FMklriCFQEnJ+Owsg9ev89DmzQRRRMpxmK7VmK7VlhtHWmtm5+YYHB5m85YtCMCVEtVkqG/VkbfGLM+Cz8/PY5XC0Zq2RILJWu22ICqFoBwEtlytElk79poQ4dstz982GC7EdlGfI7aWShiimurSjO9zfHCQdCZDLp/HNGuKYqNBsdFY5gJaay5duUKupYW29nbiKEJLyVi5zGS1+gZzX8oQqXSaZCqF7/uMT0zYtvZ2Fnyf4NYGtLWEcUyxXqdhrV2YnSWEl5dHLu+lZeRXrbpeD4KxWqlkHK1NJQhYCAImm52xTqVIptOLpEpKrpdKJLXm/vb2ZXU8CEPGJibo7etb3qySkqlqlYlq9c3rFmNoLxSolctMTkwwNTMjCoWCMdbiNUXf6VqN2UaDmVqNSAgbNhqMjI1ZC//r7eiadwXGftBfvH69EcGfj1y5Io0x1nUcKr5PUilcpQhu6Y1ExtCdTtOdySyfmpSSarWKH4a0tLYuc4TpapW2ZJL729tvm7dYKtAy2SyJVIrZ6Wl7ZXDQlqKoNjI4KKMgMI7nxQhhrRDWgHU9LxbWRpdPntQLvv8HzwtxtjlpeM/AUCOL/iYf7+o6OjI7+55aqbS6pbXVplIpobUGKalWKkzdvElffz9xs2+Rcd1lR9VaUyqXGbpxg4E1axandYHZRgOArnT6Njl/qTmsHYfyIkkzrxw5IqMo+shCrTZRmZx8r6u11I4jBIg4DEVxakqeP3VKjc7N/c9PHzjwrzh3Th68h0As0/ElGv5LjvNQLQyPOEJ4uXxeaK1FEATEYUgQBGzeto3unh7q9ToZzyPfvBGglKI4N8f/OXSIvY88QiKZJGr6fMvSrYE3WXEck0gkon/4h3/QZ0ZHn/u2EB8FeMLaX9Dw6wmttymtM1EYlhtxfCqG//GcEM8uVc33KnDeSbqiX3Hd9eUg+NZAV1figY0bmS+VhJdKETYadHd2Uq5U+P6RI2SyWTKZDFXfJzKGlNa4QFtrq0m4rpmbn1d9mYyI45h6FOFKScpx3gCINYZ0KmUuXLqkLoyOzuWTyd9+ql6Xh0A+J8SzwLMfiqJcFEUZDZUXhCgBNEcK7qlFLLvJYRC/mc+3FGu1Q6u7uu7bv2+fzWUysruri672dlqyWbTWtObzaKU4evw4nueRy+UQShFYSzUImJ6aEpMTE7JarYre/n5r4lgABE0hRyuFvEUD0Z5nr4+O2lePHZNGyk9+LQiOFUB+p3kL4ACIL0DjGpSvgv8UyALIP/kJAbF0dYonrf1SVzb7y+/dvz/MeZ4TxjEsXWG4pd3veR6j4+McP3t2ebRZCGHr1apYqFZPAH8J/N4D69atXLNxYxQGgTbGYKxFS4kSYnFiWOt49MYNefbkSVEz5ne+CZ9dctU736+pZdp77RJvCsYvwqNKiBcf3r077u/uVu6PoM7WWtymyU9OT9NoNMy1oSFxbWrq0qcPHHjwEwcPxp/Qep+NoufXDAzk+9eujRPJpMVa0dyJrddqcvTaNTk0PBxE8FvfgC+8BRD/6EuH8K97OztpbRIl0bwvxptTYYLmvERPoYCR0ly6dEkrIb72iWeeiQ9A8mtR9PIvwvsuDw//ycTNm/u6OjtJNUelq6USE9PTVILgkAP/5htwtFlt/n8HYvHaBjzSWigAyKX0d6s9aikXq8UmQEuMMwZ78swZOTY/X06nUp+nVhMHodHc3GmEeO+Tvv/h+dHRDwJrm9dkLmr4zrNCvIi196TsvqdgSGh1XRdALHW1l+oSKQSDc3OkHOc2riCV4sbMTHTt8mVHSfnUl2u1m0sbOwjxUyCfttZ8E76LEN+9c/gMu9zX+KkBAkAbKAdBkJdCWD+OxUIQ0OZ5zDYaVIOAahguB7/YGCIhmCuXwzNHjzqlKHr+W0L8/p0n3Ex94gDIKWvF0ujQFIjCYskdP/2PEBDfidL1w7mpqcf6BwaMAFVrxoRyEBDGMZs6OoiMYcH3MVKyUK2G544edaYrlWMr29p+yRaLb8UE7cGfspP/sTxjs1LTtWr109lczuRbW0UcxyJs6o0JpSgHAb4xGCnN1PS0OXfsmJ6tVg+15XI///n5+fl3qkT/VIJx3tqrG6C7Oj29K5HJmFxLyyJBklIIpaxWyvq+b4evXJGXz56VlTD8s44dOz7158PD1XejRP80LtFke/wN/JGG3+zu7qa1sxPHdQl9n9LcHNNTU1TC8LSG/3AQnr31JhL/hJa44985PBHBbwE7gbyFooVjDny1Z+3av/mjq1f9ZrA0/BNxjVvX/wNqdsdA+MX3PAAAAABJRU5ErkJggg==', 'fruits/star-fruit.png': 'iVBORw0KGgoAAAANSUhEUgAAAEcAAABACAYAAABItWqnAAAc1ElEQVR42t2ceZxV1ZXvv3ufc8eaq6xiFAsZRZlFVNBCBCEiQTQFBk3UaDomnU5MXjrpzusOXRlev9f96by8vE7aj5lDO1aM+BBEBBVQgkNERAYZiqEZq6Co4dYdz9nr/bHPrbpVFI5JXufV53M+t4q6dc8+v73Wb631W2uj+PP8UnV1OBs34gHU1xMOdzHXGBb5Ptf6hmGOQzLk8ET2LH/buJXW5cvRDQ2YD3STPzdU6utxGhvxAZbNpIISPuP73On5jEfAVRANQzoDnoFwhLeSSW56ajP/sRx0A+8foD8bcJYvRwM0NGCmTiU0+gK+YISviTBUDFxQiowdpczYceihF6OMglWrJff67wiFo7zpt3M1F5JtbMQA8v8DOGo5qBfr0HkXWjKXhY6iwQiTfR8uGoQ34yqlJ1+NHjwSwiW9P6Dha5Lb8XtCoQj//dFn+dtCy/tzAkfV16Obm1E1NUjfHa6fw3jH4dtacXMmCwOq8K+dofTMuaghY8HaFYhv/8r44ITgTDNy/71i0mlyUYfx/76OA8uXo94P/7j/GcCYNQvT0IDpu6P1dRT7EYa6cJ9W3Gd8Iqk0ZsaVsPg25Vw8AXQIMCC5nt0WQCvwMlA1ELXkDiU//ZFE3RL+Abhj1648lP+5LEfV19uF9QXi7nqqsykmY7gym+Ey32e4E2KwVgxQgpNOwqBa/MWLlXPtDRApCUDJf4oUvAbfi9gnzGaRr9wn0nwSU1zMpF+vYuf7iV7un4pMX3zR8kYelPnziVSFmGx8rhePWR0tTBGPSiNQVA6RYkicgUwXlNXgz/gE+saPK+figfbhJVtwA+lzBf+mAN+HaClqwUL8n/4YN5vjfuCzu3a9t2GoPzYou3ah8oDceSdR7wzX+j63mhyzTY6RCPgaYlVw0XjM6EmYrgRq2zOoZCuMm42avQg1eZQiBpgsKDkPKNLHigLrUS50nEW+/AVRHZ20VoYY9bN1nD333X8ay1H19eiGBgvKbfO4VCk+nTrGLXiMVAqIQukw5OIJ+NNnosaMRbefRD/3GHrH72HQBLhmMUy7VFHlADkwJthN8y6gyLm7bzJQVoOaOg1/wzoqM0XMAFbNn08klcKftRHTX/6j/phJWv1cJjku3/Ay3OoIITcO8aGYmnGYcRPRV09B15bA8aPw2IPw0gtQMwXm3A5XTFQMjYDybORBAovJX6YffunPDhR4OQhXw4Yn8H70Q3FKSvlpdDD3P/ggyX6ybj//KX9QcOrqcDduxKufQ5l2WG48vqgNoUgl1EzEqxqLHjECffkIGF4GyQ745Y9gzZMQHQxz74XZsxW1MQj5lmwlD0o/1iLv4k55YHwfQsVw7BAs/zvh6FEkHEaFQhwJh9nihlmvHV789UoOFNJBQwNG/cGStSB3WDKPOQp+rHKM0nEYOh2/4hJ0VTVqyjCYWmv/4LmVsOJn0KVg6gK4YRFcNlAR9QE/cKFCUEw/gLwHMMYHtwiaT8K3vikcPQa3LIaW07BvH7SdBc8DpUhHo7zmuKzJplnR+DzHli9Hf2RwCuuV2+bxd2L4jp+DmkvxBl+FEypFjR4A14yFqlI4fQJ+8q9wqB3Gz4Up0+CyQYpSCnil0HUKQTEfHJjTzfCdBmHbdlh8M3zl74Au6GjDHGhC3ngDdryNc+yotbJIhBbjs+zxDaxXHzUaNTRg5s8nUg4/Vz7LfIUZNQ9KR6LjLtRdAuOH2/dvfQEefQJqZ8E118PwUkWlBjwLCgLK9LYWMefhmfdypSI43QLf+baw4224ZBx8+++hrBz8HIRCoCOAA9lOpOkQZtMmzPMvEjKGNu0w2f2owCyqozwGT5JllluBN3ExbqgKLiqFOROhrMi+f/XjsOENuPFLMH20oswAopAciBGUqMBSxILSx3LOyzd9voyBUAmcOArf+2/Czt0wYTzc/1dQWQ65XJA952yuJIDjoMaOxhl7OU4kirdyJeXROH/jfhRg6q+k0omz1qSYVjqM3KVLCIVjcMWFcPmongiy6lF47QjUfxWuqFE4BsRXiDEok7cWsQ9t1Dk8c16+6WMxIhCKwzs74V/+p7D3AMy4Ej53DwysgVwWlLKfpwi+D1KDVBdEFIwdi4NCjM+N7ofimAbk9iso9ctYa5JMqxqDN24JoYoiuG40DK60ROe6sOoxeOsUzP40jIwpHKMwnqCMQUkcGAyqGFQO8Y+B3wZG9QblPdxJa3s/7cDLm+GBnwhHT8Csa+Av7oaK0mAD1HlSPmU/QwNdCZTrghEGflBw1K561NQmHL+SlSZtgZlwB25tFdSNgFikB5jfroCdx6HuLqg0UF2kMDmDEg2MBHURiEPgWyhdieR+D34SRCOBNfXHNUrZy/ehPWEfcPNW4aHHoOUMXH8tfPYuKCkCR/d2Q6X6ZHpBqWE8OHkCHAd8H8f9oNJkYyPe0nn8SjJcVz6c3JS7CI0fbF0pvymuC7/837D/NCz4S1BpGDNQIzmDkgpQ40HKbepqcnblosFrBS9nXUukN+cEz6GChC+bhUQSMlkrTax7QXhqNbR3wvV1cN+dEI9Zi+jOfVVvo8tjJMquuaMdTh6z4HgeCfcDJHjOxo14S27ge9rj9kgNuen3Erp6FIyttHqKAs6ehn/9R0hFYPFXIJNUTLxACGEwMgqlxtnVmHSwnREwWSTbhOQOBYSsUFKweLEW4nlW/kxlIJ2GUBgE4bHfwIsvQSoF866D++6CSKiHT+jHUgr/TcRKHweb4EwLBkEpzX73g2S+S+ZymyN8kwjejM/jzpkAw4rte1pb4Jnfwssvw8WXw023QzKhGFchlIRcjExF6VoLismBuNZacqeQzG7E6wBRKLHAeDnIpCGTgWzORhnf2GgkAiUl0Nwq/Hsj7Nhlf3frQrh7qXUbCUBR71Zuqx7OymVg+zZoTyC+QWvFWvV+a6X6OYx3NFtzWSKz7kffOh81KGLf8/RjsG491IyHGTdA9SDoaFOMLhcuiBUhMgPUBWCS1jJwwU9aULKHsNWMRiPkMtDeYa3A83r4IL/DjgOxOGx7W3j8KTh20uYsdy6BWxdYdxMJeEUVAKR6X0oHOZFAtBS2boJn1iAHjiKZHFlHmPhelqPyipzj0phNEJ9yB379AtQAF04dh5/+L0jEYd5XoXYoeElItCnGlAtVsUqEOpSKISYFEgrUp8NIegfiJ21kMgqN0NkJZ89aF9KB8pnnGGMsKKmM8Pj/gfWboDMJQwfCF+6E6VMhnbSgODrwXFMAROETBZcxEC2BPW/Dli3Q0o7v+7haseKx9exV78edls5jhUlxx5Cr8b76LdwLY/DqZvjVz2H8YpgwE4pyNtPNiaK2WLggXoNwHQoHMVnLLV4Hkn4Tsv9hzd5oMIIWaG2D9vaeteeJ2PjgOhCJw859whNrYM8+qKyE666E+gVQXQ3plHUP34dMzl7pLMTjUFEWGGyAuFL250gx7N0La9bAqRbMgYMopWjOwYQJz3HafU+emce92uOO0CC8e75sgVn/NDzyCCz4Ogy6EGJpCDmQQXFhUQCMmoMShYgPxCC7D0m9CX4aMbo7G1YCp89AZ6fNXLtzG9/yRjwOnWlh5dOwfrPln7nXwq3zYdRIqxN3tIPn28iV87rzSZQD0UiB/qAKgCmC/fth9Vro7EIOHgatUWK488n1NLv15wnlQQbsLZ3DJVr4YdLHv++/4Fw2EJ5fDQ8/Ap9ogLIyKMkoQiFI56A6KgyIlyLqOqyjWJ+Q1EuQORjUSto6elBHnT4DiUQAjG/BMb5tzkXisH2fsHId7G2CQTVw161QN91aSPNJyHoWGGOs5ShtwXBdKC+HWDSI5MHv8sAcOAhPrbGfs+cdjALHN/x143qeravDbWzEc/tN9Hah/mIqoQ6XFakOYrPvxZ8/DfXaJlixAuq/BxXlUOErHBcyHpSFFYNjCtHXoIghYsBvQ7o2gtcOJkjqfLHZqljpoKsr6Krk9RsP4lHoSMMTq4WXXoOOLpg0Du6ttxxz4iTkAmFdBQ/tOD0WUhS30cx1+wEmDgcPQ+NKGDEG2tswiU6ceBFHx83g+6xHB4LXuS2KINHz26r5nkkxdeg0vM/ejXNwN/zbj+Dmb8EFF0Clrwg5docHxRXD4gbtTECpgdaVTArpWh8A4wTA9BST3cAErmQ80AJFMXjrgPD9XwjPvAjJNMy5Cr70KSiOwYnmoFRQFhAVZL8iEI1CVaXlmG6w+gBz5Dg8sRqGDoe6OXDiOOI4ABwLuhHdubTbT9j2ls7jGjy+JiV4n/8bHGmDf/4uzP4SXDgcKlOKkiKIKog5Cm0MItXgTEQkCzhIcrMNXeIgxnQDowRaWguA8S0wsTCk0vDos8K6l2xeM6oW5l4N08cHfCKW2/Ku4wchOxaF4mIIRywY3RyTBwYLTEsrPLkWhg2HmXV2DUeaIBwGEU4HGOi8zOv2Ddt31hFNw4PpJGrp19HjBqIavgJjb4HJV8HglKK8LK/SKcQzGBNBR64NqvAwktoK2ZO2PvJNN8lq4PTZHmDEt59THIe9h4SHVsGu/TCmFm64GqZdCpEwdKUsCDp4cCO2yCyKQlFRb1CU6hOVFESi0NYJT6yFAYNhxjXWso4dgc4kEo5AKs0JgObmnrzRLbAa3diIv+RjNEgXY0ddj/fJubiPPQDeMPjYrTA0o4jF8/wQxE1x0eHZiCq1K8rshfTOcwpHjQ3X+ahkPJviGw9WvSA88ax9+E8thNnTbF2UTEFX0gKRf3jtWNcrKgp2XNkrzz/dwOQtJgbNrfDkOiguhxkzLYGHo3CoCXQUtNV49vfbmsnrM7fPY0wuy9dUHP+er+Ic2AFbm+Bz34NhOUUkbHUYW//4QDEqXAe6xq4wdxLp2mJFrILCUSubw7TngfEtaZ44ITz0FLzyFkwaC7ffCMMHW0tJdNlkTuWBURawkhKbERM8fHeSVwAKAWCROBw4Ams2w4BquGpajy7ke9B0EKLFqPQp0A77+gUn6P6ZnOKfvDSROX+JP7Yc9Y8/gMVfguGOJgJBcRMQqx6Jci+3DSgMmC6k6wX75EFyh1gwOjrgbHtPGVAUg9e2CStWQlsH3L4Abpxpf9+esKA4Tm8LKC6G8rKALVVvTil0I1EW0HAE3tgNm96A0cNh0qV2+b5vC9bTp+BUC4TD6I4MEiq1ljNrFmbjxgCcfO30yZuY6SX4eOlw/E/cjPPaJqieprh8qEDGgAsiLqghqPA4UANt3MWAySGJ56wOg7ZWUwBMa5u9meOANrByrXWjAVXwjXvgsoshkbI76rrnkmlxUQDMeUApBMYN2dcNr8PbTXD5BBg5zCaPElTk4TDs3wMZQRwP5QsnqsSC09DQI4d1c46f4+99D2Z/BqqBJ/fDdZ8UVMbFdSssKM6FoCoD20wHf24wiSBk09ti2gOLQdkok+0SHnoSNmyFKZfAvbdCRQl0JAtcSPUOv0XvAUzejYyy5J3MKZ7ZKrS0K2ZfIQyosmVEoVqBwI7t4EQw2ZNo1+H3Dz5Nsu/sjtvYiL/sRiZlu5hbNhwzf5ZyDu0VSi6KMiI2BfHLcZxSIBSMNGSCO4R6gMm1dAMjYom1rQ3a2u1NIhrazgg/exze2GnD8+0LLCBdGStWQY8FUABMRXkATAEQvQDKAxOB1oRi1RYhFC3jYzMyxENpMoGgLnnxPQQtp6CpCUJlSDaJUmHW9o1U3ZbjC3eLj5p+M361Qm96Cy6bWYtmOKLTtqLEs30MABUGyWG6XoDsqZ5tDrSR9nYbOkUgKsLxY/DAwzYBW7YQbqqzhWHWt26QB6aw9okXQUVFwCEF4bkXAeeJNwonWjVPbzUMHlzLjHGKbNchcr5CK+m2FrDv3bbe1lPFGifrkQ5HeRpg1kbMxgJw9B1zKfLS3ByuhOvmorNJ4WSbZszAYSBplPILpDMDKgSSxCSeg9ypghTVAtORCIDJQcQIh4/APz1oiff+u2DR9bYOEwKL0QHmeVdSECuyVXe3hTj2UsGFY3Uyx4VwTPHOccVTWw2jRl/GnGkDySYO2lxISU+VH+RJmTS8sgVCEXw/jULz7MNrOFxfj9N3mMCVCDP8LoYNm4mMKlF6x6tCRXUVYVWG8XNoR/X0R1QM/NYg++0MnqzHYhJdcLYN/KQQ1XD8JPzzT6CmCj53GwweYHVfrQt0FXq7UiwGlRUFYo7ubTkSPGTIhbSveHW3sOswTL98BpMuLuXMkbUIClUoPAe1W1Ex/O4lOHYcikrRWZtafP+8IyjG4wZRMPEKW5m+tRMumT6kRxTuJu8oeIeQ5CuIyfVYTFANJ5O2w+glhHgITp6G7/8Mpk2A2xfZSJVI060DFCp1BB8Vi1tg8uVBr4gUvDfs2kziUKfi5beFRGechTfMZUh5ljOHn0VECIcUStuSQwVavdZWJXxuLRRX45tOHF944Tfr2RTkeecMUbq+xxVuHCZchkKEllbFohHVgIfSEhAvkN2GZHYXEETgSq4FpvmYkOuyrZnms/CLx2Hh9TBnpq2Zsp4VrfqTK00ATEVFQa5SYDmirKWg4FRK8fZJ4eXNQqmuZWn9eAYV7+Xs4d3dYbq0Ukh29YBjfCgth9VPQmsGIi60J0FH+GZBnneu5Xg5RhdVwbChSrUcF4rLyyiJFNtupI6CnEWy24KI1K3mWj03BMl2OHFIkJyNLu0dsPlVWLYYJoy1+YUvloB7gZKXKvPkW96blPOgOAG3tKVhfye8vkPYuSnC9IljWLI0Slitp/NkClGKUEgoKbMrzKStteSBP3YIXtwCFYPwWnbg4rLi8bVsfbfRW9cYKsJxKHFRO/fDoCHVQBEiCZS/C8nttgVQAfGi7II7TlmLiYXt6NrufXZB9QuhqtzWMKUxkE5IBx3JXhquWB6o6JPH5F/DgVa09yzsaYFXN0Du+AXcvayGKVeegLYWkkkwogi7QrzEEnYmFUxrKBvGtYKHH4b4MEzzdrRnOOMU8dfLQTc0vtvYmxBSQYRuPQMDayuBI+BvR6S9txsF0qNkofW4kOq0csGBQ/YaUQtXTLLu4xsLRr4vl6+RCoEpKbVqovQRvkOBSHW4Ew50wds7Ydc6l0kjy/n0tzTxkt34p4SsUfi+EHKFWFHgisa2c/IpQWk5/PzfoKsEuo5ivCSuE+KvHl3Fqfp6HN5lYNtViqyfJQJCNgPkdgKJHqUoqNTycoDXBclWIdEBTYdh736oqoIFN0BNNWTTgc5SYAEePSE5D0R5qS0iu4XvQLzSCppTsD8JB1vgrfUKddzlc0tCTJ7RBp0e2VbwjcKIEA7byjvv8bmsLSoBKiqh8SHYlwSyeG2HcInwy8ee5ZG8FPquo7Za055OUpMGEYNKnE0E9VHALcY+mFZAu3CsCd7cAUeOwoAauHEeDL3QdnbTgZ+rgqZZfrIEt0dyqCi3wnkeGK0tt3TkoCkDTWdh3xua1jcVV14CN93jE4rnyLbYHbItSSEaKciu6eEalAXmNw/BGy0QL8Z/Zy2uE2e7aecL73fE39UuRzIJas62ITqEKqq2haORQBpwrSHteV3Y9LKtZMeMgvqbYeAQ62LprgDAvmGaniQv30koK7VZsSeWt1wFSR+OZhX72oXd2+DM9hDjquHee32qLjSYTsh15PtPgqstKHk6yG9ENmPvV1oEv/4J7DdQMRDzxiM4bowWY1jcuJXU8nkFPPFu4Lgu29MJLj+0B4lWw+CLBaUgFINcO7y8QXh+vY06V14B9YuhosbWnelA0eu1e6p3u1Vp2wUQrMabJ96wgrSBIyj2dwk7XhOOvxFldHmUT96cZMjIHGQEv81am+P0JIDdBWrBl++BG7b9qx8+AMmLoCKM2fJTdDhG0vdZ1LiBg/X1OP3lNP2C44R5SSvueWUT6tqPwZAKaD4G654Snltjw+yij8N119iRetMFmc5A8Xf76T2r3tYjCsKxnpERB0gLHNWKpjS89YpwcEuI2spaPr8ox4gRxyGbxbRbEArvoZ1z+9wSuKZTAu+8Dg89CTXXQews5qUH0ZE4WWO4pXEDv3s/PNOrDbN0HrV+jp1uhPiieiWtp0WtftJGoWXL4OMLwI1BLhG0aXU/8y39AaQKQmlgw10Kml3FkRS8+Tth3/MwpGwMixYN5tIR+yBxFJMJaih9bminT7j3jQUFAysfhS0HYOJSOPI7/FcfwokWk/LhlsZ1rM03KT/wSb1PXM/GsMs1HQlMLoOzcCEs+6R1n2zCZphaF1hywfSCKmjUq0KL0VYQzDqQ0HBaw9EO2PEyvLMeBsZr+fiSyUyd2ApnXsF0plGuQjlyDgh9gTHByAhlcPhtePgxMCPgkjp4ZQXevudxIyWczsEtTzzL5g8DTLdkoR2eR3FtTQ3+N76JM3oK0GF5JRw+10ok//AKfGUTQE9DTkEWSBpIeNCRgjPt0HwKju2B478PURO7kM/cMYnpM4qg9WXkQBPigA6pnkMNqn+LzEsPuhyy7bDyJ/D6YbjkZoi4yDPfxW8/jBst5e2Mof6369jzYYHpXsZtcxjkK7ZHQ1xQOx5v9Ax0RQ06HLY+rwMy9I3tRed8W8SlMzbhSiWhqxOSndDVDl1nIdVmC3eVhmKtGXFRCXNuvIhJ08rBnEFO7EE8Hx0JQDmP6xQK6brYvm7dCKtfgOJJMPYa2Ps8/muP4GgBJ0JjLsdnG9fT/kFO5Z0PHA2Y+huYqoUHcxmmBHmJ74ZxjGdT+FAYG0K1zYBdZX8OuxBxbJslFoayYjugWFUOg6phYDXUVCtCFbaVI11B1AkHJ8aUnOs+eYWPID2I2sXu32P722dLYPxCyLQiW36Bf3ovrhsjpRy++ehaftD3DMZHAad7Cr2uDvfCOJ/yhK+7irGJJP61C3Fmjwc3aKdEQvYKhywwIefc0qB7yNEPBMRgOAAHtBOMbSpzXm4RZY8BEQI/DXua4LU9cDQDg6ZCSRmy7bf4+1/EdTToCJs8jy83Psebgfwg7/eQ6/s6b1V4cu2mm4gX5/gfyvDFUBXe/f8Vd/IQy0MUTEMU9r7zIyOF88KqoMJW53GZPBh9JdBUBvYch53Ndjiq7CLQHrL3efxdz+L6SdBRTmmH7z7yDD8C5KPwy/s5jNbrMPvS+TxAhs9RRe6L3yA0cwjQGUxM0UMVvcZh6TOQqPq/RPcZQQt+TggcTMDeBKTjECuB5CnM3hcwBzbjZjtBR2jTIR4gww8efZ5ThQdT/qCHxs43iJ0/rnzbfH6lPD4tJXgfW4aefzm6xgGSQVuY3scGz5ljVQVyhOqt1+RnPLoETvpw3ECrAzltZ5xO78UceAlO7MDxbSfojBvlZwp+/PAaDv+huOXDHGPs3o3b5vMvyuerqQxUX4JfNx+uGodTGwWdxcZvUxBuVT9SaEF4ztkalhagRUG7A2kgl0Q6j2GOv4kc3Y7beSIAPcy+UIif+1l+/eh6jheAYv4Q3PJhz3h2A7R0Hrc4im/7GS7Nij2CeNkM/MkT0COr0TUhiBXyULBkT0EmyI7bFbRq+9olkMki6bNIx1HMqd2o5ndwOk/Zc5xGk3ajbFCaX/tFrGpsJJUH5fHG7gMAf9yjzB/kaGL9OMJOLbdqny94WWYaY3OP8mEweBRm8EVIVRUqVoRyIyijrVGlDaSzSDoFmQ4k2YJ0HEO1HUZ3nIRswGMGMm6E17XDb9GsemR1T3M/IFufPwEoH/iMZ1/fvuNGrvbhFj/DfC/LaAUhHHuGyYnaClm7PSNtJgd+Jri8YHpFkXVCHHDDvO66vGCEjQ+vpqmf08XmTwnKhz0Ae84h0eXLcZu2Md43XG1yTDYel4kwUAyVRogiOEBWOSSUokU7HNAuux2XbcplR7yadx58kFzfe+T/lwL+H379X7bXnPXxNxmsAAAAAElFTkSuQmCC', 'fruits/strawberry.png': 'iVBORw0KGgoAAAANSUhEUgAAAD4AAABACAYAAABC6cT1AAAbKUlEQVR42s2beZDdV3XnP+fe+/u9rXdtrX31JhnL2I533PIiG8ssBiICCQmGTEFIMlOVZFKVSSaleCoJGSrFTE2WIcxkMkyGqkBPCkPhIIwxkuUdL1i2ZXmR5La1tFot9faW33bvmT/e63bb2EZiCX5VXdXv9Xv33e8953zP95x72vLze8iOHZhKZUO8epP76oaNXbsO7m80APlx1jrTD5ifF+rt2zG3307I4+k/tZYPSqNV/LgHCOj1ty48+4bt/b2nexDm5wTaDg8Ttn5gyfnWye95r6mpuSoAO87IerJ9O2ZoB86I+Y4U7sbZQ33bWhxQ9eE/WIcYK1GWm8Vt3Kf/GBrCDg/j5YnF/65ccasUVgOMjb09LW6Gh/HbPr5oMC/s+wcGW2HteYmpT8cXArJr1+ntaWgIt3s3xcVXLV1Vq4UdLs7VZ7LybRvjQ0Pt78zrcoNzUlu6LimWbWhRrvlPALp4Mfqj3HsWNAKVGl8495J6T8+CXLLUDrztyc3nXFGqBO3q9abWl/tFy/3VV904+GvDw/jt27Fvxg2A7t5NccvHFm0eevfgPSvOSm5eeW6SiYAxoQpwGof3rw98dlNBZUNUDhKVgliLLFndDJHlv777VwZWDA8Tdux47d46hOgvv3HFwE0fXvT51pR5pHeBv3bjZVMhBKyIooh921p8eJgAIEF7nQvYSCkyzOCqRGt9oT+fcV8EdN++VwlqaAg3PIy/4f0Lt/X1pI/5zP1OqeLjC7ec8uVaMCFHvRfEkL3tXR1EAUTAe6j1Bjsw2CrE2Ju3fmjRx4eH8UNDuNl43vqhRb8lxt6ZpXZN/+JWcdm2Ce0e8NbnoApZyyLoybctq8/lWKMTRW7wmSACIcDKc1oGguLlz9/3vgXdW7YQdu+muPFDCz9JcH8dV7zfvGUiXPruSVft9lJkYBwkLaExbbFOn/95W/xNT3zWGtZyKEsMrZZRY6HIYGBJYRYuSzzYZa1IPnH77YT3fHjhRUVmv9DVn/orbzklKzakpsghFKCAi2D8SEnqk0ZLvcW3AbZsaYfTzxz4DjBD4Ha8ut4sq5ohcEPgtoOdfyDG6aN5apgai8RaUARVWL+5IRhV9ea2HTswjab9b119Ibr4ugniSjBpqx0es6DTluQv7++2KuErO798at/27djbb//RwO1P5LZg9wG7QUcg7O4A7ryudF4fgdB5zqWXIov2Yd2m2nSR8hlVccvWJaoBCQG6eoNJmiLjR0rdxyfLqJrbNg9NaO9Cb4oUjGnHtLGAku+9dyA6ccTt6xtobd+/N0v37fvRqYwfsxJqa2Qww+ABhgYHLxFjtqnq5QirglICEgNjDkaC6j415glTFE/fMzZ2fHaR629d9F1r7HWXbB33/UsKW+RtYEUuPHjngE6diDjrojqbrqhL1gLpgLYOgqd48t4Bd+xA/IPyQPbenV8+dbjjweFnBXxu8WuWLt3q4A9U5DonQlCl8IpYwQkkCqkIEeBUEdUpC09guC/P5FuVq4u1XsyXFi1rccl1UzbP2juKy/DSMxWef7yLoV8cx1pF9VXQeSp+770D9thL0R1LVqW3DX9xYmrHjna19zOpY3eAuR3CxUuXVnvgvxiRTzVEUA1aeHytZmTtWif9Ncf3n0o4q17oBQTdZwyHRcyEMSaIUAJcUAR9SQbDct8v0YXbJllzXkJrClwM9UnLzKTT5etTyTvZ2TpIGtY/8b1+e+KI++x9d43+IcCZgj5T4AYIly1evKRq7dfFmMumQvBXe8827+0BY3jKGo7FhhkPSwrlD/KM9SEwI8Ik8JIx+pQxYa8x+pIxtiEicQGRUeIlgUs+NsGSjRlFAkUmWNe2NLTdPE+Nf/SuAXviiPv9++8e/cuN24kXjRF278bPI9SfKnABZGj16h7Jst0Yc0Hhff7pooi2ed85FaVQ4STQFGERgUghEcECDogBq0pLhEMiPGmN7rFOXrGCLYQgsPLKBme/p07XYk9Wf5XBoxL+2Yd67bOPVr/0wN3HbnuLiu20DkFOl72HwQ8tXXqnNWZb5n3+h3kevct7JuTVJRSIVamKUBhDSwQNAVFtU3znxwA1IAP+qFziqRxMHsAaxFuq/Z6N26dY/a4WPgUNaJEb7vv6wFg35bWt/smaTpRXq2iPWG34hKM7v3bq8OsbHW91AHLaoAcHPxNb+7enQsh/J8+j9xcFJ0VwnfeFDujgHPtLMWItZ4uhFAKtZhPxvm2+znvLwGfjmG8Vhneuidh8cY2XD6Y8/VSTVAy2Jax+V4Pzf3mKWp+GIy+UzcM7e542hmdRuc46FihtdzA2NKzVZ4KaO3ya/5977jx1ZH5h8+PkcdkHevmKFQNRCF+bFilv8d58sihkah7oWUtPW8ufV8r8o1e+nXoeQ3mHs/SLIc9zRAQFKsBBY/h749i8MeZDHxvg5lsG8CgP7a5zYUk5XoJTL5QYf6rEwo2ZvHKoxNRYafGSlcWm1Zua1XXnN+gfTIOgJA0XF7lbHkVcb5zctn5TrdK3ruf7d99Rz4aGcCMjP0x8bwl8CNwIhPXV6r8Xa2/pDcH/SZ7ZqkIqr8o0AVSEPyvF3JcogzXD4CLHc83AWFC2OkfRAe6BigZ2WcvDzjF0bZXpLCO04M6vTXHslOdzPmedDzxaE2YmHMfur9C/IeeiW6Z0xYZGWLgsl0qXl75FhSxbl8riVYnaKGhj0vikaWvOyZaq8+9f947qM/f8S/PQjh2Y3bvb2zwd4DICOtTb26dx/H8RqrGq2adG+gVWqJJ33LZblZ1xxFeCZeidZbZ+sIdLr6lxajRn/IjnegqymTpJkpAmCXmS8I1yicM4SgJ5EP75azM8P5LxkaLJxWnOuQLvCPBk2XAyN2QvRAxsyKR3lTdpXUQVgm//lKoqS1ZlsnRty1S6vDZnjE+a0WDIzcfPOr+S/a8vNPcAOt/69kdZe1Vf36/Hzm7P0+BDJPZwj+ORXNgSAtUOcGuE/6kOljvet72Hri5L2lTuva/JmqbnislT1NMMDQFVJQDf6OpmtBQxPuo59XzKuqmEjyczXF9v0MgzJrOMZXnOlQH2lx2HC8uJB8vUlhUMrCvwaTvFzVZ2Pm9r9wVLC1m+oWW6F+ShyESyVnzD8tVdF61aWt59773N6dnYf1PgIx0CXtPT/Xe+YMnAgOXWX+0zF59f5uEnUy4qPCtos/VJEb7qIpZvtHg8i/pi9nx7hqdeLPhw3mR1s0U+ryJywNKiYLH3vLtV55caM9zYarK8KEg6oWNUqXtPJU25Mst5ruw46hzjD5fpWZnTv8ZTpK+laQ1tuQvQM+Bl2bqEviWpjyJzXh7sL609u2us75rG/rv/kcK+RfERtixdeqWI+cPgVW/5xV674dwyo8cK9j6RcJN6FgER8KgRvu0iVq6w1AUe2JPw6L6CqzXlQ5NTZPPSh9AW+Mu8551pyoqiwKmSwmsOZ9Ydc8B6z5XNFgcqEUdtxMkflOg/O6U6ELCGEMV45wg2IlgL1qKqbW/oGQhm6bqkKFV8//jR8gfj6doH155brbk3rJk7+wzCr5og9PaZMNZMTf8Jyw8ea1HJAistpIXHN5uoESr9JUaeyFkTCpZPFfyC8dwyMzOXw+V1OTQBmrPK6E3qY533t7oxbJ1qcHLAcTK1PPzXC+i6MqerL5iuXm8q3Z5yJWCjgIgSAuSp0JhyTIzF0fTJCA1CXJZNact/Qt7sSuY9S5dWp9EXCLKsVpNwzpXOnDquPL7X89ua8/6ZOieSBFSJgBPWEhSWBE+ps1DSse5biQV9C1EROqnvsXKZv+ruw8WGgaxgMjJe1dqi7L+erjd/5TJWm1JYb52ebQ1rMQyi9GggCiqFEZ1wsR4wIo+D7rzra2N3y5sJlnctWXKLs/abXjWoYrTLUGoqv5SkbJ2ZZibPX9N1iDobzwQ8Aqqv7Ty8ASjpyFixltwYgvftonzeumWB/9g1QHhnH7f9cj9f/Pwo1bFEX6mVcF6nyqN69k5GT8xf+z2fWlqVad/tbBE5Z4vqmurkl24fSea/x71pRSLyQUS0GkIAzKapgl9OWyycmWFK9TWuKZ1Y1E4LsdaxYyqCqr6hJctAiCJOVKuINSwKig2BVqPR7j7OrivQUuHmoS42nlMm6otY/sqMFLEpjselvubS4nMc5ZPvPou4+50Uw8OEb37xWLMTSfNLSzO0C7N4MTo8TJA3cvPLV6yolIvihYYxyz+TZWFrCEaylGa9QetNcmAASoBay6GuLkwcsU5BvCeZmYGieI37Pl8q8U8L+hkphKJQVjnl3xQ5a5KkLXE7B1kC/lP3ACs+upALzon5hy9M8pGRcQ5XYt1Z6yJWbYYoOmvPyy8fm9crkLkLyNvnoknfVLnNtow2dHUNIfLbpRDCr3tvutOU8UYD/xagy8B4FPE3ixby1ajEPRj2O8s7rKXbWvKk7Wkl4IhzfHbBAg5lwqKFhmUrIva34FQQhnxBlmVzwKvAS9bxg8IxcrRg5f5pbkqbTFkrh53zmXNl8X5kpF5/ZAjsyPwOzO43L1LMG7G5hHBrJqLnQFiQpow1GnPsq29ATiXguHP8Wf8Aj2RCfwUW9xgeU/gS4KwFY9oNQuBrPd1MquHGayp89Nf7+fAn+jlvTcTJvB0y81NfBlxVJBw+Gjj1WJNPNKYRoBICK4tCChFF9f1tnKffjJgf47IbioshUrg5B9mcpiZrNOYKi9mNhHmbM0Auwt/09nGq5Lh1qMK6jWWqVeGOr05zaH9K4gTpsP9h53jclFi/ynDRNVVKsWXiRMHLRwoudBDVU5LOugKkwKo85zenJ1ib5/SFwMlOx3FJnhtCEIVLrl+2bMF3jx49+Sb2eXOLb+/83rV06WUYs74SQlg/M2O8KhXgpTji+Sgi61Rl2jmAKrCzq8azUYl3XmDZdFmF3j5LkSqjJwvWRoZykpBruwP5VKlEElsWLBP2HWiQtDzf+1adqSllKGng8/yHSMcDWxtNVmd5W9mFQBRFrKuUpaoaMKY3V904H8dpW3zOzeGDhTGsbbXC+qIwp6zhH3r7eNSWME5Y6XN+b/wkPSGgwIQRvlOqsmxAiBcqB0daLF9Q5s7hKSZOKjdpQjFTb3sG8HSphAuKKWBq3PNPeyZ58ZjyAd9k80yd1jzemG+d+jwx46xltKvG9zNIVYMrGZN7PQfYM3aazRU3z839uzdsKCX1+q0JcHmrZVJj+Gz/Ag66iPVLLJMnCw5JzJHIMZBmWOC7lSrjkWOwFcgmHS8cTrn7YIuxuvIx3+S8iUmanS9qifBSHNGyhsf2BizQ1Sy4LW/y3ukZso4nVTrgJIpQY2gWBSbL5jhiNHIM58L688ssq3uOvFxgYk57KGAOeKdHHlr1+rVizNruoggXpan5q74+DriID91c4/xLq3z17yeYfiknmtdGykQorGE0UUYf9niFxaHgd5ozXNNo0upYSYFIlZtm6ozEMYN5zpo8Z0ORM+ADzY6Va8BTXV3c21VDBS7LCi71npZASDMi4BVjKSLhNz65kDvumOClAzmVkvSdMfBZgjbw8aYxXNNohEa5ZB4wZW56V4ULr6wxPVkwMR3oJ7AsL/Ad1725Xmd5UbCvVCIVYWWRc1mrxUIfeP3sVgDe12ggnSwROuRV7xxMF3B/V42/6emlmXhE4btRzG9IzrYoZirNKIAsQFcsNItAkgSMAVTtmQI3w+BvWLlyWVEU7/Uh6JY8sy/GFbr6hF+4ukqeKw99t8HhU4Ffy5oMhEB9niUvTRIuT5K5oiLpgHkjlmnMNic7fePZNFkGDkURf1frIdLAlssrDC6LuP+hJneNBq4zhgiYFsFZ6JrO+cbOCU6dyrBW8KrZGQEfArMbgs+yT/koqq1rtYoL8sI9X7W0MuX+79QZGy14asRzOSnvnZ6Zc99ZK87MA8AbVFs6P7+8Tu7Or8KGu7opYsNHfrGbVWfFlMuGZw9kJIdzJHgMcDSKUIXNRcquhyOqBFxk8ITJM7oO2g1+65IlNRX5dAIMtVpGvecqX3BW4rn7/hZPj3gusp7fmpicc1HmKatuEazIHODXu7cDuoFua+mKIub3P2YF0ItxxJOuxPo1QmVAMCIcfDbh2f0Z50WBarPFpAjHncMDG3zGhkZCIwMjoF5fPlNXVy+yDWsH+/PcX5wkdiYEetKUPwL2OQOFZ9NEncj7OWUlgDOGp7trYB3neU9oNCmKYg74rJSdihz/r7eXk9ZxiS+4Ii9Imk00z1ERIlV+UCqTR4b+xcKBVxo0TnnuvrOJ9crQzDStEDhQLuNFsKqgcEGacjCOrW9Xgs8CLD7NGxXXOfWPJiJ6cZrqwhBIgKjZpJokXAWkIcx1SGZBWRH+R38/d1FCCFxbVn7TB6Refw3o0cjxud4FHFRDOcDdJuYzsbAtj5jJc6TTfXmuFBN55dQRZWJa+N6zDSQLfLoxxfok5fFSiWlriTo9O9OWyVoYIyaEMUJ4BmD4NGWru3pwcFGAIVWVC5PEVoAZIzxQqXBCDFe0WizrqKfZ0qcb+Ea1yndKZW66rERXv+WunQ2OGsPyDrk5oGmEz3f3Ux+I2X5ZiSXLI3btabLr2cBN1sx5Ti7ChLE0I8NLBz2L8owbQsr1rSbrspwnSiXGnCPqdHNmQ+Swc96CFbh314kT9dlewmkBd3BNMGagtyjCBWlqnizF/PeuPl5xjmCFh6tV/nh8nFoI+A6gCWP4Zlxj6zUVbn5fN99/qEklU7rzrH1xJUJZla/UunmlFrP9xjIbN9fAK8W9TZap4vJ8DkSkyqcnJjgQx6zJc9YWOb1BmQIeLpeZ6lh6liQdMGktR52TSFUCfHm++jwt4AaGEmPY3GrpaOTY0bOA3gWWT93azXN7E3Y/HDgaOzYlGXURqqo84SIa3RGXXVLmxX0pw99sckPWYiBJmBGhpMqos+yKq5y1RpjSjPHjEd+/v8WB53Nu8y3yJH1NE2N9nrMpzwmdVPhMFHE4isg7MR3mVR9Olf1xHFJrTez9wUpX17dm1edpAw9wsQdWFrn8c6WLuMfykQ/3sHx9zHMvZPTnnmVFMRffBni2XGZaha/eUefEkZzLxpt8tDFFom0nLAEPVCpMRpbeSeWlA8oD90wzecLzK1mDjdMzNOZlANcJpePGMG4tJ5yjYQyxKrVOK8p3VGKsyri17IvjUAHnRf5i54svpkPgdsNpj347hbVRCBhBDhFx/bVVVmyIOXIw47EnU36BjAVFR7Co4oEVRc6mJOXcRya5NKScXeTk8xqLAaj5QI8JJKOB2sspl/ucK4qETUlKNtutoX2NPGUt49YyaS25tEvYWgjUreHJUpmmEdalOcs7HnF/peKDtc57v7c+Ovq/OwML/ozSWYB+p0qGSBoZfKo8/WiLXd9pkE55bkoac8c4q8qubTa5ttkkpn3ErdeJlyYw1GxyQZLggP4QqHTeOwMkxjBtDJPWMm0MqWkTnVXFqRKr8nIccberMIVBAuwtx9wiDcaM01eiSGshhFz1U49Bvq7dGDqjiQgnInGg3RldYAJ33tXAFYp3widb05yd5TQ63dBitjDpfDidl9PnSsZ5tfrCEMiBU8YwYwx1a2l07s2LWcGjStSRr7NXzYejiDvjGstXR9xwYYlSxfDgvU3uOVKFWIpukSgL4ffvO3784TNh8tfmcdU8GFNyqtzYaPBEuUzdGs5OUi5ttpiR9uThCWPoKzwRr37LfJYNAk1rGBdLALwIdWNIRMhF8B2NbjqS1c27aJhdx9Je455Slf5Bw3XbqqxZUybLlCcebGquFF3GRKkPX9gzOvqXQ+CGzyCuX8/qEwqDM8boiqKQK5vtrqxVeLBW5ZmoxJHCMBVZNpBzU6PRZtnOXTfAoVLMc+o4jqVRc3RpYFu9QalDTKaTsub302ef5yJzPmpVeaxUoe4MF51tGJ1McSOGe79TD4cO5XRVbJT58MV7jx37zI9r6VfJTfU5EVlywlo9B6TVGc+asoZ/Kdc45SxXb63SXRG+eVc7rq9sNudmW/bUquwNEes3xKxbbHnqsRYhtF3WvU7XK1BSJbXCQRdTiLAyy4lDoBChYQwvRRFOldEXAuMjme45lvg8xVXKhrTwf7xndPRPZ6evznTg57WuLvL1CIZGoihcnKa4Tsw9ViozGVk+/IFuzr+kwisHMyIf6AkBA1RUebpSZq9GDF1RZsvN3aStwN4nU1YmGdUQSDpxPPsooTwXxTxiy0xaQ+5hSTnmvWmTvsJTEuX8NOWJcimMHfVBRFwcGReVw5NF7n93z/Hj93RA608CGsBlql+Ovf+jGef6n45jvSJJ5JSzHPKWa6+tsvnyCqfGPd+7u0E1D5yVZ+QiIPB0cGzeXOKG9/WgwKPfa5E3POcU2Q/5YITyQFzhya4y55zvuPXCGnmqfPueJo+MeNbbXJ+NYj/unDFiTFQW40M4EtDPq43/ds/hkWQ72Nt/Avd+DfAHjx8fG1q27D9X4XNPlMv58qKIejSQRIbmdODxB5o88XCLF14uuElTyr7tli1jmBbDpgWWkycKnvp+i3sfSngHOUvyYs7a7WJFeSiu8MyCChdcCGvWRJQrRsfHMu3SEA5EkTkQl4wV40wIoPpMUYS/T5370kOHD5+af6f3Uxuv3g52DEQHB3cZa6/C+/yGZjN6MY75gcTQGdG8PCRc1myRdQAJcGdXFy87RwWYSZXV1nNzo0GkOsfesSrPl2J2lrt0QT9a6iZMjyNJU22eKKWywTkIQY8JfDuofoXR0btnVVgHcPhJXfuNroQNoNctXrw4OLcLkXO99/mmLHNlVFrWsDgvWJu2XVzmpbC6MTxaLnPSWpaGgssbLSohtL3BGMasDWPOhcPOkRrjRAUN7UHdIOqN5RmC7JYQdoZS6YHdIyOT80dROtr7pwr49VfSBgjvWrVqqSmKYSdyVUOVwaIoNqapXZ3nEneufXUeU9tOWspFKIBR5xiJonDMuTBprS2MEdNpNPgQFHjBCI+oYTee+3YfO7b/9Xd382rqnwngH7qLn00RGyFeODj4J0bkd4O1pVSVqvdhaVHoYu+l13spq8psHT1tjI5Zq6PO6Yy1FhGizkSOqB4U1QeDyC5CeIjjx/e/vpAYArcY9F8D7FtNNs6NZF89OLjRifxb4AMqssR3ZtSkE7906u7Qcf8IIAQv8Dgid9oQ/iXq6tq788UX09dbdaxdQoYz1dc/6/8dee0Q/qJFg2LtTUbk+gAXquoKFemeVZgGxgSeQ/UBtfauXUeOPPxGQLdA+Gnk35/W4/8DpFqSPLP6GiUAAAAASUVORK5CYII=', 'fruits/watermelon.png': 'iVBORw0KGgoAAAANSUhEUgAAAE4AAABACAYAAAC0oEFtAAAdgElEQVR42s2caYxdx5Xff1V139r7ym6SzVUSZVGSZS0jL7Ipydos2Z6RYwYT5EsiJJgkMwiMiWHAmCS2Ak8mieEPAQLP2IntMTKexFBgx6OMLI22kS1biyVLIiVSEsnm1k02e++3v3fvrcqHU7fv7WaTomw5mQYe+vXr++pWnTrL//zPqav4Tfx8Cc3fonmGKPvxtvsYCDRX4bjGOq5xjt02ZsI5Bhz04sgDARADsVI0lWIFzYJSnFSKY2jeMIrXaXBk8glW1tx3HwGjOB7CAo7f4I96T0fbj+EqHA9ik/F3fYIbCLjdWm4LQ66JQrZEFpwFpSAw8jIGnJLPQP5vLcSx/HZO/qc0aLAm4Iw2HFCKn6N5Khfxy6OP0l4jxGeIf1MCVO+ZwDK7vP1e3qcNv2tj7u90uKYdgYuhuwybRmHzVuzYZlzfMKrUC0ERRQ5lEcEA2BiiENduQqOGqy7hluZhYQa1NIeurkDYkeuNCH9Sax5D8/0TJZ7lIeLM3BysbubfCcFpvgSJhk18gjsDwx90OtzT6pB3FsZH4drriK69DrV1J7o8gIoDaMfQDiGMRatW9cKlM1Mq1bJEE+MImjVYmsOdPYk9+TZuahKzvIByQC4HJuCA0nzXhfyPU49xdqPN/f8nODGFKBGYMXwx7HBbvQm9XXDzB4luuw29+yq0KsNSHRar0GiJNikygtloJm7DtygF2kAQgAnEhGsrMDWJfftV7NEDmOUlVJCDXI4FFP/dWf7L6Uc59l6asPqVvvMlFA9iN3+cKwolvhJF7K/WYHgIe9+9uLs+gR7YjJqpwJl5qDdBI4tNBOXWy8im75UCdOY6t3ayzslnyRjGQK4AWsPKIrz9GvbVZ7FTxwiUgXyeGpr/apt8beoppjMaGP+/EVzmZtvu5V8q+EqtQU+pgP3MZ3Cf+XuY0iAcOQtnF2SBuQC0kkXaWASitSxceYF1FuV911aZkYuhveD9VwmU1ywHWOevcYmURdAO+V4QQL4IYRuOvYF7/nHiyUMEJg+5HHNY/tOpOf4zLxP+Otqn3q1pjt7OpnKJ/xZZPlmrwEf3Ef+Lf4YZn4A3zsD0vAgmMKkGacSs2ucg6AJdgthBrCC0sHwKlk+CKgMF6FShsyLfMUChBAM7oHcT5APQkZirKYnPa1cg7kB+wGuij8DFkkTkt17F/eT/EE8dIyh1g9a8HLf5V1NP8MwqfHrw3QUP9W6EtuVObg4KfL/ZYHuxSPQHv4/51L2o4ytweEommQt81FBgtAiuE0KtAwvzMHMcFs5BZQlcEXQfzL0KYUsgiEvM0oCLoHs7jFwPxT4Zu1gAU4X2GzAyBkOD0FOCrkEoDkPkYUxi4kpDsSwa+IuncT95mLjZJCiJUL96+hT/mkN0sj77vRGcH3DrXfxOkON7KyuUr7iS6Mv/lmDrBLxwHBYqsqjAgFEQW6i2Yb4Gs1VYbkCjI6jWWTFPE8DcC1CbBFP0fk2tnZCNoXc3FAZEo5wClRMVnn0J4gYUitA3CFfdDRPbYbgbuvLy/TASjXSxaGipC2an4dG/xL75CqqrH4XlxU7IP5p5gsPvRnjqkoR2B/8gKPC9pUXUrXcSf/mPMHUHvzgGUQzlsgii1oSzFZhegoWqwA3lzVbrtYGhPg2zL4DteJ91AUOxYQqWV2GKBu0DjbUQRZJvlEagbwTGRmHbGGwZg+4eIC/ztLEIWhn42SPw1A+JTEBgApZtyANTj/PDSxWeekeh3cn9QYEfLC1gP/1Z+KM/RB9fhoOnoRBAPg9np+HlvxWBRUUoDUFpAIKCaErUhLgNtg1hQxx/c1Y0IXGEKpAFpaE1hR/rP7Ohv9bvglLyme1IcLY++IxMwG/th20D0F8CDHR8btHVA0cOwA++SVyvYordELb4/Jkn+NqlYD5zwej5CPH4nXwkX+CvlhbQn/4s/Js/RL85D4enobsAtTYcOAMvvArTJ6E2BdE8NKehegIqk1A5CitvQ+0k1M9A6xxEDT/5LTB4tZiqbYvpOe+XbCg+zkUeAGsRdNyG3stEcJ1lMHnRyPIWeWkD2oJ20GrBYgTzeai0IGjBQL9c06jBpgnYcx36+GHs8iy21Ms93dsoVh7mcfZjOHRhjTMbZgOHcFtvZ0u+yBMrK/Tfehfu331RhHbsnPizw2fhpVMwV4GuIRi4DBpTsjDnwIV+8SEE3aBzqdNXiED6roDBa6E8Bj07oNAvn0d1GL4eenZCrgda8zKuycPQdTByo4zXnhdtNiXYcjv07pRgUt4s93RtqL0FYQ3OTsFLj8DsKRjeDIOj0KxCVy9c/Vuo00dR81PEpV4+1rWNnurDPMY+Ak5uHG3Neaa7XwTXv4eHm232Xr6H+Kt/gjlTg6PnoB7Bz4/A8XmJnLlANMFZWDkiyj3wPll03BHt2PRBwWiVSTAFH/3qUBgUv2Q74rOKI9C9TTSw/0ooDEH3Vsj3QmmTCLNrQrSzMAA9uyGsQOOs/D/XLYEn6ILyOPTugtJmaJ6D+jHZtJlpOHwAXAm2bBMBmAD23oQ6dRQ9N01Y7OGWrglM9WmevJDwzEYAd9vdfMFq/kkuIPrq1wjyZTh4Ck5V4Effg46BvnFvSjb1PbkuGLxGhFYYgt4dsvtKgylDeZNoWf8e6Nouf5NhRGwkQxVHvMbG8llxAIqjYmIuTEGyDkTDwroIrDzuQbb1v50IuP99oq3105AvQRzC0dfg5AnoH4bBfjA52HMdTB5CL88RF7u4rWuCs9Wn+cVGwjNrOLSv4ybuYbcy/K/qCupzX8B86HrU85NwcBaeewFsE3J50a5cr2gQXniFAVlMHHpGjUwAiFOTdREEJc+EuLV5aHJtNsl3cSoIMtE1uW/PDtHi1ejro7fOwfIhmH1ezN1mYmXOQGUO3ngJmhWY2AXdfbBjD+rwy6hWHRvkua+8kydrT3HS+zx3vuBG0RzC9l/OtxpNrr75Q7jP/XP0yyfgoe/AW2ehfRrCKlSOQ/0UNGcEZyXRMBFSMvk1cVtliB31DiTP+kiq1o25/rr4AvhAizCrx6CzJOPZDgRF2UR8BjL5Jpw+BhOXweYd0D+Kev150AFGwR0DV/PnK9fQ9nlGRnDeRLfcy80KvqYc9st/jIkUfOPr8NqrQAU6/kUs2tY9IY79kuG0WvveucxHG3xXBb8GjPebk+8VP5vrFoGVRsXnAtROwebbYXAnzJ6Gwy/C2HbY+wFotVBHXiEq9TAQN9ha+Q4/yGqdCG6vBITBy/jTao099/028f13o//02/DsT6HcDZFPicqbYGBvGvXcu6UHfbKuDei8mDZ6nQZ6fBbVUrD7K5FA3syVkeDRPSFBKijKuOVxMfN8PwxdCZGBN56D4TG4/kNw7BB64QxRvsx1vTt4rvIwRxPhqSTB3XE3eyLHG9ahv/OXqOMn4N//MQR58UnFERFYecyzFT6VeVf8SgbohlVx1tWTYu69lwl80Uai8Zmn5RpTgLFbJAK76CL3cxefyyqbkvCARoQXd1KfGJSguQSN0/A7nxLz/uZXiPMFdBxxaHqRD/BJYh7ESVFF8NUDjTbmo7cSDw7An3/H3yeWndlyhwBWZz1WsxeYqGcnshPNml5YgfmXYOpRmP+laEBxyGuGv65+WjCaAtpL4h4SSj17jyxrrHwKtqFm+uxC5zx54P1x3PGfef8ZNqDQDd274fFfwMAE3LQPU1vB5ors3TzIP+RBLPswmmeIb7iBXGz5LBbu+zT6x4/B5HFhFeIIenbJDeP2uui3kT/Oe+dbSn2UczLB6iSc+N+w8BrErXTS2iflSgudNPcy6IJEwe5tYk42yqRYxpub5+mUkQ2JOz5QZdlj6+ehJNNwsUcC3gV0lv31Tv52Tjg9V4Zn34Sb7oCePog6OK34QsLhGYD8DdzcbvH57dtw992P/sY3oNUUzJTvhaFr1uKtC/kTZSR7WH5bknidg6Cc+i9lRHtHbpAFNWcFvK4cEaddGJBoFzcFtGoPrnUOioPposMqLB/2oHkYKsfgzJPQmBEha5OpohWhvQjnfgZLByUguFDuVz0u3ytvFlcQVnyGUhTItbQCPYNQDtGTh7Glbka74PnqJEcCAOW4ox3CjR8kfv1NgpOTMDwBvVfIDUzxEoKAg5mfQGtRzC9uiFaZgufZYolspiC73F7yjj8vk114VQQUtwVzJflpWIWVtySdCkqy2NkXJXBUJmHLx6F2wvN+S+IGNn1YsgEbSq4893Kq4WEN5l4S82+eE01uL8h4cy+JsAtDktqVhuCtM7D7Guh6FhdbnFI8APw4ALCWWwINE5ejfvZTYTY2f1xywITWyUbEtbyPXFc/LTtf3gLDHxCBRA35vLUgwggrEp1t2ztn47UoL4Kc+ptUU1ZNTvnsIycmvnhAPgu6RBinfyx/60RQk5K/Bl0ikPaS/E/nM2ZeFhJCadmMytEUNaDFClrzMP5RyI3Bch4278QcfwuVL3D72D2MBFftp3t5ib09vdB26MkjsOUW0bKokUHw2pOIG5hr3Iblt+Q77UU48UPvozrySqKvMt6nFTIb4dMqnUsXlg0AOhAtq0+Lpupcyt+tCtfK3EbeL/NonMnwdvnUh60JLDbjH+t+fX484/3rzLOw+Q5YKUHPDpQ6jFWGQRPxsaDZYlccs6l3EJYWULYPerbIrukgdfBRA1rToj04WbzOpVoVVv21XhhR3Qs88L4qCxtcmmkUBmSs5kz6/fW+03a87/FmvyaaKgkKg1fC8I3yfvGAv9ZdeLzBa8RS5l/K3NelAUUHcs/FV6EwDHoY8kUZUiluCeKIy50j6B/EVqvo8k6ZjCmK8BqnoD4lqhs11qVNLoUZaxatMnmouwB29eB04GoxxRM/3ACnqbUYbJXAZG3UNCXBglFNINPyoY19slKez7schm8QU18+LOvaKBrrvASc5lnoGoVcN6pVAwXXBUoz4YBSERf1QFc/tFck0tVOiTPFI32TMZPEVJS6wM5eAtJXRgRu8qJ5jZnUZFEpyM71pP42XEnZ31VB7JAUMO4ICsj1eOyXEYbSommFIeHz4pZ8P9cla1xllLPYz9c9GmcEFgV9KJaBPDsCBZtwUOqFYAyWnoL621Ki0/m0/cC5NDoGJQ8L6t5/6Ysk6ZeQZiojzhybKrKLBSIMvV/KfjqQudROw+xzqSaasnB3iQnrgqRQzVkPhRI/3JHN3/RB2Szb8evLmKkN12YWLhPZlYJCP6p6HJRjKLCWMaWg0YDKz6WA0j0sPqC82ROFZ0Sd4w507xLWVgcCVs885VOlXOp4begXkb8EAXoh53szZp7UY/NeOPlUMGEl1aCoIWlgcUiiotLi1Ic/IAutnfaWoWX84etF42wnzRaCLm+WOclnTV40sFNJfasOMtfKWsqBjfmYMTB1FN21GcZuhqGr04tRQgQm0dGUvICs5K9D18H8K6L6zok2br5Vdnzp9RR2rDcD2xaNyPWIoEujck3c9vUFJ5mGyQtoTjalOZf6r3y/EA0JAZqMbYqC5QYqAqZVTgSng7SqlhCnQ9dCzzYRSq479b1hXaJ4VIfSuIBmXVhNArTaehehcwS5AnSNQ6clwujZsS7F0impmA0OOi+aVzsBrSWhwgevlkUuHRJsFzXWmUGQmmFx2MMRI7CjekL+DoqCCXt3nd/u0FkR4eUSYjQ+f2NWfajfBBdfgAhQaaaRXdsqPPEVNB1I0WnqMdAFOmrrXXSAHE40IGE+hq4Tk0zM7mKsROLkkwnbTpq32lB8RNRM6e6gLM4clVLhkDF3TwWh07HWBxUyxe2LMiaXwhW6jbulkq6CxJKW38BNP4UyBRYD4KiD9xVL2PIguloRxz//qmjcyI0+wXYXbsVapbZZ27IVt0VD8n0SNRMtvZAGrArJC9RdgFBwUTay/AoCc5lbrxtjlVzVaRE9yUw6y35uMKuV5ptxJOWyyz8iXY4oEd7SYTj3fLr7a5r/EpomSE1Ir/NlOvB+wZtC3EnrpedFX3f+IpR69yRpQjHpfFqSPI/BKXh2JZfBmi5ddxJ1w4r41OpxqLwJnUWshziTgYZnlMLWFtDFPui7DBrHJAiYguA5G8LYR+RGNvIDh9CYl8nl+8RkooY45kQoYR1asyK0wpCvm9rzAaz25GbiB5NNSQDvGr/q06ukep/VlGTjOhUBwzovvnR9QWjliPy/vFmCknOiXXETlo/4wnrdw61Qom3fOETtVQ18LXCGEzpivtFkdOUobuRa1LmqZArG70z1uHxh8No0hz33c0m/lBaGojgCM08ILTT+UVns4gFB8aYkCxrYK1HMRmnACUrQXpYct2traopzvxQAXtwEmz6U+j6dh4bvFBi6Lk3DTNFzeS/6ilZbNHzkRhi6Xv72ZiZ48BQsHpQMojyW0ludimyA9Z1SPdshUGAbENY86et43qwcodV3Ofc6x858GasCdNcuEUpU9eYWCF1UnZQbrrztdzQnStB7GZSGRfMWD0jRONcr6U9pVBL00rBMMihmolVHWiNmfiqC7N7mK1YOzj0n4w3sEQDcWZHN6r8cFg5KtB65wZuWEopo9jn5bfISrcvjHvGXPJ7zWte93VvMlFxfOSabkfB91rdZjN8Crg5xDcIGrjKFco6lXIEvBt41/FQbbp0/ixvqBRPCphth/oCkQSaXgtCwlkKKxMnOviAUThzKzc8+I/gq3yvapJCdXHhFNCNuiRlENTEJGwkmMwVwxpONsSy+OCIavvymx5JWOgWiutw33ycaVp9O/WpxGLbelbaKudi7ECdaOP9LWHwtpdJx/nSFk7mVxmHwfeKyorqggHYNi0NrzQvHH+acApi4h1uAnwJ2ZC+60C+DBP1QnZaCStzOONt1mYCLUz+kvYBdtJZOz6Y0q7XSBCt5envgKoEtlaMp9ZMsLqxC3x4p3MTttB4x83MhME1pbeAauCrtIkggUWteNKyzklbX1sOtnu3QvxOilTTrUAZmDxK1lgk0/N6pR/mmAtRV+8mtrHBYwa6uYezQVWgXe/zSJe2llZNCEur1EZZ1tLrLwA42aMFng89ZK9gkR06EtxptHYzcJKA4WfT0E+If17AbzoNqL5zVjV3XTmY7Ysp9V/gsYQnKI2Bbaze3U8PNvAI4apFm98yjzBn2Ecw9Qti7m2GTY19YIy4Po00hHdy1hUbGSSVotWKv12pdkpYlO/WuEv+Eu8v53o9OOnEbpjRV/bSkc605SfXClQ14PLW2mJ3lBROB2FA2YPQmoCN9xUHeA3KdyYwCWDxCHDUxSvEXZx7jf7Ifo5Jy8NhtbM8VOBx3KA7ugr5dKJtB9Yk52QhaFajPQNhMzTco+3wTMYvmOZ+m6bVNMKsFaLcxQHWR+KO+pN5REj5s/pUMERmm81nPo11Kld+FEORg9Hq/2VGaUmY3WOegMYubPYhTOWwUce3ME7zJl5Lb7sfU/pql3l1s0zlujJrE3ePoxP9kTUUpKHRBaVAa+MKKFGaGP+DbsLYJFsz3CqvirJh3eVQwVVJzSCKyDTOmZCXp33y7RL5cWcy1PS/+KZuDrmrPBbmqTCU/MfmOvFwD+ndJtF3dgHVCS8qhcweJCTAu5ttnH+fb7MfwdWzSO6J4Bsp7OGAsvxeH5GwE+W7UKqDN5IRJMbo4CIU+yBUhWob6SYETSeU/6c8Yej+MfczXSLdLhG0vyKLK4x5LjYuvSvBeIoD6Kek2Su656gLUukwjYxmJkFxHynyuJbBCxbJpxeFM+8YGaWTiFuYPYjsNFLAcBHx25QgN36XpRHDPeK37EUu9uwlMjts6DWIbodvnxAyDUoaFUKlfMwXJCPJ9kCuJP4yrEC7IiZeuERFs3BQ8pA2Uh+X7xSFx9sUhefXsEC1deEXGDLpk3N7d8jtc8UDWym/n/S8dUCEQeq0KQVnov8I3F46k6V7PTh9cLkJYOAeLh6C+QByUMC7k908/xrO+6dKe3z+0H719llxc4CUUe/P9xIHCRIvQf5XvQYsvcu5KnR8EVIZqX72RTitdLkrzRJ2T6yrHfKvrntQ9mIJguZW35X3XVp/FCDglqqWVfGflmpGb1tL7NhQAnu2Ty6IAFciGLB2G2iKRKRHYNg9NPc7fX9+Nvrb0sRe98ghh/25ewPBAVEPlh1GmgKqfEFyk8xsk3+odaJtMk8vqscoow5BkqetYsozCoI9wHhPaSMBpEmC6tkoOWRzyv0dTGisoQr5HaPWgkEEBZu3GZ0uCOi9Ho5YOQ2OFWBcIXMThZpFPNX+XiO9eqCMT4JCY7MrDTHfvYC4o8Kn2ElFxDKPz0J4Reiise8rFXCA6XgIPprLNghnqJtEY4vM3Jd8jvjBhi7OnUI1vkyiNyjXlsQzlrs5vIVOB+NqwmnYIVE9Dp0OMxjjLnIu589yPOcutKJ5ZG7vP7zo/hGMfQfVpXuzZSZ/J85H2ImFpHOMUVI/47vKObypMIEbimLXHTBsB5Y1ooAzeCyteq2xK7yjW0VleECY43zVktcnFF9g8l+bKy29KrtxelCNLUUzsIoxTrBBxz/STHEyi6KWdczgpmld5mEd7drFTG65vLRIWRzFBrzAFnWUxCZTUXU3R+4+OgNPmnG+azl/cfF2GOlp4RViK5owvaCeV9fy7PH22Ltq6KHMsPUjrqe15CHpBd0GnRRR3CJRi0UXcO/0kL7KPgEc2Ppp5cc+0H81DxBN38y1leMDGRN3bMUEJ1Z4WCKLzon0m7yNfTSKo89zZ4NWSNNtW2pObpDyVo7KQnp3ivFfeFsElDdZJKtd7mfixdxKWDs4PRPUpmVdxUFxMfcozIUBhk5zCac4S4Qic4hgh9089wcF3OppkLjqZQ9KNXvkWP+rZRVEbPtZeQKmAuDiOVnmBGIn0o3oGUPq0JaxIUSVbQVp5S/Bd85w4+cKA79f1TIeLMoUWK36otClNo5JImY3ccUuE0jwj2K/QLxbQWRbmpTEtxaP2EuSGID8OnTq2eQ6nA4xzPNFu8qmZp5m8lPNc5h3V/hlprq78FY/37OQ0hrujGvm4SVQYRuWGUC4SkJlw9asO2OeZrdm0UmRDWWDcFG0NqxkGxXoSMlzbQhGURcDaCHuy8Iqg+uKQ3yjtG38Oy+bFLclQUFJ9q58SfjHoh9IEOI2rnyWOaxidQznHn0w9ygONk1ST41jv+UHfbXdxg9V8Q8ENSkFxjDjfj7FNCOdEA1fNUa1jK0yGwdFrCzeJH7sQTuy/UgjN6qQk+X2Xixa6OC3qzL8i6R/aF3tiOVSc64dgAOIQ15ojDqtyYtpZXo8jPnfmCZ7MeMZLagc3lyy4k1j2Eaw8xfToZXw3UigXc3NUJxfVsKYLmx9Gm24vEF+UUS6tXbLBgwtWe+HsxcuQrUWpzKtA2hhy3ekG6Zxvxp7yuamGoAcK45AfBWdwzTni5gwGi1aaqrX8h3ief3z2Z7ztO8nf1RMi1K/0FBt/DHvLXbxfaR7E8dtKQVDC5YexuS60i1BxBaKKBAYXb3DQ451Ke+u7ljxM6fWH43Biou0l0UJyEiVNlwSnqIZtL+LiJsanUm0lT4X4j1N/w9Ff56EGv+rjM1Yjrn/0zx0u5vPOcnfSzJfvI871gs6hXQcVN8SMbctroz1/FmoD3s5tQHg6T2PpolDeuiBZAgoXNXGdCjaqYpxFeUiziOJ7yvFnpx71afqv+RwS9Ws/QwlWH9iy7V4+bC3/1MV8WhsGk5w06MIG3ViTlyNWRCjr2YvkaOZqVd6eT69jpCSoA1B5H0iUb2GJcHELF1ZxcYMgIVF9/vuSUvyFVnz/5CPMvJdPvnkvHxG0Opnt9zLmHJ+0Mfc7y4eVoV+b1B/pAug81hRwHloo9GoKrNZYrTT4OGfBhjgb4mwH4hbahqjV8xF61b+9phSPxhE/nH6cFy7y3Cf+7jyU6ktoDqGyPmPX/YxGbW6yMR91jg9iuRLHpgSnnVezUOcfAjmvcXG1iY4GiuNK8xqOn7iAn039Na9vgAbe84dTvbeCy467D8Ot2PU7PP5JhnMxe3DstZY9wGUOtijLqFP04CgBAQoNRAraTlFTsAScU5rjOI6geN0a3pwuM3mec08fhxb/htbH/wV/3RD8mmDNYwAAAABJRU5ErkJggg==', 'bombs/bomb64.png': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAYAAACqaXHeAAAMs0lEQVR4nN1ba2wcRx3/zWMf94rvnMTEkKdJ6jRxCG0pSdu0KdBSUrVVBaIqSAgh8UakpaI8xBf4wAc+AEWoICGBoPCJN+VRQIkoSZ9p3DZp0iQkcWLHsV3n4sSv273dnRk+xLMd760dn3NXXEZa7e3c7M7/9/s/Z/aO4P+kZRjJvK1grbhteXZH11J3k5SQL/Z73f8equwa9KKzvlR+2n3kjRa0Ga1o09YdHfm7PrKt7aNdG5dsWLQ0VxC+lAOHLgw9f6D8wh+OjPxuz7C325OqkryX/y8EvpK2elVx3bZtq7av71zalXGtfOCHwj43am8rhu/eeNPyjuzqdodYDlFeiJZlI4XFxVyrw6jTv2+o98hYcEgC0nzem4YAxqj1iY9f85nPffbdDyxdkls7MuKh4oXgUCiNTaDY3w8e+IAC0NIGkldwiGstHROt1wxMvPPWM+O39VaiUxORHDef+6YggHNqfePr27/9xS9sffjVI8PY3z2ACxe8SRFJ5UqR2yAmyaaiQKEaQZ7tAbMLQNsqoKhgtY/zxcsKpTUtTgenpAbvm4KAz3zq+oce2HnDw93dA3h+X//Q3qd6H+05NbLbkZJ+6rrSl9e9b9n7C8sWZZEJQQpZoPgWAAwAA3EsYmctK2OzDEmJeQuegJtuXHnHlx648StHj5axZ+/p3j/9+ejOgweHHgeAkktbyXpnuO3tRVXYvAGABSgOuAXAmwSqVSivqrwx3x8aDwalmu7/AEDfcER1NMfhuc9/dstObtHWF17oD574+/FvavAAEAmEPeXg5OTwREVVJgAnC+RbAeEDVQ/q4hiqQ6Phqd7R3j0Dk09WIjmZnGNBE3DXnZ33f+COtXcePPgadu0++fP93Wd/bn4/EcqJf56c/NurLw4f8XsGqxgZAsbPAReHIfvOqsqhM/6ZlwcGdh0q7zp00X85VCpMzrFgXcBxWO6++7o+NnLBx/7u/jP7uwd+nByjAHX6YnjqF08MPmYT7nRt8Ta4hbwjx6UcOzk6cfzAuZ7dR87t/uXxCz87H8py2jwLloCtW1bcvu2mVdtfemkQTz/T9+vBofGX08ZVIjn5x2Pjvx0cPzXwoSMXPry6LbsmmFTBiTOVE/86Nbb7+fPeM8OBGJppngVLwC03r75dCIlj/ymP/+f4+T/MNnYskqNPnqns6h7291mMcCmhIqGiSiQr1RlKYN0WJAGLFjntt9y8+vaREQ8HDg4+3d8/+uzl7gmVCkd8cb7euRZkEFyzptTVsaa0ZmBgDMeOlfeolPTVqLYgCVjfuXST43D+2vCkGnpt4sVmzrUgCbhq3eKuMJI4ceJ83/nzlcPNnGvBEUAI6MoVLR2+F2J4eLKvUglfa+Z8CyIIEkIW5fP5T1qW5Sxe3DKSy5dWj44FKJcn+6SsLV4a2RYEAZZlXdvS0vKd1tZWvnnzZhx4pYR/PXkKJ076q23bfn8QBHsBeM2Ye0G4gGVZ7+Kcc9d1kc1mcfFiBUePncfomLgpn8//NZvN/p5SuroZc7+RFkAZYys551czxtZSSvMALADUcZx7pJQghIAQAs/zUKlUEIYhpJScMbapWUI1nQDHcW51HOcO27avtSyri1L6VkJeX5YrpQAAURShr68P5XIZjuNgcnISQggopSCE2C+lPN0M+ZpCACEk77rujkwmc18mk9nBGMtp0PqslIrBm2fP8+B5XgyeMYYwDEdxyVoaHhBZox/oOM57i8XiIy0tLV93XXcDY8wmhIDSS+FGn7W568PsS37POX8HpXS9lHJIStnXSHkbRgAhJJPP5x8slUrfd113M6UUlFJo8OZ1GkjTLZL9hBDCOd9oWdbdAIpCiBcBzLrImWtrCAGc86tLpdIPFi1a9CDnPGeC1p/Nc5KEtLNuCWJyjLGbAbQJIZ5CA1LjFRPAGOsolUo/yuVydzPGwBibBtTUvj70mDSLMM/A6/FB90898xpCyFohxD6l1MUrkv+KbmZsZbFYfCSXy92ZBG6C5JxDk2NZVo1rmDHCJCEJPjH3BgCdQoinAcybhHkTQCktFovFH+ZyuQ/OpnXLsmDbNjjn8Vkf5n2U0jgzmMB1S/YppUApXaeUygshngDmt2SeNwH5fP5rhULh80kQWvsaoG3bsG0bjuMgk8nE15ZlxQQA0zWuz2bKTDY9nhByrZRyWEr5wnxwzIsAy7JuKBaL37UsK5emecuywDmHLm2z2WxsAZZlwXEcWJY1zR2mwNTUByb4tLphiqRNQohupVRvvVjmUwjZ+Xx+J+e8LZnOdIDTJu66LhzHgeu6sfkzxqCUgpQSURTBtu2YhCAIYtBRFMXj0kCbxHDOV3DOHwyC4CnU6Qp1E5DJZO7PZDL3p4E3A57ruigUCuCcI5fLIZfL6aIGQggIISClhOd504BJKSGEQBRFNRnC0Pi0uAEAjLEdlNLbpZT/aCoBjuPcqzWmmxZM+7/p967rIp/Pw3Xd2PyFEKhWq/B9H0IIhGGIarUKIURNUKSUQghRI0eKAlxK6X1NJYBzfq3jOLfOVMmZFsA5B6UUtm3HJCRjhAYrhEAQBDEBwOslsxCipkpMqw2myL9HCHG9UmrOAbGu/QDHce5gjJWS/Ukz1fleW4I2/Uwmg2w2G8cEy7JisGZWsCyrpjw2n28SoGPElAKWUEpvqwdTPQRw27bfmzR9M2UxdimpJCvCZN43A6XjOPE4KeU0d0pLf3ouE3jiu+vqwDR3AiilyznnG2bSjNnMgoZSGkd8DVAHQX2Y92hwZgyYJf3VyEAI2UgIWdoMAt5GKV2S1LopmN7VkVIiDEMEQYAoiuIjCIK4v1qtxp/1OA1YP1NKWZMGTZJN8JoUSulaQsjGueKacxBkjC0nhNjJwsT0S9MsdZ/O7YyxOMhFUYQwDOOtL8/zYguZ2gaLN0Rm2jgxZUiQwwGsbDgBlNJVGlwSvCmITmuUUlSrVXB+aQqtccZYqoVo69BaNwnQrpMkIa1Immp2wwkghGRNjSTL16R2CSGwbRthGMZ9uvTVvh8EQUxGtVqNx2nwJnDddN9l4oLVcALMdbcpgNaCDnbaAvT3QghwzmOrABC7g9a67/vwfT++jqIorhRNEvRcphymPEZsyjScACnl5EzaN82fEBJbgNaoPpsBToPU5q8zhY4FSbOfye/TrAF1BPd6CBg105OeUJOgtWOWrRqgrgs459PGSinjEliTYmrfdIPkkawTEiTN+IuQeROglBo3J9LpztzNSWpN+7omILnlZWpb+75ZK6RE+Br3S5o/AF8pdbwZBJwTQlQIIVkTfJomkilMKTXNBXTTMUNbihk39LPMSJ8WD1KC5YBSas5b53MmQAhxOIqig5zzrWbETbqDXtyYwpubJsnYkQQ7U+qbyzHlnj1KqTm/Uq9nNeiFYfisbdtbkxaQXJtrAOZuD4DUz9oF0tKbqeU0fzdJ0muOIAgOo45NkbpWg1EUPTNTlE4LWmZRo9Ojvj8t4M1F+2Z5DLweB6b2GkIp5RP1YKqLACHEs0KIV9NMNalF3WcC1IDMHG/eb7a0OZIBUPfbto22tjbYtv1UFEVPNo0ApdTZarX602SRYgqfZtZ6vLkVNhMJSSLTLMocTwhBe3s7li1bhjAMH1dKVZtGAABEUfRYGIb70gRNatzUmA6MQG36momAZKRPkqyUguu6WLFiBRhjR8fHxx+vlbjBBCilykEQ/GQmTWoBk8QkAaa5zGyBLs11LMtCZ2cn2tvbVX9//yO+7/fUi2de7wWklEcJIV2U0vXmBonp5+Y5Ccq8TvYnyUjL//q6o6MDW7duRU9Pz68OHDjwLaVU3W+H5vtmKJJS9lJKtwBo04KmbWMlwVyuL+06LdAuXrwYnZ2dmJiYOPzcc8895Pv+4HyAXMnL0X6lVA8h5GZCSNH8Ig1gcu2eHJ+8z1wvJN0pk8lg5cqVkFKeOXz48FfL5fK/5wviit4OK6VOSinPAngPISRrgjHG1Hyerdgx+9KCJWMMuVwOlNLe06dPP1Aul2f9Jfnl2hX/PkAp9aq6VHpuB5BJgjGv9ee0vrQjSYDeZCGEHB0ZGdk5MTHxlyuVvyG/EFFKHZBSHgWwBsDyqT4kz2mlbRpwIH2lN/Veca/neQ8FQbC7EbI38kdSx6SUu6WUvlKqkxCSmwvIy1V5ulFKz0gpvxcEwTeklIcaJXSz/ju8hVL6aUrp/Xr5PNvrLaCWJN0YYxWl1G+klI8KIeb1G4DZWlP/PE0IuRXAvYSQHQCuSi6JgdqYMDVmHMDLAPYC+IcQYk/TZGzWgxOtHcB1ADYTQtYBeItSagkuBU02BfocgB6lVDeAZ5VSrwCoq66fT/svI6ZK6kYprOYAAAAASUVORK5CYII='}
for relative_path, encoded in EMBEDDED_SPRITES.items():
    destination = ASSET_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        destination.write_bytes(base64.b64decode(encoded))
print(f"Runtime assets ready under {ASSET_ROOT.parent.resolve()}")

# Construct exactly one detector. It validates the active ROCm runtime and
# uses AUP_EXPECTED_GFX when set, otherwise the runtime's normal default.
EXPECTED_GFX = os.environ.get("AUP_EXPECTED_GFX")
pose_detector = PoseDetector(
    model_path=model_path,
    expected_gfx=EXPECTED_GFX,
    confidence=HAND_CONFIDENCE,
    keypoint_confidence=HAND_KEYPOINT_CONFIDENCE,
    image_size=CANVAS_WIDTH,
)

## 🧪 Parameter experiments and tips
Change one value at a time, then rerun the gameplay cell to feel the effect:

- Raise `BOMB_PROBABILITY` for a harder game (more bombs).
- Increase `TOUCH_RADIUS` if slices feel unresponsive.
- Tweak the spawn speed ranges to alter fruit trajectories.
- Lower `JPEG_QUALITY` to speed up inline display on slower machines.
- Toggle `SHOW_FPS` and change `FPS_SMOOTH_SAMPLES` for different HUD behavior.
- Leave `FRUIT_NINJA_MAX_FRAMES` unset (or `0`) for the normal unlimited loop.

## Step 2: Load Assets (Sprites)
We load fruit and bomb sprites from the Notebook-managed `runtime_assets/assets/` directory. If a sprite is missing, a colored placeholder is created so the demo still runs.

**💡 Why RGBA.** Sprites carry an alpha channel so fruit and bombs blend over the camera image instead of sitting inside an opaque rectangle. Each pixel's alpha value decides how much of the sprite shows through.

**🔍 Observe.** The cell reports how many fruit and bomb sprites loaded. A count of zero means the asset download step has not run yet.

In [ ]:
def create_placeholder_sprite(size=(64, 64), color=(0, 255, 0, 255)) -> np.ndarray:
    """Create a solid-color RGBA placeholder sprite with a small label."""
    sprite = np.zeros((size[1], size[0], 4), dtype=np.uint8)
    sprite[..., :3] = color[:3]
    sprite[..., 3] = color[3]
    cv2.putText(sprite, "??", (8, size[1] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0, 255), 2, cv2.LINE_AA)
    return sprite


def cvt_rgba_to_bgra(image: np.ndarray) -> np.ndarray:
    """Convert RGBA to BGRA if needed (OpenCV uses BGR ordering)."""
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_RGBA2BGRA)
    return image


def load_png_sprite(path: Path) -> np.ndarray:
    """Load a PNG sprite with alpha; add opaque alpha if missing."""
    sprite = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if sprite is None:
        raise FileNotFoundError(f"Failed to load sprite: {path}")
    if sprite.shape[2] == 3:
        alpha = np.full((*sprite.shape[:2], 1), 255, dtype=np.uint8)
        sprite = np.concatenate([sprite, alpha], axis=2)
    return sprite


def load_gif_first_frame(path: Path) -> np.ndarray:
    """Load the first frame from a GIF as BGRA."""
    with PILImage.open(path) as gif:
        frame = gif.convert("RGBA")
        frame_np = np.array(frame)
    return cvt_rgba_to_bgra(frame_np)


def load_assets() -> dict:
    """Load fruit and bomb assets from disk, falling back to placeholders."""
    assets = {"fruits": [], "bombs": []}

    if FRUIT_ASSET_DIR.exists():
        for fruit_path in sorted(FRUIT_ASSET_DIR.glob("*.png")):
            try:
                assets["fruits"].append(load_png_sprite(fruit_path))
            except FileNotFoundError as exc:
                print(exc)
    if not assets["fruits"]:
        assets["fruits"].append(create_placeholder_sprite(color=(0, 200, 0, 255)))

    if BOMB_ASSET_DIR.exists():
        for bomb_path in sorted(BOMB_ASSET_DIR.glob("*")):
            if bomb_path.suffix.lower() not in {".png", ".gif"}:
                continue
            try:
                if bomb_path.suffix.lower() == ".gif":
                    assets["bombs"].append(load_gif_first_frame(bomb_path))
                else:
                    assets["bombs"].append(load_png_sprite(bomb_path))
            except FileNotFoundError as exc:
                print(exc)
    if not assets["bombs"]:
        assets["bombs"].append(create_placeholder_sprite(color=(60, 60, 60, 255)))

    return assets


ASSETS = load_assets()
print(f"Loaded assets: {len(ASSETS['fruits'])} fruit(s), {len(ASSETS['bombs'])} bomb(s)")


## Step 3: Ultralytics Input Canvas
The game resizes each BGR camera frame to a 640x640 canvas used for both display and inference.

**💡 How a frame becomes a model input.** We intentionally do not reimplement model internals. `PoseDetector.predict` hands this NumPy canvas to the Ultralytics public predictor, which performs letterboxing (resize while keeping aspect ratio, then pad), BGR→RGB color conversion, pixel normalization to 0–1, and packing into an NCHW tensor placed on the PyTorch HIP device (`cuda:0`). A fixed square input keeps geometry predictable for the game.

**🔍 Observe.** This helper only returns a canvas; it does not detect anything on its own. You will see its effect once the loop runs.

In [ ]:
def prepare_canvas(frame: np.ndarray, img_size: int = 640) -> np.ndarray:
    """Create the square BGR canvas passed to Ultralytics and the renderer."""
    if frame is None or frame.ndim < 2:
        raise ValueError("frame must be a non-empty image array")
    return cv2.resize(frame, (img_size, img_size), interpolation=cv2.INTER_LINEAR)


## Step 4: Pose Results and Rendering Helpers
The public adapter reads Ultralytics `Results` fields—`result.boxes.xyxy`, `result.boxes.conf`, `result.keypoints.xy`, and `result.keypoints.conf`—and uses COCO wrist indices 9 (left) and 10 (right).

**💡 Person box vs. keypoints.** A pose model returns two related things: a bounding box that says *where a person is*, and keypoints that say *where each body joint is*. This game only needs the two wrists, so the adapter selects keypoints 9 and 10 and ignores the rest.

**💡 Confidence is a trade-off.** Every detection carries a confidence score. A higher wrist-confidence threshold rejects noisy points (steadier blades, but a fast hand can drop out); a lower threshold keeps more points (more responsive, but jitter increases). The adapter deliberately ignores low-confidence wrists so a stray point cannot fling the blade.

**🔍 Observe.** The adapter returns clipped hand dictionaries with `center`, `box`, `confidence`, and player identity, letting the lesson focus on rendering and collisions rather than private tensors.

In [ ]:
# PoseDetector.predict already extracts validated COCO wrist points into hand dictionaries.
# The remaining helpers keep rendering and collision behavior independent of inference.

def draw_transparent(canvas: np.ndarray, sprite: np.ndarray, center: Tuple[float, float]) -> None:
    """Overlay a sprite with alpha channel onto the canvas using fast blending."""
    if sprite is None or sprite.size == 0:
        return

    sprite_h, sprite_w = sprite.shape[:2]
    canvas_h, canvas_w = canvas.shape[:2]
    cx, cy = center
    top_left_x = int(cx - sprite_w / 2)
    top_left_y = int(cy - sprite_h / 2)

    # Early out when fully off-screen
    if top_left_x >= canvas_w or top_left_y >= canvas_h:
        return
    if top_left_x + sprite_w <= 0 or top_left_y + sprite_h <= 0:
        return

    # Clip to canvas
    x1 = max(top_left_x, 0)
    y1 = max(top_left_y, 0)
    x2 = min(top_left_x + sprite_w, canvas_w)
    y2 = min(top_left_y + sprite_h, canvas_h)

    sprite_x1 = x1 - top_left_x
    sprite_y1 = y1 - top_left_y
    sprite_x2 = sprite_x1 + (x2 - x1)
    sprite_y2 = sprite_y1 + (y2 - y1)

    roi = canvas[y1:y2, x1:x2]
    sprite_region = sprite[sprite_y1:sprite_y2, sprite_x1:sprite_x2]

    if sprite_region.shape[2] == 4:
        if USE_FAST_ALPHA_BLEND:
            # Vectorized integer alpha blending
            alpha_u8 = sprite_region[..., 3]
            color = sprite_region[..., :3]

            alpha_expanded = alpha_u8[..., np.newaxis].astype(np.uint16)
            inv_alpha = (255 - alpha_u8)[..., np.newaxis].astype(np.uint16)
            fg = color.astype(np.uint16)
            bg = roi.astype(np.uint16)

            blended = (alpha_expanded * fg + inv_alpha * bg) // 255
            roi[:] = blended.astype(np.uint8)
        else:
            alpha = sprite_region[..., 3:] / 255.0
            color = sprite_region[..., :3]
            roi[:] = (alpha * color + (1 - alpha) * roi).astype(np.uint8)
    else:
        roi[:] = sprite_region


def render_text(img: np.ndarray,
                text: str,
                position: Tuple[int, int],
                font_scale: float = 0.8,
                color: Tuple[int, int, int] = (255, 255, 255),
                thickness: int = 2) -> None:
    cv2.putText(img, text, position, cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness, cv2.LINE_AA)


def draw_touch_zone(canvas: np.ndarray,
                    top_left: Tuple[int, int],
                    bottom_right: Tuple[int, int],
                    label: str,
                    color: Tuple[int, int, int] = (60, 160, 255),
                    progress_seconds: Optional[float] = None) -> None:
    """Render a semi-transparent confirmation zone with centered text and progress."""
    overlay = canvas.copy()
    cv2.rectangle(overlay, top_left, bottom_right, color, -1)
    cv2.addWeighted(overlay, 0.25, canvas, 0.75, 0, canvas)

    zone_width = bottom_right[0] - top_left[0]
    zone_height = bottom_right[1] - top_left[1]

    label_scale = TOUCH_ZONE_LABEL_SCALE
    label_thickness = TOUCH_ZONE_LABEL_THICKNESS
    (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, label_scale, label_thickness)
    label_x = top_left[0] + (zone_width - label_w) // 2
    label_y = top_left[1] + (zone_height + label_h) // 2 - baseline
    render_text(canvas, label, (label_x, label_y), font_scale=label_scale, color=(255, 255, 255), thickness=label_thickness)

    if progress_seconds is not None:
        clamped = max(0.0, min(progress_seconds, TOUCH_CONFIRM_SECONDS))
        ratio = clamped / TOUCH_CONFIRM_SECONDS if TOUCH_CONFIRM_SECONDS > 0 else 0.0
        progress_width = int(zone_width * ratio)
        bar_top = bottom_right[1] - 12
        bar_bottom = bar_top + 6
        if progress_width > 0:
            cv2.rectangle(canvas,
                          (top_left[0], bar_top),
                          (top_left[0] + progress_width, bar_bottom),
                          (255, 255, 255),
                          -1)
        progress_text = f"Held: {clamped:.1f}s / {TOUCH_CONFIRM_SECONDS:.0f}s"
        progress_scale = TOUCH_ZONE_PROGRESS_SCALE
        progress_thickness = 1
        (progress_w, _), _ = cv2.getTextSize(progress_text, cv2.FONT_HERSHEY_SIMPLEX, progress_scale, progress_thickness)
        progress_x = top_left[0] + (zone_width - progress_w) // 2
        progress_y = bottom_right[1] - 18
        render_text(canvas, progress_text, (progress_x, progress_y), font_scale=progress_scale, color=(255, 255, 255), thickness=progress_thickness)


def update_touch_timer(is_active: bool, elapsed: float, dt: float) -> float:
    """Increment or reset a hold-to-confirm timer."""
    return min(elapsed + dt, TOUCH_CONFIRM_SECONDS) if is_active else 0.0


## Step 5: Game Objects and State Management
We define the game states and data structures. `FlyingObject` models fruit/bomb physics; `GameState` holds runtime state and timers.

**💡 Frame-rate-independent motion.** Each object updates its position using a time delta Δt (seconds since the last frame) instead of assuming a fixed frame rate: position += velocity × Δt, and velocity gains gravity × Δt each step. This way fruit follows the same arc whether your machine runs fast or slow.

**🔍 Observe.** This cell only defines classes and the initial state; no window appears yet.

In [ ]:
# Game states
STATE_MENU = "menu"
STATE_PLAYING = "playing"
STATE_GAME_OVER = "game_over"


@dataclass(slots=True)
class FlyingObject:
    """Represents a fruit or bomb flying across the screen."""
    sprite: np.ndarray
    position: np.ndarray
    velocity: np.ndarray
    radius: float
    score_value: int
    kind: str
    active: bool = True

    def update(self, dt: float) -> None:
        """Update object's position with gravity; deactivate when out of bounds."""
        if not self.active:
            return
        self.velocity[1] += GRAVITY * dt
        self.position += self.velocity * dt
        if (self.position[1] - self.radius) > CANVAS_HEIGHT + 80:
            self.active = False
        if self.position[0] + self.radius < -80 or self.position[0] - self.radius > CANVAS_WIDTH + 80:
            self.active = False


@dataclass
class GameState:
    """Container for the runtime state of the demo."""
    state: str = STATE_MENU
    score: int = 0
    start_time: float = 0.0
    last_spawn_time: float = 0.0
    next_spawn_interval: float = 1.0
    objects: List[FlyingObject] = field(default_factory=list)
    final_score: int = 0
    menu_touch_elapsed: float = 0.0
    restart_touch_elapsed: float = 0.0
    home_touch_elapsed: float = 0.0
    fps_samples: deque = field(default_factory=deque)
    fps_dt_sum: float = 0.0

    def reset_for_menu(self) -> None:
        self.state = STATE_MENU
        self.score = 0
        self.objects.clear()
        self.final_score = 0
        self.start_time = 0.0
        self.last_spawn_time = 0.0
        self.next_spawn_interval = random.uniform(*SPAWN_INTERVAL_RANGE)
        self.menu_touch_elapsed = 0.0
        self.restart_touch_elapsed = 0.0
        self.home_touch_elapsed = 0.0
        self.fps_samples.clear()
        self.fps_dt_sum = 0.0

    def start_game(self, current_time: float) -> None:
        self.state = STATE_PLAYING
        self.score = 0
        self.objects.clear()
        self.start_time = current_time
        self.last_spawn_time = current_time
        self.next_spawn_interval = random.uniform(*SPAWN_INTERVAL_RANGE)
        self.menu_touch_elapsed = 0.0
        self.restart_touch_elapsed = 0.0
        self.home_touch_elapsed = 0.0
        self.fps_samples.clear()
        self.fps_dt_sum = 0.0

    def finish_game(self) -> None:
        self.state = STATE_GAME_OVER
        self.final_score = self.score
        self.objects.clear()
        self.menu_touch_elapsed = 0.0
        self.restart_touch_elapsed = 0.0
        self.home_touch_elapsed = 0.0
        self.fps_samples.clear()
        self.fps_dt_sum = 0.0


def choose_spawn_preset() -> str:
    """Randomly select a spawn location preset using predefined weights."""
    return random.choices(SPAWN_PRESETS, weights=SPAWN_WEIGHTS, k=1)[0]


def compute_spawn_parameters(sprite_w: int, sprite_h: int) -> Tuple[str, np.ndarray, np.ndarray]:
    """Compute initial position and velocity based on spawn preset."""
    preset = choose_spawn_preset()

    if preset == "bottom_center":
        x_pos = random.uniform(CANVAS_WIDTH * 0.3, CANVAS_WIDTH * 0.7)
        y_pos = CANVAS_HEIGHT + sprite_h / 2 + 12
        vx = random.uniform(BOTTOM_HORIZONTAL_SPEED_RANGE[0], BOTTOM_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*BOTTOM_VERTICAL_SPEED_RANGE)
    elif preset == "bottom_left":
        x_pos = random.uniform(sprite_w / 2 + 30, CANVAS_WIDTH * 0.22)
        y_pos = CANVAS_HEIGHT + sprite_h / 2 + 12
        vx = random.uniform(240.0, SIDE_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*BOTTOM_VERTICAL_SPEED_RANGE)
    elif preset == "bottom_right":
        x_pos = random.uniform(CANVAS_WIDTH * 0.78, CANVAS_WIDTH - sprite_w / 2 - 30)
        y_pos = CANVAS_HEIGHT + sprite_h / 2 + 12
        vx = -random.uniform(240.0, SIDE_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*BOTTOM_VERTICAL_SPEED_RANGE)
    elif preset == "left_edge":
        x_pos = -sprite_w / 2 - 16
        y_pos = random.uniform(CANVAS_HEIGHT * 0.35, CANVAS_HEIGHT * 0.75)
        vx = random.uniform(SIDE_HORIZONTAL_SPEED_RANGE[0], SIDE_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*SIDE_VERTICAL_SPEED_RANGE)
    elif preset == "right_edge":
        x_pos = CANVAS_WIDTH + sprite_w / 2 + 16
        y_pos = random.uniform(CANVAS_HEIGHT * 0.35, CANVAS_HEIGHT * 0.75)
        vx = -random.uniform(SIDE_HORIZONTAL_SPEED_RANGE[0], SIDE_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*SIDE_VERTICAL_SPEED_RANGE)
    else:  # top_edge
        x_pos = random.uniform(sprite_w / 2 + 30, CANVAS_WIDTH - sprite_w / 2 - 30)
        y_pos = -sprite_h / 2 - 16
        vx = random.uniform(TOP_HORIZONTAL_SPEED_RANGE[0], TOP_HORIZONTAL_SPEED_RANGE[1])
        vy = random.uniform(*TOP_VERTICAL_SPEED_RANGE)

    position = np.array([x_pos, y_pos], dtype=np.float32)
    velocity = np.array([vx, vy], dtype=np.float32)
    return preset, position, velocity


def choose_sprite(kind: str) -> np.ndarray:
    """Select a sprite for the requested object type."""
    if kind == "bomb":
        return random.choice(ASSETS["bombs"])
    return random.choice(ASSETS["fruits"])


def create_flying_object(kind: str, current_time: float) -> FlyingObject:
    """Instantiate a fruit or bomb with randomized trajectory parameters."""
    _ = current_time  # reserved for potential time-based tuning
    sprite = choose_sprite(kind)
    sprite_h, sprite_w = sprite.shape[:2]

    preset, position, velocity = compute_spawn_parameters(sprite_w, sprite_h)

    score_value = -BOMB_PENALTY if kind == "bomb" else FRUIT_SCORE

    # Keep bombs from top visible slightly longer
    if kind == "bomb" and preset == "top_edge":
        velocity[1] *= 0.85

    radius = max(sprite_w, sprite_h) * 0.42

    return FlyingObject(sprite=sprite,
                        position=position,
                        velocity=velocity,
                        radius=radius,
                        score_value=score_value,
                        kind=kind)


def cleanup_inactive_objects(game_state: GameState) -> None:
    """Remove inactive objects from the game state."""
    game_state.objects[:] = [obj for obj in game_state.objects if obj.active]


def spawn_objects(game_state: GameState, current_time: float) -> None:
    """Spawn objects when active count and cooldown allow."""
    if len(game_state.objects) >= MAX_ACTIVE_OBJECTS:
        return
    if current_time - game_state.last_spawn_time < game_state.next_spawn_interval:
        return

    slots = MAX_ACTIVE_OBJECTS - len(game_state.objects)
    spawn_count = random.randint(1, min(3, slots))
    for _ in range(spawn_count):
        kind = "bomb" if random.random() < BOMB_PROBABILITY else "fruit"
        game_state.objects.append(create_flying_object(kind, current_time))

    game_state.last_spawn_time = current_time
    game_state.next_spawn_interval = random.uniform(*SPAWN_INTERVAL_RANGE)


def update_objects(game_state: GameState, dt: float) -> None:
    """Advance all active objects by elapsed time."""
    for obj in game_state.objects:
        obj.update(dt)


def handle_collisions(game_state: GameState, hand_points: List[dict]) -> None:
    """Resolve hand-object intersections and apply score effects."""
    if not hand_points or not game_state.objects:
        return

    if VECTORIZE_COLLISION and len(hand_points) > 0:
        # Vectorized detection for multiple hands
        hand_centers = np.array([h["center"] for h in hand_points], dtype=np.float32)
        for obj in game_state.objects:
            if not obj.active:
                continue
            deltas = hand_centers - obj.position
            distances = np.linalg.norm(deltas, axis=1)
            if np.any(distances <= (obj.radius + TOUCH_RADIUS)):
                obj.active = False
                game_state.score += obj.score_value
    else:
        # Fallback (single-hand scenarios)
        for hand in hand_points:
            hx, hy = hand["center"]
            for obj in game_state.objects:
                if not obj.active:
                    continue
                distance = math.hypot(obj.position[0] - hx, obj.position[1] - hy)
                if distance <= (obj.radius + TOUCH_RADIUS):
                    obj.active = False
                    game_state.score += obj.score_value


def draw_objects(canvas: np.ndarray, objects: List[FlyingObject]) -> None:
    """Blit all active sprites with alpha support onto the frame."""
    for obj in objects:
        if obj.active:
            draw_transparent(canvas, obj.sprite, tuple(obj.position))


## Step 6: UI and HUD Rendering
We draw the score, time left, optional FPS, the menu and game-over screens with hold-to-confirm touch zones, and highlight detected hands for feedback.

**💡 Hold-to-confirm and circle collisions.** Menu and game-over buttons require holding a hand inside a zone for a moment so an accidental pass does not trigger them. Slicing uses a simple circular test: if the distance between a wrist center and an object center is smaller than their combined radius, it counts as a hit. Circles are cheap to compute and feel fair for round fruit.

**🔍 Observe.** These are drawing helpers; you will see the HUD once the gameplay loop runs.

In [ ]:
def within_zone(point: Tuple[float, float], zone: Tuple[Tuple[int, int], Tuple[int, int]]) -> bool:
    """Check whether a point lies within an axis-aligned rectangle."""
    (x1, y1), (x2, y2) = zone
    return x1 <= point[0] <= x2 and y1 <= point[1] <= y2


def render_fps_overlay(canvas: np.ndarray, fps_values: Optional[Tuple[float, float]]) -> None:
    """Render an FPS overlay when enabled."""
    if not SHOW_FPS or fps_values is None:
        return
    inst_fps, avg_fps = fps_values
    render_text(canvas, f"FPS: {inst_fps:05.1f}", (14, 32), font_scale=0.6)
    render_text(canvas, f"Avg: {avg_fps:05.1f}", (14, 56), font_scale=0.6)


def draw_hud(canvas: np.ndarray, score: int, time_left: float, fps_values: Optional[Tuple[float, float]] = None) -> None:
    """Render score, time remaining, and optional FPS information."""
    render_text(canvas, f"Score: {score}", (CANVAS_WIDTH - 200, 32), font_scale=0.8)
    render_text(canvas, f"Time: {max(0, int(time_left))} s", (CANVAS_WIDTH - 210, CANVAS_HEIGHT - 20), font_scale=0.7)
    render_fps_overlay(canvas, fps_values)


def render_menu(canvas: np.ndarray, hold_seconds: float) -> None:
    """Render the start screen with a hold-to-start indicator."""
    overlay = canvas.copy()
    cv2.rectangle(overlay, (0, 0), (CANVAS_WIDTH, CANVAS_HEIGHT), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.35, canvas, 0.65, 0, canvas)
    render_text(canvas, "Fruit Ninja (Hand Edition)", (70, 160), font_scale=1.2, color=(255, 220, 120))
    render_text(canvas, "Hold your hand over the zone to begin", (30, 220), font_scale=0.8, color=(255, 255, 255))
    draw_touch_zone(canvas, MENU_TOUCH_ZONE[0], MENU_TOUCH_ZONE[1], "Hold 3s to Start", progress_seconds=hold_seconds)
    render_text(canvas, "Press Ctrl+C to exit", (140, CANVAS_HEIGHT - 40), font_scale=0.7, color=(255, 255, 255))


def render_game_over(canvas: np.ndarray,
                     final_score: int,
                     restart_hold: float,
                     home_hold: float) -> None:
    """Render the game-over screen with hold-to-confirm actions."""
    overlay = canvas.copy()
    cv2.rectangle(overlay, (0, 0), (CANVAS_WIDTH, CANVAS_HEIGHT), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.4, canvas, 0.6, 0, canvas)
    render_text(canvas, "Game Over", (CANVAS_WIDTH // 2 - 120, 160), font_scale=1.4, color=(255, 200, 120))
    render_text(canvas, f"Final Score: {final_score}", (CANVAS_WIDTH // 2 - 150, 220), font_scale=0.9, color=(255, 255, 255))
    draw_touch_zone(canvas, RESTART_TOUCH_ZONE[0], RESTART_TOUCH_ZONE[1], "Hold 3s: Play Again", progress_seconds=restart_hold)
    draw_touch_zone(canvas, HOME_TOUCH_ZONE[0], HOME_TOUCH_ZONE[1], "Hold 3s: Main Menu", progress_seconds=home_hold)
    render_text(canvas, "Press Ctrl+C to exit", (140, CANVAS_HEIGHT - 40), font_scale=0.7, color=(255, 255, 255))


def highlight_hands(canvas: np.ndarray, hand_points: List[dict]) -> None:
    """Draw bounding boxes around detected hands for feedback."""
    for hand in hand_points:
        x1, y1, x2, y2 = hand["box"]
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (80, 220, 255), 2)


## Step 7: ROCm Pose Inference
The import/configuration cell already constructed one `PoseDetector` with the configured detection and wrist thresholds, validated the active ROCm runtime, and loaded the downloaded `runtime_assets/model/yolov8m-pose.pt` asset. This step runs one **warm-up inference** on a blank canvas so you can confirm the accelerator before the game loop drives it every frame. Internally the adapter calls the Ultralytics public API with `device=0` (PyTorch HIP `cuda:0`) and `quantize=16` for FP16 inference.

**💡 FP16 and memory.** Half-precision (FP16) weights use about half the memory of FP32 and run well on the GPU, which is why the adapter selects it (`quantize=16`). As a practical guide for the course setting (640×640), plan for roughly **8 GB system RAM** and **2 GB GPU/shared memory** at minimum, with **16 GB / 4 GB** comfortable. On an APU the GPU shares system memory, so close other heavy apps.

**🔍 Observe.** The cell prints the active AMD device name, the model file, the FP16 input size, and how long one inference took. Zero hands on a blank frame is expected—the game loop supplies real camera frames next.

**⚠️ Fail-closed.** GPU errors are allowed to propagate. A broken accelerator stops the lesson instead of silently switching to a device that would change the results.

In [ ]:
# Prove one ROCm pose inference runs before the game loop drives it every frame.
# A blank 640x640 canvas exercises the full predict path on cuda:0 and warms up
# the model; finding no hands here is expected.
import torch

probe_canvas = prepare_canvas(np.zeros((CANVAS_HEIGHT, CANVAS_WIDTH, 3), dtype=np.uint8), img_size=CANVAS_WIDTH)
probe_start = time.perf_counter()
probe_hands = pose_detector.predict(probe_canvas)
probe_ms = (time.perf_counter() - probe_start) * 1000.0

print(f"Device:        cuda:0 -> {torch.cuda.get_device_name(0)}")
print(f"Model:         {pose_detector.model_path.name}")
print(f"Input:         {pose_detector.image_size}x{pose_detector.image_size}, FP16 (quantize=16)")
print(f"Warm-up infer: {len(probe_hands)} hand(s) on a blank frame, {probe_ms:.0f} ms")

## Step 8: Full Gameplay Loop
This cell keeps the normal webcam-at-index-0 behavior: the menu start gesture, game-over controls, sprites, physics, scoring, and inline JPEG rendering tie together into one loop—capture a frame, run pose inference, update objects, test collisions, draw, and repeat.

**💡 Why FPS matters.** A real-time game must finish all of that fast enough that motion feels smooth; the optional FPS readout tells you whether the loop keeps up. If it drops, lower `JPEG_QUALITY` or close other GPU work.

**🔍 Observe.** Raise a hand into the menu zone to start, slice fruit, avoid bombs, and watch the score and timer. Detected wrists are highlighted so you can see what the model tracks.

**🧪 Try it.** Revisit the Step 1 thresholds and the parameter tips above, change one value, and rerun this cell to feel the difference.

**🔧 Smoke run (optional).** For a finite remote smoke run, set `FRUIT_NINJA_VIDEO_SOURCE` to a video path, `FRUIT_NINJA_MAX_FRAMES` to a positive count, and `FRUIT_NINJA_AUTOSTART=1`; absent or zero max frames stays unlimited and the default state stays the menu. The outer `finally` releases OpenCV resources, and the final `FRUIT_NINJA_SMOKE_SUMMARY` line reports machine-readable frame, hand-frame, elapsed-time, and FPS values.

**✅ Recap.** You turned webcam frames into model inputs, read wrist keypoints from public results, and connected them to physics, collisions, scoring, and UI—one complete ROCm-powered game loop.

In [ ]:
# Webcam/video capture and gameplay loop (teaching run)
cap = None
frames_processed = 0
frames_with_hands = 0
run_started = time.perf_counter()

try:
    video_source = parse_video_source(os.environ.get("FRUIT_NINJA_VIDEO_SOURCE"))
    max_frames = parse_max_frames(os.environ.get("FRUIT_NINJA_MAX_FRAMES"))
    autostart = parse_autostart(os.environ.get("FRUIT_NINJA_AUTOSTART"))
    print(f"Opening Fruit Ninja video source: {video_source!r}")
    if isinstance(video_source, int) and platform.system() == "Windows":
        cap = cv2.VideoCapture(video_source, cv2.CAP_DSHOW)
    else:
        cap = cv2.VideoCapture(video_source)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CANVAS_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CANVAS_HEIGHT)

    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open video source {video_source!r}. Check the webcam/path and permissions."
        )

    # Initialize runtime state
    game_state = GameState()
    game_state.reset_for_menu()
    if autostart:
        game_state.start_game(time.perf_counter())

    last_time: Optional[float] = None
    fps_tracking_started = False


    while max_frames == 0 or frames_processed < max_frames:
        ret, frame = cap.read()
        if not ret:
            if max_frames > 0:
                raise RuntimeError(
                    f"Video source {video_source!r} ended after {frames_processed} frame(s); "
                    f"requested {max_frames}."
                )
            print("Unable to read frame from video source, stopping loop.")
            break
        # Mirror for natural interaction
        frame = cv2.flip(frame, 1)
        canvas = prepare_canvas(frame, img_size=CANVAS_WIDTH)

        # Time delta for physics and timers
        current_time = time.perf_counter()
        if last_time is None:
            last_time = current_time
            dt = 1e-4
        else:
            dt = current_time - last_time
            last_time = current_time
            dt = max(1e-4, min(dt, 0.25))
            fps_tracking_started = True

        # FPS tracking
        fps_values: Optional[Tuple[float, float]] = None
        if fps_tracking_started and SHOW_FPS and FPS_SMOOTH_SAMPLES > 0:
            while len(game_state.fps_samples) >= FPS_SMOOTH_SAMPLES:
                game_state.fps_dt_sum -= game_state.fps_samples.popleft()
            game_state.fps_samples.append(dt)
            game_state.fps_dt_sum += dt
            avg_dt = game_state.fps_dt_sum / len(game_state.fps_samples) if game_state.fps_samples else dt
            avg_fps = 1.0 / avg_dt if avg_dt > 0 else 0.0
            instant_fps = 1.0 / dt if dt > 0 else 0.0
            fps_values = (instant_fps, avg_fps)

        # Ultralytics handles preprocessing and returns public pose-derived hand dictionaries.
        # Do not catch inference failures: the outer finally still releases capture resources.
        hand_points = pose_detector.predict(canvas)
        # State machine
        if game_state.state == STATE_MENU:
            start_touch_active = any(within_zone(hand["center"], MENU_TOUCH_ZONE) for hand in hand_points)
            game_state.menu_touch_elapsed = update_touch_timer(start_touch_active, game_state.menu_touch_elapsed, dt)
            start_game_requested = game_state.menu_touch_elapsed >= TOUCH_CONFIRM_SECONDS
            render_menu(canvas, game_state.menu_touch_elapsed)
            highlight_hands(canvas, hand_points)
            render_fps_overlay(canvas, fps_values)
            if start_game_requested:
                game_state.start_game(current_time)

        elif game_state.state == STATE_PLAYING:
            spawn_objects(game_state, current_time)
            update_objects(game_state, dt)
            handle_collisions(game_state, hand_points)
            cleanup_inactive_objects(game_state)
            draw_objects(canvas, game_state.objects)
            highlight_hands(canvas, hand_points)
            time_left = GAME_DURATION - (current_time - game_state.start_time)
            draw_hud(canvas, game_state.score, time_left, fps_values)
            if time_left <= 0:
                game_state.finish_game()

        elif game_state.state == STATE_GAME_OVER:
            restart_touch_active = any(within_zone(hand["center"], RESTART_TOUCH_ZONE) for hand in hand_points)
            home_touch_active = any(within_zone(hand["center"], HOME_TOUCH_ZONE) for hand in hand_points)
            game_state.restart_touch_elapsed = update_touch_timer(restart_touch_active, game_state.restart_touch_elapsed, dt)
            game_state.home_touch_elapsed = update_touch_timer(home_touch_active, game_state.home_touch_elapsed, dt)
            restart_game_requested = game_state.restart_touch_elapsed >= TOUCH_CONFIRM_SECONDS
            home_menu_requested = game_state.home_touch_elapsed >= TOUCH_CONFIRM_SECONDS
            render_game_over(canvas, game_state.final_score, game_state.restart_touch_elapsed, game_state.home_touch_elapsed)
            highlight_hands(canvas, hand_points)
            render_fps_overlay(canvas, fps_values)
            if restart_game_requested:
                game_state.start_game(current_time)
            elif home_menu_requested:
                game_state.reset_for_menu()

        else:
            render_text(canvas, "Unexpected state detected. Returning to menu.", (40, 40))
            render_fps_overlay(canvas, fps_values)
            game_state.reset_for_menu()

        # Display inline (compressed for speed)
        clear_output(wait=True)
        output_frame = canvas
        if DISPLAY_SCALE != 1.0:
            scaled_width = int(CANVAS_WIDTH * DISPLAY_SCALE)
            scaled_height = int(CANVAS_HEIGHT * DISPLAY_SCALE)
            output_frame = cv2.resize(canvas, (scaled_width, scaled_height), interpolation=DISPLAY_INTERPOLATION)
        success, encoded_img = cv2.imencode('.jpg', output_frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        if not success:
            raise RuntimeError(
                "OpenCV failed to JPEG-encode the rendered frame; "
                "check the canvas shape, dtype, and JPEG codec support."
            )
        display(DisplayImage(data=encoded_img.tobytes()))
        # Count a frame only after inference, state update, rendering, and JPEG encoding succeed.
        frames_with_hands += int(bool(hand_points))
        frames_processed += 1

except KeyboardInterrupt:
    clear_output(wait=True)
    print("Webcam ended")
finally:
    if cap is not None:
        cap.release()
    cv2.destroyAllWindows()
    elapsed_seconds = max(0.0, time.perf_counter() - run_started)
    fps = frames_processed / elapsed_seconds if elapsed_seconds > 0 else 0.0
    summary = {
        "frames": frames_processed,
        "frames_with_hands": frames_with_hands,
        "elapsed_seconds": elapsed_seconds,
        "fps": fps,
    }
    print(f"FRUIT_NINJA_SMOKE_SUMMARY {json.dumps(summary, sort_keys=True)}")
